# فيديو ← صور ← وصف تدريبي (Qwen3-VL-32B) ← WebDataset على Hugging Face

نظام **قابل للاستئناف** يحوّل قائمة فيديوهاتك (مقدَّمة منك ومسموح لك بمعالجتها) إلى Dataset صور ضخم منقّى من التكرار والجودة المتدنية، لكل صورة وصف إنجليزي تفصيلي `training_prompt` من
`Qwen/Qwen3-VL-32B-Instruct` (bf16 كاملة على GPU واحدة؛ يُغيَّر من `MODEL_ID`)، مخزَّن في شظايا WebDataset (`<id>.jpg` + `<id>.json` داخل `train-000001.tar`) مع فهرس Parquet، ومرفوع إلى مستودع Hugging Face Dataset.

## قبل أن تبدأ
**كل الإعدادات في الخلية 00 (أول خلية)**: النموذج والعتاد، فلاتر الفيديو (سنة ≥ 2020، الضلع الأقصر ≥ 1080، مدة ≥ 10 ثوانٍ)، العيّنة (قص 10%/10% + عشوائي طبقي + `SAMPLING_RATIO`)، عتبات جودة الإطار والتكرار، وفلتر الوصف.
1. **Accelerator = GPU بذاكرة ≥ ~72GB** (RTX PRO 6000 96GB مثلًا؛ النموذج 32B بدقة bf16 كاملة بلا shard ولا تكميم — لـ GPU أصغر غيّر `MODEL_ID` أو فعّل `CAPTION_ALLOW_FALLBACK_PLANS`)، **Internet = On**، و **Add-ons ▸ Secrets ▸ `HF_TOKEN`** (صلاحية write). لا تكتب الـ Token في أي خلية.
2. **مصدر الفيديوهات (إلزامي — بدونه لا يبدأ شيء، لا Benchmark ولا Smoke Test)**: في الخلية **00** عدّل القاموس `USER_SETTINGS` واختر طريقة واحدة:
   * `"VIDEO_MANIFEST": "/kaggle/input/اسم-الداتاست"` ← **مجلد** فيه الفيديوهات (الأسهل: Add Input ثم اكتب اسمه)، أو ملف قائمة CSV/JSON/JSONL/TXT (عمود `video_url`، أو رابط/مسار في كل سطر).
   * `"VIDEO_URLS": ["https://.../a.mp4", ...]` ← روابط مباشرة لملفات فيديو.
   * **لا تضع شيئًا** (الافتراضي، `AUTO_FIND_VIDEOS=True`) ← يستعمل ما أضفتَه في Input إن وُجد، وإلا **يبحث وينزّل وحده** بترتيب المصادر في `AUTO_FIND_SOURCES` (Pexels ← Pixabay ← Wikimedia Commons ← Internet Archive؛ Pexels/Pixabay تحتاجان `PEXELS_API_KEY`/`PIXABAY_API_KEY` في Secrets وإلا تُتخطّيان) فيديوهات حديثة (≥ 2020) بدقة ≥ 1080 ومدة ≥ 10 ثوانٍ؛ كل فيديو يفشل في أي شرط يُرفض ويُسجَّل السبب ويظهر في التقرير النهائي. نوع التراخيص في `AUTO_FIND_LICENSES`: `"any"` (الافتراضي: أي فيديو يُنزَّل بلا تسجيل دخول) أو `"cc"` (أي Creative Commons) أو `"open"` (ملكية عامة/CC-BY/CC-BY-SA فقط). كلمات بحث اختيارية في `AUTO_FIND_QUERY`. يلزم Internet = On. **المسؤولية القانونية على استعمال الفيديوهات ونشر الصور عليك** (وإن كان مستودعك على HF عامًا فالصور مرئية للجميع).

   لا يتجاوز النظام DRM ولا تسجيل الدخول؛ استعمل فيديوهات تملكها أو مسموحًا لك بها. في نفس القاموس: `HF_DATASET_REPO` (مستودعك) و`TOTAL_VIDEOS` (الافتراضي 5 للتجربة؛ ارفعه بعد نجاح 24E).
3. شغّل الخلايا بالترتيب (00 ← 07 ثم الباقي). **اقرأ نتيجة الخلية 07**: يجب أن تقول «valid videos: N» و«videos in state». إن ظهر صندوق «لا توجد قائمة فيديوهات» فاتبع ما فيه.
4. شغّل الخلية **24E** (Preflight = فحوص + Benchmark + Smoke Test؛ **أول مرة 10–40 دقيقة** لأنها تنزّل أول فيديو وأوزان Qwen3-VL-32B ≈ 67GB). لا يبدأ **24F** (التشغيل الكامل) قبل نجاح Smoke Test.
5. انقطعت الجلسة؟ أعد Run All: الحالة محفوظة في المستودع (`control/`) وتُستأنف تلقائيًا. الخلية **28** تقول لك ما الذي سيحدث وأي خلية تشغّل.

## ⚠ صدق (اقرأه)
* **الجدوى والسرعة**: لا أرقام ثابتة هنا — أرقام T4 القديمة لا تنطبق. الخلية 24C **تقيس على عتادك الفعلي** (تبدأ بدفعة 16 وترفعها حتى ~80% من الذاكرة) وتطبع جدول ميزانية (10/30/100 ساعة GPU ⇒ عدد الصور). `SAMPLING_RATIO` (الافتراضي 0.05) و`MAX_IMAGES_PER_VIDEO` هما ما تضبط به الحجم قبل 24F.
* اختُبر النظام محليًا بوصّاف **تجريبي (mock)** وفك FFmpeg/CPU حقيقي، على مخزن محلي، وعلى **محاكيات صارمة** لـ `huggingface_hub` وKaggle Secrets وIPython `display` تطابق توقيعات المكتبات الحقيقية (أشكال الاستدعاء وسلوك الـ commit الذرّي والاستئناف بعد kill -9 اختُبرت؛ **اتصال Hugging Face الحقيقي لم يُختبر**: حدود المعدّل 429 وأخطاء الشبكة الفعلية غير مُحاكاة).
* **لم يُختبر هنا** (لا GPU ولا شبكة): Qwen3-VL الحقيقي/Transformers/vLLM، bitsandbytes، FlashAttention-2، PyNvVideoCodec وNVDEC، `HFStore` على Hugging Face الفعلي، واجهات Pexels/Pixabay/Wikimedia Commons/Internet Archive الحية (اختُبرت بمحاكيات فقط)، Parquet بـ pyarrow الحقيقي، IPython الحقيقي. الخلية 24D (Smoke Test) تختبر هذه الأجزاء **على عتادك الفعلي** قبل أي تشغيل جماعي.

## خريطة الخلايا
| الخلية | الوظيفة |
|---|---|
| 00 | Settings (كل الإعدادات في هذه الخلية فقط) |
| 01 | Environment Inspection |
| 02 | Dependency Installation |
| 03 | Configuration |
| 04 | Hugging Face Authentication |
| 05 | Repository Initialization |
| 06 | Resume State Loader |
| 07 | Manifest Validation |
| 08 | Video Downloader |
| 09 | Video Metadata Analyzer |
| 10 | Hardware Decoder Initialization |
| 11 | Frame Extraction Engine |
| 12 | Sampling (stratified random + trimming; deterministic per video) |
| 13 | Image Quality Control |
| 14 | Exact Deduplication |
| 15 | Perceptual Deduplication |
| 16 | Image Processing |
| 17 | Qwen3-VL Initialization |
| 18 | Caption Prompt and Structured Output |
| 19 | Batched Captioning |
| 20 | Random Sample Preview |
| 21 | WebDataset Shard Writer |
| 22 | Hugging Face Upload |
| 23 | Checkpoint Manager |
| 24 | Performance Dashboard |
| 24B | Pipeline Orchestrator |
| 24C | Benchmark Mode |
| 24D | Smoke Test (اختبار تكاملي صغير) + بوابة التشغيل الكامل |
| 24E | Preflight (قائمة الجاهزية) = فحوص + Benchmark (إن لزم) + Smoke Test (إن لزم) |
| 24F | FULL RUN (التشغيل الكامل) — محجوب ببوابة Smoke Test |
| 25 | RANDOM DATASET INSPECTOR  (+ قارئ Dataset المشترك للخليتين 26 و27) |
| 26 | DATASET INTEGRITY VALIDATOR  (فحص تكامل Dataset كاملًا في المخزن) |
| 27 | FINAL STATISTICS  (إحصاءات Dataset النهائية) |
| 28 | RESUME STATUS & INSTRUCTIONS  (حالة الاستئناف + التعليمات) |

---
## الجزء 1 — الإعدادات والبيئة (00–07)
كل إعداداتك في الخلية 00 ثم فحص البيئة، التبعيات، تحميل الإعدادات، مصادقة Hugging Face، المستودع، حالة الاستئناف، المانيفست.

### CELL 00 — Settings (كل الإعدادات في هذه الخلية فقط)
> عدّل القيم هنا ثم شغّل الخلايا بالترتيب. لا يوجد إعداد مهم في أي خلية أخرى: الخلية 03 تقرأ هذا القاموس وتفرضه على Config.
> الأولوية عند التعارض: متغيرات البيئة PIPE_<NAME> > ملف config.json > هذا القاموس > الافتراضي في الخلية 03. الـ HF_TOKEN لا يوضع هنا أبدًا (Kaggle Secrets / molab Secrets / env).
> مفاتيح مصادر الفيديو الاختيارية (PEXELS_API_KEY / PIXABAY_API_KEY) تُقرأ من Secrets/البيئة أيضًا، لا تُكتب هنا.

In [ ]:
# CELL 00 - Settings (كل الإعدادات في هذه الخلية فقط)
#: عدّل القيم هنا ثم شغّل الخلايا بالترتيب. لا يوجد إعداد مهم في أي خلية أخرى: الخلية 03 تقرأ هذا القاموس وتفرضه على Config.
#: الأولوية عند التعارض: متغيرات البيئة PIPE_<NAME> > ملف config.json > هذا القاموس > الافتراضي في الخلية 03. الـ HF_TOKEN لا يوضع هنا أبدًا (Kaggle Secrets / molab Secrets / env).
#: مفاتيح مصادر الفيديو الاختيارية (PEXELS_API_KEY / PIXABAY_API_KEY) تُقرأ من Secrets/البيئة أيضًا، لا تُكتب هنا.

USER_SETTINGS = {
    # ════════════════════════ 1) مصدر الفيديوهات (الفيديوهات يجب أن تكون ملكك أو مسموحًا لك بمعالجتها) ════════════════════════
    "VIDEO_MANIFEST": "",          # مجلد فيه فيديوهات أو ملف CSV/JSON/JSONL/TXT. فارغ = بحث داخل /kaggle/input ثم بحث تلقائي (AUTO_FIND_VIDEOS)
    "VIDEO_URLS": [],              # أو روابط مباشرة لملفات فيديو
    "AUTO_FIND_VIDEOS": True,      # True = إن لم تضع فيديوهات يبحث النظام وحده في المصادر أدناه وينزّل (يلزم Internet = On)
    "AUTO_FIND_SOURCES": "pexels,pixabay,commons,archive",   # ترتيب المصادر المفضّلة. pexels/pixabay تحتاجان مفتاحًا (PEXELS_API_KEY / PIXABAY_API_KEY) وإلا تُتخطّيان بتنبيه
    "AUTO_FIND_LICENSES": "any",   # open = ملكية عامة/CC-BY/CC-BY-SA | cc = أي Creative Commons | any = أي مصدر مسموح تنزيله (الترخيص يُسجَّل لكل صورة؛ pexels/pixabay تعملان في any فقط)
    "AUTO_FIND_QUERY": "",         # كلمات بحث بالإنجليزية (مثل "nature city people"). فارغ = الأشهر/الأحدث
    "TOTAL_VIDEOS": 5,             # 5 للتجربة الأولى؛ ارفعه بعد نجاح 24E (0 = كل ما في القائمة، وفي البحث التلقائي 0 تعني 50)

    # ════════════════════════ 2) Hugging Face ════════════════════════
    "HF_DATASET_REPO": "arttry/foxingdatasit",   # "اسمك/اسم-الداتاست"؛ فارغ = بلا رفع (تجربة محلية)
    "HF_PRIVATE_REPO": True,       # يُنشأ المستودع خاصًا إن لم يكن موجودًا (المستودع الموجود يحتفظ بخصوصيته)

    # ════════════════════════ 3) نموذج الوصف والعتاد (RTX PRO 6000 Blackwell 96GB) ════════════════════════
    "MODEL_ID": "Qwen/Qwen3-VL-32B-Instruct",
    "CAPTION_PRECISION": "bf16",            # bf16 كاملة على GPU واحدة (نسخة لكل GPU إن وُجد أكثر من واحدة). بلا shard وبلا int8/int4
    "CAPTION_ALLOW_FALLBACK_PLANS": False,  # False = لا shard ولا int8/int4 تلقائيًا (إن لم يتسع النموذج يفشل بوضوح). True = يسمح بالخطط الأضعف (GPU صغيرة)
    "ATTN_IMPLEMENTATION": "auto",          # auto = flash_attention_2 إن كانت متاحة وتعمل، وإلا sdpa تلقائيًا
    "CAPTION_BATCH_SIZE": "auto",           # auto = من نتيجة Benchmark (24C) على العتاد الحالي؛ أو رقم ثابت
    "BENCH_START_BATCH": 16,                # أول حجم دفعة يجرّبه Benchmark (ينزل إن فشل، ويصعد إن نجح)
    "BENCH_VRAM_TARGET": 0.80,              # يرفع الدفعة حتى ~80% من ذاكرة GPU ولا يعتمد ما فوقه (الباقي هامش لـ KV-cache وتقلّب طول الوصف)
    "BENCH_STABLE_FRACTION": 0.75,          # نقطة بدء الدفعة الفعلية = هذه النسبة من أفضل دفعة مقيسة (هامش أمان إضافي)
    "MODEL_MAX_PIXELS": 1_048_576,          # أقصى بكسلات تُدخَل للنموذج (≈1MP ≈ 1024 رمز بصري). خفضه يوفّر ذاكرة KV ويسمح بدفعة أكبر
    "CAPTION_MAX_NEW_TOKENS": 384,
    "CAPTION_LANGUAGE": "English",

    # ════════════════════════ 4) فلاتر الفيديو (أي شرط يفشل ⇒ يُرفض الفيديو ويُسجَّل السبب) ════════════════════════
    "MIN_VIDEO_YEAR": 2020,         # لا فيديوهات أقدم (يُفحص تاريخ المصدر عند البحث + وسم creation_time داخل الملف إن وُجد). 0 = معطّل
    "MIN_VIDEO_SHORT_SIDE": 1080,   # min(العرض, الارتفاع) ≥ هذا (الفيديو العمودي 1080×1920 مقبول). يُفحص بعد التنزيل دائمًا (حارس ثانٍ لأن سنة Internet Archive غير موثوقة). 0 = معطّل
    "MIN_VIDEO_DURATION": 10.0,     # ثوانٍ: ما هو أقصر يُرفض
    "MAX_VIDEO_DURATION": 3600.0,   # ثوانٍ: ما هو أطول يُرفض

    # ════════════════════════ 5) اختيار الإطارات ════════════════════════
    "SAMPLING_METHOD": "stratified_random",   # stratified_random (الافتراضي) | deterministic_hash | exact_per_video
    "SAMPLING_RATIO": 0.05,         # نسبة الإطارات المرشّحة (≤30fps) في الجزء الأوسط التي تُؤخذ: 5% ⇒ ~1.5 إطار/ثانية. خفّضها لتقليل التكرار والزمن
    "TRIM_START_RATIO": 0.10,       # يُقصّ 10% من بداية الفيديو (مقدّمات/شعارات)
    "TRIM_END_RATIO": 0.10,         # ويُقصّ 10% من نهايته (نهايات/اعتمادات)
    "SAMPLING_SEED": 20261105,      # مع اسم الفيديو يعطي بذرة ثابتة لكل فيديو ⇒ الاستئناف يعيد نفس الإطارات بالضبط
    "MAX_IMAGES_PER_VIDEO": 0,      # سقف اختياري للصور لكل فيديو (0 = بلا سقف)

    # ════════════════════════ 6) جودة الإطار والتكرار (كل العتبات هنا) ════════════════════════
    "QC_MIN_LUMA_STD": 20.0,            # يُرفض الإطار إن كان الانحراف المعياري للإضاءة (0-255) أقل من هذا (مسطّح/بلا تفاصيل). 0 = معطّل
    "QC_TEXT_SCREEN_ENABLE": True,      # رفض شاشات النص (خلفية داكنة + بكسلات ساطعة قليلة متجمّعة: عناوين/اعتمادات)
    "QC_TEXT_DARK_MEAN": 60.0,          # شاشة النص: متوسط الإضاءة أقل من هذا
    "QC_TEXT_DARK_LEVEL": 50,           # والبكسل «داكن» إن كانت إضاءته أقل من هذا
    "QC_TEXT_DARK_FRACTION": 0.80,      # وحصة البكسلات الداكنة ≥ هذا
    "QC_TEXT_BRIGHT_LEVEL": 160,        # والبكسل «ساطع» إن كانت إضاءته ≥ هذا
    "QC_TEXT_BRIGHT_MIN_FRACTION": 0.0005,   # حصة الساطع بين هذين الحدّين (قليل ومتجمّع)
    "QC_TEXT_BRIGHT_MAX_FRACTION": 0.12,
    "QC_TEXT_BBOX_MAX_FRACTION": 0.50,  # ومستطيل الساطع (بعد قصّ الأطراف الشاردة 0.5%) يغطي ≤ هذه الحصة من الإطار (المتجمّع)
    "QC_TEXT_MIN_COMPONENTS": 3,        # وعدد البقع الساطعة المنفصلة ≥ هذا (حروف). مصباح واحد في ظلام ليس نصًّا
    "DEDUP_LAST_N": 20,                 # الإطار يُقارَن بآخر N إطارًا مقبولًا في نفس الفيديو (0 = النافذة الزمنية القديمة DEDUP_WINDOW_SECONDS)
    "DEDUP_HAMMING_THRESHOLD": 8,       # أقصى مسافة هامنغ على pHash/dHash لاعتباره مرشّحًا للتكرار
    "DEDUP_SSIM_THRESHOLD": 0.95,       # وتأكيد التكرار بـ SSIM ≥ هذه القيمة (بديل CLIP≈0.95: لا يتوفر CLIP بلا تنزيل نموذج إضافي)

    # ════════════════════════ 7) فلتر الوصف بعد التوليد ════════════════════════
    "CAPTION_BANNED_PHRASES": ["devoid of", "no other", "solid black background", "no discernible", "blank"],   # وصف يحوي أيًّا منها (كلمة/عبارة كاملة، بلا حساسية لحالة الأحرف) ⇒ إعادة محاولة
    "CAPTION_BANNED_RETRIES": 1,        # عدد إعادات المحاولة (برومبت أشد). إن بقيت العبارة المحظورة يُتخطّى الإطار كله (صورة + وصف)
}

USER_SETTINGS.update(globals().get("USER_SETTINGS_EXTERNAL") or {})      # دفتر marimo يمرّر إعداداته من خلية الإعدادات (بلا تعديل هذا الملف)
print("settings ready:", {k: USER_SETTINGS[k] for k in ("MODEL_ID", "MIN_VIDEO_YEAR", "MIN_VIDEO_SHORT_SIDE", "MIN_VIDEO_DURATION", "SAMPLING_METHOD", "SAMPLING_RATIO",
                                                         "TRIM_START_RATIO", "TRIM_END_RATIO") if k in USER_SETTINGS})

### CELL 01 — Environment Inspection
> فحص كامل للبيئة (Python/CUDA/GPU/CPU/RAM/Disk/FFmpeg/NVDEC/PyNvVideoCodec/الإنترنت) + أدوات مساعدة مشتركة.

In [ ]:
# CELL 01 - Environment Inspection
#: فحص كامل للبيئة (Python/CUDA/GPU/CPU/RAM/Disk/FFmpeg/NVDEC/PyNvVideoCodec/الإنترنت) + أدوات مساعدة مشتركة.
from __future__ import annotations

import os, sys, re, io, json, time, math, shutil, socket, struct, hashlib, logging, platform
import subprocess, threading, queue, tempfile, traceback, random, gc, tarfile, dataclasses
import importlib, importlib.util, importlib.metadata, collections, uuid, itertools
from concurrent.futures import ThreadPoolExecutor, Future, as_completed
from pathlib import Path
from typing import Any, Dict, Iterator, List, Optional, Tuple, Callable, Iterable, Sequence

# ------------------------------------------------------------------ logging / helpers
_SECRETS: set = set()          # أي قيمة تُسجَّل هنا تُحجب من السجلات والمخرجات

class _RedactFilter(logging.Filter):
    def filter(self, record: logging.LogRecord) -> bool:
        try:
            msg = record.getMessage()
            for s in _SECRETS:
                if s and s in msg:
                    msg = msg.replace(s, "***")
            record.msg, record.args = msg, ()
        except Exception:
            pass
        return True

class _LiveStdoutHandler(logging.StreamHandler):
    """يكتب إلى sys.stdout *الحالي* وقت الإصدار (لا وقت إنشاء المعالج): ضروري في marimo/Jupyter حيث يُستبدل sys.stdout لكل خلية/تشغيل."""
    def __init__(self): logging.Handler.__init__(self)
    @property
    def stream(self): return sys.stdout
    @stream.setter
    def stream(self, v): pass

def get_logger(name: str = "vid2ds") -> logging.Logger:
    lg = logging.getLogger(name)
    if not lg.handlers:
        h = _LiveStdoutHandler()
        h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
        h.addFilter(_RedactFilter())
        lg.addHandler(h)
        lg.setLevel(os.environ.get("PIPE_LOG_LEVEL", "INFO"))
        lg.propagate = False
    return lg

log = get_logger()

_HF_TOKEN_RE = re.compile(r"hf_[A-Za-z0-9]{20,}")
_URL_QUERY_RE = re.compile(r"(https?://[^\s?#'\"<>]+)\?[^\s'\"<>]*")

def redact_text(x: Any, limit: int = 0) -> str:
    """نص آمن للتخزين/العرض: يحجب الأسرار المسجَّلة وأي HF token وسلاسل استعلام الروابط (قد تحمل توقيعات/مفاتيح). limit>0 يقصّ الطول."""
    s = "" if x is None else str(x)
    for sec in list(_SECRETS):
        if sec: s = s.replace(sec, "***")
    s = _URL_QUERY_RE.sub(r"\1?…", _HF_TOKEN_RE.sub("hf_***", s))
    return s[:limit] if limit else s

_URL_SCHEME_RE = re.compile(r"^[A-Za-z][A-Za-z0-9+.\-]*://")

def redact_url(u: Any) -> Any:
    """رابط آمن للحفظ في control/ (يُرفع إلى المستودع): بلا user:pass@ وبلا سلسلة الاستعلام/الجزء (قد تحمل توقيعات مسبقة التوقيع أو مفاتيح).
    المسارات المحلية وغير النصوص تمرّ كما هي. الرابط الكامل يبقى في الذاكرة ويُقرأ من المانيفست في كل جلسة (STATE.init_videos)."""
    if not isinstance(u, str) or not _URL_SCHEME_RE.match(u): return u
    s = re.sub(r"^([A-Za-z][A-Za-z0-9+.\-]*://)[^/@?#]*@", r"\1", u)
    return re.split(r"[?#]", s, maxsplit=1)[0]

def human_bytes(n: float) -> str:
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024 or u == "TB":
            return f"{n:.2f} {u}" if u != "B" else f"{int(n)} B"
        n /= 1024.0
    return f"{n:.2f} TB"

def human_time(sec: float) -> str:
    if sec is None or not math.isfinite(sec):
        return "n/a"
    sec = int(sec); h, r = divmod(sec, 3600); m, s = divmod(r, 60)
    return f"{h:d}h {m:02d}m {s:02d}s" if h else f"{m:d}m {s:02d}s"

def run_cmd(cmd: Sequence[str], timeout: float = 60, **kw) -> Tuple[int, str, str]:
    try:
        p = subprocess.run(list(cmd), capture_output=True, text=True, timeout=timeout, **kw)
        return p.returncode, p.stdout, p.stderr
    except FileNotFoundError:
        return 127, "", f"not found: {cmd[0]}"
    except subprocess.TimeoutExpired:
        return 124, "", "timeout"
    except Exception as e:  # pragma: no cover
        return 1, "", repr(e)

def pkg_version(name: str) -> Optional[str]:
    try:
        return importlib.metadata.version(name)
    except Exception:
        return None

def has_module(name: str) -> bool:
    """هل يمكن استيراد الوحدة؟ importlib.util.find_spec يرمي ValueError إن كانت الوحدة محقونة في sys.modules بلا __spec__
    (يحدث مع بعض الأدوات والمحاكيات) أو ImportError إن كانت حزمتها الأب مكسورة؛ لا نريد أن تسقط خلية كاملة بسبب فحص اختياري."""
    try:
        return importlib.util.find_spec(name) is not None
    except (ValueError, ImportError, AttributeError):
        return name in sys.modules

def atomic_write_bytes(path: os.PathLike, data: bytes) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f".tmp{os.getpid()}{threading.get_ident()}")
    with open(tmp, "wb") as f:
        f.write(data); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def atomic_write_json(path: os.PathLike, obj: Any) -> None:
    atomic_write_bytes(path, json.dumps(obj, ensure_ascii=False, indent=2, default=str).encode("utf-8"))

def sha256_file(path: os.PathLike, chunk: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

# ------------------------------------------------------------------ environment probes
def _cpu_limit() -> int:
    n = os.cpu_count() or 1
    try:                                   # cgroup v2 / v1 limits (Kaggle/Docker)
        q = Path("/sys/fs/cgroup/cpu.max")
        if q.exists():
            a, b = q.read_text().split()
            if a != "max": n = min(n, max(1, math.ceil(int(a) / int(b))))
        else:
            qa, pa = Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us"), Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us")
            if qa.exists() and int(qa.read_text()) > 0:
                n = min(n, max(1, math.ceil(int(qa.read_text()) / int(pa.read_text()))))
    except Exception:
        pass
    try:
        n = min(n, len(os.sched_getaffinity(0)))
    except Exception:
        pass
    return max(1, n)

def _ram_gb() -> Tuple[float, float]:
    try:
        import psutil
        vm = psutil.virtual_memory(); tot, av = vm.total, vm.available
    except Exception:
        info = {k: int(v.split()[0]) * 1024 for k, v in
                (l.split(":", 1) for l in Path("/proc/meminfo").read_text().splitlines() if ":" in l)}
        tot, av = info.get("MemTotal", 0), info.get("MemAvailable", 0)
    try:                                   # cgroup memory limit
        for p in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
            if Path(p).exists():
                v = Path(p).read_text().strip()
                if v.isdigit() and int(v) < tot: tot = int(v); av = min(av, tot)
    except Exception:
        pass
    return tot / 1e9, av / 1e9

def _gpus() -> List[Dict[str, Any]]:
    rc, out, _ = run_cmd(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,driver_version,compute_cap",
                          "--format=csv,noheader,nounits"], timeout=20)
    gpus = []
    if rc != 0:                            # بعض التعريفات القديمة لا تدعم compute_cap
        rc, out, _ = run_cmd(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,driver_version",
                              "--format=csv,noheader,nounits"], timeout=20)
    if rc == 0:
        for line in out.strip().splitlines():
            p = [x.strip() for x in line.split(",")]
            if len(p) >= 5:
                gpus.append(dict(index=int(p[0]), name=p[1], vram_total_mb=float(p[2]), vram_free_mb=float(p[3]),
                                 driver=p[4], compute_cap=(p[5] if len(p) > 5 else None)))
    return gpus

def _disk_speed(path: Path, mb: int = 64) -> Dict[str, float]:
    res = {"write_MBps": float("nan"), "read_MBps": float("nan")}
    try:
        path.mkdir(parents=True, exist_ok=True)
        f = path / f".speedtest_{os.getpid()}"; buf = os.urandom(1 << 20)
        t = time.perf_counter()
        with open(f, "wb") as fh:
            for _ in range(mb): fh.write(buf)
            fh.flush(); os.fsync(fh.fileno())
        res["write_MBps"] = mb / max(time.perf_counter() - t, 1e-6)
        try:
            fd = os.open(f, os.O_RDONLY); os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED); os.close(fd)
        except Exception:
            pass
        t = time.perf_counter()
        with open(f, "rb") as fh:
            while fh.read(1 << 22): pass
        res["read_MBps"] = mb / max(time.perf_counter() - t, 1e-6)
        f.unlink(missing_ok=True)
    except Exception as e:
        res["error"] = repr(e)
    return res

def _check_nvdec_ffmpeg(tmp: Path) -> Dict[str, Any]:
    r = {"hwaccel_listed": False, "decode_test": False, "note": ""}
    rc, out, err = run_cmd(["ffmpeg", "-hide_banner", "-hwaccels"], timeout=20)
    r["hwaccel_listed"] = rc == 0 and "cuda" in out
    if not r["hwaccel_listed"]:
        r["note"] = "ffmpeg بدون hwaccel cuda"; return r
    clip = tmp / "nvdec_probe.mp4"
    rc, _, e = run_cmd(["ffmpeg", "-y", "-v", "error", "-f", "lavfi", "-i", "testsrc=size=320x240:rate=30:duration=1",
                        "-c:v", "libx264", "-pix_fmt", "yuv420p", str(clip)], timeout=60)
    if rc != 0:
        r["note"] = "تعذّر إنشاء مقطع اختبار: " + e[-120:]; return r
    rc, _, e = run_cmd(["ffmpeg", "-v", "error", "-hwaccel", "cuda", "-i", str(clip), "-f", "null", "-"], timeout=60)
    r["decode_test"] = rc == 0
    r["note"] = "" if rc == 0 else e.strip()[-160:]
    clip.unlink(missing_ok=True)
    return r

def _internet() -> Dict[str, Any]:
    out = {"internet": False, "huggingface": False}
    for host, key in (("huggingface.co", "huggingface"), ("pypi.org", "internet")):
        try:
            socket.create_connection((host, 443), timeout=6).close(); out[key] = True
        except Exception:
            pass
    out["internet"] = out["internet"] or out["huggingface"]
    return out

def _cuda_kernel_check(env: Dict[str, Any]) -> Dict[str, Any]:
    """يشغّل نواة CUDA صغيرة فعلًا ويقارن معمارية الكرت بما بُنيت له PyTorch. Blackwell (cc 12.x) يتطلب PyTorch ≥2.7 مبنيًا بـ cu128+."""
    r: Dict[str, Any] = {"ok": None, "arch_list": [], "note": ""}
    if not env.get("cuda_available"): r["note"] = "لا CUDA"; return r
    try:
        import torch
        r["arch_list"] = list(torch.cuda.get_arch_list())
        x = torch.ones(16, device="cuda") * 2; torch.cuda.synchronize()
        r["ok"] = abs(float(x.sum().item()) - 32.0) < 1e-3
        if not r["ok"]: r["note"] = "نتيجة نواة CUDA خاطئة"
    except Exception as e:
        r["ok"] = False; r["note"] = f"{type(e).__name__}: {str(e)[:160]}"
    try: ccs = [float(g.get("compute_cap") or 0) for g in env.get("gpus", [])]
    except (TypeError, ValueError): ccs = []
    if ccs and max(ccs) >= 12.0 and r["arch_list"] and not any(a in ("sm_120", "compute_120", "sm_121") for a in r["arch_list"]):
        r["ok"] = False if r["ok"] is None else r["ok"]
        r["note"] = (r["note"] + " | " if r["note"] else "") + ("PyTorch بُني بلا sm_120 (Blackwell): ثبّت إصدارًا مبنيًا بـ cu128 أو أحدث "
                     "(uv pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu128) ثم أعد تشغيل الـ kernel")
    return r

def inspect_environment(work_dir: Optional[str] = None, quiet: bool = False) -> Dict[str, Any]:
    wd = Path(work_dir or ("/kaggle/working" if Path("/kaggle/working").exists() else "./work"))
    env: Dict[str, Any] = {"python": sys.version.split()[0], "platform": platform.platform()}
    env["on_kaggle"] = Path("/kaggle").exists()
    env["on_marimo"] = ("marimo" in sys.modules) or any(k.startswith("MARIMO") for k in os.environ)         # molab أو marimo محلي: يغيّر طريقة عرض اللوحة وحدّ الوقت
    try:
        import torch
        env["torch"] = torch.__version__
        env["cuda_runtime"] = torch.version.cuda
        env["cuda_available"] = bool(torch.cuda.is_available())
        env["gpu_count_torch"] = torch.cuda.device_count() if env["cuda_available"] else 0
        env["bf16_supported"] = bool(env["cuda_available"] and torch.cuda.is_bf16_supported())
    except Exception as e:
        env.update(torch=None, cuda_runtime=None, cuda_available=False, gpu_count_torch=0, bf16_supported=False,
                   torch_error=repr(e)[:120])
    env["gpus"] = _gpus()
    env["cuda_check"] = _cuda_kernel_check(env)
    env["cpu_cores"] = _cpu_limit()
    env["ram_total_gb"], env["ram_available_gb"] = _ram_gb()
    env["disk"] = {}
    for name, p in (("working", wd), ("tmp", Path(tempfile.gettempdir()))):
        try:
            p.mkdir(parents=True, exist_ok=True); u = shutil.disk_usage(p)
            env["disk"][name] = dict(path=str(p), total_gb=u.total / 1e9, free_gb=u.free / 1e9)
        except Exception as e:
            env["disk"][name] = dict(path=str(p), error=repr(e))
    env["disk_speed"] = _disk_speed(Path(tempfile.gettempdir()) / "vid2ds_speed")
    rc, out, _ = run_cmd(["ffmpeg", "-version"], timeout=15)
    env["ffmpeg"] = out.splitlines()[0] if rc == 0 and out else None
    env["ffprobe"] = shutil.which("ffprobe")
    env["nvdec_ffmpeg"] = _check_nvdec_ffmpeg(Path(tempfile.gettempdir())) if env["gpus"] and env["ffmpeg"] else \
        {"hwaccel_listed": False, "decode_test": False, "note": "no GPU/ffmpeg"}
    env["pynvvideocodec"] = pkg_version("PyNvVideoCodec")
    env["libs"] = {k: pkg_version(k) for k in ("transformers", "accelerate", "bitsandbytes", "flash-attn", "vllm",
                   "huggingface_hub", "hf_xet", "webdataset", "pyarrow", "opencv-python", "opencv-python-headless",
                   "Pillow", "pillow-simd", "pyvips", "numpy", "psutil", "tqdm", "rich")}
    env["network"] = _internet()
    if not quiet:
        print_env_report(env)
    return env

def print_env_report(env: Dict[str, Any]) -> None:
    L = lambda k, v: print(f"  {k:<26}{v}")
    print("=" * 78); print(" ENVIRONMENT REPORT"); print("=" * 78)
    L("Python", env["python"]); L("Platform", env["platform"]); L("Kaggle / marimo", f'{env["on_kaggle"]} / {env.get("on_marimo")}')
    L("PyTorch / CUDA runtime", f'{env.get("torch")} / {env.get("cuda_runtime")}')
    if env.get("torch_error"): L("⚠ PyTorch غير قابل للاستيراد", env["torch_error"] + " — بدونه لا يعمل Qwen؛ ثبّت torch يدويًا ثم أعد تشغيل الـ kernel")
    L("CUDA available", f'{env["cuda_available"]}  (BF16: {env["bf16_supported"]})')
    if env["gpus"]:
        for g in env["gpus"]:
            L(f'GPU[{g["index"]}]', f'{g["name"]} | VRAM {g["vram_total_mb"]/1024:.1f} GB (free {g["vram_free_mb"]/1024:.1f}) | '
                                   f'driver {g["driver"]} | cc {g.get("compute_cap")}')
    else:
        L("GPU", "لا يوجد (سيعمل المشروع على CPU بأقصى ما يمكن)")
    cc = env.get("cuda_check") or {}
    if cc.get("ok") is not None or cc.get("note"): L("CUDA kernel test", ("✔ ok" if cc.get("ok") else "✖ فشل") + (f" | {cc['note']}" if cc.get("note") else ""))
    L("CPU cores (effective)", env["cpu_cores"])
    L("RAM", f'{env["ram_total_gb"]:.1f} GB total / {env["ram_available_gb"]:.1f} GB available')
    for k, d in env["disk"].items():
        L(f"Disk[{k}]", f'{d.get("path")}  free {d.get("free_gb", float("nan")):.1f} GB / {d.get("total_gb", float("nan")):.1f} GB')
    ds = env["disk_speed"]; L("Disk speed (approx.)", f'write {ds["write_MBps"]:.0f} MB/s | read {ds["read_MBps"]:.0f} MB/s')
    L("FFmpeg", env["ffmpeg"]); L("FFprobe", env["ffprobe"])
    n = env["nvdec_ffmpeg"]; L("FFmpeg NVDEC", f'listed={n["hwaccel_listed"]} decode_test={n["decode_test"]} {n["note"]}')
    L("PyNvVideoCodec", env["pynvvideocodec"] or "غير مثبت")
    L("Network", env["network"])
    print("  Libraries: " + ", ".join(f"{k}={v}" for k, v in env["libs"].items() if v))
    print("  Missing  : " + ", ".join(k for k, v in env["libs"].items() if not v))
    print("=" * 78)

def make_resource_plan(env: Dict[str, Any]) -> Dict[str, Any]:
    """يحوّل نتائج الفحص إلى إعدادات تشغيل: لا نفتح عمليات أكثر مما يتحمله العتاد."""
    cores, ram = env["cpu_cores"], env["ram_available_gb"]
    plan = dict(
        qc_threads=max(1, min(cores - 1 if cores > 2 else cores, 8)),
        io_threads=max(2, min(cores, 6)),
        ready_queue_chunks=2 if ram < 12 else 3,
        chunk_images=48 if ram < 12 else (192 if (ram >= 24 and env['gpus'] and max(g['vram_total_mb'] for g in env['gpus']) / 1024 >= 40) else 64),     # GPU 96GB: دفعات وصف حتى 192 صورة تحتاج حزمًا بهذا الحجم
        decode_queue_frames=16 if ram < 12 else 32,
        upload_queue=1,
        has_gpu=bool(env["gpus"]) and env.get("cuda_available", False),
        total_vram_gb=(env["gpus"][0]["vram_total_mb"] / 1024 if env["gpus"] else 0.0),
        n_gpus=len(env["gpus"]),
    )
    return plan

ENV = inspect_environment()
RESOURCE_PLAN = make_resource_plan(ENV)
log.info("resource plan: %s", RESOURCE_PLAN)

### CELL 02 — Dependency Installation
> تثبيت ما ينقص فقط (لا نعيد تثبيت torch أبدًا). الحزم الاختيارية قد تفشل دون إيقاف المشروع.

In [ ]:
# CELL 02 - Dependency Installation
#: تثبيت ما ينقص فقط (لا نعيد تثبيت torch أبدًا). الحزم الاختيارية قد تفشل دون إيقاف المشروع.

def _vtuple(v: str) -> Tuple[int, ...]:
    return tuple(int(x) for x in re.findall(r"\d+", v.split("+")[0])[:3])

# (اسم pip، اسم الاستيراد، أدنى إصدار، إلزامي؟)
DEPENDENCIES = [
    ("numpy",           "numpy",           None,     True),     # الثلاثة التالية موجودة مسبقًا على Kaggle؛ تُثبَّت فقط إن غابت (بلا حدّ أدنى كي لا نستبدل نسخة المنصة)
    ("Pillow",          "PIL",             None,     True),
    ("opencv-python-headless", "cv2",      None,     True),
    ("huggingface_hub", "huggingface_hub", "0.34.0", True),
    ("hf_xet",          "hf_xet",          None,     False),
    ("transformers",    "transformers",    "4.57.0", True),    # أول إصدار يدعم Qwen3-VL
    ("accelerate",      "accelerate",      "1.0.0",  True),
    ("pyarrow",         "pyarrow",         None,     True),
    ("psutil",          "psutil",          None,     True),
    ("tqdm",            "tqdm",            None,     False),
    ("rich",            "rich",            None,     False),
    ("nvidia-ml-py",    "pynvml",          None,     False),
    ("webdataset",      "webdataset",      None,     False),    # للتحقق فقط؛ الكتابة بـ tarfile
    ("bitsandbytes",    "bitsandbytes",    "0.43.0", False),    # 4/8-bit عند الحاجة
    ("PyNvVideoCodec",  "PyNvVideoCodec",  None,     False),    # يُثبَّت فقط عند وجود GPU
]

def _is_ok(pip_name: str, mod: str, min_ver: Optional[str]) -> bool:
    if not has_module(mod):
        return False
    if min_ver:
        v = pkg_version(pip_name)
        return bool(v) and _vtuple(v) >= _vtuple(min_ver)
    return True

def _pip_install(spec: str, timeout: float = 1200) -> Tuple[int, str, str]:
    """pip إن وُجد؛ وإلا uv (بيئات marimo/molab غالبًا uv بلا pip)؛ وإلا فشل واضح."""
    if has_module("pip"):
        return run_cmd([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", spec], timeout=timeout)
    uv = shutil.which("uv")
    if uv: return run_cmd([uv, "pip", "install", "-q", "--python", sys.executable, spec], timeout=timeout)
    return 127, "", "لا pip ولا uv في هذه البيئة"

def ensure_ffmpeg() -> Dict[str, Any]:
    """ffmpeg/ffprobe إلزاميان. إن غابا (molab مثلًا) نحاول حزمة static-ffmpeg (تنزّل ثنائيات ثابتة عند أول استعمال؛ تحتاج إنترنت)."""
    res: Dict[str, Any] = {"had": bool(shutil.which("ffmpeg") and shutil.which("ffprobe")), "installed": False, "note": ""}
    if res["had"] or os.environ.get("PIPELINE_SKIP_INSTALL") == "1": return res
    log.info("ffmpeg/ffprobe غير موجودين → محاولة static-ffmpeg ...")
    if not has_module("static_ffmpeg"):
        rc, out, err = _pip_install("static-ffmpeg", timeout=600)
        importlib.invalidate_caches()
        if rc != 0: res["note"] = "تعذّر تثبيت static-ffmpeg: " + (err or out).strip()[-200:]; return res
    try:
        import static_ffmpeg; static_ffmpeg.add_paths()
    except Exception as e:
        res["note"] = f"static_ffmpeg.add_paths فشل: {type(e).__name__}: {str(e)[:160]}"; return res
    res["installed"] = bool(shutil.which("ffmpeg") and shutil.which("ffprobe"))
    if not res["installed"]: res["note"] = "بعد static-ffmpeg ما زال ffmpeg/ffprobe غير موجودين في PATH"
    return res

def install_dependencies(has_gpu: bool, skip: bool = False) -> Dict[str, str]:
    status: Dict[str, str] = {}
    todo: List[Tuple[str, str, Optional[str], bool]] = []
    for pip_name, mod, min_ver, required in DEPENDENCIES:
        if pip_name in ("PyNvVideoCodec", "bitsandbytes") and not has_gpu:
            status[pip_name] = "skipped (no GPU)"; continue
        if _is_ok(pip_name, mod, min_ver):
            status[pip_name] = "ok"
        else:
            todo.append((pip_name, mod, min_ver, required)); status[pip_name] = "missing"
    if skip or os.environ.get("PIPELINE_SKIP_INSTALL") == "1":
        for p, *_ in todo: status[p] = "missing (install skipped)"
        return status
    for pip_name, mod, min_ver, required in todo:
        spec = f"{pip_name}>={min_ver}" if min_ver else pip_name
        log.info("pip install %s ...", spec)
        rc, out, err = _pip_install(spec)
        importlib.invalidate_caches()
        if rc == 0 and _is_ok(pip_name, mod, min_ver):
            status[pip_name] = "installed"
        else:
            status[pip_name] = "FAILED"
            (log.error if required else log.warning)("تعذّر تثبيت %s: %s", pip_name, (err or out).strip()[-300:])
    return status

DEP_STATUS = install_dependencies(RESOURCE_PLAN["has_gpu"])
for _k, _v in DEP_STATUS.items():
    print(f"  {_k:<18}{_v}")

# اختياري: vLLM لا يُثبَّت تلقائيًا لأنه قد يستبدل torch. فعّله يدويًا: PIPE_INSTALL_VLLM=1
if os.environ.get("PIPE_INSTALL_VLLM") == "1" and not has_module("vllm"):
    log.warning("تثبيت vllm قد يغيّر إصدار torch. أعد تشغيل الـ kernel بعد انتهاء هذه الخلية.")
    _pip_install("vllm", timeout=2400)

FFMPEG_FIX = ensure_ffmpeg()
if FFMPEG_FIX["installed"]:
    _rc, _o, _ = run_cmd(["ffmpeg", "-version"], timeout=15); ENV["ffmpeg"] = _o.splitlines()[0] if _rc == 0 and _o else ENV.get("ffmpeg"); ENV["ffprobe"] = shutil.which("ffprobe")
    print("  ffmpeg/ffprobe    installed via static-ffmpeg")
elif FFMPEG_FIX["note"]: log.error("%s", FFMPEG_FIX["note"])

_missing_required = [k for k, v in DEP_STATUS.items() if v.startswith(("FAILED", "missing"))
                     and k in ("numpy", "Pillow", "opencv-python-headless", "huggingface_hub", "transformers", "accelerate", "pyarrow", "psutil")]
if _missing_required:
    log.warning("حزم إلزامية غير متوفرة: %s  → سيعمل ما أمكن (JSONL بدل Parquet / بدون Qwen) وسيُوضَّح ذلك لاحقًا.",
                _missing_required)

### CELL 03 — Configuration
> تعريف حقول Config وتحميلها. **الإعدادات التي تغيّرها أنت كلها في الخلية 00** (USER_SETTINGS)؛ هنا القيم الافتراضية والتحقق فقط.
> للتغيير دون لمس الكود: ملف config.json أو متغيرات بيئة بالشكل PIPE_<NAME>=value (الأولوية: بيئة > config.json > الخلية 00 > الافتراضي هنا).

In [ ]:
# CELL 03 - Configuration
#: تعريف حقول Config وتحميلها. **الإعدادات التي تغيّرها أنت كلها في الخلية 00** (USER_SETTINGS)؛ هنا القيم الافتراضية والتحقق فقط.
#: للتغيير دون لمس الكود: ملف config.json أو متغيرات بيئة بالشكل PIPE_<NAME>=value (الأولوية: بيئة > config.json > الخلية 00 > الافتراضي هنا).

USER_SETTINGS = globals().get("USER_SETTINGS") or {}        # يُعرَّف في الخلية 00 (كل الإعدادات هناك). إن شُغّلت الخلية 03 وحدها تُستعمل الافتراضيات أدناه

@dataclasses.dataclass
class Config:
    # ---------------- المصدر
    VIDEO_MANIFEST: str = ""                                          # مجلد فيديوهات أو CSV/JSON/JSONL/TXT؛ فارغ = بحث تلقائي (الخلية 07)
    AUTO_FIND_VIDEOS: bool = False                                    # بحث تلقائي عن فيديوهات مفتوحة الترخيص (الخلية 07)
    AUTO_FIND_QUERY: str = ""
    AUTO_FIND_LICENSES: str = "open"                                  # open | cc | any (الخلية 07)
    AUTO_FIND_SOURCES: str = "pexels,pixabay,commons,archive"          # ترتيب مصادر البحث التلقائي (الخلية 07)
    AUTO_PREFLIGHT: bool = True                                       # 24F يشغّل Preflight (قياس + Smoke Test) تلقائيًا إن لم يكن قد نجح، ثم يكمل
    TOTAL_VIDEOS: int = 2000                                          # 0 = كل ما في المانيفست
    MODEL_ID: str = "Qwen/Qwen3-VL-32B-Instruct"
    # ---------------- الإطارات والعينة
    TARGET_FPS: float = 30.0
    SAMPLING_RATIO: float = 0.05                     # نسبة الإطارات المرشّحة (في الجزء الأوسط بعد القص) التي تُؤخذ
    TRIM_START_RATIO: float = 0.10                   # قص من بداية الفيديو (نسبة من مدته)
    TRIM_END_RATIO: float = 0.10                     # وقص من نهايته
    SAMPLING_SEED: int = 20261105
    SAMPLING_METHOD: str = "stratified_random"       # stratified_random | deterministic_hash | exact_per_video
    EXACT_PER_VIDEO_SAMPLING: bool = False           # مرادف لـ SAMPLING_METHOD="exact_per_video"
    MIN_VIDEO_DURATION: float = 10.0
    MAX_VIDEO_DURATION: float = 3600.0
    ENFORCE_DURATION_LIMITS: bool = True
    MIN_VIDEO_YEAR: int = 2020                       # 0 = معطّل (سنة المصدر عند البحث + creation_time داخل الملف)
    MIN_VIDEO_SHORT_SIDE: int = 1080                 # min(w,h) ≥ هذا بعد التنزيل (0 = معطّل)
    # ---------------- الصور
    MIN_IMAGE_WIDTH: int = 256
    MIN_IMAGE_HEIGHT: int = 256
    IMAGE_FORMAT: str = "jpg"                        # jpg | webp
    IMAGE_QUALITY: int = 95
    MAX_STORED_IMAGE_DIMENSION: int = 4096           # لا تصغير إلا إذا تجاوز الضلع الأطول هذا الحد
    # ---------------- الوصف
    CAPTION_LANGUAGE: str = "English"
    CAPTION_MAX_NEW_TOKENS: int = 384
    CAPTION_BATCH_SIZE: Any = "auto"                 # "auto" أو رقم
    CAPTION_PRECISION: str = "bf16"                  # auto | bf16 | fp16 | int8 | int4
    CAPTION_ALLOW_FALLBACK_PLANS: bool = False       # False: لا shard ولا int8/int4 تلقائيًا (bf16 كاملة على GPU واحدة فقط)
    BENCH_START_BATCH: int = 16                      # أول دفعة في قياس Benchmark
    BENCH_VRAM_TARGET: float = 0.80                  # سقف ذاكرة GPU عند اختيار الدفعة (الباقي هامش KV-cache)
    BENCH_STABLE_FRACTION: float = 0.75              # نقطة البدء الفعلية = هذه النسبة من أفضل دفعة مقيسة
    ATTN_IMPLEMENTATION: str = "auto"                # auto | sdpa | flash_attention_2
    CAPTION_ENGINE: str = "auto"                     # auto | transformers | vllm
    MODEL_MIN_PIXELS: int = 65_536
    MODEL_MAX_PIXELS: int = 1_048_576                # ≈1MP ≈ 1024 رمز بصري (العامل 32)
    CAPTION_REPETITION_PENALTY: float = 1.05
    CAPTION_VALIDATION: bool = True
    ENABLE_VLM_SECOND_REVIEW: bool = False           # مراجعة ثانية بالـVLM (مكلفة، اختيارية)
    # ---------------- الشظايا
    SHARD_TARGET_SIZE_GB: float = 0.75
    # ---------------- المفاتيح
    ENABLE_EXACT_DEDUPLICATION: bool = True
    ENABLE_PERCEPTUAL_DEDUPLICATION: bool = True
    ENABLE_IMAGE_QUALITY_FILTER: bool = True
    ENABLE_CAPTIONING: bool = True
    ENABLE_GPU_DECODING: bool = True
    ENABLE_RANDOM_SAMPLE_PREVIEW: bool = True
    PREVIEW_SAMPLES_PER_BATCH: int = 1
    UPLOAD_AFTER_COMPLETED_SHARD: bool = True
    AUTO_RESUME: bool = True
    DRY_RUN: bool = False                            # لا رفع إلى HF: يُستعمل مخزن محلي بديل
    # ---------------- Hugging Face
    HF_DATASET_REPO: str = "YOUR_USERNAME/YOUR_DATASET"
    HF_TOKEN: Optional[str] = None                   # يُقرأ من Kaggle Secrets / البيئة، لا تكتبه هنا
    HF_PRIVATE_REPO: bool = True
    HF_XET_HIGH_PERFORMANCE: bool = False
    # ---------------- إزالة التكرار والجودة
    DEDUP_HAMMING_THRESHOLD: int = 8                 # على 64-bit pHash+dHash (يجب أن يتحقق كلاهما)
    DEDUP_LAST_N: int = 20                           # مقارنة بآخر N إطارًا مقبولًا (0 = النافذة الزمنية DEDUP_WINDOW_SECONDS)
    DEDUP_WINDOW_SECONDS: float = 12.0
    DEDUP_SSIM_VERIFY: bool = True
    DEDUP_SSIM_THRESHOLD: float = 0.95
    QC_MIN_LUMA_STD: float = 20.0                    # رفض الإطار المسطّح (std الإضاءة أقل من هذا). 0 = معطّل
    QC_TEXT_SCREEN_ENABLE: bool = True               # رفض شاشات النص (داكن + بقع ساطعة قليلة متجمّعة)
    QC_TEXT_DARK_MEAN: float = 60.0
    QC_TEXT_DARK_LEVEL: int = 50
    QC_TEXT_DARK_FRACTION: float = 0.80
    QC_TEXT_BRIGHT_LEVEL: int = 160
    QC_TEXT_BRIGHT_MIN_FRACTION: float = 0.0005
    QC_TEXT_BRIGHT_MAX_FRACTION: float = 0.12
    QC_TEXT_BBOX_MAX_FRACTION: float = 0.50
    QC_TEXT_MIN_COMPONENTS: int = 3
    CROSS_VIDEO_DEDUP: bool = False                  # فهرس BK-tree/LSH عالمي (اختياري)
    BLUR_THRESHOLD: Optional[float] = None           # None = قيمة محافظة/معايَرة من benchmark_results.json
    MAX_JPEG_FAILS_PER_VIDEO: int = 50
    # ---------------- التنزيل والتخزين
    WORK_DIR: Optional[str] = None
    TEMP_DIR: Optional[str] = None
    MIN_FREE_DISK_GB: float = 6.0
    DOWNLOAD_TIMEOUT: float = 30.0
    DOWNLOAD_RETRIES: int = 5
    DOWNLOAD_BACKOFF: float = 2.0
    KEEP_VIDEOS: bool = False
    UPLOAD_RETRIES: int = 5
    # ---------------- التشغيل والتوزيع
    WORKER_ID: int = 0                               # لتشغيل عدة Notebooks: كلٌّ يأخذ شريحة من الفيديوهات
    NUM_WORKERS: int = 1
    CHECKPOINT_EVERY_SECONDS: float = 120.0
    DASHBOARD_REFRESH_SECONDS: float = 10.0
    MAX_VIDEOS_THIS_RUN: int = 0                     # 0 = بلا حد (مفيد لجلسات Kaggle المحدودة)
    MAX_RUNTIME_HOURS: float = 0.0                   # 0 = بلا حد؛ عند بلوغه يُنهي النظام الجلسة بأمان
    DECODER_PREFERENCE: str = "auto"                 # auto | pynvvideocodec | ffmpeg_nvdec | ffmpeg_cpu
    SKIP_HDR_VIDEOS: bool = True                     # HDR بلا tone-mapping يعطي ألوانًا باهتة
    JPEG_SUBSAMPLING: int = -1                       # -1 افتراضي Pillow (4:2:0) | 0 = 4:4:4
    MAX_IMAGES_PER_VIDEO: int = 0                    # 0 = بلا سقف (35% كاملة). اختياري لتقليل زمن Qwen
    QC_ENABLE_SSIM_VERIFY: bool = True
    ALLOW_MOCK_CAPTIONER: bool = False               # للاختبارات المحلية فقط؛ لا يُستعمل أبدًا افتراضيًا
    CAPTION_MIN_WORDS: int = 18
    CAPTION_REVIEW_SCORE: float = 0.55               # أقل من ذلك → سجل المراجعة
    CAPTION_BANNED_PHRASES: List[str] = dataclasses.field(default_factory=lambda: ["devoid of", "no other", "solid black background", "no discernible", "blank"])
    CAPTION_BANNED_RETRIES: int = 1                  # إعادات المحاولة قبل تخطّي الإطار
    FFMPEG_THREADS: int = 0                          # 0 = تلقائي
    # ---------------- الموثوقية (تغيّرها فقط إن لزم)
    SHARD_MAX_AGE_MINUTES: float = 40.0              # إغلاق الشظية مبكرًا حتى لو لم تبلغ الحجم ⇒ سقف لما قد يضيع عند موت الجلسة. 0 = معطّل
    STOP_MARGIN_MINUTES: float = 12.0                # هامش قبل MAX_RUNTIME_HOURS لتفريغ الطوابير ورفع آخر شظية
    UPLOAD_PATIENCE_MINUTES: float = 90.0            # مدة الصبر على انقطاع الشبكة/HF قبل اعتبار الرفع فاشلًا (الضغط الخلفي يوقف الفك مؤقتًا)
    CHUNK_LINGER_SECONDS: float = 4.0                # أقصى انتظار لتجميع دفعة وصف جزئية
    PREFETCH_VIDEOS: int = 1                         # كم فيديو يُنزَّل مسبقًا أثناء معالجة الحالي
    DECODE_BYTE_BUDGET_GB: float = 0.0               # ميزانية RAM لإطارات RGB الحيّة. 0 = تلقائي (25% من المتاح، بحد أقصى 3GB)
    DISK_WAIT_MINUTES: float = 15.0                  # أقصى انتظار لتفريغ مساحة قبل اعتبار التنزيل فاشلًا مؤقتًا
    REQUIRE_SMOKE_TEST: bool = True                  # لا تبدأ التشغيل الكامل قبل نجاح الاختبار التكاملي الصغير (يُسجَّل في benchmark_results.json)
    SMOKE_TEST_IMAGES: int = 24                      # عدد صور اختبار الدخان (يُنفَّذ على أول فيديو)
    MAX_UNEXPECTED_VIDEO_ERRORS: int = 5             # أخطاء برمجية غير متوقعة (غير فك/تنزيل) قبل إيقاف التشغيل كله

    # ----------------------------------------------------------------
    def sampling_method(self) -> str:
        if self.EXACT_PER_VIDEO_SAMPLING: return "exact_per_video"
        return self.SAMPLING_METHOD

    def to_dict(self, redact: bool = True) -> Dict[str, Any]:
        d = dataclasses.asdict(self)
        if redact and d.get("HF_TOKEN"): d["HF_TOKEN"] = "***"
        return d

    def validate(self) -> List[str]:
        errs = []
        if not (0 < self.SAMPLING_RATIO <= 1): errs.append("SAMPLING_RATIO يجب أن يكون في (0,1]")
        if self.TARGET_FPS <= 0: errs.append("TARGET_FPS يجب أن يكون > 0")
        if self.IMAGE_FORMAT not in ("jpg", "webp"): errs.append("IMAGE_FORMAT: jpg أو webp")
        if self.SAMPLING_METHOD not in ("stratified_random", "deterministic_hash", "exact_per_video"): errs.append("SAMPLING_METHOD غير معروف (stratified_random | deterministic_hash | exact_per_video)")
        if not (0 <= self.TRIM_START_RATIO < 1 and 0 <= self.TRIM_END_RATIO < 1 and self.TRIM_START_RATIO + self.TRIM_END_RATIO < 1): errs.append("TRIM_START_RATIO/TRIM_END_RATIO: كل واحدة في [0,1) ومجموعهما < 1")
        if not (0 < self.BENCH_VRAM_TARGET <= 0.95): errs.append("BENCH_VRAM_TARGET يجب أن يكون في (0, 0.95]")
        if not (0 < self.BENCH_STABLE_FRACTION <= 1): errs.append("BENCH_STABLE_FRACTION يجب أن يكون في (0,1]")
        if self.BENCH_START_BATCH < 1: errs.append("BENCH_START_BATCH ≥ 1")
        if self.MIN_VIDEO_SHORT_SIDE < 0 or self.MIN_VIDEO_YEAR < 0 or self.DEDUP_LAST_N < 0 or self.CAPTION_BANNED_RETRIES < 0: errs.append("MIN_VIDEO_SHORT_SIDE/MIN_VIDEO_YEAR/DEDUP_LAST_N/CAPTION_BANNED_RETRIES لا تكون سالبة")
        if self.AUTO_FIND_LICENSES not in ("open", "cc", "any"): errs.append("AUTO_FIND_LICENSES: open | cc | any")
        if not (0 <= self.WORKER_ID < max(1, self.NUM_WORKERS)): errs.append("WORKER_ID خارج النطاق")
        if self.SHARD_TARGET_SIZE_GB <= 0: errs.append("SHARD_TARGET_SIZE_GB > 0")
        return errs


def _coerce(value: str, default: Any) -> Any:
    if isinstance(default, (list, tuple)): return [x.strip() for x in value.split("|") if x.strip()]       # PIPE_CAPTION_BANNED_PHRASES="devoid of|blank"
    if isinstance(default, bool): return value.strip().lower() in ("1", "true", "yes", "on")
    if isinstance(default, int) and not isinstance(default, bool): return int(float(value))
    if isinstance(default, float): return float(value)
    if default is None or isinstance(default, str):
        return None if value.strip().lower() in ("none", "null", "") and default is None else value
    return value

def load_config(path: Optional[str] = None) -> Config:
    cfg = Config()
    fields = {f.name: f for f in dataclasses.fields(Config)}
    for k, v in USER_SETTINGS.items():                # أدنى أولوية فوق الافتراضي؛ HF_TOKEN لا يُقرأ من هنا أبدًا، وVIDEO_URLS تُعالَج بعد تحديد WORK_DIR
        if k in fields and k != "HF_TOKEN" and v is not None and v != "": setattr(cfg, k, v)
    if ENV.get("on_marimo") and not ENV.get("on_kaggle"): cfg.DASHBOARD_REFRESH_SECONDS = 60.0     # كل تحديث يُطبع سطورًا في console الخلية: لا نغرقه
    wd = Path(os.environ.get("PIPE_WORK_DIR") or ("/kaggle/working" if Path("/kaggle/working").exists() else "./work"))
    cand = path or os.environ.get("PIPE_CONFIG_FILE") or str(wd / "config.json")
    if Path(cand).exists():
        try:
            for k, v in json.loads(Path(cand).read_text(encoding="utf-8")).items():
                if k in fields and k != "HF_TOKEN": setattr(cfg, k, v)
            log.info("config overrides loaded from %s", cand)
        except Exception as e:
            log.error("فشل قراءة %s: %r", cand, e)
    for k, f in fields.items():                      # متغيرات البيئة تتفوق على الملف
        ev = os.environ.get(f"PIPE_{k}")
        if ev is not None and k != "HF_TOKEN":
            cur = getattr(cfg, k)
            if k == "CAPTION_BATCH_SIZE": setattr(cfg, k, ev if ev == "auto" else int(ev))
            else: setattr(cfg, k, _coerce(ev, cur))
    if cfg.WORK_DIR is None: cfg.WORK_DIR = str(wd)
    if cfg.TEMP_DIR is None: cfg.TEMP_DIR = str(Path(tempfile.gettempdir()) / "vid2ds")
    urls = [str(u).strip() for u in (USER_SETTINGS.get("VIDEO_URLS") or []) if str(u).strip()]
    if urls:
        if (cfg.VIDEO_MANIFEST or "").strip(): log.warning("VIDEO_URLS تُجاهلت لأن VIDEO_MANIFEST مضبوط (%s): استعمل واحدًا فقط", cfg.VIDEO_MANIFEST)
        else:
            f = Path(cfg.WORK_DIR) / "my_videos.txt"; f.parent.mkdir(parents=True, exist_ok=True); f.write_text("\n".join(urls) + "\n", encoding="utf-8"); cfg.VIDEO_MANIFEST = str(f)
    errs = cfg.validate()
    if errs: raise ValueError("إعدادات غير صالحة: " + "; ".join(errs))
    return cfg

CONFIG = load_config()
Path(CONFIG.WORK_DIR).mkdir(parents=True, exist_ok=True)
Path(CONFIG.TEMP_DIR).mkdir(parents=True, exist_ok=True)
CONTROL_DIR = Path(CONFIG.WORK_DIR) / "control"                    # نسخة محلية من control/
SHARD_DIR = Path(CONFIG.WORK_DIR) / "shards"
VIDEO_DIR = Path(CONFIG.TEMP_DIR) / "videos"
for _d in (CONTROL_DIR, SHARD_DIR, VIDEO_DIR): _d.mkdir(parents=True, exist_ok=True)

atomic_write_json(CONTROL_DIR / "pipeline_config.json", {"config": CONFIG.to_dict(redact=True), "created": time.time()})
print(json.dumps(CONFIG.to_dict(redact=True), indent=1, ensure_ascii=False)[:3500])

### CELL 04 — Hugging Face Authentication
> قراءة HF_TOKEN من Kaggle Secrets أو البيئة. لا يُطبع الـ Token أبدًا ويُحجب من السجلات.

In [ ]:
# CELL 04 - Hugging Face Authentication
#: قراءة HF_TOKEN من Kaggle Secrets أو البيئة. لا يُطبع الـ Token أبدًا ويُحجب من السجلات.

def _dotenv_token() -> Optional[str]:
    """molab: لوحة Secrets تكتب ملف .env بجانب الدفتر. نقرأ HF_TOKEN منه فقط (لا نطبع ولا نحفظ شيئًا)."""
    for d in (Path.cwd(), Path(CONFIG.WORK_DIR)):
        f = d / ".env"
        try:
            if not f.is_file(): continue
            for line in f.read_text(encoding="utf-8", errors="ignore").splitlines():
                m = re.match(r"\s*(?:export\s+)?(HF_TOKEN|HUGGING_FACE_HUB_TOKEN)\s*=\s*(.*?)\s*$", line)
                if m:
                    v = m.group(2).strip().strip("'\"")
                    if v: return v
        except OSError: pass
    return None

def get_hf_token() -> Optional[str]:
    if CONFIG.HF_TOKEN:
        tok = CONFIG.HF_TOKEN
    else:
        tok = None
        try:                                           # Kaggle Secrets (اسم السر: HF_TOKEN)
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
        tok = tok or os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or _dotenv_token()
    if tok:
        _SECRETS.add(tok)
    return tok or None

def get_api_key(name: str) -> Optional[str]:
    """مفاتيح اختيارية لمصادر الفيديو (PEXELS_API_KEY / PIXABAY_API_KEY): Kaggle Secrets ← متغير بيئة ← .env (molab). لا تُطبع ولا تُكتب في أي ملف، وتُحجب من السجلات."""
    key = None
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret(name)
    except Exception: pass
    key = key or os.environ.get(name)
    if not key:
        for d in (Path.cwd(), Path(CONFIG.WORK_DIR)):
            try:
                f = d / ".env"
                if not f.is_file(): continue
                for line in f.read_text(encoding="utf-8", errors="ignore").splitlines():
                    m = re.match(r"\s*(?:export\s+)?" + re.escape(name) + r"\s*=\s*(.*?)\s*$", line)
                    if m and m.group(1).strip().strip("'\""): key = m.group(1).strip().strip("'\""); break
            except OSError: pass
            if key: break
    if key: _SECRETS.add(key)
    return key or None

def login_huggingface() -> Dict[str, Any]:
    """يعيد {'ok','user','reason'}. في DRY_RUN أو بلا Token لا يُعدّ ذلك خطأ قاتلًا."""
    res = {"ok": False, "user": None, "reason": ""}
    if CONFIG.DRY_RUN:
        res["reason"] = "DRY_RUN: لن يُستعمل Hugging Face"; return res
    tok = get_hf_token()
    if not tok:
        res["reason"] = "لا يوجد HF_TOKEN (Kaggle: Secrets باسم HF_TOKEN | molab: لوحة Secrets أو ملف .env يحوي HF_TOKEN=... | أو متغير بيئة)"; return res
    if CONFIG.HF_XET_HIGH_PERFORMANCE:
        os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
    try:
        from huggingface_hub import HfApi
        who = HfApi(token=tok).whoami()
        res.update(ok=True, user=who.get("name"))
    except Exception as e:
        res["reason"] = f"فشل التحقق من الـ Token: {type(e).__name__}"
    return res

HF_LOGIN = login_huggingface()
print("HF login:", {k: v for k, v in HF_LOGIN.items()})
if not HF_LOGIN["ok"] and not CONFIG.DRY_RUN:
    log.warning("لا اتصال بـ HF → سيعمل النظام كـ DRY_RUN (مخزن محلي) إلى أن تُصلح المصادقة. السبب: %s", HF_LOGIN["reason"])

### CELL 05 — Repository Initialization
> طبقة تخزين عن بُعد (HFStore) + بديل محلي (LocalStore) للتجارب. الالتزام (commit) واحد وذري يحمل الشظية + الفهرس + ملفات control.

In [ ]:
# CELL 05 - Repository Initialization
#: طبقة تخزين عن بُعد (HFStore) + بديل محلي (LocalStore) للتجارب. الالتزام (commit) واحد وذري يحمل الشظية + الفهرس + ملفات control.

def with_retries(fn: Callable[[], Any], what: str, retries: int = 5, backoff: float = 2.0,
                 no_retry: Tuple[type, ...] = ()) -> Any:
    last: Optional[BaseException] = None
    for attempt in range(1, retries + 1):
        try:
            return fn()
        except no_retry:
            raise
        except Exception as e:
            last = e
            wait = min(120.0, backoff ** attempt) + random.random()
            log.warning("%s فشل (محاولة %d/%d): %s: %s — إعادة بعد %.0fs", what, attempt, retries,
                        type(e).__name__, str(e)[:160], wait)
            time.sleep(wait if attempt < retries else 0)
    raise RuntimeError(f"{what}: استُنفدت المحاولات") from last


class RemoteStore:
    name = "abstract"
    def exists(self, path: str) -> bool: raise NotImplementedError
    def size(self, path: str) -> Optional[int]: raise NotImplementedError
    def read_bytes(self, path: str) -> Optional[bytes]: raise NotImplementedError
    def list_files(self, prefix: str = "") -> List[str]: raise NotImplementedError
    def sizes(self, paths: Sequence[str]) -> Dict[str, Optional[int]]:
        return {p: self.size(p) for p in paths}
    def download(self, path: str, local: Path) -> bool:
        b = self.read_bytes(path)
        if b is None: return False
        atomic_write_bytes(local, b); return True
    def read_range(self, path: str, start: int, length: int) -> Optional[bytes]:
        """بايتات [start, start+length) من ملف بعيد دون تنزيله كاملًا (قراءة عشوائية عبر إزاحات الفهرس). الأساس: بديل بطيء يقرأ الملف كله."""
        b = self.read_bytes(path)
        return None if b is None else b[start:start + length]
    def commit(self, adds: Dict[str, Any], message: str, deletes: Sequence[str] = ()) -> None:
        """adds: {مسار_في_المستودع: Path | bytes}. إما أن ينجح كله أو يفشل كله."""
        raise NotImplementedError
    def verify(self, path: str, size: int, sha256: Optional[str] = None) -> bool:
        s = self.size(path)
        return s is not None and s == size


class LocalStore(RemoteStore):
    """يحاكي مستودع HF داخل مجلد محلي (DRY_RUN والاختبارات). fail_next_commits لحقن أعطال في الاختبارات."""
    name = "local"
    def __init__(self, root: os.PathLike):
        self.root = Path(root); self.root.mkdir(parents=True, exist_ok=True)
        self.fail_next_commits = 0
        self.commit_count = 0
    def _p(self, path: str) -> Path: return self.root / path
    def exists(self, path): return self._p(path).is_file()
    def size(self, path): return self._p(path).stat().st_size if self.exists(path) else None
    def read_bytes(self, path): return self._p(path).read_bytes() if self.exists(path) else None
    def read_range(self, path, start, length):
        if not self.exists(path): return None
        with open(self._p(path), "rb") as f: f.seek(start); return f.read(length)
    def list_files(self, prefix=""):
        return sorted(str(p.relative_to(self.root)) for p in self.root.rglob("*")
                      if p.is_file() and not p.name.startswith(".tmp") and ".tmp" not in p.name
                      and str(p.relative_to(self.root)).startswith(prefix))
    def commit(self, adds, message, deletes=()):
        if self.fail_next_commits > 0:
            self.fail_next_commits -= 1
            raise ConnectionError("simulated upload failure")
        staged: List[Tuple[Path, Path]] = []
        try:
            for rp, src in adds.items():                       # 1) نكتب كل شيء مؤقتًا
                dst = self._p(rp); dst.parent.mkdir(parents=True, exist_ok=True)
                tmp = dst.with_name(dst.name + ".tmpcommit")
                if isinstance(src, (bytes, bytearray)): tmp.write_bytes(bytes(src))
                else: shutil.copyfile(src, tmp)
                staged.append((tmp, dst))
            for tmp, dst in staged: os.replace(tmp, dst)       # 2) ثم نُبدّل بسرعة
        finally:
            for tmp, _ in staged: tmp.unlink(missing_ok=True)
        for d in deletes:
            f = self._p(d); f.unlink(missing_ok=True); q = f.parent
            while q != self.root and q.is_dir() and not any(q.iterdir()):          # HF لا يحمل مجلدات فارغة: نحاكيه بحذف الآباء الفارغة صعودًا
                q.rmdir(); q = q.parent
        self.commit_count += 1


class HFStore(RemoteStore):
    name = "huggingface"
    def __init__(self, repo_id: str, token: str, private: bool = True):
        from huggingface_hub import HfApi
        self.repo_id, self.token = repo_id, token
        self.api = HfApi(token=token)
        with_retries(lambda: self.api.create_repo(repo_id, repo_type="dataset", private=private, exist_ok=True),
                     "create_repo", retries=3)
        self.private: Optional[bool] = None            # الخصوصية الفعلية: مستودع موجود مسبقًا يحتفظ بخصوصيته ولا يتبع HF_PRIVATE_REPO
        try: self.private = bool(with_retries(lambda: self.api.dataset_info(repo_id), "dataset_info", retries=2).private)
        except Exception: pass
    def _info(self, path: str):
        r = with_retries(lambda: self.api.get_paths_info(self.repo_id, [path], repo_type="dataset"),
                         f"get_paths_info({path})", retries=3)
        return r[0] if r else None
    def exists(self, path): return self._info(path) is not None
    def size(self, path):
        i = self._info(path); return getattr(i, "size", None) if i else None
    def sizes(self, paths):
        out: Dict[str, Optional[int]] = {p: None for p in paths}
        for i in range(0, len(paths), 200):
            part = list(paths[i:i + 200])
            for info in with_retries(lambda: self.api.get_paths_info(self.repo_id, part, repo_type="dataset"),
                                     "get_paths_info(batch)", retries=3):
                out[info.path] = getattr(info, "size", None)
        return out
    def read_bytes(self, path):
        if not self.exists(path): return None
        from huggingface_hub import hf_hub_download
        p = with_retries(lambda: hf_hub_download(self.repo_id, path, repo_type="dataset", token=self.token,
                                                 force_download=True), f"download({path})", retries=4)
        return Path(p).read_bytes()
    def download(self, path, local):
        """تنزيل ملف كبير (شظية) مباشرة إلى مجلد محلي مؤقت بجانب الوجهة بلا نسخة ثانية في كاش HF (لا يضاعف استهلاك القرص)، ثم نقله ذريًا إلى `local`."""
        if not self.exists(path): return False
        from huggingface_hub import hf_hub_download
        Path(local).parent.mkdir(parents=True, exist_ok=True)
        tmp = Path(local).parent / f".dl-{uuid.uuid4().hex[:8]}"; tmp.mkdir(parents=True, exist_ok=True)
        try:
            p = with_retries(lambda: hf_hub_download(self.repo_id, path, repo_type="dataset", token=self.token, local_dir=str(tmp), force_download=True), f"download({path})", retries=4)
            os.replace(p, local); return True
        finally: shutil.rmtree(tmp, ignore_errors=True)
    def read_range(self, path, start, length):
        """HfFileSystem: طلبات HTTP Range فقط (لا تنزيل كامل للشظية). عند الفشل: قراءة الملف كاملًا (بطيئة لكنها صحيحة)."""
        def _go():
            from huggingface_hub import HfFileSystem
            return HfFileSystem(token=self.token).cat_file(f"datasets/{self.repo_id}/{path}", start=start, end=start + length)
        try: return with_retries(_go, f"range({path})", retries=3)
        except Exception as e:
            log.warning("قراءة Range عبر HfFileSystem فشلت (%s) → قراءة كاملة للملف", type(e).__name__)
            return RemoteStore.read_range(self, path, start, length)
    def list_files(self, prefix=""):
        fs = with_retries(lambda: self.api.list_repo_files(self.repo_id, repo_type="dataset"), "list_repo_files", retries=3)
        return sorted(f for f in fs if f.startswith(prefix))
    def commit(self, adds, message, deletes=()):
        from huggingface_hub import CommitOperationAdd, CommitOperationDelete
        def _do():
            ops = [CommitOperationAdd(path_in_repo=rp, path_or_fileobj=(io.BytesIO(bytes(src)) if isinstance(src, (bytes, bytearray)) else str(src)))
                   for rp, src in adds.items()]
            ops += [CommitOperationDelete(path_in_repo=d) for d in deletes]
            return self.api.create_commit(repo_id=self.repo_id, repo_type="dataset", operations=ops, commit_message=message)
        with_retries(_do, f"create_commit({message[:40]})", retries=CONFIG.UPLOAD_RETRIES, backoff=CONFIG.DOWNLOAD_BACKOFF)
    def verify(self, path, size, sha256=None):
        i = self._info(path)
        if i is None or getattr(i, "size", None) != size: return False
        lfs = getattr(i, "lfs", None)
        lsha = (getattr(lfs, "sha256", None) or (lfs.get("sha256") if isinstance(lfs, dict) else None)) if lfs else None
        return (sha256 is None) or (lsha is None) or (lsha == sha256)


# ---------------------------------------------------------------- مسارات المستودع
def control_prefix() -> str:
    return "control" if CONFIG.NUM_WORKERS == 1 else f"control/w{CONFIG.WORKER_ID:02d}"

def shard_basename(n: int, kind: str = "train") -> str:
    return f"{kind}-{n:06d}" if CONFIG.NUM_WORKERS == 1 else f"{kind}-w{CONFIG.WORKER_ID:02d}-{n:06d}"

CONTROL_FILES = ["progress.json", "videos_manifest.jsonl", "completed_shards.json", "failed_videos.jsonl",
                 "pipeline_config.json", "benchmark_results.json"]

DATASET_CARD = """---
license: other
task_categories:
- text-to-image
- image-to-text
language:
- en
tags:
- webdataset
- video-frames
- captions
size_categories:
- 1M<n<10M
---
# Video-frame image/caption dataset

Frames sampled from user-provided videos (deterministic hash sampling), quality filtered and de-duplicated,
captioned with `{model}`. Stored as WebDataset shards (`data/train-*.tar`: `<id>.jpg` + `<id>.json`),
with per-shard Parquet indexes in `index/` and resume state in `control/`.
Load with `webdataset` (`wds.WebDataset("hf://datasets/<repo>/data/train-{{000001..00000N}}.tar")`) or stream the tar files.
Respect the licenses of the source videos (see `source_name` / `license` fields in each JSON).
"""

def make_store() -> RemoteStore:
    if (not CONFIG.DRY_RUN) and HF_LOGIN.get("ok"):
        try:
            st = HFStore(CONFIG.HF_DATASET_REPO, get_hf_token(), private=CONFIG.HF_PRIVATE_REPO)
            log.info("remote store: Hugging Face dataset %s", CONFIG.HF_DATASET_REPO)
            return st
        except Exception as e:
            log.error("تعذّر تهيئة مستودع HF (%s) → المخزن المحلي", type(e).__name__)
    st = LocalStore(Path(CONFIG.WORK_DIR) / "local_repo")
    log.warning("remote store: LOCAL (%s) — لا رفع إلى Hugging Face في هذه الجلسة", st.root)
    return st

STORE: RemoteStore = make_store()
if CONFIG.WORKER_ID == 0 and not STORE.exists("README.md"):
    try: STORE.commit({"README.md": DATASET_CARD.format(model=CONFIG.MODEL_ID).encode("utf-8")}, "init dataset card")
    except Exception as e: log.warning("لم يُنشأ README: %r", e)
print("store:", STORE.name, "| control prefix:", control_prefix(), "| next shard name:", shard_basename(1))

### CELL 06 — Resume State Loader
> حالة المشروع (control/). الحالة تُلتزَم عن بُعد داخل نفس commit الشظية ⇒ دائمًا متسقة مع البيانات الموجودة فعلًا.

In [ ]:
# CELL 06 - Resume State Loader
#: حالة المشروع (control/). الحالة تُلتزَم عن بُعد داخل نفس commit الشظية ⇒ دائمًا متسقة مع البيانات الموجودة فعلًا.

MAX_VIDEO_ATTEMPTS = 3
REMOTE_CONTROL_MIN_INTERVAL = 600.0

def stable_hash_int(text: str, mod: int = 1 << 62) -> int:
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8"), digest_size=8).digest(), "big") % mod

def new_video_record(row: Dict[str, Any]) -> Dict[str, Any]:
    return dict(video_id=row["video_id"], state="pending", url=row.get("video_url"), source_name=row.get("source_name"),
                license=row.get("license"), source_reference=row.get("source_reference"),
                expected_duration=row.get("expected_duration"), sha256=row.get("sha256"), expected_size=row.get("expected_size"),
                download_status="pending", decode_status="pending",
                duration=None, width=None, height=None, fps_avg=None, fps_mode=None, codec=None,
                candidate_frames=0, selected_frames=0, sampling_ratio_actual=None,
                accepted=0, rejected=0, rejected_by_reason={}, captioned=0, caption_failures=0, review_flagged=0,
                cursor=-1, shards=[], upload_status="pending", attempts=0, error=None, updated_at=time.time())


class StateManager:
    def __init__(self, store: RemoteStore, control_dir: Path):
        self.store, self.dir, self.lock = store, Path(control_dir), threading.RLock()
        self.videos: Dict[str, Dict[str, Any]] = {}          # بالترتيب (ترتيب الإدراج = ترتيب المانيفست)
        self.shards: List[Dict[str, Any]] = []
        self.failed: List[Dict[str, Any]] = []
        self.benchmark: Dict[str, Any] = {}
        self.progress: Dict[str, Any] = dict(total_videos=0, completed_videos=0, failed_videos=0, total_candidate_frames=0,
                                             selected_frames=0, accepted_images=0, rejected_images=0, captioned_images=0,
                                             uploaded_shards=0, current_video=None, current_stage="initialization",
                                             rejects_by_reason={}, bytes_uploaded=0, sessions=0)
        self.resumed = False
        self._last_local_save = 0.0
        self._last_remote_control = 0.0
        self._dirty_since_commit = False

    # ------------------------------------------------------------ تحميل / حفظ
    def _remote(self, name: str) -> str: return f"{control_prefix()}/{name}"

    def load(self) -> bool:
        """يحمّل حالة الاستئناف من المخزن البعيد (إن وُجدت). يعيد True عند الاستئناف الفعلي."""
        if not CONFIG.AUTO_RESUME:
            log.info("AUTO_RESUME=False → بداية جديدة"); return False
        raw = {n: self.store.read_bytes(self._remote(n)) for n in CONTROL_FILES}
        if not raw["progress.json"] or not raw["videos_manifest.jsonl"]:
            log.info("لا حالة سابقة في المخزن البعيد → بداية جديدة"); return False
        try:
            self.progress.update(json.loads(raw["progress.json"]))
            for line in raw["videos_manifest.jsonl"].decode("utf-8").splitlines():
                if line.strip():
                    r = json.loads(line); self.videos[r["video_id"]] = r
            self.shards = json.loads(raw["completed_shards.json"] or b"[]")
            self.failed = [json.loads(l) for l in (raw["failed_videos.jsonl"] or b"").decode("utf-8").splitlines() if l.strip()]
            self.benchmark = json.loads(raw["benchmark_results.json"] or b"{}")
        except Exception as e:
            raise RuntimeError(f"ملفات control تالفة في المخزن البعيد: {e!r}") from e
        self.progress["sessions"] = int(self.progress.get("sessions", 0)) + 1
        self.resumed = True
        self._verify_shards()
        log.info("RESUME: %d فيديو في الحالة | %d شظية مكتملة | %d فيديو done", len(self.videos), len(self.shards),
                 sum(v["state"] == "done" for v in self.videos.values()))
        return True

    def _verify_shards(self) -> None:
        bad = []
        sizes = self.store.sizes([s["repo_path"] for s in self.shards]) if self.shards else {}
        for s in self.shards:
            if sizes.get(s["repo_path"]) != s["size"]: bad.append(s["name"])
        if bad:
            log.error("شظايا مسجَّلة لكنها مفقودة/مختلفة الحجم في المخزن: %s ... (%d) — راجع المستودع قبل المتابعة", bad[:5], len(bad))
        self.progress["damaged_shards"] = bad

    def next_shard_index(self, kind: str = "train") -> int:
        pat = re.compile(r"^%s-(?:w\d+-)?(\d{6})$" % re.escape(kind))
        nums = [int(m.group(1)) for s in self.shards for m in [pat.match(s["name"])] if m]
        return (max(nums) + 1) if nums else 1

    # ------------------------------------------------------------ الفيديوهات
    def init_videos(self, rows: List[Dict[str, Any]]) -> None:
        with self.lock:
            for r in rows:
                if CONFIG.NUM_WORKERS > 1 and stable_hash_int(r["video_id"]) % CONFIG.NUM_WORKERS != CONFIG.WORKER_ID:
                    continue
                if r["video_id"] not in self.videos:
                    self.videos[r["video_id"]] = new_video_record(r)
                else:
                    v = self.videos[r["video_id"]]
                    v["url"] = r.get("video_url") or v.get("url")
            self.recompute()

    def pending_videos(self) -> List[Dict[str, Any]]:
        with self.lock:
            return [v for v in self.videos.values() if v["state"] not in ("done", "failed_permanent", "skipped")]

    def update_video(self, vid: str, **kw) -> None:
        with self.lock:
            v = self.videos[vid]; v.update(kw); v["updated_at"] = time.time(); self._dirty_since_commit = True

    def add_reject(self, vid: str, reason: str, n: int = 1) -> None:
        with self.lock:
            v = self.videos[vid]; v["rejected"] += n
            v["rejected_by_reason"][reason] = v["rejected_by_reason"].get(reason, 0) + n

    def mark_failed(self, vid: str, reason: str, permanent: bool = False) -> None:
        reason = redact_text(reason)                       # الخطأ يُحفظ في control/ (يُرفع إلى المستودع): بلا توقيعات روابط ولا Tokens
        with self.lock:
            v = self.videos[vid]; v["attempts"] += 1; v["error"] = reason[:500]
            v["state"] = "failed_permanent" if (permanent or v["attempts"] >= MAX_VIDEO_ATTEMPTS) else "failed"
            self.failed.append(dict(video_id=vid, reason=reason[:500], attempt=v["attempts"], state=v["state"], time=time.time()))
            self._dirty_since_commit = True
        log.error("video %s → %s: %s", vid, v["state"], reason[:200])

    def set_stage(self, stage: str, current_video: Optional[str] = None) -> None:
        with self.lock:
            self.progress["current_stage"] = stage
            if current_video is not None or stage in ("idle", "completed"): self.progress["current_video"] = current_video

    # ------------------------------------------------------------ تجميع وحفظ
    def recompute(self) -> Dict[str, Any]:
        with self.lock:
            vs = list(self.videos.values()); p = self.progress
            p["total_videos"] = len(vs)
            p["completed_videos"] = sum(v["state"] == "done" for v in vs)
            p["failed_videos"] = sum(v["state"] in ("failed", "failed_permanent") for v in vs)
            p["total_candidate_frames"] = sum(v["candidate_frames"] for v in vs)
            p["selected_frames"] = sum(v["selected_frames"] for v in vs)
            p["accepted_images"] = sum(v["accepted"] for v in vs)
            p["rejected_images"] = sum(v["rejected"] for v in vs)
            p["captioned_images"] = sum(v["captioned"] for v in vs)
            p["uploaded_shards"] = len(self.shards)
            rb: Dict[str, int] = {}
            for v in vs:
                for k, n in v["rejected_by_reason"].items(): rb[k] = rb.get(k, 0) + n
            p["rejects_by_reason"] = rb
            p["updated_at"] = time.time()
            return p

    def control_bytes(self) -> Dict[str, bytes]:
        """لقطة متسقة لكل ملفات control (تُستدعى من الخيط الرئيسي لحظة إغلاق الشظية)."""
        with self.lock:
            self.recompute()
            out = {
                "progress.json": json.dumps(self.progress, ensure_ascii=False, indent=2, default=str).encode(),
                # الرابط الكامل (قد يكون موقَّعًا مسبقًا) لا يُكتب في control/؛ المانيفست يعيده في كل جلسة (init_videos)
                "videos_manifest.jsonl": "\n".join(json.dumps({**v, "url": redact_url(v.get("url"))}, ensure_ascii=False, default=str) for v in self.videos.values()).encode(),
                "completed_shards.json": json.dumps(self.shards, ensure_ascii=False, indent=1).encode(),
                "failed_videos.jsonl": "\n".join(json.dumps(f, ensure_ascii=False) for f in self.failed).encode(),
                "pipeline_config.json": json.dumps({"config": CONFIG.to_dict(redact=True), "env": {k: ENV.get(k) for k in
                                         ("python", "torch", "cuda_runtime", "gpus", "cpu_cores", "ram_total_gb")},
                                         "updated_at": time.time()}, ensure_ascii=False, indent=2, default=str).encode(),
                "benchmark_results.json": json.dumps(self.benchmark, ensure_ascii=False, indent=2, default=str).encode(),
            }
            return out

    def save_local(self, force: bool = False) -> None:
        if not force and time.time() - self._last_local_save < CONFIG.CHECKPOINT_EVERY_SECONDS: return
        for name, data in self.control_bytes().items():
            atomic_write_bytes(self.dir / name, data)
        self._last_local_save = time.time()

    def maybe_commit_control_only(self, force: bool = False) -> bool:
        """Checkpoint بعيد بلا شظية (فيديوهات فاشلة/مكتملة بلا عينات). محدود التكرار لحماية حدود HF."""
        with self.lock:
            if not self._dirty_since_commit and not force: return False
            if not force and time.time() - self._last_remote_control < REMOTE_CONTROL_MIN_INTERVAL: return False
            snap = self.control_bytes()
        try:
            self.store.commit({f"{control_prefix()}/{k}": v for k, v in snap.items()}, "control checkpoint")
            self._last_remote_control = time.time(); self._dirty_since_commit = False
            return True
        except Exception as e:
            log.warning("فشل checkpoint للحالة (سيُعاد لاحقًا): %r", e); return False

    def register_shard(self, info: Dict[str, Any]) -> None:
        with self.lock:
            self.shards.append(info)
            for vid, n in info.get("videos", {}).items():
                if vid in self.videos and info["name"] not in self.videos[vid]["shards"]:
                    self.videos[vid]["shards"].append(info["name"])
            self._dirty_since_commit = False; self._last_remote_control = time.time()

    def mark_committed(self, info: Dict[str, Any]) -> None:
        with self.lock:
            for s in self.shards:
                if s["name"] == info["name"]: s["uploaded_at"] = time.time(); s["upload_ok"] = True
            self.progress["bytes_uploaded"] = self.progress.get("bytes_uploaded", 0) + info.get("size", 0)

    def reload_from_remote(self, manifest_rows: Optional[List[Dict[str, Any]]] = None) -> bool:
        """بعد فشل/إيقاف غير سليم: نتخلى عن أي تقدّم لم يُلتزَم عن بُعد (قد يشير لشظايا لم تُرفع) ونعود للحالة البعيدة المتسقة مع البيانات الفعلية."""
        with self.lock:
            bench = dict(self.benchmark)
            self.videos.clear(); self.shards.clear(); self.failed.clear(); self.benchmark = {}; self.resumed = False
            ok = self.load()
            if ok: self.fail_unfinished_pending_for_retry()
            if not self.benchmark: self.benchmark = bench
            if manifest_rows: self.init_videos(manifest_rows)
            self.progress["current_video"] = None; self.progress["current_stage"] = "reloaded"
            self.recompute()
            return ok

    def fail_unfinished_pending_for_retry(self) -> None:
        """عند الاستئناف: أي فيديو بحالة وسيطة يعود pending مع الحفاظ على المؤشر المُلتزَم (cursor)."""
        with self.lock:
            for v in self.videos.values():
                if v["state"] in ("downloading", "downloaded", "analyzing", "processing", "failed"):
                    v["state"] = "pending"
                    v["download_status"] = "pending" if v["download_status"] != "done" else "done"
            self.recompute()


STATE = StateManager(STORE, CONTROL_DIR)
RESUMED = STATE.load()
if RESUMED: STATE.fail_unfinished_pending_for_retry()
print("resumed:", RESUMED, "| next shard index:", STATE.next_shard_index(), "| progress:",
      {k: STATE.progress[k] for k in ("total_videos", "completed_videos", "failed_videos", "uploaded_shards")})

### CELL 07 — Manifest Validation
> قراءة قائمة الفيديوهات: CSV/JSON/JSONL/TXT أو مجلد فيديوهات (Kaggle Input)، مع تحقق صارم وتقرير بالأخطاء.
> لا قائمة مضبوطة؟ يبحث تلقائيًا (بحث محدود الزمن) داخل /kaggle/input عن ملف قائمة أو فيديوهات، وإلا يطبع بالضبط ما عليك فعله.

In [ ]:
# CELL 07 - Manifest Validation
#: قراءة قائمة الفيديوهات: CSV/JSON/JSONL/TXT أو مجلد فيديوهات (Kaggle Input)، مع تحقق صارم وتقرير بالأخطاء.
#: لا قائمة مضبوطة؟ يبحث تلقائيًا (بحث محدود الزمن) داخل /kaggle/input عن ملف قائمة أو فيديوهات، وإلا يطبع بالضبط ما عليك فعله.
import csv

VIDEO_EXTS = (".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v", ".ts", ".flv", ".wmv")
KAGGLE_INPUT = Path("/kaggle/input")
_MANIFEST_EXTS = (".csv", ".json", ".jsonl", ".txt")
_MANIFEST_NAME_RE = re.compile(r"(manifest|video|url|link|list)", re.I)
_ID_RE = re.compile(r"^[A-Za-z0-9._\-]{1,128}$")
MANIFEST_FIELDS = ("video_id", "video_url", "source_name", "license", "source_reference", "expected_duration",
                   "sha256", "expected_size")

def _slug(text: str) -> str:
    s = re.sub(r"[^A-Za-z0-9._\-]+", "_", text).strip("._-")
    return s[:100] or hashlib.blake2b(text.encode(), digest_size=6).hexdigest()

def _read_rows(path: Path) -> List[Dict[str, Any]]:
    suf = path.suffix.lower()
    if suf == ".csv":
        with open(path, newline="", encoding="utf-8-sig") as f:
            return [dict(r) for r in csv.DictReader(f)]
    if suf == ".jsonl":
        return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
    if suf == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(data, dict):
            data = data.get("videos") or data.get("items") or data.get("data") or []
        return list(data)
    if suf == ".txt":                       # رابط أو مسار ملف في كل سطر؛ الأسطر الفارغة وما يبدأ بـ # تُهمل
        return [dict(video_url=l.strip()) for l in path.read_text(encoding="utf-8-sig").splitlines() if l.strip() and not l.strip().startswith("#")]
    raise ValueError(f"صيغة قائمة غير مدعومة: {suf} (المدعوم: csv, json, jsonl, txt، أو مجلد فيديوهات)")

def _scan_kaggle_input(root: Path) -> List[Dict[str, Any]]:
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
            rows.append(dict(video_id=_slug(p.stem), video_url=str(p), source_name="kaggle_input", license="", source_reference=str(p)))
    return rows

def _walk_limited(root: Path, max_seconds: float = 20.0, max_depth: int = 6) -> Iterator[Path]:
    """مسح مجلد كبير (قد يحوي ملايين الملفات) بلا تعليق: يتوقف عند انقضاء الزمن أو بلوغ العمق."""
    t0, stack = time.time(), [(root, 0)]
    while stack:
        d, depth = stack.pop()
        try: entries = sorted(os.scandir(d), key=lambda e: e.name)
        except OSError: continue
        for e in entries:
            if time.time() - t0 > max_seconds: return
            try: is_dir = e.is_dir(follow_symlinks=False)
            except OSError: continue
            if is_dir:
                if depth < max_depth: stack.append((Path(e.path), depth + 1))
            else: yield Path(e.path)

def discover_manifest() -> Tuple[Optional[str], str]:
    """بحث تلقائي حين لا يُضبط VIDEO_MANIFEST: (1) ملف قائمة صالح داخل المرفقات، (2) مجلد المرفق الذي فيه أكثر فيديوهات. يعيد (المسار، وصف) أو (None، سبب)."""
    roots = [KAGGLE_INPUT] if KAGGLE_INPUT.is_dir() else [d for d in (Path(CONFIG.WORK_DIR) / "videos", Path("videos")) if d.is_dir()]
    if not roots: return None, "لا يوجد /kaggle/input ولا مجلد videos"
    lists: List[Path] = []; per_dir: "collections.Counter[str]" = collections.Counter()
    for root in roots:
        for f in _walk_limited(root):
            suf = f.suffix.lower()
            if suf in VIDEO_EXTS:
                rel = f.relative_to(root).parts; per_dir[str(root / rel[0]) if len(rel) > 1 else str(root)] += 1
            elif suf in _MANIFEST_EXTS and _MANIFEST_NAME_RE.search(f.stem):
                try:
                    if f.stat().st_size <= 50_000_000: lists.append(f)
                except OSError: pass
    best: Optional[Tuple[Path, int]] = None
    for f in lists:
        try: rows, _ = load_manifest(str(f))
        except Exception: continue
        if rows and (best is None or len(rows) > best[1]): best = (f, len(rows))
    if best: return str(best[0]), f"ملف قائمة داخل المرفقات: {best[1]} فيديو صالح"
    if per_dir:
        d, n = per_dir.most_common(1)[0]; return d, f"مجلد فيه {n} ملف فيديو (+{sum(per_dir.values()) - n} في مجلدات أخرى)" if len(per_dir) > 1 else f"مجلد فيه {n} ملف فيديو"
    return None, "لا فيديوهات ولا ملف قائمة في " + ", ".join(str(r) for r in roots)

# ---------------------------------------------------------------- بحث تلقائي: Pexels / Pixabay / Wikimedia Commons / Internet Archive
_IA_PERMISSIVE_RE = re.compile(r"(publicdomain|creativecommons\.org/licenses/(by|by-sa)/)", re.I)
_IA_LICENSE_Q = "(licenseurl:*publicdomain* OR licenseurl:*creativecommons.org/licenses/by/* OR licenseurl:*creativecommons.org/licenses/by-sa/*)"
SEARCH_REJECTIONS: "collections.Counter[str]" = collections.Counter()      # مرشّحون رُفضوا أثناء البحث (قبل التنزيل): السبب → العدد. يُحفظ في STATE.progress["search_rejections"] ويظهر في التقرير النهائي

def _rej(code: str, detail: str = "") -> None:
    SEARCH_REJECTIONS[code] += 1
    if detail and SEARCH_REJECTIONS[code] <= 3: log.info("بحث: رُفض مرشّح (%s): %s", code, detail[:150])

def _http_json(url: str, timeout: float = 30.0, retries: int = 3, headers: Optional[Dict[str, str]] = None) -> Any:
    import urllib.request, urllib.parse
    last: Optional[Exception] = None
    for i in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "vid2ds-dataset-builder/1.0 (research dataset pipeline)", **(headers or {})})
            with urllib.request.urlopen(req, timeout=timeout) as r: return json.loads(r.read().decode("utf-8"))
        except Exception as e:
            last = e; time.sleep(1.5 * (i + 1))
    raise RuntimeError(f"تعذّر الاتصال بـ {urllib.parse.urlparse(url).netloc} ({type(last).__name__}: {redact_text(str(last), 120)})")

_IA_CC_RE = re.compile(r"(publicdomain|creativecommons\.org)", re.I)
_IA_CC_Q = "(licenseurl:*publicdomain* OR licenseurl:*creativecommons.org*)"

def _ia_license_ok(lic: str, mode: str) -> bool:
    if mode == "any": return True
    if mode == "cc": return bool(_IA_CC_RE.search(lic))
    return bool(_IA_PERMISSIVE_RE.search(lic)) and not re.search(r"licenses/by-(nc|nd)", lic, re.I)

def _year_of(v: Any) -> Optional[int]:
    """أول سنة من 4 أرقام في قيمة تاريخ نصية ('2021-05-04', '2019', 'Mar 2020'...) إن كانت معقولة (1800..العام القادم)، وإلا None."""
    m = re.search(r"(?<!\d)(1[89]\d\d|20\d\d)(?!\d)", str(v)) if v not in (None, "") else None
    return int(m.group(1)) if m and 1800 <= int(m.group(1)) <= time.gmtime().tm_year + 1 else None

def _short_side_ok(w: Any, h: Any, min_side: int) -> Optional[bool]:
    """None = أبعاد غير معروفة (يُحسم بعد التنزيل بـ ffprobe)، وإلا هل min(w,h) ≥ min_side."""
    try: w, h = float(w), float(h)
    except (TypeError, ValueError): return None
    return min(w, h) >= min_side if w > 0 and h > 0 else None

def find_open_videos(n: int, query: str = "", min_s: float = 600.0, max_s: float = 3600.0, max_seconds: float = 420.0, licenses: str = "open",
                     min_year: Optional[int] = None, min_side: Optional[int] = None) -> List[Dict[str, Any]]:
    """Internet Archive: n فيديو mp4 مدّته بين min_s وmax_s. licenses: open (ملكية عامة/CC-BY/CC-BY-SA) | cc | any.
    min_year: يضيف إلى الاستعلام date:[min_year-01-01 TO 9999-12-31] ويرفض عنصرًا تاريخه (date/year في ميتاداتا IA) أقدم؛ min_side: يرفض الملف إن عُرفت أبعاده وأقصر ضلع أقل منه.
    تاريخ IA غير موثوق (أرشيفات قديمة كثيرًا ما يكون تاريخها فارغًا أو خاطئًا) ⇒ الفحص الحاسم للدقة والسنة يُعاد بعد التنزيل (الخلية 09). روابط تنزيل علنية فقط."""
    import urllib.parse as up
    min_year = CONFIG.MIN_VIDEO_YEAR if min_year is None else int(min_year); min_side = CONFIG.MIN_VIDEO_SHORT_SIDE if min_side is None else int(min_side)
    t0, rows, seen, page = time.time(), [], set(), 1
    licenses = licenses if licenses in ("open", "cc", "any") else "open"
    q = "mediatype:movies" + {"open": f" AND {_IA_LICENSE_Q}", "cc": f" AND {_IA_CC_Q}", "any": ""}[licenses] + (f" AND ({query})" if query.strip() else "")
    if min_year: q += f" AND date:[{min_year}-01-01 TO 9999-12-31]"
    while len(rows) < n and time.time() - t0 < max_seconds and page <= 20:
        url = ("https://archive.org/advancedsearch.php?" + up.urlencode({"q": q, "fl[]": "identifier", "sort[]": "downloads desc", "rows": 50, "page": page, "output": "json"}))
        docs = (_http_json(url).get("response") or {}).get("docs") or []
        if not docs: break
        for d in docs:
            ident = d.get("identifier")
            if not ident or ident in seen: continue
            seen.add(ident)
            if len(rows) >= n or time.time() - t0 > max_seconds: break
            try: meta = _http_json("https://archive.org/metadata/" + up.quote(ident, safe=""))
            except Exception: continue
            md = meta.get("metadata") or {}
            lic = md.get("licenseurl") or ""
            if isinstance(lic, list): lic = lic[0] if lic else ""
            if str(md.get("access-restricted-item", "")).lower() == "true" or meta.get("is_dark"): continue      # قاعدة الإعارة: لا تنزيل علني
            if not _ia_license_ok(lic, licenses): continue
            if min_year:
                yr = _year_of(md.get("date")) or _year_of(md.get("year"))
                if yr is not None and yr < min_year: _rej("year", f"IA {ident}: {yr} < {min_year}"); continue
            best, why = None, "no_mp4"
            for f in meta.get("files") or []:
                name = f.get("name") or ""
                try: ln, sz = float(f.get("length")), float(f.get("size") or 0)
                except (TypeError, ValueError): continue
                if not name.lower().endswith(".mp4") or "/" in name or sz > 4e9: continue
                if not (min_s <= ln <= max_s): why = "duration" if why == "no_mp4" else why; continue
                ok_side = _short_side_ok(f.get("width"), f.get("height"), min_side) if min_side else True
                if ok_side is None and f.get("height") and not f.get("width"): ok_side = float(f["height"]) >= min_side if min_side else True     # عرض مجهول: الارتفاع بديل (أفقي غالبًا)
                if ok_side is False: why = "resolution"; continue
                if f.get("height") and float(f["height"]) < CONFIG.MIN_IMAGE_HEIGHT: continue
                if best is None or sz > best[2]: best = (name, ln, sz)
            if not best: _rej(why, f"IA {ident}"); continue
            rows.append(dict(video_id="ia_" + _slug(ident), video_url=f"https://archive.org/download/{up.quote(ident, safe='')}/{up.quote(best[0])}",
                             source_name="Internet Archive", license=lic or "unspecified (Internet Archive)", source_reference=f"https://archive.org/details/{ident}",
                             expected_duration=best[1], expected_size=best[2]))
        page += 1
    return rows

def find_pexels_videos(n: int, query: str = "", min_s: float = 10.0, max_s: float = 3600.0, min_side: Optional[int] = None, max_seconds: float = 150.0) -> List[Dict[str, Any]]:
    """Pexels API (يحتاج PEXELS_API_KEY). يختار لكل فيديو أصغر ملف mp4 يحقق min(w,h) ≥ min_side (1080p بدل 4K: نفس الجودة للنموذج وتخزين أصغر). ترخيص Pexels يسمح بالاستعمال
    بلا إسناد (ليس ملكية عامة/CC) ⇒ يُستعمل فقط حين AUTO_FIND_LICENSES='any'. الـAPI لا يعطي تاريخ التصوير ⇒ فحص السنة يتم بوسم الملف بعد التنزيل إن وُجد."""
    import urllib.parse as up
    key = get_api_key("PEXELS_API_KEY"); min_side = CONFIG.MIN_VIDEO_SHORT_SIDE if min_side is None else int(min_side)
    if not key: return []
    t0, rows, seen, page = time.time(), [], set(), 1
    while len(rows) < n and page <= 10 and time.time() - t0 < max_seconds:
        qs = {"per_page": 80, "page": page}
        if query.strip(): url = "https://api.pexels.com/videos/search?" + up.urlencode({**qs, "query": query, "size": "medium"})          # medium = Full HD فأعلى
        else: url = "https://api.pexels.com/videos/popular?" + up.urlencode({**qs, "min_duration": max(1, int(min_s)), "max_duration": int(max_s)})
        vids = (_http_json(url, headers={"Authorization": key}) or {}).get("videos") or []
        if not vids: break
        for v in vids:
            vid = v.get("id")
            if vid is None or vid in seen: continue
            seen.add(vid)
            if len(rows) >= n: break
            dur = v.get("duration")
            if isinstance(dur, (int, float)) and not (min_s <= dur <= max_s): _rej("duration", f"Pexels {vid}: {dur}s"); continue
            files = [f for f in (v.get("video_files") or []) if str(f.get("file_type", "")).lower() == "video/mp4" and f.get("link") and f.get("width") and f.get("height")]
            good = sorted((f for f in files if not min_side or min(f["width"], f["height"]) >= min_side), key=lambda f: f["width"] * f["height"])
            if not good: _rej("resolution", f"Pexels {vid}: لا ملف ≥{min_side}"); continue
            rows.append(dict(video_id=f"pexels_{vid}", video_url=good[0]["link"], source_name="Pexels", license="Pexels License (free to use, no attribution required)",
                             source_reference=v.get("url") or f"https://www.pexels.com/video/{vid}/", expected_duration=(float(dur) if dur else None), expected_size=None))
        page += 1
    return rows

def find_pixabay_videos(n: int, query: str = "", min_s: float = 10.0, max_s: float = 3600.0, min_side: Optional[int] = None, max_seconds: float = 150.0) -> List[Dict[str, Any]]:
    """Pixabay API (يحتاج PIXABAY_API_KEY). نفس منطق Pexels؛ ترخيص Pixabay Content License (ليس ملكية عامة/CC) ⇒ AUTO_FIND_LICENSES='any' فقط. بلا تاريخ تصوير في الـAPI."""
    import urllib.parse as up
    key = get_api_key("PIXABAY_API_KEY"); min_side = CONFIG.MIN_VIDEO_SHORT_SIDE if min_side is None else int(min_side)
    if not key: return []
    t0, rows, seen, page = time.time(), [], set(), 1
    while len(rows) < n and page <= 10 and time.time() - t0 < max_seconds:
        url = "https://pixabay.com/api/videos/?" + up.urlencode({"key": key, "q": query.strip(), "video_type": "film", "order": "popular", "per_page": 100, "page": page})
        hits = (_http_json(url) or {}).get("hits") or []
        if not hits: break
        for h in hits:
            vid = h.get("id")
            if vid is None or vid in seen: continue
            seen.add(vid)
            if len(rows) >= n: break
            dur = h.get("duration")
            if isinstance(dur, (int, float)) and not (min_s <= dur <= max_s): _rej("duration", f"Pixabay {vid}: {dur}s"); continue
            vs = [x for x in (h.get("videos") or {}).values() if isinstance(x, dict) and x.get("url") and x.get("width") and x.get("height")]
            good = sorted((x for x in vs if not min_side or min(x["width"], x["height"]) >= min_side), key=lambda x: x["width"] * x["height"])
            if not good: _rej("resolution", f"Pixabay {vid}: لا ملف ≥{min_side}"); continue
            rows.append(dict(video_id=f"pixabay_{vid}", video_url=good[0]["url"], source_name="Pixabay", license="Pixabay Content License",
                             source_reference=h.get("pageURL") or f"https://pixabay.com/videos/id-{vid}/", expected_duration=(float(dur) if dur else None),
                             expected_size=(float(good[0]["size"]) if good[0].get("size") else None)))
        page += 1
    return rows

def _commons_license(ext: Dict[str, Any]) -> Tuple[str, str]:
    """(نص الترخيص للتسجيل، نص للفحص بـ_ia_license_ok) من extmetadata في Commons."""
    short = str((ext.get("LicenseShortName") or {}).get("value") or "").strip(); url = str((ext.get("LicenseUrl") or {}).get("value") or "").strip()
    if not url and re.search(r"public domain|cc0|\bpd\b", short, re.I): url = "https://creativecommons.org/publicdomain/mark/1.0/"
    return (f"{short} ({url})" if short and url else short or url or "unspecified (Wikimedia Commons)"), (url or short)

def find_commons_videos(n: int, query: str = "", min_s: float = 10.0, max_s: float = 3600.0, licenses: str = "any", min_year: Optional[int] = None,
                        min_side: Optional[int] = None, max_seconds: float = 150.0) -> List[Dict[str, Any]]:
    """Wikimedia Commons (بلا مفتاح): ملفات webm/mp4 الأحدث رفعًا أولًا (create_timestamp_desc). يرفض: سنة التصوير (DateTimeOriginal) أو سنة الرفع < min_year، دقة < min_side، مدة خارج النطاق،
    وترخيصًا لا يطابق licenses (open/cc/any). الترخيص الفعلي لكل ملف يُسجَّل مع الصورة."""
    import urllib.parse as up
    min_year = CONFIG.MIN_VIDEO_YEAR if min_year is None else int(min_year); min_side = CONFIG.MIN_VIDEO_SHORT_SIDE if min_side is None else int(min_side)
    t0, rows, seen, offset = time.time(), [], set(), 0
    search = "filetype:video" + (f" {query.strip()}" if query.strip() else "")
    for _ in range(12):
        if len(rows) >= n or time.time() - t0 > max_seconds: break
        params = dict(action="query", format="json", formatversion=2, generator="search", gsrnamespace=6, gsrsearch=search, gsrlimit=50, gsroffset=offset, gsrsort="create_timestamp_desc",
                      prop="imageinfo", iiprop="url|size|mime|timestamp|extmetadata", iiextmetadatafilter="LicenseShortName|LicenseUrl|DateTimeOriginal")
        data = _http_json("https://commons.wikimedia.org/w/api.php?" + up.urlencode(params)) or {}
        pages = (data.get("query") or {}).get("pages") or []
        if isinstance(pages, dict): pages = list(pages.values())
        for pg in sorted(pages, key=lambda p: p.get("index", 0)):
            info = (pg.get("imageinfo") or [None])[0]; title = str(pg.get("title") or "")
            if not info or title in seen or len(rows) >= n: continue
            seen.add(title)
            if str(info.get("mime", "")).lower() not in ("video/webm", "video/mp4") or not info.get("url"): continue
            ext = info.get("extmetadata") or {}
            yr = _year_of((ext.get("DateTimeOriginal") or {}).get("value")); up_yr = _year_of(info.get("timestamp"))
            if min_year and ((yr is not None and yr < min_year) or (up_yr is not None and up_yr < min_year)): _rej("year", f"Commons {title}: صُوّر {yr} / رُفع {up_yr}"); continue
            dur = info.get("duration")
            if isinstance(dur, (int, float)) and not (min_s <= dur <= max_s): _rej("duration", f"Commons {title}: {dur}s"); continue
            if min_side and _short_side_ok(info.get("width"), info.get("height"), min_side) is False: _rej("resolution", f"Commons {title}: {info.get('width')}x{info.get('height')}"); continue
            lic_txt, lic_check = _commons_license(ext)
            if not _ia_license_ok(lic_check, licenses): _rej("license", f"Commons {title}: {lic_txt}"); continue
            if float(info.get("size") or 0) > 4e9: continue
            rows.append(dict(video_id="commons_" + _slug(re.sub(r"^File:", "", title))[:90], video_url=info["url"], source_name="Wikimedia Commons", license=lic_txt,
                             source_reference=info.get("descriptionurl") or "https://commons.wikimedia.org/wiki/" + up.quote(title.replace(" ", "_")),
                             expected_duration=(float(dur) if isinstance(dur, (int, float)) else None), expected_size=(float(info["size"]) if info.get("size") else None)))
        cont = (data.get("continue") or {}).get("gsroffset")
        if cont is None: break
        offset = int(cont)
    return rows

def find_videos(n: int, query: str = "", min_s: Optional[float] = None, max_s: Optional[float] = None, licenses: Optional[str] = None,
                sources: Optional[str] = None, max_seconds: float = 420.0) -> List[Dict[str, Any]]:
    """يجمع n فيديو من المصادر بالترتيب المفضّل (AUTO_FIND_SOURCES). كل مصدر يُعطى حصة متساوية أولًا (تنوّع)، ثم يُكمَّل النقص من المصادر بالترتيب. مصدر فاشل/بلا مفتاح لا يوقف الباقي.
    تُسجَّل أسباب رفض المرشّحين في SEARCH_REJECTIONS."""
    min_s = CONFIG.MIN_VIDEO_DURATION if min_s is None else min_s; max_s = CONFIG.MAX_VIDEO_DURATION if max_s is None else max_s
    licenses = licenses or CONFIG.AUTO_FIND_LICENSES; names = [x.strip().lower() for x in (sources if sources is not None else CONFIG.AUTO_FIND_SOURCES).split(",") if x.strip()]
    finders: Dict[str, Callable[[int, float], List[Dict[str, Any]]]] = {
        "pexels": lambda k, t: find_pexels_videos(k, query, min_s, max_s, max_seconds=t),
        "pixabay": lambda k, t: find_pixabay_videos(k, query, min_s, max_s, max_seconds=t),
        "commons": lambda k, t: find_commons_videos(k, query, min_s, max_s, licenses, max_seconds=t),
        "archive": lambda k, t: find_open_videos(k, query, min_s, max_s, max_seconds=t, licenses=licenses)}
    active: List[str] = []
    for nm in names:
        if nm not in finders: print(f"  ⚠ مصدر غير معروف في AUTO_FIND_SOURCES: '{nm}' (المتاح: {', '.join(finders)})"); continue
        if nm in ("pexels", "pixabay"):
            if licenses != "any": print(f"  ℹ {nm}: متخطّى لأن AUTO_FIND_LICENSES='{licenses}' (ترخيصه ليس ملكية عامة/Creative Commons)"); continue
            if not get_api_key(f"{nm.upper()}_API_KEY"): print(f"  ℹ {nm}: متخطّى — لا يوجد {nm.upper()}_API_KEY (Kaggle Secrets / molab Secrets / متغير بيئة)"); continue
        if nm not in active: active.append(nm)
    if not active: return []
    quota = -(-n // len(active)); per: Dict[str, List[Dict[str, Any]]] = {}; budget = max_seconds / (len(active) + 1)
    for nm in active:
        try: per[nm] = finders[nm](quota, budget)
        except Exception as e: per[nm] = []; print(f"  ⚠ {nm}: فشل البحث ({redact_text(str(e), 140)})")
        print(f"  • {nm}: {len(per[nm])} فيديو مقبول")
    rows = [r for nm in active for r in per[nm]]
    for nm in active:                                                       # تكميل النقص من المصدر الذي أعطى حصته كاملة (قد يملك المزيد)
        need = n - len(rows)
        if need <= 0: break
        if len(per[nm]) < quota: continue
        try: more = finders[nm](quota + need, budget)
        except Exception: continue
        have = {r["video_id"] for r in rows}; rows += [r for r in more if r["video_id"] not in have][:need]
    return rows[:n]

def auto_find_manifest() -> Tuple[Optional[str], str]:
    """AUTO_FIND_VIDEOS: يحفظ النتيجة في WORK_DIR/auto_videos.jsonl (وتُعاد استعمالها عند الاستئناف كي لا تتغير القائمة)."""
    f = Path(CONFIG.WORK_DIR) / "auto_videos.jsonl"
    if f.exists() and f.stat().st_size > 0: return str(f), f"استُعملت قائمة البحث التلقائي المحفوظة سابقًا: {f}"
    n = CONFIG.TOTAL_VIDEOS if CONFIG.TOTAL_VIDEOS > 0 else 50
    print(f"  🔎 بحث تلقائي عن {n} فيديو | المصادر: {CONFIG.AUTO_FIND_SOURCES} | تراخيص: {CONFIG.AUTO_FIND_LICENSES} | مدة {CONFIG.MIN_VIDEO_DURATION:.0f}–{CONFIG.MAX_VIDEO_DURATION:.0f}ث"
          f" | الضلع الأقصر ≥ {CONFIG.MIN_VIDEO_SHORT_SIDE} | سنة ≥ {CONFIG.MIN_VIDEO_YEAR}" + (f" | الكلمات: {CONFIG.AUTO_FIND_QUERY}" if CONFIG.AUTO_FIND_QUERY else "") + " … قد يأخذ عدة دقائق")
    SEARCH_REJECTIONS.clear()
    try: rows = find_videos(n, CONFIG.AUTO_FIND_QUERY)
    except Exception as e: return None, f"فشل البحث التلقائي: {e}"
    if SEARCH_REJECTIONS:
        STATE.progress["search_rejections"] = dict(SEARCH_REJECTIONS)
        print("  مرشّحون رُفضوا أثناء البحث: " + " | ".join(f"{k}: {v}" for k, v in SEARCH_REJECTIONS.most_common()))
    if not rows: return None, "البحث التلقائي لم يجد فيديوهات مطابقة للشروط (جرّب كلمات أخرى في AUTO_FIND_QUERY، أو أضف PEXELS_API_KEY/PIXABAY_API_KEY، أو خفّف MIN_VIDEO_SHORT_SIDE/MIN_VIDEO_YEAR)"
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in rows) + "\n", encoding="utf-8")
    print(f"  ✔ وُجد {len(rows)} فيديو (الترخيص مسجَّل لكل فيديو ويُنشر مع كل صورة). القائمة: {f}")
    return str(f), f"بحث تلقائي: {len(rows)} فيديو"

def resolve_manifest() -> Tuple[Optional[str], str]:
    """(المسار الفعلي أو None، ملاحظة للطباعة). الملاحظة غير فارغة حين لم يُستعمل VIDEO_MANIFEST كما هو."""
    p = (CONFIG.VIDEO_MANIFEST or "").strip()
    if p and Path(p).exists(): return p, ""
    cached = Path(CONFIG.WORK_DIR) / "auto_videos.jsonl"
    if CONFIG.AUTO_FIND_VIDEOS and cached.exists() and cached.stat().st_size > 0: return auto_find_manifest()      # استئناف: القائمة المحفوظة تتقدم على كل شيء
    found, how = discover_manifest(); head = f"VIDEO_MANIFEST='{p}' غير موجود" if p else "VIDEO_MANIFEST غير مضبوط"
    if found: return found, f"{head} ← اكتُشف تلقائيًا: {found} ({how})"
    if CONFIG.AUTO_FIND_VIDEOS:                                  # لا فيديوهات مرفقة ولا قائمة ← بحث في Internet Archive
        if p: print(f"  ⚠ {head}؛ سيُجرى البحث التلقائي بدلًا منه (AUTO_FIND_VIDEOS=True)")
        return auto_find_manifest()
    return None, f"{head}، ولم يُعثر على قائمة تلقائيًا ({how})"

def print_manifest_help(reason: str = "") -> None:
    on_kaggle = KAGGLE_INPUT.is_dir(); attached: List[str] = []
    try: attached = sorted(q.name for q in KAGGLE_INPUT.iterdir())[:12] if on_kaggle else []
    except OSError: pass
    print("=" * 78); print(" ✖ لا توجد قائمة فيديوهات — لا يمكن بدء المعالجة بدونها (ولا يبدأ Benchmark ولا Smoke Test)"); print("=" * 78)
    if reason: print(f"  السبب: {reason}")
    print("  الحل: اختر طريقة واحدة في الخلية 00 (قاموس USER_SETTINGS) ثم شغّل الخلية 00 ثم 03 ثم 07:")
    print('   1) مجلد فيه الفيديوهات (الأسهل):  "VIDEO_MANIFEST": "/kaggle/input/اسم-الداتاست"   — يُقرأ كل mp4/mkv/webm/mov/avi داخله')
    print('   2) ملف قائمة CSV/JSON/JSONL/TXT:  عمود video_url (رابط مباشر أو مسار)؛ في TXT: رابط أو مسار في كل سطر')
    print('   3) روابط مباشرة لملفات فيديو:     "VIDEO_URLS": ["https://.../a.mp4", "https://.../b.mp4"]')
    print('   4) بحث وتنزيل تلقائي (Pexels/Pixabay/Wikimedia Commons/Internet Archive حسب AUTO_FIND_SOURCES): "AUTO_FIND_VIDEOS": True  (يلزم Internet = On؛ pexels/pixabay تحتاجان مفتاحًا في Secrets)')
    if on_kaggle:
        print("  مرفقات Kaggle الحالية (/kaggle/input): " + (", ".join(attached) if attached else "لا شيء — اضغط «Add Input» وأضف Dataset فيه فيديوهاتك أو ملف القائمة"))
    else: print("  ضع فيديوهاتك في مجلد videos بجانب الـ Notebook، أو اضبط VIDEO_MANIFEST على مسار مجلد/ملف.")
    print("  تذكير: استعمل فيديوهات تملكها أو مسموحًا لك بمعالجتها فقط.")
    print("=" * 78)

def _to_float(x: Any) -> Optional[float]:
    try:
        v = float(x); return v if math.isfinite(v) else None
    except (TypeError, ValueError):
        return None

def load_manifest(path: str, max_videos: int = 0) -> Tuple[List[Dict[str, Any]], List[str]]:
    """يعيد (الصفوف الصالحة، قائمة المشاكل). لا يتجاوز DRM ولا يسجّل دخولًا: الروابط يقدّمها المستخدم ومسموحة له."""
    problems: List[str] = []
    p = Path(path)
    if p.is_dir():
        raw = _scan_kaggle_input(p)
    elif p.exists():
        raw = _read_rows(p)
    else:
        raise FileNotFoundError(f"المانيفست غير موجود: {path}")
    rows, seen = [], set()
    for i, r in enumerate(raw):
        r = {k.strip(): (v.strip() if isinstance(v, str) else v) for k, v in r.items() if k}
        url = r.get("video_url") or r.get("url") or r.get("path")
        if not url:
            problems.append(f"سطر {i + 1}: لا يوجد video_url"); continue
        vid = r.get("video_id") or _slug(Path(str(url).split("?")[0]).stem or url)
        if not _ID_RE.match(vid):
            vid2 = _slug(vid); problems.append(f"سطر {i + 1}: video_id '{vid}' غير صالح → '{vid2}'"); vid = vid2
        if vid in seen:
            problems.append(f"سطر {i + 1}: video_id مكرر '{vid}' (تم تجاهله)"); continue
        is_http = str(url).startswith(("http://", "https://"))
        if not is_http and not Path(str(url)).exists():
            problems.append(f"سطر {i + 1} [{vid}]: ملف محلي غير موجود: {url}"); continue
        if not is_http and Path(str(url)).suffix.lower() not in VIDEO_EXTS:
            problems.append(f"سطر {i + 1} [{vid}]: امتداد غير مدعوم")
        seen.add(vid)
        rows.append(dict(video_id=vid, video_url=str(url), source_name=r.get("source_name", ""), license=r.get("license", ""),
                         source_reference=r.get("source_reference", ""), expected_duration=_to_float(r.get("expected_duration")),
                         sha256=(r.get("sha256") or "").lower() or None, expected_size=_to_float(r.get("expected_size"))))
        if max_videos and len(rows) >= max_videos: break
    return rows, problems

def validate_manifest_report(rows: List[Dict[str, Any]], problems: List[str]) -> None:
    print("=" * 78); print(" MANIFEST REPORT"); print("=" * 78)
    http = sum(r["video_url"].startswith("http") for r in rows)
    print(f"  valid videos : {len(rows)}   (remote: {http}, local: {len(rows) - http})")
    print(f"  missing license field: {sum(not r['license'] for r in rows)}   missing source_name: {sum(not r['source_name'] for r in rows)}")
    if CONFIG.TOTAL_VIDEOS and len(rows) != CONFIG.TOTAL_VIDEOS:
        print(f"  ⚠ المتوقع TOTAL_VIDEOS={CONFIG.TOTAL_VIDEOS} لكن الصالح {len(rows)}")
    for pr in problems[:25]: print("  -", pr)
    if len(problems) > 25: print(f"  ... و{len(problems) - 25} مشكلة أخرى")
    print("  تذكير: المصادر يجب أن تكون مقدّمة منك ومسموحًا لك بتنزيلها ومعالجتها؛ لا يتجاوز النظام DRM أو تسجيل الدخول.")
    print("=" * 78)

def prepare_manifest() -> List[Dict[str, Any]]:
    """جلسة جديدة: تُقرأ القائمة وتُتحقَّق. استئناف: مجموعة الفيديوهات هي المخزّنة في control (لا تتغير بتعديل المانيفست)، لكن الروابط تُحدَّث من المانيفست
    إن وُجد — control لا يحفظ توقيعات الروابط ولا user:pass@ (انظر redact_url)، والروابط الموقّعة تنتهي صلاحيتها، فالمانيفست هو مصدر الروابط الصالحة."""
    if STATE.resumed and STATE.videos and CONFIG.AUTO_FIND_VIDEOS and not (CONFIG.VIDEO_MANIFEST or "").strip() and not (Path(CONFIG.WORK_DIR) / "auto_videos.jsonl").exists():
        path, note = None, ""          # استئناف بعد مسح القرص المحلي (molab): القائمة المخزّنة في control/ هي المرجع؛ لا حاجة لبحث جديد قد يختار فيديوهات أخرى
    else: path, note = resolve_manifest()
    if path and note:
        if note.startswith(("بحث تلقائي", "استُعملت قائمة البحث")): print(f"  ✔ {note}")
        else: print(f"  ⚠ {note}\n    اضبط VIDEO_MANIFEST في الخلية 00 لتثبيت الاختيار.")
        CONFIG.VIDEO_MANIFEST = path
    if STATE.resumed and STATE.videos:
        stored = [dict(video_id=v["video_id"], video_url=v["url"], source_name=v["source_name"], license=v["license"],
                       source_reference=v["source_reference"], expected_duration=v["expected_duration"],
                       sha256=v.get("sha256"), expected_size=v.get("expected_size")) for v in STATE.videos.values()]
        refreshed = 0
        if path:
            try:
                fresh = {r["video_id"]: r for r in load_manifest(path)[0]}
                for r in stored:
                    f = fresh.get(r["video_id"])
                    if f and f["video_url"] != r["video_url"]: r["video_url"] = f["video_url"]; refreshed += 1
            except Exception as e: log.warning("استئناف: تعذّرت قراءة المانيفست (%s) — تُستعمل الروابط المخزّنة (بلا توقيعات/معاملات استعلام)", redact_text(str(e), 160))
        else: log.info("استئناف بلا مانيفست: تُستعمل الروابط المخزّنة (روابط موقّعة تحتاج المانيفست الأصلي لتعمل)")
        log.info("استئناف: قائمة الفيديوهات المخزّنة (%d)؛ حُدّثت روابط %d منها من المانيفست.", len(stored), refreshed)
        return stored
    if not path: raise FileNotFoundError(note)
    rows, problems = load_manifest(path, CONFIG.TOTAL_VIDEOS)
    validate_manifest_report(rows, problems)
    if not rows: raise FileNotFoundError(f"القائمة '{path}' لا تحوي فيديوهات صالحة (انظر المشاكل أعلاه)")
    return rows

try:
    MANIFEST_ROWS = prepare_manifest()
    STATE.init_videos(MANIFEST_ROWS)
    STATE.save_local(force=True)
    print(f"videos in state: {len(STATE.videos)} | pending: {len(STATE.pending_videos())}")
except FileNotFoundError as _e:
    MANIFEST_ROWS = []
    print_manifest_help(str(_e))

---
## الجزء 2 — المعالجة (08–19)
تنزيل، ميتاداتا، مفكّكات الفيديو، الاستخراج، العيّنات الحتمية، الجودة، إزالة التكرار، معالجة الصور، Qwen3-VL، البرومبت، الوصف الدفعي.

### CELL 08 — Video Downloader
> تنزيل متدفق مع Range/استئناف جزئي وSHA256 وفحص الحجم وإعادة المحاولة. فيديو واحد (أو اثنان مسبقًا) في كل مرة.

In [ ]:
# CELL 08 - Video Downloader
#: تنزيل متدفق مع Range/استئناف جزئي وSHA256 وفحص الحجم وإعادة المحاولة. فيديو واحد (أو اثنان مسبقًا) في كل مرة.
import urllib.request, urllib.error, urllib.parse, ssl, http.client

class DownloadError(Exception):
    pass
class PermanentDownloadError(DownloadError):
    """أخطاء لا تُعاد (404، 403، DRM/تسجيل دخول...)."""

def free_disk_gb(path: os.PathLike) -> float:
    return shutil.disk_usage(path).free / 1e9

def ensure_disk(path: os.PathLike, need_gb: float, what: str) -> None:
    f = free_disk_gb(path)
    if f < need_gb:
        raise DownloadError(f"مساحة غير كافية لـ{what}: متاح {f:.1f}GB مطلوب {need_gb:.1f}GB")

def _local_name(video_id: str, url: str) -> str:
    ext = Path(urllib.parse.urlparse(url).path).suffix.lower()
    return video_id + (ext if ext in VIDEO_EXTS else ".mp4")

class Downloader:
    CHUNK = 1 << 20
    def __init__(self, dest: Path):
        self.dest = Path(dest); self.dest.mkdir(parents=True, exist_ok=True)
        self.stats = dict(bytes=0, files=0, retries=0, seconds=0.0)
        self._ctx = ssl.create_default_context()

    def local_path(self, rec: Dict[str, Any]) -> Path:
        return self.dest / _local_name(rec["video_id"], rec["url"])

    def _head(self, url: str) -> Dict[str, Any]:
        req = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "vid2ds/1.0"})
        try:
            with urllib.request.urlopen(req, timeout=CONFIG.DOWNLOAD_TIMEOUT, context=self._ctx) as r:
                return dict(size=int(r.headers.get("Content-Length") or 0) or None,
                            ranges=(r.headers.get("Accept-Ranges", "").lower() == "bytes"))
        except Exception:
            return dict(size=None, ranges=False)

    def _fetch(self, url: str, part: Path, total: Optional[int], ranges: bool) -> None:
        have = part.stat().st_size if part.exists() else 0
        headers = {"User-Agent": "vid2ds/1.0"}
        if have and ranges: headers["Range"] = f"bytes={have}-"
        else: have = 0
        req = urllib.request.Request(url, headers=headers)
        try:
            resp = urllib.request.urlopen(req, timeout=CONFIG.DOWNLOAD_TIMEOUT, context=self._ctx)
        except urllib.error.HTTPError as e:
            if e.code == 416 and total and have == total: return          # مكتمل أصلًا
            if e.code in (401, 403, 404, 410, 451):
                raise PermanentDownloadError(f"HTTP {e.code} (يتطلب تصريحًا/غير موجود): لا يحاول النظام تجاوز القيود")
            raise DownloadError(f"HTTP {e.code}")
        with resp:
            if have and resp.status != 206:           # الخادم تجاهل Range → نبدأ من الصفر
                have = 0
            mode = "ab" if have else "wb"
            with open(part, mode) as f:
                while True:
                    buf = resp.read(self.CHUNK)
                    if not buf: break
                    f.write(buf); self.stats["bytes"] += len(buf)
                f.flush(); os.fsync(f.fileno())

    def ensure(self, rec: Dict[str, Any], expected_sha256: Optional[str] = None,
               expected_size: Optional[float] = None) -> Path:
        url, final = rec["url"], self.local_path(rec)
        t0 = time.time()
        # ملف محلي من Kaggle Input (أو مسار مباشر): لا نسخ ولا حذف لاحقًا
        if not str(url).startswith(("http://", "https://")):
            if not Path(url).is_file(): raise PermanentDownloadError(f"الملف غير موجود: {url}")
            return Path(url)
        if final.exists() and final.stat().st_size > 0:
            if self._valid(final, expected_sha256, expected_size, None):
                return final                            # مكتمل مسبقًا
            final.unlink(missing_ok=True)
        ensure_disk(self.dest, CONFIG.MIN_FREE_DISK_GB, "التنزيل")
        head = self._head(url); total = head["size"] or (int(expected_size) if expected_size else None)
        if total: ensure_disk(self.dest, total / 1e9 + CONFIG.MIN_FREE_DISK_GB / 2, "حجم الفيديو")
        part = final.with_name(final.name + ".part")
        last: Optional[BaseException] = None
        for attempt in range(1, CONFIG.DOWNLOAD_RETRIES + 1):
            try:
                self._fetch(url, part, total, head["ranges"])
                if total and part.stat().st_size != total:
                    raise DownloadError(f"حجم غير مطابق: {part.stat().st_size} من {total}")
                if not self._valid(part, expected_sha256, expected_size, total):
                    part.unlink(missing_ok=True)
                    raise DownloadError("فشل التحقق من SHA256/الحجم")
                os.replace(part, final)
                self.stats["files"] += 1; self.stats["seconds"] += time.time() - t0
                return final
            except PermanentDownloadError:
                part.unlink(missing_ok=True); raise
            except (DownloadError, urllib.error.URLError, socket.timeout, ConnectionError, OSError, ssl.SSLError, http.client.HTTPException) as e:
                last = e; self.stats["retries"] += 1
                wait = min(90.0, CONFIG.DOWNLOAD_BACKOFF ** attempt) + random.random()
                log.warning("تنزيل %s محاولة %d/%d فشلت: %s — إعادة بعد %.0fs", rec["video_id"], attempt,
                            CONFIG.DOWNLOAD_RETRIES, str(e)[:120], wait)
                time.sleep(wait)
        raise DownloadError(f"فشل التنزيل بعد {CONFIG.DOWNLOAD_RETRIES} محاولات: {last!r}")

    @staticmethod
    def _valid(path: Path, sha: Optional[str], size: Optional[float], total: Optional[int]) -> bool:
        s = path.stat().st_size
        if s <= 0: return False
        if size and int(size) != s: return False
        if total and total != s: return False
        if sha and sha256_file(path) != sha.lower(): return False
        return True

    def release(self, rec: Dict[str, Any]) -> None:
        """يحذف الفيديو المؤقت (فقط إن كان منزَّلًا بواسطتنا، وبعد التأكد من التزام نتائجه — يُستدعى من المنسّق)."""
        if CONFIG.KEEP_VIDEOS: return
        p = self.local_path(rec)
        if str(rec["url"]).startswith(("http://", "https://")):
            p.unlink(missing_ok=True); p.with_name(p.name + ".part").unlink(missing_ok=True)

    def sweep_orphans(self, keep: Iterable[str] = ()) -> int:
        """Garbage collector: يحذف .part القديمة والفيديوهات اليتيمة (لا نتائج غير مرفوعة تعتمد عليها)."""
        n = 0; keep = set(keep)
        for p in self.dest.glob("*"):
            if p.name in keep: continue
            try: p.unlink(); n += 1
            except OSError: pass
        return n

DOWNLOADER = Downloader(VIDEO_DIR)
print("downloader ready →", VIDEO_DIR, "| free disk:", f"{free_disk_gb(VIDEO_DIR):.1f} GB")

### CELL 09 — Video Metadata Analyzer
> ffprobe: خصائص الفيديو + قائمة الـ PTS لكل إطار (ترتيب العرض). هذه القائمة هي مرجع "frame_index" والتوقيت في كل المشروع.

In [ ]:
# CELL 09 - Video Metadata Analyzer
#: ffprobe: خصائص الفيديو + قائمة الـ PTS لكل إطار (ترتيب العرض). هذه القائمة هي مرجع "frame_index" والتوقيت في كل المشروع.
import numpy as np

class VideoAnalysisError(Exception):
    pass

@dataclasses.dataclass
class VideoMeta:
    video_id: str
    path: str
    file_size: int
    duration: float
    width: int                  # أبعاد العرض بعد الدوران التلقائي
    height: int
    rotation: int
    codec: str
    pix_fmt: str
    avg_fps: float
    r_fps: float
    effective_fps: float
    is_vfr: bool
    n_frames: int
    pts_sec: np.ndarray         # ثوانٍ نسبةً لأول إطار، بترتيب العرض (index = frame_index)
    pts_reliable: bool
    t0: float                   # PTS المطلق لأول إطار (ثوانٍ) — لمطابقة showinfo مع -copyts
    color_space: Optional[str]
    color_range: Optional[str]
    color_transfer: Optional[str]
    hdr: bool
    sar: str
    bit_depth: Optional[int]
    warnings: List[str] = dataclasses.field(default_factory=list)
    creation_year: Optional[int] = None   # سنة وسم creation_time في الحاوية (إن وُجد وكان معقولًا): حارس سنة إضافي بعد التنزيل

    def summary(self) -> Dict[str, Any]:
        d = {k: getattr(self, k) for k in ("video_id", "duration", "width", "height", "rotation", "codec", "pix_fmt", "avg_fps",
                                           "r_fps", "effective_fps", "is_vfr", "n_frames", "pts_reliable", "color_space",
                                           "color_range", "hdr", "sar", "bit_depth", "warnings")}
        return d


def _frac(s: Optional[str]) -> float:
    try:
        if not s or s in ("0/0", "N/A"): return 0.0
        if "/" in s:
            a, b = s.split("/"); return float(a) / float(b) if float(b) else 0.0
        return float(s)
    except (ValueError, ZeroDivisionError):
        return 0.0

def _ffprobe_json(path: str, extra: Sequence[str], timeout: float = 120) -> Dict[str, Any]:
    rc, out, err = run_cmd(["ffprobe", "-v", "error", "-hide_banner", "-select_streams", "v:0", *extra, "-of", "json", path],
                           timeout=timeout)
    if rc != 0: raise VideoAnalysisError(f"ffprobe فشل ({rc}): {err.strip()[-200:]}")
    try: return json.loads(out)
    except json.JSONDecodeError as e: raise VideoAnalysisError(f"ffprobe إخراج غير صالح: {e}")

def read_packet_timestamps(path: str, time_base: float, timeout: float = 900) -> Tuple[np.ndarray, bool]:
    """PTS لكل حزمة فيديو (بوحدات time_base كأعداد صحيحة) مرتبةً بترتيب العرض. يتجاهل الحزم المعلّمة بـ D (discard)."""
    rc, out, err = run_cmd(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "packet=pts,dts,flags",
                            "-of", "csv=p=0", path], timeout=timeout)
    if rc != 0: raise VideoAnalysisError(f"ffprobe packets فشل: {err.strip()[-200:]}")
    vals: List[int] = []; missing = 0
    for line in out.splitlines():
        p = line.strip().split(",")
        if len(p) < 3 or "D" in p[2]: continue
        v = p[0] if p[0] not in ("", "N/A") else p[1]                 # pts أو dts عند غيابه
        if v in ("", "N/A"): missing += 1; continue
        try: vals.append(int(v))
        except ValueError: missing += 1
    reliable = missing == 0 and len(vals) > 0
    return np.sort(np.asarray(vals, dtype=np.int64)), reliable

def _creation_year(s: Dict[str, Any], fmt: Dict[str, Any]) -> Optional[int]:
    """سنة التسجيل من وسوم ffprobe (creation_time/date). قيم غير معقولة (أقل من 1990 أو في المستقبل البعيد، مثل 1904/1970 من أجهزة بلا ساعة) تُعدّ مجهولة."""
    for tags in ((fmt.get("tags") or {}), (s.get("tags") or {})):
        for k in ("creation_time", "date", "com.apple.quicktime.creationdate", "DATE_RECORDED"):
            v = next((tags[t] for t in tags if t.lower() == k.lower()), None)
            m = re.match(r"\s*(\d{4})", str(v)) if v else None
            if m and 1990 <= int(m.group(1)) <= time.gmtime().tm_year + 1: return int(m.group(1))
    return None

def analyze_video(path: str, video_id: str) -> VideoMeta:
    p = Path(path)
    if not p.is_file() or p.stat().st_size == 0: raise VideoAnalysisError("ملف الفيديو غير موجود أو فارغ")
    info = _ffprobe_json(str(p), ["-show_streams", "-show_format"])
    streams = info.get("streams") or []
    if not streams: raise VideoAnalysisError("لا يوجد مسار فيديو")
    s, fmt = streams[0], info.get("format", {})
    warnings: List[str] = []
    tb = _frac(s.get("time_base")) or 1e-6
    avg, rfr = _frac(s.get("avg_frame_rate")), _frac(s.get("r_frame_rate"))
    rot = 0
    try:
        cw_deg = None
        for sd in s.get("side_data_list", []) or []:                  # display matrix: ffprobe يعطي rotation عكس عقارب الساعة
            if "rotation" in sd: cw_deg = (-int(round(float(sd["rotation"])))) % 360
        if cw_deg is None and (s.get("tags") or {}).get("rotate"):   # وسم rotate القديم: مع عقارب الساعة
            cw_deg = int(float(s["tags"]["rotate"])) % 360
        rot = cw_deg or 0
    except (ValueError, TypeError): pass
    cw, ch = int(s.get("width") or 0), int(s.get("height") or 0)
    if cw <= 0 or ch <= 0: raise VideoAnalysisError("أبعاد غير صالحة")
    w, h = (ch, cw) if rot in (90, 270) else (cw, ch)
    ticks, reliable = read_packet_timestamps(str(p), tb)
    n = int(len(ticks))
    if n == 0: raise VideoAnalysisError("لا توجد حزم فيديو قابلة للقراءة")
    if len(np.unique(ticks)) != n:
        warnings.append("PTS مكررة: سيُستعمل العدّ التسلسلي"); reliable = False
        ticks = np.unique(ticks); n = int(len(ticks))
    if not reliable and avg > 0:                                      # لا PTS موثوقة → توليد زمن ثابت
        ticks = np.round(np.arange(n) / avg / tb).astype(np.int64)
        warnings.append("PTS غير متوفرة؛ زمن اصطناعي من avg_fps (لا يُستخدم لرفع FPS، فقط للترقيم)")
    sec = (ticks - ticks[0]).astype(np.float64) * tb
    span = float(sec[-1]) if n > 1 else 0.0
    gaps = np.diff(sec) if n > 1 else np.array([1.0 / max(avg, 25.0)])
    eff = (n - 1) / span if span > 0 else (avg or 0.0)
    med = float(np.median(gaps)) if len(gaps) else 0.0
    vfr = bool(med > 0 and (float(gaps.max()) > 1.6 * med or float(gaps.min()) < 0.6 * med)) or \
        (avg > 0 and rfr > 0 and abs(avg - rfr) / avg > 0.05)
    dur = float(fmt.get("duration") or s.get("duration") or 0) or (span + (med or 0.0))
    ctr = s.get("color_transfer")
    hdr = ctr in ("smpte2084", "arib-std-b67")
    bd = s.get("bits_per_raw_sample")
    nbf = s.get("nb_frames")
    if nbf and str(nbf).isdigit() and abs(int(nbf) - n) > max(2, 0.01 * n):
        warnings.append(f"nb_frames في الحاوية={nbf} ≠ عدد الحزم={n}")
    return VideoMeta(video_id=video_id, path=str(p), file_size=p.stat().st_size, duration=dur, width=w, height=h, rotation=rot,
                     codec=s.get("codec_name", "?"), pix_fmt=s.get("pix_fmt", "?"), avg_fps=avg, r_fps=rfr, effective_fps=eff,
                     is_vfr=vfr, n_frames=n, pts_sec=sec, pts_reliable=reliable, t0=float(ticks[0]) * tb, color_space=s.get("color_space"),
                     color_range=s.get("color_range"), color_transfer=ctr, hdr=hdr, sar=s.get("sample_aspect_ratio") or "1:1",
                     bit_depth=int(bd) if bd and str(bd).isdigit() else None, warnings=warnings, creation_year=_creation_year(s, fmt))


def check_video_limits(meta: VideoMeta) -> Optional[str]:
    """يعيد سبب الرفض (أو None) بالشكل "<code>: <تفصيل>" حيث code ∈ duration | resolution | year | hdr | low_resolution. يُسجَّل الفيديو كـ skipped (لا يُعدّ فشلًا) ويُحسب في التقرير النهائي.
    الشروط (من الخلية 00): المدة ضمن [MIN_VIDEO_DURATION, MAX_VIDEO_DURATION]، min(w,h) ≥ MIN_VIDEO_SHORT_SIDE (العمودي مقبول)، سنة الحاوية ≥ MIN_VIDEO_YEAR إن وُجدت."""
    if CONFIG.ENFORCE_DURATION_LIMITS and not (CONFIG.MIN_VIDEO_DURATION <= meta.duration <= CONFIG.MAX_VIDEO_DURATION):
        return f"duration: {meta.duration:.1f}s خارج [{CONFIG.MIN_VIDEO_DURATION:.0f},{CONFIG.MAX_VIDEO_DURATION:.0f}]"
    if CONFIG.MIN_VIDEO_SHORT_SIDE and min(meta.width, meta.height) < CONFIG.MIN_VIDEO_SHORT_SIDE:
        return f"resolution: {meta.width}x{meta.height} (الضلع الأقصر {min(meta.width, meta.height)} < {CONFIG.MIN_VIDEO_SHORT_SIDE})"
    if CONFIG.MIN_VIDEO_YEAR and meta.creation_year is not None and meta.creation_year < CONFIG.MIN_VIDEO_YEAR:
        return f"year: سنة التسجيل {meta.creation_year} < {CONFIG.MIN_VIDEO_YEAR}"
    if meta.hdr and getattr(CONFIG, "SKIP_HDR_VIDEOS", True):
        return "hdr: HDR (PQ/HLG) بلا tone-mapping: تخطٍّ لحماية جودة الألوان (SKIP_HDR_VIDEOS=False لتعطيله)"
    if min(meta.width, meta.height) < CONFIG.MIN_IMAGE_WIDTH // 2:
        return f"low_resolution: دقة الفيديو {meta.width}x{meta.height} أقل من الحد الأدنى"
    return None

def skip_reason_code(reason: Optional[str]) -> str:
    """رمز سبب الرفض من نص check_video_limits/_predict_skip ("duration: ..." → "duration"). نص قديم بلا رمز يُصنَّف بالكلمة الأولى."""
    t = (reason or "").strip()
    m = re.match(r"^([a-z_]+)\s*:", t) or re.match(r"^([A-Za-z_]+)\b", t)
    return m.group(1).lower() if m else "unknown"

print("metadata analyzer ready (ffprobe:", ENV.get("ffprobe"), ")")

### CELL 10 — Hardware Decoder Initialization
> اختبار فعلي (لا افتراض) لكل مفكّك: PyNvVideoCodec → FFmpeg NVDEC → FFmpeg CPU → (طوارئ) OpenCV.

In [ ]:
# CELL 10 - Hardware Decoder Initialization
#: اختبار فعلي (لا افتراض) لكل مفكّك: PyNvVideoCodec → FFmpeg NVDEC → FFmpeg CPU → (طوارئ) OpenCV.
import numpy as np

DECODER_ORDER_DEFAULT = ["pynvvideocodec", "ffmpeg_nvdec", "ffmpeg_cpu", "opencv"]
NVDEC_CODECS = {"h264", "hevc", "vp9", "av1", "mpeg2video", "mpeg4", "vc1", "vp8"}   # NVDEC قد لا يدعم كلها على كل GPU: نجرّب ونرجع
PYNV_MAX_COLOR_MAD = 4.0          # أقصى فرق لوني متوسط مقبول بين PyNvVideoCodec وFFmpeg (مقياس 0-255)

def make_probe_clip(path: Path, seconds: int = 2, size: str = "320x240", colorspace: str = "bt709") -> bool:
    """مقطع H.264 ملوّن بوسوم BT.709 يُستعمل لاختبار المفكّكات ومطابقة الألوان."""
    vf = "format=yuv420p"
    rc, _, e = run_cmd(["ffmpeg", "-y", "-v", "error", "-f", "lavfi", "-i", f"testsrc2=size={size}:rate=30:duration={seconds}",
                        "-vf", vf, "-c:v", "libx264", "-pix_fmt", "yuv420p", "-colorspace", colorspace, "-color_primaries", colorspace,
                        "-color_trc", colorspace, "-color_range", "tv", "-g", "15", str(path)], timeout=60)
    return rc == 0 and path.exists()

def ffmpeg_reference_rgb(path: Path, n_frames: int = 4, w: int = 320, h: int = 240) -> Optional[np.ndarray]:
    p = subprocess.run(["ffmpeg", "-v", "error", "-i", str(path), "-frames:v", str(n_frames), "-vf", "scale=flags=accurate_rnd+bicubic",
                        "-pix_fmt", "rgb24", "-f", "rawvideo", "pipe:1"], capture_output=True, timeout=60)
    if p.returncode != 0 or len(p.stdout) < n_frames * w * h * 3: return None
    return np.frombuffer(p.stdout[: n_frames * w * h * 3], np.uint8).reshape(n_frames, h, w, 3)

def probe_pynvvideocodec(clip: Path) -> Dict[str, Any]:
    r: Dict[str, Any] = dict(available=False, threaded_ok=False, color_mad=None, reason="", version=ENV.get("pynvvideocodec"))
    if not RESOURCE_PLAN["has_gpu"]: r["reason"] = "لا يوجد GPU"; return r
    if not has_module("PyNvVideoCodec"): r["reason"] = "PyNvVideoCodec غير مثبت"; return r
    if not has_module("torch"): r["reason"] = "torch مطلوب لتحويل DLPack"; return r
    try:
        import PyNvVideoCodec as nvc, torch
        def grab(dec, n):
            fr = dec.get_batch_frames(n)
            return np.stack([torch.from_dlpack(f).cpu().numpy() for f in fr]) if fr else None
        dec = nvc.SimpleDecoder(str(clip), gpu_id=0, use_device_memory=True, output_color_type=nvc.OutputColorType.RGB)
        arr = grab(dec, 4)
        if arr is None or arr.ndim != 4 or arr.shape[-1] != 3 or arr.dtype != np.uint8:
            r["reason"] = f"شكل غير متوقع: {None if arr is None else (arr.shape, arr.dtype)}"; return r
        r["available"] = True
        ref = ffmpeg_reference_rgb(clip, 4, arr.shape[2], arr.shape[1])
        if ref is not None and ref.shape == arr.shape:
            r["color_mad"] = float(np.abs(ref.astype(np.int16) - arr.astype(np.int16)).mean())
            if r["color_mad"] > PYNV_MAX_COLOR_MAD and not getattr(CONFIG, "PYNV_ALLOW_COLOR_MISMATCH", False):
                r["available"] = False
                r["reason"] = f"اختلاف لون عن FFmpeg (MAD={r['color_mad']:.1f}>{PYNV_MAX_COLOR_MAD}): يُعطَّل حفاظًا على الألوان"
        try:
            td = nvc.ThreadedDecoder(str(clip), buffer_size=8, gpu_id=0, use_device_memory=True, output_color_type=nvc.OutputColorType.RGB)
            r["threaded_ok"] = grab(td, 4) is not None
        except Exception as e:
            r["threaded_reason"] = f"{type(e).__name__}: {str(e)[:100]}"
    except Exception as e:
        r["available"] = False; r["reason"] = f"{type(e).__name__}: {str(e)[:160]}"
    return r

def probe_decoders() -> Dict[str, Dict[str, Any]]:
    res: Dict[str, Dict[str, Any]] = {}
    clip = Path(CONFIG.TEMP_DIR) / "decoder_probe.mp4"
    have_clip = bool(ENV.get("ffmpeg")) and make_probe_clip(clip)
    res["pynvvideocodec"] = probe_pynvvideocodec(clip) if (have_clip and CONFIG.ENABLE_GPU_DECODING) else \
        dict(available=False, reason="GPU decoding معطّل أو لا مقطع اختبار")
    nv = ENV.get("nvdec_ffmpeg", {})
    res["ffmpeg_nvdec"] = dict(available=bool(nv.get("decode_test")) and CONFIG.ENABLE_GPU_DECODING, reason=nv.get("note", ""))
    res["ffmpeg_cpu"] = dict(available=bool(ENV.get("ffmpeg")), reason="" if ENV.get("ffmpeg") else "ffmpeg غير موجود")
    res["opencv"] = dict(available=has_module("cv2"), reason="")
    if have_clip: clip.unlink(missing_ok=True)
    return res

def probe_ffmpeg_features() -> Dict[str, Any]:
    """خيارات تختلف بين إصدارات ffmpeg: -fps_mode مقابل -vsync، وطريقة تمرير فلتر كبير (-filter_script أو -/filter)."""
    f: Dict[str, Any] = {"vsync_args": ["-vsync", "0"], "filter_file_opt": None}
    base = ["ffmpeg", "-v", "error", "-f", "lavfi", "-i", "color=c=black:s=16x16:d=0.2:r=10"]
    if run_cmd(base + ["-fps_mode", "passthrough", "-f", "null", "-"], timeout=30)[0] == 0: f["vsync_args"] = ["-fps_mode", "passthrough"]
    tmp = Path(CONFIG.TEMP_DIR) / "vf_probe.txt"; tmp.write_text("select='1',format=rgb24")
    for opt in (["-filter_script:v", str(tmp)], ["-/filter:v", str(tmp)]):
        if run_cmd(base + opt + ["-f", "null", "-"], timeout=30)[0] == 0: f["filter_file_opt"] = opt[0]; break
    tmp.unlink(missing_ok=True)
    return f

DECODER_STATUS = probe_decoders()
FFMPEG_FEATURES = probe_ffmpeg_features() if ENV.get("ffmpeg") else {"vsync_args": [], "filter_file_opt": None}
DECODER_HEALTH: Dict[Tuple[str, str], int] = {}          # (backend, codec) → عدد الإخفاقات؛ ≥2 ⇒ نتجاوز المفكّك لهذا الترميز

def decoder_chain(meta: Optional["VideoMeta"] = None) -> List[str]:
    pref = CONFIG.DECODER_PREFERENCE
    order = DECODER_ORDER_DEFAULT if pref == "auto" else [pref] + [d for d in DECODER_ORDER_DEFAULT if d != pref]
    if not CONFIG.ENABLE_GPU_DECODING: order = [d for d in order if d in ("ffmpeg_cpu", "opencv")]
    chain = []
    for d in order:
        if not DECODER_STATUS.get(d, {}).get("available"): continue
        if meta is not None:
            if d in ("pynvvideocodec", "ffmpeg_nvdec") and meta.codec not in NVDEC_CODECS: continue
            if DECODER_HEALTH.get((d, meta.codec), 0) >= 2: continue
        chain.append(d)
    return chain or ["ffmpeg_cpu"]

print("=" * 78); print(" DECODER STATUS"); print("=" * 78)
for _k, _v in DECODER_STATUS.items():
    print(f"  {_k:<16}{'OK ' if _v.get('available') else '-- '} {({kk: vv for kk, vv in _v.items() if kk not in ('available',) and vv not in ('', None)})}")
print("  ffmpeg features:", FFMPEG_FEATURES); print("  decoder chain (generic):", decoder_chain()); print("=" * 78)

### CELL 11 — Frame Extraction Engine
> بثّ الإطارات المختارة فقط إلى الذاكرة (لا كتابة على القرص). كل مفكّك يعيد (frame_index, timestamp, RGB).
> FFmpeg: الاختيار داخل الفلتر (select بقناع بتّات على رقم الإطار) فلا يُحوَّل لـRGB إلا المختار؛ التحقق بالـ PTS من showinfo.

In [ ]:
# CELL 11 - Frame Extraction Engine
#: بثّ الإطارات المختارة فقط إلى الذاكرة (لا كتابة على القرص). كل مفكّك يعيد (frame_index, timestamp, RGB).
#: FFmpeg: الاختيار داخل الفلتر (select بقناع بتّات على رقم الإطار) فلا يُحوَّل لـRGB إلا المختار؛ التحقق بالـ PTS من showinfo.
import numpy as np
try:
    import fcntl
except ImportError:                                   # pragma: no cover
    fcntl = None

class FrameExtractionError(Exception):
    pass
class PlanMismatch(FrameExtractionError):
    """الإطار المُخرَج لا يطابق الخطة (تحوّل المفكّك أو فُقد إطار): نعيد المحاولة بوضع المطابقة بالـ PTS."""
class DecoderFailure(FrameExtractionError):
    pass

@dataclasses.dataclass
class RawFrame:
    video_id: str
    frame_index: int
    ts: float                       # ثوانٍ نسبةً لأول إطار
    rgb: np.ndarray                 # HxWx3 uint8
    seed_only: bool = False         # يُستعمل لتهيئة نافذة إزالة التكرار بعد الاستئناف فقط
    backend: str = ""

EXTRACT_STATS: Dict[str, Any] = dict(frames=0, bytes=0, seconds=0.0, last_index=0, backend="", fallbacks=0, plan_mismatches=0)

# ---------------------------------------------------------------- قناع البتّات لاختيار الإطارات داخل ffmpeg
def build_select_expr(wanted: np.ndarray, n_frames: int, bits: int = 48) -> str:
    """تعبير select يعيد 1 إذا كان رقم الإطار n ضمن wanted. كل كلمة تحمل `bits` بتًّا (≤53 لتبقى دقيقة في double)،
    والبحث عن الكلمة شجرة if() ثنائية (عمق ~log2). تكلفة التقييم لكل إطار صغيرة جدًا حتى لو كان العدد عشرات الآلاف."""
    nw = max(1, (n_frames + bits - 1) // bits)
    words = [0] * nw
    for i in wanted.tolist():
        if 0 <= i < n_frames: words[i // bits] |= 1 << (i % bits)
    nz = np.cumsum([0] + [1 if w else 0 for w in words])               # عدد الكلمات غير الصفرية (لتقليم الأفرع الفارغة)
    def tree(lo: int, hi: int) -> str:
        if nz[hi] - nz[lo] == 0: return "0"
        if hi - lo == 1: return str(words[lo])
        mid = (lo + hi) // 2
        return f"if(lt(ld(0),{mid}),{tree(lo, mid)},{tree(mid, hi)})"
    return f"st(0,floor(n/{bits}));lt(n,{n_frames})*mod(floor({tree(0, nw)}/pow(2,n-ld(0)*{bits})),2)"

_SHOWINFO_RE = re.compile(r"n:\s*(\d+)\s+pts:\s*(-?\d+|NOPTS)\s+pts_time:\s*(-?[\d.]+(?:[eE][-+]?\d+)?|NOPTS)")

def _scale_filter(meta: "VideoMeta") -> str:
    cm = "auto"
    if (meta.color_space in (None, "", "unknown")) and meta.height >= 720: cm = "bt709"      # سلوك المشغّلات للـ HD غير الموسوم
    return f"scale=in_color_matrix={cm}:in_range=auto:out_range=full:flags=bicubic+accurate_rnd+full_chroma_int"

def _tolerance(pts_abs: np.ndarray) -> float:
    g = np.diff(pts_abs)
    mg = float(g[g > 0].min()) if len(g) and (g > 0).any() else 0.01
    return max(5e-6, min(2e-3, 0.45 * mg))

def iter_ffmpeg(meta: "VideoMeta", wanted: np.ndarray, seed_set: set, hw: bool, use_select: bool = True) -> Iterator[RawFrame]:
    W, H = meta.width, meta.height; fb = W * H * 3
    if len(wanted) == 0: return
    pts_abs = meta.pts_sec + meta.t0; tol = _tolerance(pts_abs)
    wanted_set = set(int(x) for x in wanted.tolist()); last_needed = int(wanted[-1])
    tmp_script: Optional[Path] = None
    cmd = ["ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "info", "-nostats", "-copyts"]
    thr = CONFIG.FFMPEG_THREADS or max(1, (RESOURCE_PLAN.get("qc_threads", 2) + 1) // 2 + 1)
    cmd += ["-threads", str(thr)]
    if hw: cmd += ["-hwaccel", "cuda", "-hwaccel_device", "0"]
    cmd += ["-i", meta.path, "-map", "0:v:0", "-an", "-sn", "-dn"] + FFMPEG_FEATURES["vsync_args"]
    chain = f"showinfo,{_scale_filter(meta)},format=rgb24"
    if use_select:
        expr = build_select_expr(wanted, meta.n_frames)
        vf = f"select='{expr}',{chain}"
        if len(vf) > 90_000:                                           # حد وسيط الأمر (128KB) ⇒ ملف فلتر
            opt = FFMPEG_FEATURES.get("filter_file_opt")
            if not opt: raise DecoderFailure("تعبير select أكبر من حد سطر الأوامر ولا يوجد دعم لملف فلتر")
            tmp_script = Path(CONFIG.TEMP_DIR) / f"vf_{meta.video_id}_{os.getpid()}.txt"; tmp_script.write_text(vf, encoding="utf-8")
            cmd += [opt, str(tmp_script)]
        else:
            cmd += ["-vf", vf]
    else:
        cmd += ["-vf", chain]
    cmd += ["-f", "rawvideo", "-pix_fmt", "rgb24", "pipe:1"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, bufsize=0)
    if fcntl is not None:
        try: fcntl.fcntl(proc.stdout.fileno(), 1031, 4 << 20)           # F_SETPIPE_SZ
        except Exception: pass
    info_q: "queue.Queue" = queue.Queue(); tail: "collections.deque" = __import__("collections").deque(maxlen=40)
    def _err_reader() -> None:
        try:
            for raw in iter(proc.stderr.readline, b""):
                line = raw.decode("utf-8", "replace")
                m = _SHOWINFO_RE.search(line) if "showinfo" in line else None
                if m: info_q.put((int(m.group(1)), None if m.group(3) == "NOPTS" else float(m.group(3))))
                else: tail.append(line.rstrip())
        finally: info_q.put(None)
    th = threading.Thread(target=_err_reader, daemon=True); th.start()
    next_pos = 0                                                       # مؤشر داخل wanted (وضع select)
    try:
        while True:
            buf = np.empty((H, W, 3), np.uint8); mv = memoryview(buf).cast("B"); got = 0
            while got < fb:
                r = proc.stdout.readinto(mv[got:])
                if not r: break
                got += r
            if got < fb:
                if got == 0 and next_pos >= len(wanted) if use_select else got == 0:
                    break                                                # نهاية طبيعية
                th.join(timeout=2)
                if use_select and next_pos < len(wanted):
                    raise DecoderFailure(f"ffmpeg انتهى مبكرًا بعد {next_pos}/{len(wanted)} إطارًا: {' | '.join(list(tail)[-3:])[:300]}")
                break
            try: rec = info_q.get(timeout=30)
            except queue.Empty: raise DecoderFailure("لا معلومات showinfo للإطار (مهلة)")
            if rec is None: raise DecoderFailure("انتهى stderr قبل المتوقع")
            _, pt = rec
            if pt is None:                                               # بلا PTS: نعتمد الترتيب فقط
                idx = int(wanted[next_pos]) if use_select and next_pos < len(wanted) else -1
            else:
                j = int(np.searchsorted(pts_abs, pt - tol))
                idx = j if (j < len(pts_abs) and abs(pts_abs[j] - pt) <= tol) else -1
            if use_select:
                if next_pos >= len(wanted): break
                if idx != int(wanted[next_pos]):
                    raise PlanMismatch(f"متوقع الإطار {int(wanted[next_pos])} لكن المفكّك أعطى {idx} (pts={pt})")
                next_pos += 1; emit = True
            else:
                emit = idx in wanted_set
            if emit:
                EXTRACT_STATS["frames"] += 1; EXTRACT_STATS["bytes"] += fb; EXTRACT_STATS["last_index"] = idx
                yield RawFrame(meta.video_id, idx, float(meta.pts_sec[idx]), buf, idx in seed_set, "ffmpeg_nvdec" if hw else "ffmpeg_cpu")
            if idx >= last_needed and (not use_select or next_pos >= len(wanted)): break    # لا حاجة لفك الباقي
    finally:
        try: proc.kill()
        except Exception: pass
        try: proc.wait(timeout=10)
        except Exception: pass
        th.join(timeout=2)
        if tmp_script is not None: tmp_script.unlink(missing_ok=True)

def _rotate_to_display(rgb: np.ndarray, rotation_cw: int) -> np.ndarray:
    return rgb if rotation_cw % 360 == 0 else np.ascontiguousarray(np.rot90(rgb, k=-(rotation_cw // 90)))

def iter_pynv(meta: "VideoMeta", wanted: np.ndarray, seed_set: set) -> Iterator[RawFrame]:
    """GPU: PyNvVideoCodec (SimpleDecoder أو ThreadedDecoder). الترقيم بالعدّ التسلسلي لإطارات العرض + تحقق مبدئي من التوقيت."""
    import PyNvVideoCodec as nvc, torch
    if len(wanted) == 0: return
    out_color = nvc.OutputColorType.RGB
    st = DECODER_STATUS["pynvvideocodec"]
    if st.get("threaded_ok"):
        dec = nvc.ThreadedDecoder(meta.path, buffer_size=12, gpu_id=0, use_device_memory=True, output_color_type=out_color)
        batch = 12
    else:
        dec = nvc.SimpleDecoder(meta.path, gpu_id=0, use_device_memory=True, output_color_type=out_color); batch = 16
    wi, idx = 0, 0; first_ts: List[float] = []
    while wi < len(wanted):
        frames = dec.get_batch_frames(batch)
        if not frames: break
        for f in frames:
            if len(first_ts) < 48 and hasattr(f, "timestamp"):          # فحص التناسب بين وحدات الـ timestamp والـ PTS المتوقعة
                first_ts.append(float(f.timestamp))
                if len(first_ts) == 48 and meta.pts_reliable and meta.n_frames >= 48:
                    d_dec = np.diff(first_ts); d_plan = np.diff(meta.pts_sec[:48])
                    ok = (d_dec > 0).all() and d_plan.min() > 0 and (np.std(d_dec / d_plan) / max(1e-12, np.mean(d_dec / d_plan)) < 0.02)
                    if not ok: raise PlanMismatch("توقيت إطارات PyNvVideoCodec لا يتناسب مع قائمة PTS (فقدان/إعادة ترتيب)")
            if idx == int(wanted[wi]):
                rgb = torch.from_dlpack(f).cpu().numpy()                 # HWC uint8 (نسخة؛ الإطار صالح حتى الاستدعاء التالي)
                rgb = _rotate_to_display(rgb, meta.rotation)             # NVDEC لا يطبّق display-matrix
                EXTRACT_STATS["last_index"] = idx; EXTRACT_STATS["frames"] += 1; EXTRACT_STATS["bytes"] += rgb.nbytes
                yield RawFrame(meta.video_id, idx, float(meta.pts_sec[idx]), rgb, idx in seed_set, "pynvvideocodec")
                wi += 1
                if wi >= len(wanted): break
            idx += 1
    if wi < len(wanted): raise DecoderFailure(f"PyNvVideoCodec أنتج {idx} إطارًا فقط (المخطط {meta.n_frames})")

def iter_opencv(meta: "VideoMeta", wanted: np.ndarray, seed_set: set) -> Iterator[RawFrame]:
    """مسار طوارئ أخير (ألوان BT.601 افتراضيًا): grab() لكل إطار و retrieve() للمختار فقط."""
    import cv2
    cap = cv2.VideoCapture(meta.path, cv2.CAP_FFMPEG)
    if not cap.isOpened(): raise DecoderFailure("OpenCV لم يفتح الفيديو")
    try: cap.set(cv2.CAP_PROP_ORIENTATION_AUTO, 1)
    except Exception: pass
    idx, wi = 0, 0
    try:
        while wi < len(wanted):
            if not cap.grab(): break
            if idx == int(wanted[wi]):
                ok, bgr = cap.retrieve()
                if not ok: raise DecoderFailure(f"OpenCV فشل retrieve للإطار {idx}")
                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                if rgb.shape[:2] == (meta.width, meta.height) and (meta.width, meta.height) != (meta.height, meta.width):
                    rgb = _rotate_to_display(rgb, 90)
                EXTRACT_STATS["last_index"] = idx; EXTRACT_STATS["frames"] += 1
                yield RawFrame(meta.video_id, idx, float(meta.pts_sec[idx]), rgb, idx in seed_set, "opencv")
                wi += 1
            idx += 1
    finally:
        cap.release()
    if wi < len(wanted): raise DecoderFailure(f"OpenCV أنتج {idx} إطارًا فقط")

def extract_frames(meta: "VideoMeta", plan: "FramePlan", cursor: int = -1, chain: Optional[List[str]] = None) -> Iterator[RawFrame]:
    """المولّد الرئيسي. cursor: آخر frame_index حُسم والتُزم (استئناف). إطارات نافذة الـ dedup قبله تُبثّ كـ seed_only."""
    sel = plan.sel
    new = sel[sel > cursor]
    if cursor >= 0 and len(sel):
        if CONFIG.DEDUP_LAST_N:                                          # نافذة «آخر N مقبول»: نُهيّئها من آخر 3N إطارًا مختارًا قبل المؤشّر (بعضها كان مرفوضًا أصلًا فيُعاد حسمه بنفس القواعد)
            seeds = sel[sel <= cursor][-3 * CONFIG.DEDUP_LAST_N:]
        else:
            t_cut = float(meta.pts_sec[min(cursor, meta.n_frames - 1)]) - CONFIG.DEDUP_WINDOW_SECONDS
            seeds = sel[(sel <= cursor) & (meta.pts_sec[sel] >= t_cut)]
    else:
        seeds = sel[:0]
    wanted = np.concatenate([seeds, new]).astype(np.int64); seed_set = set(int(x) for x in seeds.tolist())
    last_idx = -1; chain = list(chain or decoder_chain(meta)); errors: List[str] = []
    for bi, backend in enumerate(chain):
        remaining = wanted[wanted > last_idx]
        if len(remaining) == 0: return
        EXTRACT_STATS["backend"] = backend
        try:
            if backend == "pynvvideocodec": it = iter_pynv(meta, remaining, seed_set)
            elif backend == "ffmpeg_nvdec": it = iter_ffmpeg(meta, remaining, seed_set, hw=True)
            elif backend == "ffmpeg_cpu": it = iter_ffmpeg(meta, remaining, seed_set, hw=False)
            else: it = iter_opencv(meta, remaining, seed_set)
            for fr in it:
                last_idx = fr.frame_index
                yield fr
            return
        except PlanMismatch as e:                                        # نعيد بالمطابقة بالـ PTS (أبطأ لكنه صحيح) قبل تبديل المفكّك
            EXTRACT_STATS["plan_mismatches"] += 1; errors.append(f"{backend}: {e}")
            log.warning("%s: عدم تطابق مع الخطة (%s) → وضع المطابقة بالـ PTS", meta.video_id, str(e)[:160])
            if backend.startswith("ffmpeg"):
                try:
                    for fr in iter_ffmpeg(meta, wanted[wanted > last_idx], seed_set, hw=backend == "ffmpeg_nvdec", use_select=False):
                        last_idx = fr.frame_index
                        yield fr
                    return
                except FrameExtractionError as e2: errors.append(f"{backend}/pts-match: {e2}")
        except (FrameExtractionError, ImportError, RuntimeError, OSError, ValueError) as e:
            errors.append(f"{backend}: {type(e).__name__}: {str(e)[:200]}")
        DECODER_HEALTH[(backend, meta.codec)] = DECODER_HEALTH.get((backend, meta.codec), 0) + 1
        EXTRACT_STATS["fallbacks"] += 1
        log.warning("%s: المفكّك %s فشل → %s", meta.video_id, backend, "المفكّك التالي" if bi + 1 < len(chain) else "لا بدائل")
    raise FrameExtractionError(f"فشلت كل المفكّكات ({'; '.join(errors)[:600]})")

print("frame extraction engine ready | chain for h264:", [d for d in decoder_chain()])

### CELL 12 — Sampling (stratified random + trimming; deterministic per video)
> خطة الإطارات: (1) المرشّحة بمعدل ≤ TARGET_FPS اعتمادًا على PTS، (2) قص TRIM_START_RATIO من بداية الفيديو وTRIM_END_RATIO من نهايته (بنسبة من المدة)،
> (3) SAMPLING_METHOD="stratified_random" (الافتراضي): الجزء الأوسط يُقسَّم إلى k قطعة متساوية (k = SAMPLING_RATIO × عدد المرشّحة فيه) ويُؤخذ إطار عشوائي واحد من كل قطعة،
> والبذرة ثابتة لكل فيديو = BLAKE2b(SAMPLING_SEED|اسم الفيديو) ⇒ الاستئناف/إعادة التشغيل يعطيان نفس الإطارات بالضبط. (الطرق القديمة deterministic_hash / exact_per_video متاحة.)
> frame_index = ترتيب الإطار في العرض داخل الفيديو كله (قبل أي تخفيض FPS) ⇒ ثابت بين الجلسات والمفكّكات.

In [ ]:
# CELL 12 - Sampling (stratified random + trimming; deterministic per video)
#: خطة الإطارات: (1) المرشّحة بمعدل ≤ TARGET_FPS اعتمادًا على PTS، (2) قص TRIM_START_RATIO من بداية الفيديو وTRIM_END_RATIO من نهايته (بنسبة من المدة)،
#: (3) SAMPLING_METHOD="stratified_random" (الافتراضي): الجزء الأوسط يُقسَّم إلى k قطعة متساوية (k = SAMPLING_RATIO × عدد المرشّحة فيه) ويُؤخذ إطار عشوائي واحد من كل قطعة،
#:     والبذرة ثابتة لكل فيديو = BLAKE2b(SAMPLING_SEED|اسم الفيديو) ⇒ الاستئناف/إعادة التشغيل يعطيان نفس الإطارات بالضبط. (الطرق القديمة deterministic_hash / exact_per_video متاحة.)
#: frame_index = ترتيب الإطار في العرض داخل الفيديو كله (قبل أي تخفيض FPS) ⇒ ثابت بين الجلسات والمفكّكات.
import numpy as np

def hash_unit(seed: int, video_id: str, frame_index: int) -> float:
    """قيمة في [0,1) حتمية: BLAKE2b(seed|video_id|frame_index). لا نستعمل hash() المدمجة لأنها تتغير بين العمليات."""
    d = hashlib.blake2b(f"{seed}|{video_id}|{frame_index}".encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(d, "big") / 18446744073709551616.0          # 2**64

def candidate_indices(pts_sec: np.ndarray, target_fps: float) -> np.ndarray:
    """إطارات مرشحة بمعدل لا يتجاوز target_fps، مختارة بالتوقيت (أقرب إطار لكل نبضة 1/target_fps).
    - FPS أعلى من الهدف: خفض بالتوقيت. - FPS أقل أو يساويه: كل الإطارات الأصلية (لا تكرار ولا إطارات اصطناعية).
    - VFR: يعمل محليًا (المقاطع البطيئة تُبقي كل إطاراتها، السريعة تُخفَّض)."""
    n = len(pts_sec)
    if n <= 1: return np.arange(n, dtype=np.int64)
    t = np.asarray(pts_sec, dtype=np.float64)
    ticks = t[0] + np.arange(0, int(np.floor((t[-1] - t[0]) * target_fps + 1e-9)) + 1, dtype=np.float64) / target_fps
    j = np.searchsorted(t, ticks)                                       # أول إطار ≥ النبضة
    j0, j1 = np.clip(j - 1, 0, n - 1), np.clip(j, 0, n - 1)
    pick = np.unique(np.where(np.abs(t[j0] - ticks) <= np.abs(t[j1] - ticks), j0, j1)).astype(np.int64)
    # الإطار الأخير يملك نصف فترة فقط فقد لا تلتقطه أي نبضة؛ نُلحقه إذا لم يخلق معدلًا يتجاوز الهدف (فجوة ≥ 0.9/target)
    if pick[-1] != n - 1 and (t[-1] - t[pick[-1]]) >= 0.9 / target_fps:
        pick = np.append(pick, n - 1)
    return pick

def trim_candidates(cand: np.ndarray, pts_sec: np.ndarray, start_ratio: float, end_ratio: float) -> np.ndarray:
    """يُبقي المرشّحة التي توقيتها داخل [t0 + start·span ، t1 − end·span] حيث span = مدة الفيديو (بالتوقيت لا بالعدد ⇒ صحيح مع VFR). بلا قص إن كانت النسبتان 0."""
    if len(cand) == 0 or (start_ratio <= 0 and end_ratio <= 0): return cand
    t = np.asarray(pts_sec, dtype=np.float64)[cand]; t0, t1 = float(np.asarray(pts_sec)[0]), float(np.asarray(pts_sec)[-1]); span = t1 - t0
    keep = (t >= t0 + start_ratio * span - 1e-9) & (t <= t1 - end_ratio * span + 1e-9)
    return cand[keep]

def video_rng(video_id: str, seed: int) -> "np.random.Generator":
    """مولّد عشوائي ثابت لكل فيديو (BLAKE2b لا hash() المدمجة): نفس الفيديو + نفس SAMPLING_SEED ⇒ نفس الإطارات في أي جلسة/جهاز."""
    d = hashlib.blake2b(f"{seed}|{video_id}|stratified".encode("utf-8"), digest_size=8).digest()
    return np.random.default_rng(int.from_bytes(d, "big"))

def select_stratified(video_id: str, cand: np.ndarray, ratio: float, seed: int, max_k: int = 0) -> np.ndarray:
    """k = max(1, round(m·ratio)) (وبحد أقصى max_k إن وُجد) قطعة متساوية من m مرشّحًا مرتّبًا زمنيًا؛ إطار عشوائي واحد من كل قطعة. النتيجة مرتّبة تصاعديًا وبلا تكرار."""
    m = len(cand)
    if m == 0 or ratio <= 0: return cand[:0]
    k = min(m, max(1, int(round(m * ratio))))
    if max_k and max_k > 0: k = min(k, int(max_k))
    edges = np.linspace(0, m, k + 1).astype(np.int64); rng = video_rng(video_id, seed)
    picks = np.array([edges[i] + int(rng.integers(0, max(1, edges[i + 1] - edges[i]))) for i in range(k)], dtype=np.int64)
    return cand[picks]

def select_hash(video_id: str, cand: np.ndarray, ratio: float, seed: int) -> np.ndarray:
    u = np.fromiter((hash_unit(seed, video_id, int(i)) for i in cand), dtype=np.float64, count=len(cand))
    return cand[u < ratio]

def select_exact(video_id: str, cand: np.ndarray, ratio: float, seed: int) -> np.ndarray:
    """EXACT_PER_VIDEO_SAMPLING: بالضبط round(N*ratio) إطارًا دون تكرار (أقل قيم hash)، حتمي أيضًا."""
    k = int(round(len(cand) * ratio))
    if k <= 0 or len(cand) == 0: return cand[:0]
    u = np.fromiter((hash_unit(seed, video_id, int(i)) for i in cand), dtype=np.float64, count=len(cand))
    keep = np.argsort(u, kind="stable")[:k]
    return np.sort(cand[keep])

@dataclasses.dataclass
class FramePlan:
    video_id: str
    n_frames: int
    cand: np.ndarray            # frame_index للمرشحة
    sel: np.ndarray             # frame_index المختارة (مرتبة تصاعديًا)
    pts_sec: np.ndarray         # لكل الإطارات
    method: str
    ratio_actual: float
    capped: int = 0             # كم إطارًا أُسقط بسبب MAX_IMAGES_PER_VIDEO
    trimmed: int = 0            # كم مرشّحًا أُسقط بقص البداية/النهاية (TRIM_*)

    def stats(self) -> Dict[str, Any]:
        return dict(total_candidate_frames=int(len(self.cand)), selected_frames=int(len(self.sel)),
                    sampling_ratio_actual=self.ratio_actual, frames_rejected_before_captioning=int(len(self.cand) - len(self.sel)),
                    source_frames=self.n_frames, method=self.method, capped=self.capped, trimmed=self.trimmed)

def build_plan(meta: "VideoMeta") -> FramePlan:
    cand = candidate_indices(meta.pts_sec, CONFIG.TARGET_FPS)
    mid = trim_candidates(cand, meta.pts_sec, CONFIG.TRIM_START_RATIO, CONFIG.TRIM_END_RATIO)          # قص البداية والنهاية قبل أي اختيار (كل الطرق)
    method = CONFIG.sampling_method(); capped = 0
    if method == "stratified_random":
        sel = select_stratified(meta.video_id, mid, CONFIG.SAMPLING_RATIO, CONFIG.SAMPLING_SEED, CONFIG.MAX_IMAGES_PER_VIDEO)       # السقف يحدّد k نفسه فتبقى القطع متساوية
        if CONFIG.MAX_IMAGES_PER_VIDEO:
            capped = max(0, min(len(mid), max(1, int(round(len(mid) * CONFIG.SAMPLING_RATIO)))) - len(sel))
    else:
        sel = (select_exact if method == "exact_per_video" else select_hash)(meta.video_id, mid, CONFIG.SAMPLING_RATIO, CONFIG.SAMPLING_SEED)
        if CONFIG.MAX_IMAGES_PER_VIDEO and len(sel) > CONFIG.MAX_IMAGES_PER_VIDEO:
            # سقف اختياري (معطّل افتراضيًا): نُبقي الأقل قيمة hash حتى يبقى الاختيار عشوائيًا حتميًا
            u = np.fromiter((hash_unit(CONFIG.SAMPLING_SEED, meta.video_id, int(i)) for i in sel), dtype=np.float64, count=len(sel))
            keep = np.sort(np.argsort(u, kind="stable")[:CONFIG.MAX_IMAGES_PER_VIDEO]); capped = len(sel) - len(keep); sel = sel[keep]
    ratio = (len(sel) / len(cand)) if len(cand) else 0.0
    return FramePlan(meta.video_id, meta.n_frames, cand, sel, meta.pts_sec, method, ratio, capped, int(len(cand) - len(mid)))

if __name__ == "__main__" or True:
    _t = np.arange(0, 10, 1 / 60.0)                                        # اختبار ذاتي سريع عند تحميل الخلية
    assert abs(len(candidate_indices(_t, 30.0)) - 300) <= 1, "60fps → 30fps"
    assert len(candidate_indices(np.arange(0, 10, 1 / 24.0), 30.0)) == 240, "24fps تبقى كما هي"
    assert hash_unit(1, "a", 5) == hash_unit(1, "a", 5) and 0 <= hash_unit(1, "a", 5) < 1
    _c = candidate_indices(_t, 30.0); _m = trim_candidates(_c, _t, 0.1, 0.1)
    assert abs(len(_m) - 240) <= 2 and _t[_m[0]] >= 1.0 - 1e-6 and _t[_m[-1]] <= 9.0 + 0.05, "قص 10%/10%"
    _s1 = select_stratified("demo", _m, 0.05, 1); assert (_s1 == select_stratified("demo", _m, 0.05, 1)).all() and len(_s1) == round(len(_m) * 0.05), "stratified حتمي"
    print("sampling ready | hash sample:", [round(hash_unit(CONFIG.SAMPLING_SEED, "demo", i), 4) for i in range(3)])

### CELL 13 — Image Quality Control
> تحليل لكل إطار (يعمل في خيوط متوازية، دوال نقية): فحوص أساسية + مقاييس الجودة + هاشات التكرار.
> القرارات المعتمدة على السياق (ضبابية نسبية، تكرار) تُتّخذ لاحقًا بالترتيب في VideoDecider (الخلية 15).

In [ ]:
# CELL 13 - Image Quality Control
#: تحليل لكل إطار (يعمل في خيوط متوازية، دوال نقية): فحوص أساسية + مقاييس الجودة + هاشات التكرار.
#: القرارات المعتمدة على السياق (ضبابية نسبية، تكرار) تُتّخذ لاحقًا بالترتيب في VideoDecider (الخلية 15).
import numpy as np
import cv2
cv2.setNumThreads(1)                      # التوازي عبر خيوط Python؛ لا نريد oversubscription داخل OpenCV

REJECT_BLUR, REJECT_BLACK, REJECT_BRIGHT = "blur", "black_frame", "overexposed"
REJECT_UNIFORM, REJECT_CORRUPT, REJECT_INVALID, REJECT_LOWRES = "uniform_frame", "corrupt_frame", "invalid_image", "low_resolution"
REJECT_EXACT, REJECT_NEAR, REJECT_ENCODE = "exact_duplicate", "near_duplicate", "encode_failed"
REJECT_LOWSTD, REJECT_TEXT, REJECT_CAPTION_BANNED = "low_luma_std", "text_screen", "caption_banned_phrase"
QC_WORK_SIDE = 512                        # الضلع الأطول لصورة العمل (يجعل العتبات مستقلة عن الدقة)

@dataclasses.dataclass
class QCThresholds:
    lap_floor: float = 3.0                # سقف مطلق للضبابية النسبية: لا يُرفض إطار ضبابيّ نسبيًا إن كان lap > 3×هذا (أي حاد فعليًا)
    ten_floor: float = 40.0               # ونفس الفكرة لـ Tenengrad
    rel_ratio: float = 0.12               # الإطار أقل من 12% من وسيط الفيديو نفسه (حماية الأنمي/الكرتون/الخلفيات البسيطة)
    black_mean: float = 12.0; black_p99: float = 40.0
    bright_mean: float = 243.0; bright_p1: float = 215.0
    uniform_std: float = 1.2
    min_luma_std: float = 20.0            # std الإضاءة (0-255) أقل من هذا ⇒ low_luma_std (0 = معطّل) — CONFIG.QC_MIN_LUMA_STD
    text_screen: bool = True              # CONFIG.QC_TEXT_SCREEN_ENABLE
    text_dark_mean: float = 60.0          # شاشة النص: متوسط الإضاءة < هذا
    text_dark_level: int = 50             # بكسل «داكن» < هذا
    text_dark_fraction: float = 0.80      # حصة الداكن ≥ هذا
    text_bright_level: int = 160          # بكسل «ساطع» ≥ هذا
    text_bright_min: float = 0.0005       # حصة الساطع بين الحدّين
    text_bright_max: float = 0.12
    text_bbox_max: float = 0.50           # مستطيل الساطع (بعد قصّ الأطراف 0.5%) ≤ هذه الحصة من الإطار ⇒ «متجمّع»
    text_min_components: int = 3          # وعدد البقع الساطعة المنفصلة ≥ هذا (حروف/كلمات لا مصباح واحد)

def load_qc_thresholds() -> QCThresholds:
    t = QCThresholds()
    cal = (STATE.benchmark.get("qc_calibration") if "STATE" in globals() else None) or {}
    for k in ("lap_floor", "ten_floor"):
        if k in cal: setattr(t, k, float(cal[k]))
    if CONFIG.BLUR_THRESHOLD is not None: t.lap_floor = float(CONFIG.BLUR_THRESHOLD)
    t.min_luma_std = float(CONFIG.QC_MIN_LUMA_STD); t.text_screen = bool(CONFIG.QC_TEXT_SCREEN_ENABLE)
    t.text_dark_mean, t.text_dark_level, t.text_dark_fraction = float(CONFIG.QC_TEXT_DARK_MEAN), int(CONFIG.QC_TEXT_DARK_LEVEL), float(CONFIG.QC_TEXT_DARK_FRACTION)
    t.text_bright_level, t.text_bright_min, t.text_bright_max = int(CONFIG.QC_TEXT_BRIGHT_LEVEL), float(CONFIG.QC_TEXT_BRIGHT_MIN_FRACTION), float(CONFIG.QC_TEXT_BRIGHT_MAX_FRACTION)
    t.text_bbox_max, t.text_min_components = float(CONFIG.QC_TEXT_BBOX_MAX_FRACTION), int(CONFIG.QC_TEXT_MIN_COMPONENTS)
    return t

QC = load_qc_thresholds()

@dataclasses.dataclass
class FrameAnalysis:
    frame_index: int
    ts: float
    width: int
    height: int
    reject: Optional[str]
    metrics: Dict[str, float]
    pixel_sha256: Optional[str]
    phash: int
    dhash: int
    thumb: Optional[np.ndarray]           # رمادي 128x128 للتحقق بـSSIM
    seed_only: bool = False

def _work_image(rgb: np.ndarray) -> np.ndarray:
    h, w = rgb.shape[:2]; s = QC_WORK_SIDE / max(h, w)
    return rgb if s >= 1 else cv2.resize(rgb, (max(1, int(round(w * s))), max(1, int(round(h * s)))), interpolation=cv2.INTER_AREA)

def phash64(gray: np.ndarray) -> int:
    g = cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    low = cv2.dct(g)[:8, :8].ravel(); med = float(np.median(low[1:]))        # الوسيط بدون DC
    bits = (low > med)
    return int("".join("1" if b else "0" for b in bits), 2)

def dhash64(gray: np.ndarray) -> int:
    g = cv2.resize(gray, (9, 8), interpolation=cv2.INTER_AREA)
    bits = (g[:, 1:] > g[:, :-1]).ravel()
    return int("".join("1" if b else "0" for b in bits), 2)

def pixel_sha256(rgb: np.ndarray) -> str:
    """هاش الصورة بعد فك الترميز بصيغة معيارية: الأبعاد + بايتات RGB (uint8، HWC، متجاورة)."""
    h = hashlib.sha256(); h.update(struct.pack("<II", rgb.shape[0], rgb.shape[1]))
    h.update(np.ascontiguousarray(rgb).data); return h.hexdigest()

def analyze_frame(fr: "RawFrame") -> FrameAnalysis:
    rgb = fr.rgb
    def rej(reason: str, **m) -> FrameAnalysis:
        return FrameAnalysis(fr.frame_index, fr.ts, int(rgb.shape[1]) if rgb.ndim == 3 else 0, int(rgb.shape[0]) if rgb.ndim == 3 else 0,
                             reason, m, None, 0, 0, None, fr.seed_only)
    if not isinstance(rgb, np.ndarray) or rgb.ndim != 3 or rgb.shape[2] != 3 or rgb.dtype != np.uint8 or rgb.size == 0:
        return rej(REJECT_INVALID)
    h, w = rgb.shape[:2]
    if w < CONFIG.MIN_IMAGE_WIDTH or h < CONFIG.MIN_IMAGE_HEIGHT: return rej(REJECT_LOWRES)
    small = _work_image(rgb)
    gray = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY)
    g32 = gray.astype(np.float32)
    mean, std = float(g32.mean()), float(g32.std())
    p1, p99 = (float(x) for x in np.percentile(gray, (1, 99)))
    lap = float(cv2.Laplacian(g32, cv2.CV_32F, ksize=3).var())
    gx, gy = cv2.Sobel(g32, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(g32, cv2.CV_32F, 0, 1, ksize=3)
    ten = float(np.mean(gx * gx + gy * gy)) / 16.0                              # تطبيع تقريبي لتدرّج Sobel
    rg, yb = small[..., 0].astype(np.float32) - small[..., 1], 0.5 * (small[..., 0].astype(np.float32) + small[..., 1]) - small[..., 2]
    colorful = float(np.sqrt(rg.var() + yb.var()) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2))    # Hasler-Süsstrunk
    sharp = min(1.0, np.log1p(lap) / np.log1p(300.0)); expo = 1.0 - min(1.0, abs(mean - 118.0) / 118.0); contrast = min(1.0, std / 60.0)
    metrics = dict(lap_var=lap, tenengrad=ten, mean_luma=mean, std_luma=std, p1=p1, p99=p99, colorfulness=colorful,
                   quality_score=float(round(0.5 * sharp + 0.3 * expo + 0.2 * contrast, 4)))
    reason: Optional[str] = None
    if mean < QC.black_mean and p99 < QC.black_p99: reason = REJECT_BLACK
    elif mean > QC.bright_mean and p1 > QC.bright_p1: reason = REJECT_BRIGHT
    elif std < QC.uniform_std and lap < 0.5: reason = REJECT_UNIFORM                  # لون واحد تمامًا: لا معلومات
    elif _looks_corrupt(small): reason = REJECT_CORRUPT
    elif QC.text_screen and is_text_screen(gray, mean, QC): reason = REJECT_TEXT                          # عنوان/اعتمادات: خلفية داكنة + بكسلات ساطعة قليلة متجمّعة
    elif QC.min_luma_std > 0 and std < QC.min_luma_std: reason = REJECT_LOWSTD                           # مسطّح بلا تفاصيل (انحراف إضاءة أقل من العتبة)
    # الضبابية لا تُحسم هنا: الخلفيات البسيطة/الأنمي/الإضاءة المسطحة تبدو 'ناعمة' مطلقًا. القرار نسبي لكل فيديو في VideoDecider.
    thumb = cv2.resize(gray, (128, 128), interpolation=cv2.INTER_AREA)
    sha = pixel_sha256(rgb) if CONFIG.ENABLE_EXACT_DEDUPLICATION or CONFIG.ENABLE_PERCEPTUAL_DEDUPLICATION else None
    ph = phash64(gray) if CONFIG.ENABLE_PERCEPTUAL_DEDUPLICATION else 0
    dh = dhash64(gray) if CONFIG.ENABLE_PERCEPTUAL_DEDUPLICATION else 0
    return FrameAnalysis(fr.frame_index, fr.ts, w, h, reason, metrics, sha, ph, dh, thumb, fr.seed_only)

def is_text_screen(gray: np.ndarray, mean: float, t: "QCThresholds") -> bool:
    """شاشة نص (عنوان/اعتمادات/شعار على خلفية داكنة): متوسط الإضاءة منخفض، ≥ text_dark_fraction من البكسلات داكنة، الساطع (≥ text_bright_level) قليل (بين الحدّين)
    ومتجمّع (مستطيله بعد قصّ 0.5% من الأطراف ≤ text_bbox_max من الإطار) ومكوَّن من ≥ text_min_components بقعة منفصلة (حروف). مشهد ليلي تتوزّع أضواؤه على الإطار أو مصباح واحد لا يُرفضان."""
    if mean >= t.text_dark_mean: return False
    dark = float((gray < t.text_dark_level).mean())
    if dark < t.text_dark_fraction: return False
    mask = gray >= t.text_bright_level; frac = float(mask.mean())
    if not (t.text_bright_min <= frac <= t.text_bright_max): return False
    ys, xs = np.nonzero(mask)
    if len(xs) == 0: return False
    x0, x1 = np.percentile(xs, (0.5, 99.5)); y0, y1 = np.percentile(ys, (0.5, 99.5)); H, W = gray.shape
    if ((x1 - x0 + 1) * (y1 - y0 + 1)) / float(H * W) > t.text_bbox_max: return False
    n, _, st, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    return int((st[1:, cv2.CC_STAT_AREA] >= 3).sum()) >= t.text_min_components

def _looks_corrupt(small: np.ndarray) -> bool:
    """أخضر/أرجواني مسطّح يغطي ≥45% من الإطار: توقيع شائع لأخطاء فك H.264/HEVC. محافظ: يتطلب لونًا واحدًا مهيمنًا فعلًا."""
    t = cv2.resize(small, (48, 48), interpolation=cv2.INTER_AREA).reshape(-1, 3).astype(np.int16)
    q = (t // 16); key = q[:, 0] * 256 + q[:, 1] * 16 + q[:, 2]
    vals, cnt = np.unique(key, return_counts=True); k = int(vals[cnt.argmax()]); frac = float(cnt.max()) / len(key)
    if frac < 0.45: return False
    r, g, b = (k // 256) * 16 + 8, ((k // 16) % 16) * 16 + 8, (k % 16) * 16 + 8
    return bool((g > 100 and r < 60 and b < 60) or (r > 180 and b > 180 and g < 70))

def reject_if_relative_blur(m: Dict[str, float], median_lap: Optional[float]) -> bool:
    """ضبابية نسبية: إطار أقل بكثير من وسيط الفيديو نفسه وأقل من أرضية مطلقة معقولة (3×lap_floor)."""
    if median_lap is None or median_lap <= 0: return False
    return m["lap_var"] < QC.rel_ratio * median_lap and m["lap_var"] < max(QC.lap_floor * 3.0, 12.0) and m["tenengrad"] < QC.ten_floor * 3

def calibrate_blur_thresholds(laps: Sequence[float], tens: Sequence[float]) -> Dict[str, float]:
    """معايرة من عينات حقيقية (تُستدعى في Benchmark): أرضية Laplacian = 15% من المئين 1 بحدّ أدنى 2، وبسقف 8 كي لا تكون عدوانية."""
    a, b = np.asarray(laps, dtype=np.float64), np.asarray(tens, dtype=np.float64)
    if len(a) < 30: return {}
    return dict(lap_floor=float(min(8.0, max(2.0, 0.15 * np.percentile(a, 1)))), ten_floor=float(min(120.0, max(20.0, 0.15 * np.percentile(b, 1)))),
                n=int(len(a)), lap_p1=float(np.percentile(a, 1)), lap_p50=float(np.percentile(a, 50)), lap_p99=float(np.percentile(a, 99)))

print("QC ready | thresholds:", dataclasses.asdict(QC))

### CELL 14 — Exact Deduplication
> SHA256 لبكسلات الصورة بعد فك الترميز (أبعاد + RGB uint8). نطاق الفيديو الواحد دائمًا؛ نطاق عالمي اختياري (CROSS_VIDEO_DEDUP).

In [ ]:
# CELL 14 - Exact Deduplication
#: SHA256 لبكسلات الصورة بعد فك الترميز (أبعاد + RGB uint8). نطاق الفيديو الواحد دائمًا؛ نطاق عالمي اختياري (CROSS_VIDEO_DEDUP).

class ExactDeduper:
    def __init__(self, global_scope: bool = False, max_global: int = 20_000_000):
        self.global_scope, self.max_global = global_scope, max_global
        self._video: set = set(); self._global: set = set()
        self.hits = 0

    @staticmethod
    def _key(sha_hex: str) -> int:
        return int(sha_hex[:16], 16)                  # 64 بت تكفي هنا (احتمال تصادم ضئيل مقابل ملايين الصور)

    def new_video(self) -> None:
        self._video = set()

    def contains(self, sha_hex: Optional[str]) -> bool:
        """True إذا سبق قبول صورة مطابقة تمامًا (الرفض لا يسجّل أي هاش: الصورة المرفوضة ليست في Dataset)."""
        if not sha_hex or not CONFIG.ENABLE_EXACT_DEDUPLICATION: return False
        k = self._key(sha_hex)
        hit = k in self._video or (self.global_scope and k in self._global)
        if hit: self.hits += 1
        return hit

    def add(self, sha_hex: Optional[str]) -> None:
        if not sha_hex: return
        k = self._key(sha_hex); self._video.add(k)
        if self.global_scope and len(self._global) < self.max_global: self._global.add(k)

EXACT_DEDUPER = ExactDeduper(global_scope=CONFIG.CROSS_VIDEO_DEDUP)
print("exact dedup ready | scope:", "global+video" if CONFIG.CROSS_VIDEO_DEDUP else "per-video")

### CELL 15 — Perceptual Deduplication
> نافذة متحركة داخل الفيديو (لا O(N²)): آخر DEDUP_LAST_N إطارًا مقبولًا (الافتراضي 20؛ 0 = نافذة زمنية DEDUP_WINDOW_SECONDS)، pHash+dHash (يجب أن يتحقق الاثنان) ثم SSIM للتحقق
> (DEDUP_SSIM_THRESHOLD=0.95 بديل عملي لعتبة CLIP≈0.95: CLIP يحتاج نموذجًا إضافيًا).
> اختياري: فهرس MIH (multi-index hashing) للتشابه بين الفيديوهات بزمن شبه ثابت.

In [ ]:
# CELL 15 - Perceptual Deduplication
#: نافذة متحركة داخل الفيديو (لا O(N²)): آخر DEDUP_LAST_N إطارًا مقبولًا (الافتراضي 20؛ 0 = نافذة زمنية DEDUP_WINDOW_SECONDS)، pHash+dHash (يجب أن يتحقق الاثنان) ثم SSIM للتحقق
#: (DEDUP_SSIM_THRESHOLD=0.95 بديل عملي لعتبة CLIP≈0.95: CLIP يحتاج نموذجًا إضافيًا).
#: اختياري: فهرس MIH (multi-index hashing) للتشابه بين الفيديوهات بزمن شبه ثابت.
import collections

POPCOUNT8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def hamming(a: int, b: int) -> int:
    return (a ^ b).bit_count()

def ssim_gray(a: np.ndarray, b: np.ndarray) -> float:
    """SSIM قياسي (نافذة Gaussian 11×11، σ=1.5) على صورتين رماديتين uint8 بالحجم نفسه."""
    C1, C2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    x, y = a.astype(np.float32), b.astype(np.float32)
    g = lambda z: cv2.GaussianBlur(z, (11, 11), 1.5)
    mx, my = g(x), g(y); mxx, myy, mxy = mx * mx, my * my, mx * my
    sx, sy, sxy = g(x * x) - mxx, g(y * y) - myy, g(x * y) - mxy
    return float((((2 * mxy + C1) * (2 * sxy + C2)) / ((mxx + myy + C1) * (sx + sy + C2))).mean())

class MIHIndex:
    """فهرس متعدد الجداول لمسافة هامنغ ≤ d على هاشات 64 بت (مبدأ العصفور: d+1 قطعة ⇒ قطعة واحدة على الأقل متطابقة)."""
    def __init__(self, d: int = 5, bucket_cap: int = 4000):
        self.d, self.cap = d, bucket_cap
        n = d + 1; base, extra = divmod(64, n)
        self.spans: List[Tuple[int, int]] = []; pos = 0
        for i in range(n):
            w = base + (1 if i < extra else 0); self.spans.append((pos, w)); pos += w
        self.tables: List[Dict[int, List[int]]] = [collections.defaultdict(list) for _ in self.spans]
        self.hashes: List[int] = []

    def _chunks(self, h: int) -> List[int]:
        return [(h >> s) & ((1 << w) - 1) for s, w in self.spans]

    def add(self, h: int) -> None:
        i = len(self.hashes); self.hashes.append(h)
        for t, c in zip(self.tables, self._chunks(h)):
            if len(t[c]) < self.cap: t[c].append(i)

    def query(self, h: int) -> Optional[int]:
        """يعيد أصغر مسافة وجدت (≤ d) أو None."""
        best: Optional[int] = None; seen: set = set()
        for t, c in zip(self.tables, self._chunks(h)):
            for i in t.get(c, ()):
                if i in seen: continue
                seen.add(i); dist = hamming(h, self.hashes[i])
                if dist <= self.d and (best is None or dist < best): best = dist
        return best

class VideoDecider:
    """يُنشأ لكل فيديو. يستقبل FrameAnalysis بالترتيب الزمني ويقرر: قبول/رفض + سبب. الإطارات seed_only تُسجَّل فقط."""
    def __init__(self, video_id: str, mih: Optional[MIHIndex] = None):
        self.video_id = video_id
        self.window: "collections.deque" = collections.deque(maxlen=(CONFIG.DEDUP_LAST_N or None))          # (ts, phash, dhash, thumb): آخر N مقبول (أو زمنية إن N=0)
        self.laps: "collections.deque" = collections.deque(maxlen=400)
        self.exact = EXACT_DEDUPER; self.exact.new_video()
        self.mih = mih
        self.stats = collections.Counter()

    def _median_lap(self) -> Optional[float]:
        return float(np.median(self.laps)) if len(self.laps) >= 30 else None

    def _is_near_duplicate(self, fa: FrameAnalysis) -> bool:
        T = CONFIG.DEDUP_HAMMING_THRESHOLD
        if not CONFIG.DEDUP_LAST_N:
            while self.window and fa.ts - self.window[0][0] > CONFIG.DEDUP_WINDOW_SECONDS: self.window.popleft()
        for ts, ph, dh, th in self.window:
            if hamming(fa.phash, ph) <= T and hamming(fa.dhash, dh) <= T + 2:
                if not (CONFIG.DEDUP_SSIM_VERIFY and CONFIG.QC_ENABLE_SSIM_VERIFY) or ssim_gray(fa.thumb, th) >= CONFIG.DEDUP_SSIM_THRESHOLD:
                    return True
        if self.mih is not None and self.mih.query(fa.phash) is not None: return True
        return False

    def decide(self, fa: FrameAnalysis) -> Optional[str]:
        """None = مقبول. وإلا سبب الرفض."""
        if fa.reject is None:
            self.laps.append(fa.metrics["lap_var"])                       # الوسيط يتكوّن من الإطارات الصالحة فقط
        reason = fa.reject
        if reason is None and reject_if_relative_blur(fa.metrics, self._median_lap()): reason = REJECT_BLUR
        if reason is None and CONFIG.ENABLE_EXACT_DEDUPLICATION and self.exact.contains(fa.pixel_sha256): reason = REJECT_EXACT
        if reason is None and CONFIG.ENABLE_PERCEPTUAL_DEDUPLICATION and fa.thumb is not None and self._is_near_duplicate(fa): reason = REJECT_NEAR
        if fa.seed_only:                                                  # تهيئة نافذة بعد استئناف: تدخل النافذة ولا تُبثّ
            if reason is None or reason in (REJECT_EXACT, REJECT_NEAR): self._register(fa)
            return "seed"
        if reason is None: self._register(fa)
        self.stats[reason or "accepted"] += 1
        return reason

    def _register(self, fa: FrameAnalysis) -> None:
        self.exact.add(fa.pixel_sha256)
        if fa.thumb is not None: self.window.append((fa.ts, fa.phash, fa.dhash, fa.thumb))
        if self.mih is not None and fa.phash: self.mih.add(fa.phash)

CROSS_MIH = MIHIndex(d=CONFIG.DEDUP_HAMMING_THRESHOLD) if CONFIG.CROSS_VIDEO_DEDUP else None
print("perceptual dedup ready | window:", (f"last {CONFIG.DEDUP_LAST_N} accepted" if CONFIG.DEDUP_LAST_N else f"{CONFIG.DEDUP_WINDOW_SECONDS}s"), "| hamming<=", CONFIG.DEDUP_HAMMING_THRESHOLD,
      "| ssim verify:", CONFIG.DEDUP_SSIM_VERIFY, f"(>= {CONFIG.DEDUP_SSIM_THRESHOLD})", "| cross-video:", CONFIG.CROSS_VIDEO_DEDUP)

### CELL 16 — Image Processing
> ثلاث صور منفصلة: STORED_IMAGE (تُحفظ)، MODEL_INPUT_IMAGE (للـVLM فقط، في الذاكرة)، PREVIEW_IMAGE (عند العرض فقط).
> لا قص، لا تغيير نسبة، لا تكبير. التصغير فقط إذا تجاوز الضلع الأطول MAX_STORED_IMAGE_DIMENSION.

In [ ]:
# CELL 16 - Image Processing
#: ثلاث صور منفصلة: STORED_IMAGE (تُحفظ)، MODEL_INPUT_IMAGE (للـVLM فقط، في الذاكرة)، PREVIEW_IMAGE (عند العرض فقط).
#: لا قص، لا تغيير نسبة، لا تكبير. التصغير فقط إذا تجاوز الضلع الأطول MAX_STORED_IMAGE_DIMENSION.
from PIL import Image

def _pillow_jpeg(rgb: np.ndarray, quality: int, subsampling: int) -> bytes:
    bio = io.BytesIO()
    kw = dict(format="JPEG", quality=quality, optimize=False)
    if subsampling >= 0: kw["subsampling"] = subsampling
    Image.fromarray(rgb).save(bio, **kw); return bio.getvalue()

def _pillow_webp(rgb: np.ndarray, quality: int, subsampling: int) -> bytes:
    bio = io.BytesIO(); Image.fromarray(rgb).save(bio, format="WEBP", quality=quality, method=4); return bio.getvalue()

def _cv2_jpeg(rgb: np.ndarray, quality: int, subsampling: int) -> bytes:
    ok, buf = cv2.imencode(".jpg", cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    if not ok: raise RuntimeError("cv2.imencode فشل")
    return buf.tobytes()

def _vips_jpeg(rgb: np.ndarray, quality: int, subsampling: int) -> bytes:
    import pyvips
    h, w = rgb.shape[:2]
    im = pyvips.Image.new_from_memory(np.ascontiguousarray(rgb).data, w, h, 3, "uchar")
    kw = dict(Q=quality, optimize_coding=False)
    if subsampling == 0: kw["subsample_mode"] = "off"
    return im.write_to_buffer(".jpg", **kw)

ENCODERS: Dict[str, Callable[[np.ndarray, int, int], bytes]] = {"pillow": _pillow_jpeg}
if CONFIG.IMAGE_FORMAT == "jpg":
    ENCODERS["cv2"] = _cv2_jpeg                          # نفس libjpeg-turbo تقريبًا؛ يُعتمد فقط إن ثبتت سرعته وتكافؤ جودته في Benchmark
    if has_module("pyvips"): ENCODERS["pyvips"] = _vips_jpeg
else:
    ENCODERS = {"pillow": _pillow_webp}
ENCODER_CHOICE = "pillow"                                # الافتراضي؛ يغيّره benchmark_encoders() عند الإثبات

def encode_image(rgb: np.ndarray) -> bytes:
    fn = ENCODERS.get(ENCODER_CHOICE) or ENCODERS["pillow"]
    return fn(np.ascontiguousarray(rgb), CONFIG.IMAGE_QUALITY, CONFIG.JPEG_SUBSAMPLING)

def stored_size(w: int, h: int) -> Tuple[int, int]:
    m = CONFIG.MAX_STORED_IMAGE_DIMENSION; s = m / max(w, h)
    return (w, h) if s >= 1 else (max(1, int(round(w * s))), max(1, int(round(h * s))))

def prepare_stored_image(rgb: np.ndarray) -> Tuple[bytes, int, int]:
    """يعيد (bytes, saved_w, saved_h). تصغير بـ INTER_AREA فقط عند تجاوز الحد الأقصى، وإلا الصورة كما هي."""
    h, w = rgb.shape[:2]; sw, sh = stored_size(w, h)
    img = rgb if (sw, sh) == (w, h) else cv2.resize(rgb, (sw, sh), interpolation=cv2.INTER_AREA)
    return encode_image(img), int(sw), int(sh)

def model_input_size(w: int, h: int, factor: int = 32) -> Tuple[int, int]:
    """أبعاد مضاعفات factor تقع بين MODEL_MIN_PIXELS وMODEL_MAX_PIXELS مع حفظ النسبة (بلا قص). يمنع إعادة ضبط داخل المعالج."""
    area = w * h; s = 1.0
    if area > CONFIG.MODEL_MAX_PIXELS: s = (CONFIG.MODEL_MAX_PIXELS / area) ** 0.5
    elif area < CONFIG.MODEL_MIN_PIXELS: s = (CONFIG.MODEL_MIN_PIXELS / area) ** 0.5
    nw, nh = max(factor, int(round(w * s / factor)) * factor), max(factor, int(round(h * s / factor)) * factor)
    while nw * nh > CONFIG.MODEL_MAX_PIXELS and (nw > factor or nh > factor):     # التقريب قد يرفع المساحة قليلًا
        if nw >= nh and nw > factor: nw -= factor
        elif nh > factor: nh -= factor
        else: break
    return nw, nh

def prepare_model_input(rgb: np.ndarray, factor: int = 32) -> np.ndarray:
    h, w = rgb.shape[:2]; nw, nh = model_input_size(w, h, factor)
    if (nw, nh) == (w, h): return rgb
    return cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_AREA if nw * nh < w * h else cv2.INTER_CUBIC)

def make_preview_jpeg(jpeg_bytes: bytes, max_side: int = 768) -> bytes:
    im = Image.open(io.BytesIO(jpeg_bytes)).convert("RGB"); im.thumbnail((max_side, max_side))
    bio = io.BytesIO(); im.save(bio, format="JPEG", quality=88); return bio.getvalue()

def decode_image_bytes(data: bytes) -> np.ndarray:
    return np.asarray(Image.open(io.BytesIO(data)).convert("RGB"))

def benchmark_encoders(frames: Sequence[np.ndarray], repeat: int = 2) -> Dict[str, Any]:
    """يقيس كل مشفّر متاح على إطارات حقيقية ويعتمد الأسرع فقط إن كانت جودته مكافئة (PSNR ≥ 45dB مقابل Pillow)."""
    global ENCODER_CHOICE
    res: Dict[str, Any] = {}
    if not frames: return res
    ref = [decode_image_bytes(ENCODERS["pillow"](f, CONFIG.IMAGE_QUALITY, CONFIG.JPEG_SUBSAMPLING)) for f in frames[:3]]
    for name, fn in ENCODERS.items():
        try:
            t = time.perf_counter()
            for _ in range(repeat):
                for f in frames: fn(np.ascontiguousarray(f), CONFIG.IMAGE_QUALITY, CONFIG.JPEG_SUBSAMPLING)
            dt = (time.perf_counter() - t) / (repeat * len(frames))
            out = [decode_image_bytes(fn(np.ascontiguousarray(f), CONFIG.IMAGE_QUALITY, CONFIG.JPEG_SUBSAMPLING)) for f in frames[:3]]
            mse = float(np.mean([np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2) for a, b in zip(ref, out)]))
            psnr = 99.0 if mse < 1e-9 else float(10 * np.log10(255.0 ** 2 / mse))
            size = float(np.mean([len(fn(np.ascontiguousarray(f), CONFIG.IMAGE_QUALITY, CONFIG.JPEG_SUBSAMPLING)) for f in frames[:3]]))
            res[name] = dict(ms_per_image=dt * 1000, psnr_vs_pillow=psnr, avg_bytes=size)
        except Exception as e:
            res[name] = dict(error=f"{type(e).__name__}: {str(e)[:80]}")
    ok = {k: v for k, v in res.items() if "error" not in v and v["psnr_vs_pillow"] >= 45.0 and
          v["avg_bytes"] <= 1.25 * res.get("pillow", v).get("avg_bytes", v["avg_bytes"])}
    if ok:
        best = min(ok, key=lambda k: ok[k]["ms_per_image"])
        if best != "pillow" and ok[best]["ms_per_image"] <= 0.85 * res["pillow"]["ms_per_image"]: ENCODER_CHOICE = best   # فارق ≥15% فقط
    res["chosen"] = ENCODER_CHOICE
    return res

print("image processing ready | format:", CONFIG.IMAGE_FORMAT, "q=", CONFIG.IMAGE_QUALITY, "| encoders:", list(ENCODERS), "| max stored dim:", CONFIG.MAX_STORED_IMAGE_DIMENSION)

### CELL 17 — Qwen3-VL Initialization
> فحص التوافق → اختيار الدقة/التوزيع حسب العتاد الفعلي → محرك Transformers (أساسي) أو vLLM (اختياري) → فحص سلامة بعد التحميل.
> الافتراضي الآن: Qwen3-VL-32B بدقة bf16 كاملة على GPU واحدة (نسخة لكل GPU إن وُجد أكثر من واحدة)، بلا shard وبلا int8/int4 (CAPTION_ALLOW_FALLBACK_PLANS=False)،
> وانتباه flash_attention_2 إن كانت متاحة وتعمل وإلا sdpa تلقائيًا (أيضًا عند فشل FA2 أثناء التحميل أو فحص السلامة).
> ملاحظة صدق: هذه الخلية تحتاج GPU وأوزان النموذج (~67GB لـ32B) ولا يمكن اختبارها في بيئة التطوير بلا GPU؛ منطق اختيار الخطة يُختبر محليًا.

In [ ]:
# CELL 17 - Qwen3-VL Initialization
#: فحص التوافق → اختيار الدقة/التوزيع حسب العتاد الفعلي → محرك Transformers (أساسي) أو vLLM (اختياري) → فحص سلامة بعد التحميل.
#: الافتراضي الآن: Qwen3-VL-32B بدقة bf16 كاملة على GPU واحدة (نسخة لكل GPU إن وُجد أكثر من واحدة)، بلا shard وبلا int8/int4 (CAPTION_ALLOW_FALLBACK_PLANS=False)،
#: وانتباه flash_attention_2 إن كانت متاحة وتعمل وإلا sdpa تلقائيًا (أيضًا عند فشل FA2 أثناء التحميل أو فحص السلامة).
#: ملاحظة صدق: هذه الخلية تحتاج GPU وأوزان النموذج (~67GB لـ32B) ولا يمكن اختبارها في بيئة التطوير بلا GPU؛ منطق اختيار الخطة يُختبر محليًا.
import numpy as np

QWEN_FACTOR = 32                       # patch_size(16) × merge_size(2) — من preprocessor_config الرسمي (يُتحقق منه وقت التحميل)
WEIGHT_GB = {"bf16": 17.6, "fp16": 17.6, "int8": 9.6, "int4": 6.2}          # أوزان 8B (الافتراضي القديم؛ انظر model_weight_gb لباقي الأحجام)
_WEIGHT_TABLE = {8.0: WEIGHT_GB, 32.0: {"bf16": 67.0, "fp16": 67.0, "int8": 35.0, "int4": 19.5}}
ACT_GB = 3.0                           # هامش تفعيلات/KV-cache أدنى لكل GPU عند دفعة صغيرة

def model_weight_gb(model_id: Optional[str] = None) -> Dict[str, float]:
    """حجم الأوزان التقريبي (GB) لكل دقة. 8B و32B من جدول (32B ≈ 33B معامل ⇒ ~67GB bf16)؛ غيرهما تقدير = معاملات×2.1 (bf16)."""
    m = re.search(r"(\d+(?:\.\d+)?)\s*[bB](?![a-zA-Z])", model_id or CONFIG.MODEL_ID)
    if not m: return dict(WEIGHT_GB)
    n = float(m.group(1))
    if n in _WEIGHT_TABLE: return dict(_WEIGHT_TABLE[n])
    return {"bf16": 2.1 * n, "fp16": 2.1 * n, "int8": 1.2 * n, "int4": 0.78 * n}

class CaptionOOM(RuntimeError):
    pass
class ModelSanityError(RuntimeError):
    pass

def is_oom(e: BaseException) -> bool:
    return "out of memory" in str(e).lower() or type(e).__name__ == "OutOfMemoryError"

def check_vlm_compat() -> Dict[str, Any]:
    c: Dict[str, Any] = {k: pkg_version(k) for k in ("torch", "transformers", "accelerate", "bitsandbytes", "flash-attn", "vllm", "qwen-vl-utils")}
    c["flash_attn_import_ok"] = False
    if c["flash-attn"]:
        try: import flash_attn; c["flash_attn_import_ok"] = True            # الحزمة مثبّتة لكن قد تفشل في الاستيراد (ABI/CUDA غير متوافق)
        except Exception as e: c["flash_attn_import_error"] = f"{type(e).__name__}: {str(e)[:100]}"
    c["cuda_available"] = bool(ENV.get("cuda_available"))
    c["sdpa"] = False
    try:
        import torch; c["sdpa"] = hasattr(torch.nn.functional, "scaled_dot_product_attention")
    except Exception: pass
    c["qwen3vl_class"] = False
    try:
        import transformers; c["qwen3vl_class"] = hasattr(transformers, "Qwen3VLForConditionalGeneration")
    except Exception: pass
    cc = [g.get("compute_cap") for g in ENV.get("gpus", [])]
    c["compute_caps"] = cc
    ampere = bool(cc) and all((float(x) if x else 0.0) >= 8.0 for x in cc)
    c["flash_attn_usable"] = bool(c["flash-attn"]) and bool(c["flash_attn_import_ok"]) and ampere
    c["bf16_ok"] = ampere
    c["vllm_usable"] = bool(c["vllm"]) and bool(cc)
    return c

VLM_COMPAT = check_vlm_compat()
print("VLM compat:", VLM_COMPAT)

# ---------------------------------------------------------------- خطة الدقة والتوزيع (دالة نقية، قابلة للاختبار)
def choose_precision_plan(gpus: List[Dict[str, float]], override: str = "auto", allow_fallback: bool = True, model_id: Optional[str] = None) -> List[Dict[str, Any]]:
    """gpus: [{'vram_gb':..., 'cc':...}]. يعيد قائمة خطط مرتبة بالجودة (الأعلى أولًا). المحاولة بالترتيب عند فشل التحميل.
    allow_fallback=False (الإعداد الافتراضي في الخلية 00): خطة واحدة فقط = الدقة الكاملة (bf16؛ fp16 على GPU بلا bf16) كاملةً على كل GPU (single/replicate)،
    بلا shard وبلا int8/int4؛ إن لم يتسع النموذج تُعاد قائمة فارغة (فشل واضح بدل تدهور صامت في الجودة)."""
    if not gpus: return []
    W = model_weight_gb(model_id)
    n = len(gpus); vmax = max(g["vram_gb"] for g in gpus); vsum = sum(g["vram_gb"] for g in gpus)
    full = "bf16" if all(g.get("cc", 0) >= 8.0 for g in gpus) else "fp16"          # Turing/Pascal/Volta لا تدعم bf16 أصليًا
    need_full = W[full] + ACT_GB + 1
    plans: List[Dict[str, Any]] = []
    def add(prec: str, layout: str, n_gpu: Optional[int] = None) -> None:
        plans.append(dict(precision=prec, layout=layout, n_gpu=(n_gpu if n_gpu is not None else (n if layout in ("replicate", "shard") else 1)), dtype=full))
    all_fit = all(g["vram_gb"] >= need_full for g in gpus)
    if all_fit: add(full, "single" if n == 1 else "replicate", n)                   # نسخة كاملة على كل GPU
    elif vmax >= need_full: add(full, "single", 1)
    if allow_fallback:
        if n >= 2 and vsum >= W[full] + ACT_GB * n and not all_fit: add(full, "shard")
        if vmax >= W["int8"] + ACT_GB: add("int8", "replicate" if n > 1 else "single")
        if vmax >= W["int4"] + ACT_GB * 0.8: add("int4", "replicate" if n > 1 else "single")
    if override != "auto":
        want = full if override in ("bf16", "fp16") else override
        forced = [p for p in plans if p["precision"] == want]
        if forced: return forced
        if want in ("int8", "int4"):                                                   # اختيار صريح من المستخدم لدقة مكمّمة: يُحترم دائمًا
            return [dict(precision=want, layout=("replicate" if n > 1 else "single"), n_gpu=(n if n > 1 else 1), dtype=full, forced=True)]
        return [] if not allow_fallback else [dict(precision=want, layout="single", n_gpu=1, dtype=full, forced=True)]
    return plans

def current_gpu_specs() -> List[Dict[str, float]]:
    out = []
    for g in ENV.get("gpus", []):
        try: cc = float(g.get("compute_cap") or 0)
        except ValueError: cc = 0.0
        out.append(dict(vram_gb=float(g["vram_total_mb"]) / 1024 * 0.94, cc=cc, name=g["name"], index=g["index"]))
    return out

# ---------------------------------------------------------------- المحرّكات
class CaptionEngine:
    name = "abstract"
    def __init__(self, plan: Dict[str, Any], gpu_index: int = 0):
        self.plan, self.gpu_index = plan, gpu_index
        self.model_id = CONFIG.MODEL_ID; self.precision = plan.get("precision", "?"); self.loaded = False
        self.prompt_text: Optional[str] = None
    def load(self) -> None: raise NotImplementedError
    def generate(self, images: List[np.ndarray], max_new_tokens: int, texts: Optional[List[str]] = None) -> List[str]: raise NotImplementedError
    def free(self) -> None: self.loaded = False
    def generate_retry(self, images: List[np.ndarray], max_new_tokens: int) -> List[str]:
        """إعادة المحاولة بعد فلتر العبارات المحظورة: نفس الصور مع برومبت أشد (الفك greedy ⇒ تكرار نفس البرومبت يعطي نفس النص). يعيد مخرجات تبدأ بـ PREFILL كـ generate()."""
        if getattr(self, "processor", None) is None: return self.generate(images, max_new_tokens)
        text = build_prompt_text(self.processor, full_instruction() + RETRY_SUFFIX)
        return [PREFILL + o for o in self.generate(images, max_new_tokens, texts=[text] * len(images))]
    @staticmethod
    def visual_tokens(w: int, h: int) -> int: return max(1, (w // QWEN_FACTOR) * (h // QWEN_FACTOR))

class TransformersEngine(CaptionEngine):
    name = "transformers"
    def load(self) -> None:
        import torch, transformers
        from transformers import AutoProcessor
        cls = next((getattr(transformers, n) for n in ("Qwen3VLForConditionalGeneration", "AutoModelForImageTextToText", "AutoModelForVision2Seq")
                    if hasattr(transformers, n)), None)
        if cls is None: raise ImportError("transformers لا يحتوي Qwen3VLForConditionalGeneration (يلزم ≥4.57)")
        tok = get_hf_token()
        dt = torch.bfloat16 if self.plan.get("dtype") == "bf16" else torch.float16
        kw: Dict[str, Any] = dict(dtype=dt, token=tok)
        prec = self.plan["precision"]
        if prec in ("int8", "int4"):
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True) if prec == "int8" else BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dt)
        layout = self.plan.get("layout", "single")
        if layout == "shard":
            kw["device_map"] = "auto"
            kw["max_memory"] = {int(g["index"]): f"{int(g['vram_gb'] - ACT_GB)}GiB" for g in current_gpu_specs()}
        else:
            kw["device_map"] = {"": self.gpu_index}
        attn = CONFIG.ATTN_IMPLEMENTATION
        if attn == "auto": attn = self.plan.get("attn") or ("flash_attention_2" if VLM_COMPAT.get("flash_attn_usable") else "sdpa")     # FA2 إن كانت متاحة وإلا SDPA
        if attn == "flash_attention_2" and not VLM_COMPAT.get("flash_attn_usable"): attn = "sdpa"
        kw["attn_implementation"] = attn; self.attn = attn
        log.info("تحميل %s | %s | layout=%s | attn=%s | GPU=%s", self.model_id, prec, layout, attn, self.gpu_index)
        try:
            self.model = cls.from_pretrained(self.model_id, **kw).eval()
        except Exception as e:
            if attn != "flash_attention_2" or is_oom(e) or not (isinstance(e, ImportError) or re.search(r"flash", str(e), re.I)): raise
            log.warning("flash_attention_2 فشلت عند التحميل (%s: %s) → التحويل تلقائيًا إلى sdpa", type(e).__name__, str(e)[:140])
            gc.collect()
            try:
                import torch as _t; _t.cuda.empty_cache()
            except Exception: pass
            kw["attn_implementation"] = attn = "sdpa"; self.attn = attn
            self.model = cls.from_pretrained(self.model_id, **kw).eval()
        self.processor = AutoProcessor.from_pretrained(self.model_id, token=tok)
        self.processor.tokenizer.padding_side = "left"                      # ضروري للتوليد المجمّع
        ip = getattr(self.processor, "image_processor", None)
        f = int(getattr(ip, "patch_size", 16)) * int(getattr(ip, "merge_size", 2))
        global QWEN_FACTOR
        if f != QWEN_FACTOR: log.warning("عامل الصورة الفعلي %d (المفترض %d) — يُعتمد الفعلي", f, QWEN_FACTOR); QWEN_FACTOR = f
        self.prompt_text = build_prompt_text(self.processor)
        self._gen_cfg = None
        try:
            import copy
            g = copy.deepcopy(self.model.generation_config); g.temperature = None; g.top_p = None; g.top_k = None; self._gen_cfg = g
        except Exception: pass
        self.loaded = True

    def generate(self, images: List[np.ndarray], max_new_tokens: int, texts: Optional[List[str]] = None) -> List[str]:
        import torch
        from PIL import Image as _Im
        pil = [_Im.fromarray(a) for a in images]; custom = texts is not None
        try:
            with torch.inference_mode():
                inputs = self.processor(text=(texts if custom else [self.prompt_text] * len(pil)), images=pil, padding=True, return_tensors="pt")
                inputs = inputs.to(self.model.device)
                kw = dict(max_new_tokens=max_new_tokens, do_sample=False, use_cache=True, repetition_penalty=CONFIG.CAPTION_REPETITION_PENALTY)
                if self._gen_cfg is not None: kw["generation_config"] = self._gen_cfg
                out = self.model.generate(**inputs, **kw)
                new = out[:, inputs["input_ids"].shape[1]:]
                decoded = self.processor.batch_decode(new, skip_special_tokens=True, clean_up_tokenization_spaces=False)
            del inputs, out, new
            return list(decoded) if custom else [PREFILL + t for t in decoded]
        except torch.cuda.OutOfMemoryError as e:
            torch.cuda.empty_cache(); raise CaptionOOM(str(e)[:200]) from e
        except RuntimeError as e:
            if is_oom(e):
                import torch as _t; _t.cuda.empty_cache(); raise CaptionOOM(str(e)[:200]) from e
            raise

    def free(self) -> None:
        super().free()
        try:
            import torch
            for a in ("model", "processor"):
                if hasattr(self, a): delattr(self, a)
            gc.collect(); torch.cuda.empty_cache()
        except Exception: pass

class VLLMEngine(CaptionEngine):
    """اختياري (CAPTION_ENGINE=vllm). يفشل بأمان ⇒ المنسّق يعود إلى Transformers تلقائيًا."""
    name = "vllm"
    def load(self) -> None:
        from vllm import LLM, SamplingParams
        from transformers import AutoProcessor
        n = int(self.plan.get("n_gpu", 1)) if self.plan.get("layout") == "shard" else 1
        self.processor = AutoProcessor.from_pretrained(self.model_id, token=get_hf_token())
        self.prompt_text = build_prompt_text(self.processor)
        dt = "bfloat16" if self.plan.get("dtype") == "bf16" else "float16"
        self.llm = LLM(model=self.model_id, trust_remote_code=True, dtype=dt, gpu_memory_utilization=0.88, enforce_eager=False,
                       tensor_parallel_size=n, max_model_len=4096, limit_mm_per_prompt={"image": 1}, seed=0)
        self._SP = SamplingParams
        self.loaded = True
    def generate(self, images: List[np.ndarray], max_new_tokens: int, texts: Optional[List[str]] = None) -> List[str]:
        from PIL import Image as _Im
        sp = self._SP(temperature=0.0, max_tokens=max_new_tokens, top_k=-1, repetition_penalty=CONFIG.CAPTION_REPETITION_PENALTY)
        reqs = [{"prompt": (texts[i] if texts else self.prompt_text), "multi_modal_data": {"image": _Im.fromarray(a)}} for i, a in enumerate(images)]
        try: outs = self.llm.generate(reqs, sampling_params=sp, use_tqdm=False)
        except Exception as e:
            if is_oom(e): raise CaptionOOM(str(e)[:200]) from e
            raise
        return [(o.outputs[0].text if texts else PREFILL + o.outputs[0].text) for o in outs]
    def free(self) -> None:
        super().free()
        try:
            import torch
            del self.llm; gc.collect(); torch.cuda.empty_cache()
        except Exception: pass

class MockEngine(CaptionEngine):
    """للاختبارات المحلية فقط (ALLOW_MOCK_CAPTIONER=True). نصوصه مولَّدة من إحصاءات الصورة وليست من Qwen، وتُوسَم دائمًا mock-test-only."""
    name = "mock"
    def __init__(self, plan: Optional[Dict[str, Any]] = None, gpu_index: int = 0):
        super().__init__(plan or dict(precision="mock"), gpu_index); self.model_id = "mock-test-only"; self.calls = 0
        self.fail_with_oom_above: Optional[int] = None
        self.inject_fn: Optional[Callable[[np.ndarray], Optional[str]]] = None       # اختبارات: عبارة تُحقن في وصف صورة معيّنة (لاختبار فلتر العبارات المحظورة)
        self.inject_in_retry = False; self.retry_calls = 0; self._in_retry = False
    def load(self) -> None: self.prompt_text = "mock"; self.loaded = True
    def generate_retry(self, images: List[np.ndarray], max_new_tokens: int) -> List[str]:
        self.retry_calls += 1; self._in_retry = True
        try: return self.generate(images, max_new_tokens)
        finally: self._in_retry = False
    def generate(self, images: List[np.ndarray], max_new_tokens: int, texts: Optional[List[str]] = None) -> List[str]:
        self.calls += 1
        if texts is not None: return ['{"accurate": true, "unsupported_claims": [], "missing_major_elements": []}'] * len(images)
        if self.fail_with_oom_above is not None and len(images) > self.fail_with_oom_above: raise CaptionOOM("CUDA out of memory (simulated)")
        outs = []
        for a in images:
            m = a.reshape(-1, 3).mean(0); h, w = a.shape[:2]
            tone = "dark" if m.mean() < 85 else ("bright" if m.mean() > 170 else "mid-tone")
            dom = ["red", "green", "blue"][int(np.argmax(m))]
            tp = (f"A synthetic test frame {w} by {h} pixels with a {tone} overall exposure and a dominant {dom} cast. In the foreground, a smooth "
                  f"gradient band sits left of a high-contrast barcode-like strip along the top edge, while the background holds soft sinusoidal "
                  f"texture. The composition is centered and flat, lit evenly from the front with no visible shadows, rendered in a digital "
                  f"illustration style with clean edges and a limited palette of {dom} and gray.")
            if self.inject_fn is not None and (self.inject_in_retry or not self._in_retry):
                ph = self.inject_fn(a)
                if ph: tp = tp.replace("The composition is centered", f"The scene is {ph} here. The composition is centered")
            outs.append(PREFILL + tp + '", "visual_style": "digital illustration", "objects": ["gradient band", "strip"], '
                        f'"lighting": "even flat lighting", "camera_view": "frontal flat view", "color_palette": ["{dom}", "gray"], "visible_text": []}}')
        return outs

def build_engine(plan: Dict[str, Any], gpu_index: int = 0, prefer: Optional[str] = None) -> CaptionEngine:
    pref = prefer or CONFIG.CAPTION_ENGINE
    if CONFIG.ALLOW_MOCK_CAPTIONER and not ENV.get("gpus"): return MockEngine(plan, gpu_index)
    if pref == "vllm" and VLM_COMPAT.get("vllm_usable"): return VLLMEngine(plan, gpu_index)
    return TransformersEngine(plan, gpu_index)

def sanity_check_engine(engine: CaptionEngine) -> Tuple[bool, str]:
    """صورة اصطناعية بسيطة: نتحقق أن الإخراج نصٌّ سليم (يكشف NaN/تلف fp16: فراغ، رموز مكررة، نص غير لاتيني)."""
    import cv2
    img = np.zeros((384, 512, 3), np.uint8); img[:] = (30, 60, 160); cv2.circle(img, (256, 192), 90, (220, 40, 40), -1); cv2.rectangle(img, (40, 40), (160, 120), (240, 240, 60), -1)
    try: txt = engine.generate([img], 160)[0]
    except Exception as e: return False, f"{type(e).__name__}: {str(e)[:150]}"
    body = txt[len(PREFILL):]
    letters = sum(ch.isalpha() and ch.isascii() for ch in body); ratio = letters / max(1, len(body))
    uniq = len(set(body.split())) / max(1, len(body.split()))
    ok = len(body.split()) >= 8 and ratio > 0.55 and uniq > 0.35
    return ok, f"words={len(body.split())} alpha={ratio:.2f} uniq={uniq:.2f} sample={body[:90]!r}"

print("engines available: transformers | vllm:", VLM_COMPAT.get("vllm_usable"), "| mock:", CONFIG.ALLOW_MOCK_CAPTIONER)
_specs = current_gpu_specs()
CAPTION_PLANS = choose_precision_plan(_specs, CONFIG.CAPTION_PRECISION, CONFIG.CAPTION_ALLOW_FALLBACK_PLANS, CONFIG.MODEL_ID)
print("GPU specs:", [(s["name"], round(s["vram_gb"], 1), s["cc"]) for s in _specs])
print("candidate caption plans (best quality first):", CAPTION_PLANS or f"لا خطة: لا GPU، أو GPU لا يتسع لـ{CONFIG.MODEL_ID} بدقة كاملة (~{model_weight_gb()['bf16']:.0f}GB أوزان + هامش). CAPTION_ALLOW_FALLBACK_PLANS=True يسمح بخطط أضعف")

### CELL 18 — Caption Prompt and Structured Output
> البرومبت (نصّك الأصلي + قواعد: المرئي فقط، بلا ألفاظ تحوّط، بلا نفي فارغ + مخطط JSON)، تعبئة مسبقة للإجابة لضمان بدء JSON سليم، محلّل مقاوم للأعطال،
> ومدقّق جودة آلي للأوصاف + فلتر العبارات المحظورة (caption_banned_hits) الذي تستعمله الخلية 19 لإعادة المحاولة/التخطّي.

In [ ]:
# CELL 18 - Caption Prompt and Structured Output
#: البرومبت (نصّك الأصلي + قواعد: المرئي فقط، بلا ألفاظ تحوّط، بلا نفي فارغ + مخطط JSON)، تعبئة مسبقة للإجابة لضمان بدء JSON سليم، محلّل مقاوم للأعطال،
#: ومدقّق جودة آلي للأوصاف + فلتر العبارات المحظورة (caption_banned_hits) الذي تستعمله الخلية 19 لإعادة المحاولة/التخطّي.
BASE_INSTRUCTION = """You are an expert visual data annotator creating high-quality image-to-text training descriptions for advanced text-to-image models.

Analyze the provided image carefully.

Write an extremely detailed, objective, information-dense visual description.

Describe the main subject, secondary elements, objects, visible appearance, clothing, poses, actions, environment, foreground, midground, background, spatial relationships, composition, camera angle, perspective, lighting, shadows, colors, materials, textures, depth of field, and artistic style.

Describe ONLY what is visibly present in the image: its objects, colors, composition and lighting (and, where visible, materials, textures, poses, actions, environment and style).
Be precise about what is actually visible, and state it plainly and with certainty.

Never use hedging words such as "likely", "possibly", "perhaps", "probably", "might", "may be", "seems", "appears to" or "suggests".
Never describe what is absent: no empty negations such as "no other objects", "devoid of ...", "nothing else", "without any ..." or "no visible ...". If an area has nothing notable, simply leave it out.

Preserve important visual relationships and small details.

Describe anime, cartoon, illustration, photography, and 3D rendering styles accurately when present.

Do not invent details.
Do not infer hidden context.
Do not guess identities.
Do not invent text.
Do not add events or objects that cannot be verified from the image.

Produce one coherent, highly descriptive training prompt in {language}.

Make the description as detailed as the image warrants, without filler or repetition."""

JSON_SUFFIX = """

Output format: return ONLY one JSON object, with exactly these keys and nothing before or after it:
"training_prompt": the single coherent paragraph described above in {language} (typically 70-150 words for detail-rich images; shorter for simple images, longer only when there is a lot to see). Describe only what is visible, with no hedging words and no statements about what is absent: no feelings or relationships unless visibly evident, no identities, no events before or after the frame, and quote visible text only if it is clearly legible.
"visual_style": a short phrase naming the visible artistic style (for example photorealistic, cinematic photograph, anime, cartoon, illustration, 3D render, watercolor, oil painting), only if actually visible.
"objects": a list of up to 12 short noun phrases for the main visible subjects and objects.
"lighting": a short phrase describing the lighting type and direction.
"camera_view": a short phrase describing the camera angle, distance and perspective.
"color_palette": a list of 3 to 6 dominant color names.
"visible_text": a list of legible text strings exactly as seen, or an empty list."""

PREFILL = '{"training_prompt": "'
RETRY_SUFFIX = """

IMPORTANT: your previous description was rejected for unsupported wording. Write it again from scratch using only concrete, directly visible details (objects, colors, composition, lighting). Describe only what is present; never describe absence, emptiness or missing elements, and never hedge."""
REVIEW_INSTRUCTION = """You are a strict quality reviewer for image captions. Compare the caption with the image.
Caption: \"\"\"{caption}\"\"\"
Return ONLY JSON: {{"accurate": true or false, "unsupported_claims": [short strings], "missing_major_elements": [short strings]}}.
Mark accurate=false only if the caption states something clearly not visible or contradicts the image."""

def full_instruction() -> str:
    return (BASE_INSTRUCTION + JSON_SUFFIX).format(language=CONFIG.CAPTION_LANGUAGE)

def build_prompt_text(processor: Any, instruction: Optional[str] = None, prefill: str = PREFILL) -> str:
    """نص الدردشة الكامل (مع علامة الصورة) + التعبئة المسبقة. العلامة <|image_pad|> تُوسَّع داخل المعالج حسب أبعاد كل صورة."""
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction or full_instruction()}]}]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + prefill

# ---------------------------------------------------------------- فلتر العبارات المحظورة (CAPTION_BANNED_PHRASES)
_BANNED_RE_CACHE: Dict[Tuple[str, ...], Any] = {}

def caption_banned_hits(parsed_or_text: Any, phrases: Optional[Sequence[str]] = None) -> List[str]:
    """العبارات المحظورة الموجودة في الوصف (training_prompt وحقول النص المنظَّمة: visual_style/lighting/camera_view/objects/color_palette). مطابقة عبارة كاملة وبلا حساسية لحالة الأحرف
    ("blank" لا تطابق "blanket"؛ المسافات داخل العبارة تطابق أي فراغ). تعيد القائمة (فارغة = نظيف). visible_text (نص مقروء في الصورة) لا يُفحص."""
    phrases = tuple(p.strip().lower() for p in (CONFIG.CAPTION_BANNED_PHRASES if phrases is None else phrases) if p and p.strip())
    if not phrases: return []
    rx = _BANNED_RE_CACHE.get(phrases)
    if rx is None:
        rx = _BANNED_RE_CACHE[phrases] = re.compile(r"(?<![A-Za-z0-9])(" + "|".join(r"\s+".join(re.escape(w) for w in p.split()) for p in phrases) + r")(?![A-Za-z0-9])", re.I)
    if isinstance(parsed_or_text, dict):
        d = parsed_or_text; parts = [str(d.get("training_prompt") or ""), str(d.get("visual_style") or ""), str(d.get("lighting") or ""), str(d.get("camera_view") or "")]
        for k in ("objects", "color_palette"): parts += [str(x) for x in (d.get(k) or [])]
        text = " \n ".join(parts)
    else: text = str(parsed_or_text or "")
    return sorted({re.sub(r"\s+", " ", m.group(1).lower()) for m in rx.finditer(text)})

# ---------------------------------------------------------------- المحلّل
_STYLE_MAP = [("anime", "anime"), ("manga", "anime"), ("cartoon", "cartoon"), ("watercolor", "watercolor"), ("oil painting", "oil painting"),
              ("3d render", "3D render"), ("cgi", "3D render"), ("rendered", "3D render"), ("pixel art", "pixel art"), ("sketch", "sketch"),
              ("illustration", "illustration"), ("vector", "vector illustration"), ("cinematic", "cinematic photograph"),
              ("photorealistic", "photorealistic"), ("photograph", "photograph"), ("photo", "photograph"), ("screenshot", "screenshot")]
_COLORS = ["red", "orange", "yellow", "green", "teal", "cyan", "blue", "navy", "purple", "violet", "pink", "magenta", "brown", "beige", "tan",
           "white", "black", "gray", "grey", "silver", "gold", "golden", "cream", "maroon", "turquoise"]

def _clean_text(s: str) -> str:
    s = s.replace("\\n", " ").replace("\\t", " ").replace('\\"', '"').replace("\\/", "/")
    return re.sub(r"\s+", " ", s).strip().strip('"').strip()

def _as_list(v: Any, limit: int = 16) -> List[str]:
    if isinstance(v, str): v = [x for x in re.split(r"[;,]\s*", v) if x.strip()]
    if not isinstance(v, (list, tuple)): return []
    out = []
    for x in v:
        x = _clean_text(str(x))
        if x and x.lower() not in ("none", "n/a") and len(x) < 120: out.append(x)
    return out[:limit]

def _regex_field(raw: str, key: str) -> Optional[str]:
    m = re.search(r'"%s"\s*:\s*"((?:[^"\\]|\\.)*)' % re.escape(key), raw, re.S)
    return m.group(1) if m else None

def parse_caption_output(raw: str) -> Dict[str, Any]:
    """يحوّل مخرجات النموذج (تبدأ بالتعبئة المسبقة) إلى حقول منظمة. parse_status: json | repaired | prose | failed."""
    txt = raw.strip()
    txt = re.sub(r"^```(?:json)?\s*|\s*```$", "", txt)
    res: Dict[str, Any] = dict(training_prompt="", visual_style="", objects=[], lighting="", camera_view="", color_palette=[], visible_text=[],
                               parse_status="failed", truncated=False)
    a, b = txt.find("{"), txt.rfind("}")
    obj: Optional[Dict[str, Any]] = None
    if a >= 0 and b > a:
        try: obj = json.loads(txt[a:b + 1]); res["parse_status"] = "json"
        except json.JSONDecodeError: obj = None
    if obj is None and a >= 0:                                               # مقطوع/معطوب: نُصلح بالتعابير النمطية
        tp = _regex_field(txt, "training_prompt")
        if tp is not None:
            obj = {"training_prompt": tp}
            for k in ("visual_style", "lighting", "camera_view"):
                v = _regex_field(txt, k)
                if v: obj[k] = v
            for k in ("objects", "color_palette", "visible_text"):
                m = re.search(r'"%s"\s*:\s*\[(.*?)\]' % k, txt, re.S)
                if m: obj[k] = re.findall(r'"((?:[^"\\]|\\.)*)"', m.group(1))
            res["parse_status"] = "repaired"
            res["truncated"] = not re.search(r'"training_prompt"\s*:\s*"(?:[^"\\]|\\.)*"\s*[,}]', txt, re.S)
    if obj is None:
        prose = _clean_text(txt)
        if len(prose.split()) >= 5 and not prose.startswith("{"): res.update(training_prompt=prose, parse_status="prose")
        return res
    res["training_prompt"] = _clean_text(str(obj.get("training_prompt", "")))
    res["visual_style"] = _clean_text(str(obj.get("visual_style", "")))[:80]
    res["lighting"] = _clean_text(str(obj.get("lighting", "")))[:160]
    res["camera_view"] = _clean_text(str(obj.get("camera_view", "")))[:160]
    res["objects"], res["color_palette"], res["visible_text"] = _as_list(obj.get("objects")), _as_list(obj.get("color_palette"), 8), _as_list(obj.get("visible_text"), 12)
    return res

def fill_structured_fallbacks(r: Dict[str, Any]) -> Dict[str, Any]:
    """يملأ الحقول المنظمة الناقصة بقواعد بسيطة من نص الوصف (لا اختراع: فقط استخراج كلمات موجودة فيه)."""
    tp = r["training_prompt"]; low = tp.lower()
    if not r["visual_style"]:
        r["visual_style"] = next((v for k, v in _STYLE_MAP if k in low), "unspecified")
    if not r["color_palette"]:
        found = [(low.count(c), c) for c in _COLORS if re.search(r"\b%s\b" % c, low)]
        r["color_palette"] = [c for _, c in sorted(found, reverse=True)[:5]]
    sents = re.split(r"(?<=[.!?])\s+", tp)
    if not r["lighting"]: r["lighting"] = next((s[:160] for s in sents if re.search(r"\b(light|lighting|lit|shadow|sunlight|glow|illuminat)", s.lower())), "")
    if not r["camera_view"]: r["camera_view"] = next((s[:160] for s in sents if re.search(r"\b(angle|shot|view|perspective|close-up|wide)\b", s.lower())), "")
    return r

# ---------------------------------------------------------------- مدقّق الجودة الآلي (قواعد فقط؛ لا يثبت صحة الوصف)
_REFUSAL = re.compile(r"\b(i'?m sorry|i cannot|i can'?t|as an ai|unable to (?:see|view|describe|process)|cannot (?:see|view|describe)|no image)\b", re.I)
_GENERIC_OPEN = re.compile(r"^(?:the|this)\s+(?:image|picture|photo|photograph|frame)\s+(?:shows|depicts|features|displays|is)|^(?:an?\s+)?(?:image|picture)\s+of\b|^in this (?:image|picture)", re.I)
_FLUFF = re.compile(r"\b(stunning|breathtaking|amazing|beautiful|gorgeous|masterpiece|captivating|mesmerizing|awe-inspiring|very very)\b", re.I)
_HEDGE = re.compile(r"\b(possibly|probably|perhaps|seems? to|might be|may be|likely|suggests?|appears? to be (?:feeling|happy|sad|angry))\b", re.I)
_IDENT = re.compile(r"\b(?:looks like|resembles|appears to be|is)\s+(?:the\s+)?(?:actor|actress|celebrity|singer|president|famous)\b|\b[A-Z][a-z]+\s+[A-Z][a-z]+\s+(?:himself|herself)\b", re.I)
_STYLE_WORDS = re.compile(r"\b(photo|photograph|photorealistic|cinematic|anime|cartoon|illustration|3d|render|rendered|watercolor|painting|sketch|pixel|vector|drawing|digital art|screenshot|film|footage|still)\b", re.I)
_SPATIAL = re.compile(r"\b(left|right|foreground|background|midground|center|centre|top|bottom|behind|in front|beside|above|below|next to|upper|lower|middle|edge|corner)\b", re.I)
_LIGHT = re.compile(r"\b(light|lighting|lit|shadow|illuminat|sunlight|glow|backlit|overcast|dim|bright|highlight)\w*", re.I)

def validate_caption(r: Dict[str, Any]) -> Dict[str, Any]:
    """يعيد {'score','flags','hard_fail','words'}. hard_fail ⇒ لا يدخل Dataset الرئيسي (يذهب لشظية المراجعة)."""
    tp = r.get("training_prompt", ""); words = tp.split(); n = len(words); flags: List[str] = []; hard: Optional[str] = None
    if n == 0: return dict(score=0.0, flags=["empty"], hard_fail="empty", words=0)
    if _REFUSAL.search(tp): hard = "refusal"
    letters = [ch for ch in tp if ch.isalpha()]
    non_ascii = sum(not ch.isascii() for ch in letters) / max(1, len(letters))
    if CONFIG.CAPTION_LANGUAGE.lower() == "english":
        if non_ascii > 0.20: hard = hard or "wrong_language"
        elif non_ascii > 0.05: flags.append("mixed_language")
    low = [w.lower().strip(".,;:!?\"'()") for w in words]
    if n >= 12:
        tri = [tuple(low[i:i + 3]) for i in range(n - 2)]; rep = 1 - len(set(tri)) / len(tri)
        sentences = [s.strip().lower() for s in re.split(r"(?<=[.!?])\s+", tp) if len(s.split()) >= 4]
        run = max((len(list(g)) for _, g in __import__("itertools").groupby(low)), default=1)
        if run >= 6 or (sentences and len(set(sentences)) < len(sentences) * 0.7) or rep > 0.45: hard = hard or "degenerate_repetition"
        elif rep > 0.22: flags.append("repetitive")
    if n < CONFIG.CAPTION_MIN_WORDS: flags.append("too_short")
    if n > 420: flags.append("too_long")
    if _GENERIC_OPEN.search(tp): flags.append("generic_opening")
    if _FLUFF.search(tp): flags.append("subjective_fluff")
    if len(_HEDGE.findall(tp)) >= 3: flags.append("speculation")
    if _IDENT.search(tp): flags.append("possible_identity")
    if r.get("truncated") or not re.search(r"[.!?\"')]$", tp.strip()): flags.append("truncated")
    if n >= 25:
        if not _STYLE_WORDS.search(tp): flags.append("no_style")
        if not _SPATIAL.search(tp): flags.append("no_spatial")
        if not _LIGHT.search(tp): flags.append("no_lighting")
    if r.get("parse_status") in ("prose", "failed"): flags.append("unstructured")
    elif r.get("parse_status") == "repaired": flags.append("json_repaired")
    pen = dict(too_short=.50, too_long=.1, generic_opening=.10, subjective_fluff=.05, speculation=.10, possible_identity=.15, truncated=.20,
               no_style=.10, no_spatial=.10, no_lighting=.05, unstructured=.05, repetitive=.25, mixed_language=.10, json_repaired=.0)
    score = max(0.0, 1.0 - sum(pen.get(f, 0.0) for f in flags))
    if hard: flags.append(f"HARD:{hard}"); score = 0.0
    return dict(score=round(score, 3), flags=flags, hard_fail=hard, words=n)

def process_raw_caption(raw: str) -> Dict[str, Any]:
    """خام النموذج → سجل جاهز: الحقول المنظمة + نتيجة التدقيق + صفة 'review' (للشظية المنفصلة)."""
    r = fill_structured_fallbacks(parse_caption_output(raw)); v = validate_caption(r)
    r.update(caption_score=v["score"], caption_flags=v["flags"], caption_words=v["words"])
    r["review"] = bool(v["hard_fail"]) or v["score"] < CONFIG.CAPTION_REVIEW_SCORE
    r["review_reason"] = v["hard_fail"] or ("low_score" if r["review"] else "")
    return r

print("caption prompt ready | language:", CONFIG.CAPTION_LANGUAGE, "| instruction words:", len(full_instruction().split()))

### CELL 19 — Batched Captioning
> دفعات تكيفية (ميزانية رموز بصرية + حجم دفعة)، استرداد من OOM بلا فقدان أي صورة، ومجمّع محركات (نسخة لكل GPU عند "replicate").
> بعد كل وصف: فلتر العبارات المحظورة (CAPTION_BANNED_PHRASES) ⇒ إعادة محاولة واحدة ببرومبت أشد، وإن بقيت العبارة يُوسَم الإطار للتخطّي (الخلية 24B تسقطه).

In [ ]:
# CELL 19 - Batched Captioning
#: دفعات تكيفية (ميزانية رموز بصرية + حجم دفعة)، استرداد من OOM بلا فقدان أي صورة، ومجمّع محركات (نسخة لكل GPU عند "replicate").
#: بعد كل وصف: فلتر العبارات المحظورة (CAPTION_BANNED_PHRASES) ⇒ إعادة محاولة واحدة ببرومبت أشد، وإن بقيت العبارة يُوسَم الإطار للتخطّي (الخلية 24B تسقطه).
import numpy as np, collections

@dataclasses.dataclass
class CapItem:
    sample_id: str
    image: np.ndarray                 # MODEL_INPUT_IMAGE (RGB uint8)

@dataclasses.dataclass
class CapResult:
    sample_id: str
    raw: str = ""
    parsed: Optional[Dict[str, Any]] = None
    error: Optional[str] = None
    latency: float = 0.0
    model: str = ""

class AdaptiveBatcher:
    """يتحكم بحجم الدفعة وميزانية الرموز البصرية. OOM ⇒ خفض فوري وسقف جديد؛ نجاحات متتالية ⇒ رفع تدريجي حتى السقف."""
    def __init__(self, start_bs: int = 4, max_bs: int = 16, token_budget: int = 16384, grow_after: int = 15):
        self.bs, self.max_bs, self.ceiling = max(1, start_bs), max(1, max_bs), max(1, max_bs)
        self.token_budget, self.grow_after, self.streak, self.oom_events = token_budget, grow_after, 0, 0
        self.history: "collections.deque" = collections.deque(maxlen=50)

    def plan(self, tokens: Sequence[int]) -> List[List[int]]:
        """يقسم الفهارس (مرتّبة سلفًا بالحجم) إلى دفعات بحيث (عدد ≤ bs) و(مجموع الرموز ≤ ميزانية الرموز)."""
        batches: List[List[int]] = []; cur: List[int] = []; cur_tok = 0
        for i, t in enumerate(tokens):
            if cur and (len(cur) >= self.bs or cur_tok + t > self.token_budget):
                batches.append(cur); cur, cur_tok = [], 0
            cur.append(i); cur_tok += t
        if cur: batches.append(cur)
        return batches

    def on_success(self, n: int) -> None:
        self.streak += 1; self.history.append(n)
        if self.streak >= self.grow_after and self.bs < min(self.ceiling, self.max_bs):
            self.bs += 1; self.streak = 0
        if self.streak >= self.grow_after * 6 and self.ceiling < self.max_bs:       # السقف يتراخى ببطء (قد يكون OOM ظرفيًا)
            self.ceiling += 1; self.streak = 0

    def on_oom(self, failed_n: int) -> None:
        self.oom_events += 1; self.streak = 0
        self.ceiling = max(1, failed_n - 1)
        self.bs = max(1, min(self.bs, failed_n) // 2) if failed_n > 1 else 1
        self.token_budget = max(1024, int(self.token_budget * 0.75))

class Captioner:
    """يشغّل محركًا واحدًا على قائمة عناصر. لا يفقد أي عنصر: كل عنصر ينتهي إما بوصف أو بخطأ موثَّق."""
    def __init__(self, engine: CaptionEngine, batcher: AdaptiveBatcher):
        self.engine, self.batcher = engine, batcher
        self.stats = collections.Counter(); self.lat: "collections.deque" = collections.deque(maxlen=200)

    def _tokens(self, it: CapItem) -> int:
        h, w = it.image.shape[:2]; return self.engine.visual_tokens(w, h) + 400 + CONFIG.CAPTION_MAX_NEW_TOKENS

    def _run(self, items: List[CapItem], depth: int = 0) -> List[CapResult]:
        t0 = time.time()
        try:
            raws = self.engine.generate([it.image for it in items], CONFIG.CAPTION_MAX_NEW_TOKENS)
            if len(raws) != len(items): raise RuntimeError(f"عدد المخرجات {len(raws)} ≠ المدخلات {len(items)}")
            dt = (time.time() - t0) / len(items); self.batcher.on_success(len(items)); self.lat.append(dt)
            return [CapResult(it.sample_id, raw=r, latency=dt, model=self.engine.model_id) for it, r in zip(items, raws)]
        except CaptionOOM as e:
            self.stats["oom"] += 1; self.stats["retries"] += 1; self.batcher.on_oom(len(items))
            self._free_cache()
            if len(items) > 1:                                    # قسّم الدفعة وأعد
                mid = len(items) // 2
                return self._run(items[:mid], depth + 1) + self._run(items[mid:], depth + 1)
            it = items[0]; small = self._shrink(it.image)
            if small is not None and depth < 6:                   # صورة وحيدة: نصغّرها 0.7× مرة بعد مرة
                return self._run([CapItem(it.sample_id, small)], depth + 1)
            return [CapResult(it.sample_id, error=f"OOM حتى بعد التصغير: {str(e)[:100]}", model=self.engine.model_id)]
        except Exception as e:
            self.stats["errors"] += 1
            if len(items) > 1:                                    # عنصر معطوب قد يُفشل الدفعة كلها: نعزله
                self.stats["retries"] += 1; self._free_cache()
                return [r for it in items for r in self._run([it], depth + 1)]
            return [CapResult(items[0].sample_id, error=f"{type(e).__name__}: {str(e)[:160]}", model=self.engine.model_id)]

    @staticmethod
    def _shrink(img: np.ndarray) -> Optional[np.ndarray]:
        h, w = img.shape[:2]; nw, nh = max(QWEN_FACTOR, int(w * 0.7) // QWEN_FACTOR * QWEN_FACTOR), max(QWEN_FACTOR, int(h * 0.7) // QWEN_FACTOR * QWEN_FACTOR)
        return None if (nw, nh) == (w, h) or nw * nh < CONFIG.MODEL_MIN_PIXELS // 4 else cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)

    @staticmethod
    def _free_cache() -> None:
        try:
            import torch
            if torch.cuda.is_available(): gc.collect(); torch.cuda.empty_cache()
        except Exception: pass

    def caption_items(self, items: List[CapItem]) -> List[CapResult]:
        if not items: return []
        order = sorted(range(len(items)), key=lambda i: items[i].image.shape[0] * items[i].image.shape[1])     # تقليل الحشو (padding)
        sorted_items = [items[i] for i in order]
        out: Dict[str, CapResult] = {}
        pending = list(range(len(sorted_items)))
        while pending:
            toks = [self._tokens(sorted_items[i]) for i in pending]
            b = self.batcher.plan(toks)[0]                                    # نأخذ الدفعة الأولى ثم نعيد الحساب (قد يتغير bs بعد OOM)
            idx = [pending[j] for j in b]; pending = pending[len(b):]
            for r in self._run([sorted_items[i] for i in idx]): out[r.sample_id] = r
        results = [out[it.sample_id] for it in items]                         # إرجاع بترتيب الإدخال الأصلي
        for r in results:
            if r.error is None:
                r.parsed = process_raw_caption(r.raw)
                if r.parsed["training_prompt"]: self.stats["captioned"] += 1
                else: self.stats["empty"] += 1
            else: self.stats["failed"] += 1
        if CONFIG.CAPTION_BANNED_PHRASES: self._banned_pass(items, results)
        if CONFIG.ENABLE_VLM_SECOND_REVIEW: self._vlm_review(items, results)
        return results

    def _retry_chunk(self, chunk: List[CapItem]) -> Optional[List[str]]:
        """يولّد وصفًا ثانيًا لمجموعة صور ببرومبت أشد. OOM ⇒ تقسيم؛ أي خطأ آخر ⇒ None (يُعدّ الإطار غير مُصلَح)."""
        try: return self.engine.generate_retry([it.image for it in chunk], CONFIG.CAPTION_MAX_NEW_TOKENS)
        except CaptionOOM:
            self._free_cache(); self.batcher.on_oom(len(chunk))
            if len(chunk) == 1: return None
            mid = len(chunk) // 2; a, b = self._retry_chunk(chunk[:mid]), self._retry_chunk(chunk[mid:])
            return None if a is None or b is None else a + b
        except Exception as e:
            log.warning("إعادة محاولة الوصف فشلت (%s): %s", type(e).__name__, str(e)[:120]); return None

    def _banned_pass(self, items: List[CapItem], results: List[CapResult]) -> None:
        """فلتر ما بعد الوصف: أي وصف يحوي عبارة محظورة ⇒ إعادة محاولة (CAPTION_BANNED_RETRIES مرة) ببرومبت أشد؛ إن بقيت العبارة ⇒ parsed['reject_reason']=caption_banned_phrase."""
        bad = [(it, r) for it, r in zip(items, results) if r.parsed and r.parsed.get("training_prompt") and caption_banned_hits(r.parsed)]
        if not bad: return
        self.stats["banned_first"] += len(bad)
        for _ in range(max(0, CONFIG.CAPTION_BANNED_RETRIES)):
            todo = [(it, r) for it, r in bad if caption_banned_hits(r.parsed)]
            if not todo: break
            self.stats["banned_retried"] += len(todo)
            for i in range(0, len(todo), max(1, self.batcher.bs)):
                part = todo[i:i + max(1, self.batcher.bs)]; raws = self._retry_chunk([it for it, _ in part])
                if raws is None or len(raws) != len(part): continue
                for (it, r), raw in zip(part, raws):
                    new = process_raw_caption(raw)
                    if new["training_prompt"]: r.raw, r.parsed = raw, new; r.parsed["retried"] = True
        for it, r in bad:
            hits = caption_banned_hits(r.parsed)
            if hits: r.parsed["reject_reason"] = REJECT_CAPTION_BANNED; r.parsed["banned_hits"] = hits; self.stats["banned_skipped"] += 1
            else: self.stats["banned_fixed"] += 1

    def _vlm_review(self, items: List[CapItem], results: List[CapResult]) -> None:
        """مراجعة ثانية اختيارية بالـVLM (تكلفة إضافية). أي خطأ هنا غير قاتل ولا يحذف شيئًا؛ يضيف وسم مراجعة فقط."""
        todo = [(it, r) for it, r in zip(items, results) if r.parsed and r.parsed["training_prompt"] and not r.parsed["review"] and not r.parsed.get("reject_reason")]
        if not todo or not hasattr(self.engine, "processor"): return
        try:
            texts = [build_prompt_text(self.engine.processor, REVIEW_INSTRUCTION.format(caption=r.parsed["training_prompt"]), prefill='{"accurate": ') for _, r in todo]
            outs = self.engine.generate([it.image for it, _ in todo], 120, texts=texts)
            for (it, r), o in zip(todo, outs):
                o = ('{"accurate": ' + o) if not o.lstrip().startswith("{") else o
                m = re.search(r'"accurate"\s*:\s*(true|false)', o, re.I)
                if m and m.group(1).lower() == "false":
                    r.parsed["caption_flags"].append("vlm_review_rejected"); r.parsed["review"] = True; r.parsed["review_reason"] = "vlm_review"
                    self.stats["vlm_rejected"] += 1
        except Exception as e:
            log.warning("المراجعة الثانية بالـVLM تعذّرت (غير قاتلة): %s", str(e)[:120])

class CaptionPool:
    """نسخة محرك لكل GPU (أو محرك واحد). يُستدعى من خيوط متعددة؛ كل مهمة تستعير محركًا خاملًا."""
    def __init__(self, engines: List[CaptionEngine], batchers: List[AdaptiveBatcher], plan: Dict[str, Any]):
        self.engines, self.plan = engines, plan
        self.captioners = [Captioner(e, b) for e, b in zip(engines, batchers)]
        self._idle: "queue.Queue" = queue.Queue()
        for c in self.captioners: self._idle.put(c)
        self.model_id = engines[0].model_id; self.engine_name = engines[0].name
        self.exec = ThreadPoolExecutor(max_workers=len(engines), thread_name_prefix="caption")
        self.busy_seconds = 0.0; self.images_done = 0; self._lock = threading.Lock()

    def _task(self, items: List[CapItem]) -> List[CapResult]:
        c = self._idle.get()
        t0 = time.time()
        try: return c.caption_items(items)
        finally:
            self._idle.put(c)
            with self._lock: self.busy_seconds += time.time() - t0; self.images_done += len(items)

    def submit(self, items: List[CapItem]) -> "Future":
        return self.exec.submit(self._task, items)

    def metrics(self) -> Dict[str, Any]:
        lat = [x for c in self.captioners for x in c.lat]
        tot = collections.Counter()
        for c in self.captioners: tot.update(c.stats)
        return dict(batch_size=[c.batcher.bs for c in self.captioners], avg_latency_s=(float(np.mean(lat)) if lat else 0.0),
                    failures=tot["failed"] + tot["empty"], retries=tot["retries"], oom=tot["oom"], captioned=tot["captioned"], model=self.model_id,
                    banned_first=tot["banned_first"], banned_retried=tot["banned_retried"], banned_fixed=tot["banned_fixed"], banned_skipped=tot["banned_skipped"],
                    engine=self.engine_name, precision=self.plan.get("precision"), images_done=self.images_done, busy_seconds=self.busy_seconds)

    def close(self) -> None:
        self.exec.shutdown(wait=True)
        for e in self.engines: e.free()

def initial_batcher() -> AdaptiveBatcher:
    bench = STATE.benchmark.get("caption", {}) if "STATE" in globals() else {}
    start = CONFIG.CAPTION_BATCH_SIZE if isinstance(CONFIG.CAPTION_BATCH_SIZE, int) else int(bench.get("stable_batch", 4))
    mx = CONFIG.CAPTION_BATCH_SIZE if isinstance(CONFIG.CAPTION_BATCH_SIZE, int) else max(start, int(bench.get("max_tested_batch", 8)))
    return AdaptiveBatcher(start_bs=start, max_bs=mx, token_budget=int(bench.get("token_budget", 16384)))

def _load_engines(plan: Dict[str, Any], run_sanity: bool) -> List[CaptionEngine]:
    engines: List[CaptionEngine] = []
    try:
        n = plan["n_gpu"] if plan["layout"] == "replicate" else 1
        gpu_ids = [g["index"] for g in current_gpu_specs()][:n] if plan["layout"] != "shard" else [0]
        for gi in gpu_ids:
            eng = build_engine(plan, gi); engines.append(eng); eng.load()
            if run_sanity:
                ok, info = sanity_check_engine(eng)
                log.info("فحص سلامة [%s/%s/GPU%s/%s]: %s | %s", eng.name, plan["precision"], gi, getattr(eng, "attn", "-"), "OK" if ok else "FAIL", info)
                if not ok: raise ModelSanityError(info)
        return engines
    except Exception:
        for en in engines:
            try: en.free()
            except Exception: pass
        raise

def load_caption_pool(plans: Optional[List[Dict[str, Any]]] = None, run_sanity: bool = True) -> CaptionPool:
    """يجرّب الخطط بالترتيب (جودة أعلى أولًا) ويعتمد أول ما يُحمَّل وينجح في فحص السلامة. يرمي خطأ واضحًا إن لم ينجح شيء.
    flash_attention_2 (الافتراضي إن توفّرت) تفشل في التحميل أو فحص السلامة ⇒ إعادة تحميل نفس الخطة بـ sdpa تلقائيًا قبل الانتقال للخطة التالية."""
    if CONFIG.ALLOW_MOCK_CAPTIONER and not ENV.get("gpus"):
        e = MockEngine(); e.load(); return CaptionPool([e], [initial_batcher()], dict(precision="mock", layout="cpu"))
    plans = plans if plans is not None else CAPTION_PLANS
    if not plans:
        raise RuntimeError(f"لا توجد خطة تشغيل مناسبة لـ{CONFIG.MODEL_ID}: يلزم GPU واحدة على الأقل بذاكرة ≥ ~{model_weight_gb()['bf16'] + ACT_GB + 1:.0f}GB (bf16 كاملة، بلا shard) "
                           f"أو MODEL_ID أصغر، أو CAPTION_ALLOW_FALLBACK_PLANS=True (shard/int8/int4 بجودة أقل). GPU الحالية: {[(s['name'], round(s['vram_gb'], 1)) for s in current_gpu_specs()] or 'لا يوجد'}")
    errors: List[str] = []
    for plan in plans:
        attempts = [plan]
        for k, cur in enumerate(attempts):
            try:
                engines = _load_engines(cur, run_sanity)
                return CaptionPool(engines, [initial_batcher() for _ in engines], cur)
            except Exception as e:
                errors.append(f"{cur['precision']}/{cur['layout']}/{cur.get('attn') or CONFIG.ATTN_IMPLEMENTATION}: {type(e).__name__}: {str(e)[:150]}")
                log.warning("الخطة %s فشلت. %s", cur, errors[-1])
                fa2_possible = VLM_COMPAT.get("flash_attn_usable") and CONFIG.ATTN_IMPLEMENTATION == "auto" and cur.get("attn") != "sdpa"
                fa2_suspect = isinstance(e, ModelSanityError) or bool(re.search(r"flash|kernel image|invalid device function|illegal memory|cuda error|sm_\d+", str(e), re.I))
                if k == 0 and fa2_possible and fa2_suspect and not is_oom(e):
                    log.warning("→ إعادة المحاولة بنفس الخطة مع attn=sdpa"); attempts.append(dict(cur, attn="sdpa")); continue
                if CONFIG.CAPTION_ENGINE == "vllm":                    # فشل vLLM ⇒ نعود إلى Transformers تلقائيًا
                    try:
                        CONFIG.CAPTION_ENGINE = "transformers"; return load_caption_pool([cur], run_sanity)
                    except Exception as e2: errors.append(f"transformers-fallback: {e2}")
    raise RuntimeError("تعذّر تحميل Qwen3-VL بأي خطة: " + " | ".join(errors))

print("batched captioning ready (pool is created by load_caption_pool(); not loaded yet)")

---
## الجزء 3 — المعاينة والمخرجات والرفع (20–23)
معاينة عيّنة حقيقية بعد كل دفعة، كاتب شظايا WebDataset، الرفع إلى Hugging Face، مدير نقاط الحفظ.

### CELL 20 — Random Sample Preview
> بعد كل دفعة مكتملة: صورة حقيقية مختارة عشوائيًا من الدفعة نفسها + وصف Qwen الفعلي + بياناتها، داخل مخرجات الخلية.

In [ ]:
# CELL 20 - Random Sample Preview
#: بعد كل دفعة مكتملة: صورة حقيقية مختارة عشوائيًا من الدفعة نفسها + وصف Qwen الفعلي + بياناتها، داخل مخرجات الخلية.
import html as _html, base64, uuid
try:
    from IPython.display import display, HTML, Image as IPImage, Markdown, clear_output, update_display
    HAVE_IPY = True
except Exception:                                    # بيئة بلا IPython (اختبارات): نكتب ملف HTML ونطبع نصًا
    HAVE_IPY = False
if ENV.get("on_marimo"): HAVE_IPY = False            # marimo لا يعرض display() الخاصة بـ IPython: نكتب ملف HTML ونطبع نصًا

PREVIEW_DIR = Path(CONFIG.WORK_DIR) / "previews"; PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
MAX_APPENDED_PREVIEWS = 25                           # بعدها نحدّث بطاقة واحدة بدل تضخيم ملف الـ notebook

_CSS = """<style>
.rs{font-family:system-ui,sans-serif;border:1px solid #8884;border-radius:12px;padding:14px;margin:10px 0;max-width:980px}
.rs h4{margin:0 0 10px;letter-spacing:.08em;font-size:13px;opacity:.75}
.rs img{max-width:100%;max-height:520px;border-radius:8px;display:block;margin:0 0 10px}
.rs blockquote{margin:8px 0;padding:10px 14px;border-left:4px solid #3b82f6;background:#8881;border-radius:6px;line-height:1.5;font-size:14px}
.rs table{border-collapse:collapse;font-size:13px}.rs td{padding:2px 12px 2px 0;vertical-align:top}.rs td:first-child{opacity:.65}
.rs .fl{color:#b45309;font-size:12px;margin-top:6px}
</style>"""

def render_sample_html(jpeg: bytes, meta: Dict[str, Any], caption: str, title: str = "RANDOM DATASET SAMPLE", flags: Sequence[str] = ()) -> str:
    img = base64.b64encode(make_preview_jpeg(jpeg)).decode("ascii")
    rows = "".join(f"<tr><td>{_html.escape(k)}</td><td>{_html.escape(str(v))}</td></tr>" for k, v in meta.items())
    fl = f'<div class="fl">flags: {_html.escape(", ".join(flags))}</div>' if flags else ""
    return (f'{_CSS}<div class="rs"><h4>{_html.escape(title)}</h4><img src="data:image/jpeg;base64,{img}"/>'
            f'<div><b>Generated Training Prompt:</b></div><blockquote>{_html.escape(caption)}</blockquote>{fl}'
            f'<div><b>Metadata:</b></div><table>{rows}</table></div>')

def sample_meta_table(rec: Dict[str, Any]) -> Dict[str, Any]:
    return {"Sample ID": rec["sample_id"], "Video ID": rec["video_id"], "Frame": rec["frame_index"],
            "Timestamp": f'{rec["timestamp_seconds"]:.3f}s', "Resolution": f'{rec["width"]}x{rec["height"]}',
            "Caption Model": rec.get("caption_model", "?"), "Caption words": rec.get("caption_words", "")}

class PreviewManager:
    def __init__(self):
        self.shown = 0; self.rng = random.Random(); self.last_html = ""; self.new_run()
    def new_run(self) -> None:
        """كل تشغيل (خلية) له عدّاد ومعرّف عرض خاص: وإلا حدّثت المعاينة بعد 25 عرضًا مخرجات خلية سابقة بدل الخلية الحالية."""
        self.run_shown = 0; self.latest_id = "latest_preview-" + uuid.uuid4().hex[:8]
    def show_from_batch(self, samples: Sequence[Dict[str, Any]]) -> Optional[str]:
        """samples: سجلات مكتملة {jpeg, rec(json dict)}. نختار PREVIEW_SAMPLES_PER_BATCH عشوائيًا من الدفعة المنتهية فعلًا."""
        if not CONFIG.ENABLE_RANDOM_SAMPLE_PREVIEW or not samples: return None
        good = [s for s in samples if s["rec"].get("training_prompt")]
        if not good: return None
        out = None
        for s in self.rng.sample(good, min(CONFIG.PREVIEW_SAMPLES_PER_BATCH, len(good))):
            rec = s["rec"]; htm = render_sample_html(s["jpeg"], sample_meta_table(rec), rec["training_prompt"], flags=rec.get("caption_flags", []))
            self.last_html = htm; self.shown += 1; self.run_shown += 1; out = htm
            (PREVIEW_DIR / f"preview_{self.shown % 5}.html").write_text(htm, encoding="utf-8")
            if HAVE_IPY:
                try:
                    if self.run_shown <= MAX_APPENDED_PREVIEWS: display(HTML(htm))
                    elif self.run_shown == MAX_APPENDED_PREVIEWS + 1: display(HTML(htm), display_id=self.latest_id)
                    else: update_display(HTML(htm), display_id=self.latest_id)
                except Exception: pass
            else:
                print(f"\n---- RANDOM DATASET SAMPLE #{self.shown} ----\n[{rec['sample_id']}] {rec['width']}x{rec['height']} @ {rec['timestamp_seconds']:.2f}s "
                      f"| model: {rec.get('caption_model')}\n{rec['training_prompt']}\n----")
        return out

PREVIEW = PreviewManager()
print("preview ready | IPython display:", HAVE_IPY, "| per batch:", CONFIG.PREVIEW_SAMPLES_PER_BATCH)

### CELL 21 — WebDataset Shard Writer
> كل عينة = <sample_id>.jpg + <sample_id>.json داخل TAR. sample_id = <video_id>_<frame_index:08d> (ثابت وفريد؛ النقاط في video_id → '~' لأن مفتاح WebDataset يتوقف عند أول نقطة).
> التحقق من سلامة الشظية (TAR/صور/JSON/Checksum/UTF-8/تكرار المفاتيح) + قراءة عشوائية عبر الإزاحات قبل أي رفع.

In [ ]:
# CELL 21 - WebDataset Shard Writer
#: كل عينة = <sample_id>.jpg + <sample_id>.json داخل TAR. sample_id = <video_id>_<frame_index:08d> (ثابت وفريد؛ النقاط في video_id → '~' لأن مفتاح WebDataset يتوقف عند أول نقطة).
#: التحقق من سلامة الشظية (TAR/صور/JSON/Checksum/UTF-8/تكرار المفاتيح) + قراءة عشوائية عبر الإزاحات قبل أي رفع.
from PIL import Image

IMG_EXT = "jpg" if CONFIG.IMAGE_FORMAT == "jpg" else "webp"
TAR_BLOCK = 512

def make_sample_id(video_id: str, frame_index: int) -> str:
    return video_id.replace(".", "~") + f"_{int(frame_index):08d}"

@dataclasses.dataclass
class Sample:
    video_id: str
    frame_index: int
    ts: float
    jpeg: bytes
    width: int
    height: int
    orig_w: int
    orig_h: int
    pixel_sha256: str
    phash: int
    dhash: int
    metrics: Dict[str, float]
    backend: str = ""
    model_input: Optional[np.ndarray] = None
    caption: Optional[Dict[str, Any]] = None
    caption_model: str = ""
    caption_precision: str = ""
    caption_error: Optional[str] = None
    @property
    def sample_id(self) -> str: return make_sample_id(self.video_id, self.frame_index)

def build_record(s: Sample, vrec: Dict[str, Any]) -> Dict[str, Any]:
    c = s.caption or {}
    rec = {
        "sample_id": s.sample_id, "video_id": s.video_id, "frame_index": int(s.frame_index), "timestamp_seconds": round(float(s.ts), 6),
        "training_prompt": c.get("training_prompt", ""), "caption_language": CONFIG.CAPTION_LANGUAGE,
        "width": int(s.width), "height": int(s.height), "original_width": int(s.orig_w), "original_height": int(s.orig_h),
        "image_format": IMG_EXT, "file_size": len(s.jpeg), "image_sha256": hashlib.sha256(s.jpeg).hexdigest(), "pixel_sha256": s.pixel_sha256,
        "phash": f"{s.phash:016x}", "dhash": f"{s.dhash:016x}",
        "visual_style": c.get("visual_style", ""), "objects": c.get("objects", []), "lighting": c.get("lighting", ""), "camera_view": c.get("camera_view", ""),
        "color_palette": c.get("color_palette", []), "visible_text": c.get("visible_text", []),
        "quality_score": float(s.metrics.get("quality_score", 0.0)), "caption_score": float(c.get("caption_score", 0.0)),
        "caption_flags": c.get("caption_flags", []), "caption_words": int(c.get("caption_words", 0)),
        "caption_status": ("disabled" if not CONFIG.ENABLE_CAPTIONING else ("ok" if c.get("training_prompt") else "failed")),
        "caption_model": s.caption_model, "caption_precision": s.caption_precision,
        "source_name": vrec.get("source_name"), "license": vrec.get("license"), "source_reference": vrec.get("source_reference"),
        "sampling": {"seed": CONFIG.SAMPLING_SEED, "ratio": CONFIG.SAMPLING_RATIO, "method": CONFIG.sampling_method(),
                     "hash_u": round(hash_unit(CONFIG.SAMPLING_SEED, s.video_id, s.frame_index), 6)},
        "decoder": s.backend, "created_at": int(time.time()),
    }
    if s.caption_error: rec["caption_error"] = str(s.caption_error)[:200]
    return rec

INDEX_COLUMNS = ["sample_id", "video_id", "frame_index", "timestamp_seconds", "shard", "kind", "width", "height", "original_width", "original_height",
                 "image_format", "file_size", "image_sha256", "pixel_sha256", "phash", "dhash", "visual_style", "quality_score", "caption_score",
                 "caption_words", "caption_flags", "caption_model", "caption_status", "training_prompt", "source_name", "license",
                 "img_offset", "img_size", "json_offset", "json_size", "review_reason"]

def _member_cost(n: int) -> int:
    return TAR_BLOCK + ((n + TAR_BLOCK - 1) // TAR_BLOCK) * TAR_BLOCK

@dataclasses.dataclass
class ClosedShard:
    kind: str
    name: str
    local_path: Path
    repo_path: str
    index_repo_path: str
    size: int
    sha256: str
    n_samples: int
    rows: List[Dict[str, Any]]
    videos: Dict[str, int]
    created_at: float
    info: Dict[str, Any] = dataclasses.field(default_factory=dict)

class ShardWriter:
    def __init__(self, kind: str = "train", target_bytes: Optional[int] = None):
        self.kind = kind
        self.target = int((target_bytes if target_bytes is not None else CONFIG.SHARD_TARGET_SIZE_GB * 1e9))
        self._tf: Optional[tarfile.TarFile] = None; self._partial: Optional[Path] = None; self._final: Optional[Path] = None
        self.rows: List[Dict[str, Any]] = []; self.videos: "collections.Counter" = collections.Counter(); self.keys: set = set(); self.est = 0
        self.name = ""

    @property
    def n(self) -> int: return len(self.rows)

    def _open(self) -> None:
        idx = STATE.next_shard_index(self.kind); self.name = shard_basename(idx, self.kind)
        d = SHARD_DIR / "data"; d.mkdir(parents=True, exist_ok=True)
        ensure_disk(d, self.target / 1e9 * 1.15 + 0.5, "الشظية القادمة")          # نتحقق قبل الكتابة (لا نملأ /kaggle/working)
        self._final, self._partial = d / f"{self.name}.tar", d / f"{self.name}.tar.partial"
        self._tf = tarfile.open(self._partial, "w")

    def add(self, s: Sample, record: Dict[str, Any], review_reason: str = "") -> bool:
        """يضيف عينة (jpg + json). يعيد True عندما تبلغ الشظية الحجم المستهدف."""
        sid = s.sample_id
        if sid in self.keys: raise ValueError(f"sample_id مكرر داخل الشظية: {sid}")
        if self._tf is None: self._open()
        jb = json.dumps(record, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
        for ext, data in ((IMG_EXT, s.jpeg), ("json", jb)):
            ti = tarfile.TarInfo(f"{sid}.{ext}"); ti.size = len(data); ti.mtime = 0; ti.mode = 0o644; ti.uid = ti.gid = 0
            self._tf.addfile(ti, io.BytesIO(data))
        self.keys.add(sid); self.videos[s.video_id] += 1; self.est += _member_cost(len(s.jpeg)) + _member_cost(len(jb))
        row = {k: record.get(k) for k in INDEX_COLUMNS if k in record}
        row.update(shard=self.name, kind=self.kind, review_reason=review_reason, caption_flags=list(record.get("caption_flags", [])))
        self.rows.append(row)
        return self.est >= self.target

    def close(self) -> Optional[ClosedShard]:
        if self._tf is None: return None
        self._tf.close(); self._tf = None
        if not self.rows:
            self._partial.unlink(missing_ok=True); return None
        with open(self._partial, "rb") as f: os.fsync(f.fileno())
        os.replace(self._partial, self._final)
        size = self._final.stat().st_size
        cs = ClosedShard(kind=self.kind, name=self.name, local_path=self._final, repo_path=f"data/{self.name}.tar", index_repo_path=f"index/{self.name}.parquet",
                         size=size, sha256=sha256_file(self._final), n_samples=len(self.rows), rows=self.rows, videos=dict(self.videos), created_at=time.time())
        self.rows, self.videos, self.keys, self.est, self._partial, self._final = [], collections.Counter(), set(), 0, None, None
        return cs

# ---------------------------------------------------------------- التحقق من سلامة الشظية
def validate_tar_shard(path: os.PathLike, kind: str = "train", check_images: bool = True, max_errors: int = 20) -> Dict[str, Any]:
    """يقرأ الشظية كاملة ويتحقق من كل شرط في المواصفات. يعيد offsets لكل عينة (تُخزَّن في الفهرس للقراءة العشوائية لاحقًا)."""
    res: Dict[str, Any] = dict(ok=False, n_samples=0, errors=[], offsets={}, bytes=Path(path).stat().st_size if Path(path).exists() else 0)
    err = lambda m: res["errors"].append(m) if len(res["errors"]) < max_errors else None
    groups: Dict[str, Dict[str, Any]] = {}
    try:
        with tarfile.open(path, "r:") as tf:
            for m in tf:
                if not m.isfile(): err(f"عنصر غير ملف: {m.name}"); continue
                base = os.path.basename(m.name); key, _, ext = base.partition(".")
                f = tf.extractfile(m); data = f.read() if f else b""
                if len(data) != m.size: err(f"{m.name}: حجم مقروء {len(data)} ≠ {m.size}"); continue
                g = groups.setdefault(key, {})
                role = "img" if ext in ("jpg", "jpeg", "webp", "png") else ("json" if ext == "json" else None)
                if role is None: err(f"امتداد غير متوقع: {m.name}"); continue
                if role in g: err(f"تكرار {role} للمفتاح {key}"); continue
                g[role] = (m.offset_data, m.size)
                if role == "json":
                    try: g["rec"] = json.loads(data.decode("utf-8"))
                    except (UnicodeDecodeError, json.JSONDecodeError) as e: err(f"{key}: JSON/UTF-8 غير صالح ({type(e).__name__})")
                else:
                    g["sha"] = hashlib.sha256(data).hexdigest()
                    if check_images:
                        try:
                            im = Image.open(io.BytesIO(data)); im.load(); g["dims"] = im.size
                        except Exception as e: err(f"{key}: صورة غير قابلة للفتح ({type(e).__name__})")
    except (tarfile.TarError, OSError, EOFError) as e:
        err(f"TAR غير صالح/مبتور: {type(e).__name__}: {str(e)[:100]}"); return res
    for key, g in groups.items():
        if "img" not in g or "json" not in g: err(f"{key}: ينقص صورة أو JSON"); continue
        rec = g.get("rec")
        if rec is None: continue
        if rec.get("sample_id") != key: err(f"{key}: sample_id في JSON ≠ اسم الملف")
        if kind == "train" and rec.get("caption_status") != "disabled" and not str(rec.get("training_prompt", "")).strip(): err(f"{key}: training_prompt فارغ (لا يجوز في الشظية الرئيسية)")
        if not isinstance(rec.get("frame_index"), int): err(f"{key}: frame_index غير صحيح")
        if rec.get("image_sha256") != g.get("sha"): err(f"{key}: checksum الصورة لا يطابق JSON")
        if check_images and "dims" in g and (rec.get("width"), rec.get("height")) != g["dims"]: err(f"{key}: الأبعاد {g['dims']} ≠ JSON {(rec.get('width'), rec.get('height'))}")
        res["offsets"][key] = dict(img_offset=g["img"][0], img_size=g["img"][1], json_offset=g["json"][0], json_size=g["json"][1])
    res["n_samples"] = len(res["offsets"]); res["ok"] = not res["errors"] and res["n_samples"] > 0
    return res

def random_sample_check(path: os.PathLike, offsets: Dict[str, Dict[str, int]], k: int = 5, seed: Optional[int] = None) -> List[str]:
    """اختبار قراءة عشوائية: يفتح الملف الخام ويقرأ k عينات عبر الإزاحات المخزنة، ويقارن الصورة بـ JSON (checksum + أبعاد + وصف)."""
    errs: List[str] = []; rng = random.Random(seed)
    keys = rng.sample(sorted(offsets), min(k, len(offsets)))
    with open(path, "rb") as f:
        for key in keys:
            o = offsets[key]
            f.seek(o["json_offset"]); rec = json.loads(f.read(o["json_size"]).decode("utf-8"))
            f.seek(o["img_offset"]); img = f.read(o["img_size"])
            if hashlib.sha256(img).hexdigest() != rec.get("image_sha256"): errs.append(f"{key}: checksum (قراءة عشوائية)")
            try:
                im = Image.open(io.BytesIO(img)); im.load()
                if im.size != (rec["width"], rec["height"]): errs.append(f"{key}: أبعاد (قراءة عشوائية)")
            except Exception as e: errs.append(f"{key}: صورة ({type(e).__name__})")
            if not str(rec.get("training_prompt", "")).strip() and rec.get("caption_status") == "ok": errs.append(f"{key}: وصف فارغ")
    return errs

_INDEX_INT_COLS = {"frame_index", "width", "height", "original_width", "original_height", "file_size", "caption_words", "img_offset", "img_size", "json_offset", "json_size"}
_INDEX_FLOAT_COLS = {"timestamp_seconds", "quality_score", "caption_score"}

def _coerce_index_row(r: Dict[str, Any]) -> Dict[str, Any]:
    """أنواع ثابتة لكل عمود (نفسها في كل الشظايا): Arrow صارم (int لا يقبل float بكسر، string لا يقبل int)، واستنتاج النوع يفشل مع الأعمدة الفارغة كليًا."""
    out: Dict[str, Any] = {}
    for c in INDEX_COLUMNS:
        v = r.get(c)
        if v is None: out[c] = None
        elif c in _INDEX_INT_COLS: out[c] = int(v)
        elif c in _INDEX_FLOAT_COLS: out[c] = float(v)
        elif c == "caption_flags": out[c] = [str(x) for x in v]
        else: out[c] = str(v)
    return out

def _index_schema():
    import pyarrow as pa
    S, I, F = pa.string(), pa.int64(), pa.float64()
    return pa.schema([(c, I if c in _INDEX_INT_COLS else F if c in _INDEX_FLOAT_COLS else pa.list_(S) if c == "caption_flags" else S) for c in INDEX_COLUMNS])

def index_bytes(rows: List[Dict[str, Any]], offsets: Dict[str, Dict[str, int]]) -> Tuple[bytes, str]:
    """فهرس الشظية: Parquet (zstd، مخطط صريح ثابت) إن توفر pyarrow ونجحت الكتابة، وإلا JSONL (بديل موثَّق؛ فشل الفهرس لا يوقف التشغيل)."""
    for r in rows: r.update(offsets.get(r["sample_id"], {}))
    full = [_coerce_index_row(r) for r in rows]
    try:
        import pyarrow as pa, pyarrow.parquet as pq
        bio = io.BytesIO(); pq.write_table(pa.Table.from_pylist(full, schema=_index_schema()), bio, compression="zstd"); return bio.getvalue(), "parquet"
    except ImportError:
        log.warning("pyarrow غير متوفر → فهرس JSONL بدل Parquet")
    except Exception as e:
        log.warning("كتابة Parquet فشلت (%s: %s) → فهرس JSONL لهذه الشظية", type(e).__name__, str(e)[:160])
    return "\n".join(json.dumps(r, ensure_ascii=False) for r in full).encode("utf-8"), "jsonl"

def shard_stale_gc() -> int:
    n = 0
    for p in (SHARD_DIR / "data").glob("*.partial"): p.unlink(missing_ok=True); n += 1
    return n

shard_stale_gc()
print("shard writer ready | target:", f"{CONFIG.SHARD_TARGET_SIZE_GB} GB", "| image ext:", IMG_EXT, "| sample_id example:", make_sample_id("vid.01", 12))

### CELL 22 — Hugging Face Upload
> خيط رفع واحد (FIFO ⇒ ترتيب الالتزامات محفوظ). كل مجموعة شظايا: تحقق كامل ← فهرس ← commit ذري واحد (شظايا + فهارس + control) ← تحقق من الوجود والحجم ← حذف المحلي.
> عمليات الرفع Idempotent: نفس المسارات تُستبدل عند الاستئناف فلا تتكرر الشظايا.
> خطأ قاتل (تحقق فاشل / صلاحيات / انتهاء الصبر) ⇒ يتوقف العامل فورًا ولا يُلتزم أي شيء لاحق (كي لا تسبق حالةٌ لقطةً لشظية ناقصة).

In [ ]:
# CELL 22 - Hugging Face Upload
#: خيط رفع واحد (FIFO ⇒ ترتيب الالتزامات محفوظ). كل مجموعة شظايا: تحقق كامل ← فهرس ← commit ذري واحد (شظايا + فهارس + control) ← تحقق من الوجود والحجم ← حذف المحلي.
#: عمليات الرفع Idempotent: نفس المسارات تُستبدل عند الاستئناف فلا تتكرر الشظايا.
#: خطأ قاتل (تحقق فاشل / صلاحيات / انتهاء الصبر) ⇒ يتوقف العامل فورًا ولا يُلتزم أي شيء لاحق (كي لا تسبق حالةٌ لقطةً لشظية ناقصة).
INDEX_EXT = "parquet" if has_module("pyarrow") else "jsonl"
KEEP_LOCAL_SHARDS = False

@dataclasses.dataclass
class UploadJob:
    kind: str                                  # "shard" | "control"
    control: Dict[str, bytes]                  # {control/..: bytes} لقطة متسقة أُخذت لحظة إغلاق المجموعة
    message: str
    shards: List[ClosedShard] = dataclasses.field(default_factory=list)
    on_commit: Optional[Callable[[], None]] = None      # بعد نجاح الالتزام والتحقق (مثلًا: حذف الفيديوهات المنزّلة)
    @property
    def shard(self) -> Optional[ClosedShard]: return self.shards[0] if self.shards else None

def _http_status(e: Optional[BaseException]) -> Optional[int]:
    seen = 0
    while e is not None and seen < 6:
        r = getattr(e, "response", None); st = getattr(r, "status_code", None)
        if isinstance(st, int): return st
        e = e.__cause__ or e.__context__; seen += 1
    return None

class UploadWorker:
    def __init__(self, store: RemoteStore, state: "StateManager"):
        self.store, self.state = store, state
        self.q: "queue.Queue" = queue.Queue(maxsize=1)                 # تغذية راجعة: لا نكوّن أكثر من مجموعة واحدة معلّقة
        self.fatal: Optional[BaseException] = None
        self.stats: Dict[str, Any] = dict(shards_uploaded=0, bytes_uploaded=0, failures=0, last_speed_MBps=0.0, avg_speed_MBps=0.0, upload_seconds=0.0,
                                          control_commits=0, validation_failures=0, commit_retries=0)
        self._idle = threading.Event(); self._idle.set(); self._stop = False
        self.thread = threading.Thread(target=self._run, name="uploader", daemon=True); self.thread.start()

    # ---- واجهة المنتج
    def submit(self, job: UploadJob, block: bool = True) -> bool:
        if self.fatal: raise RuntimeError(f"الرفع متوقف بخطأ قاتل سابق: {self.fatal!r}")
        self._idle.clear()
        while True:
            try:
                self.q.put(job, block=block, timeout=(1.0 if block else 0)); return True
            except queue.Full:
                if not block: return False
                if self.fatal: raise RuntimeError(f"الرفع توقف بخطأ قاتل أثناء الانتظار: {self.fatal!r}")

    def drain(self, timeout: Optional[float] = None) -> bool:
        t0 = time.time()
        while not (self.q.empty() and self._idle.is_set()):
            if self.fatal: raise RuntimeError(f"الرفع فشل: {self.fatal!r}")
            if timeout and time.time() - t0 > timeout: return False
            time.sleep(0.2)
        if self.fatal: raise RuntimeError(f"الرفع فشل: {self.fatal!r}")      # الخطأ القاتل لا يُبتلع حتى لو انتهى الطابور
        return True

    def close(self) -> None:
        self._stop = True
        try: self.q.put_nowait(None)
        except queue.Full: pass

    # ---- العامل
    def _run(self) -> None:
        while True:
            job = self.q.get()
            if job is None: return
            self._idle.clear()
            try:
                if job.kind == "shard": self._process_shards(job)
                else: self._process_control(job)
                if job.on_commit is not None:
                    try: job.on_commit()
                    except Exception as e: log.warning("on_commit فشل (غير قاتل): %r", e)
            except BaseException as e:
                self.fatal = e; self.stats["failures"] += 1
                log.error("فشل رفع/تحقق قاتل: %s: %s", type(e).__name__, str(e)[:300])
                try:                                                   # لا نلتزم أي مهمة لاحقة (لقطتها تفترض نجاح هذه) ⇒ نفرّغ الطابور ونخرج
                    while True: self.q.get_nowait()
                except queue.Empty: pass
                return
            finally:
                if self.q.empty(): self._idle.set()

    def _patient(self, fn: Callable[[], Any], what: str) -> Any:
        """with_retries داخل حلقة صبر طويلة (UPLOAD_PATIENCE_MINUTES): انقطاع الشبكة أو ضغط HF لا يُسقط الجلسة؛ الأخطاء الدائمة (401/403/404) تُرمى فورًا."""
        deadline = time.time() + CONFIG.UPLOAD_PATIENCE_MINUTES * 60
        while True:
            try:
                return with_retries(fn, what, retries=CONFIG.UPLOAD_RETRIES, backoff=CONFIG.DOWNLOAD_BACKOFF)
            except RuntimeError as e:
                st = _http_status(e)
                if st in (401, 403, 404) or time.time() > deadline or self._stop: raise
                self.stats["commit_retries"] += 1
                log.warning("%s: ما زال فاشلًا (%s) — إعادة بعد 20s (الصبر حتى %.0f دقيقة)", what, str(e)[:100], CONFIG.UPLOAD_PATIENCE_MINUTES)
                time.sleep(20)

    def _process_control(self, job: UploadJob) -> None:
        self._patient(lambda: self.store.commit(job.control, job.message), "control commit")
        self.stats["control_commits"] += 1

    def _quarantine(self, sh: ClosedShard) -> Path:
        q = Path(CONFIG.WORK_DIR) / "quarantine"; q.mkdir(exist_ok=True); dst = q / sh.local_path.name
        shutil.move(str(sh.local_path), dst); return dst

    def _process_shards(self, job: UploadJob) -> None:
        adds: Dict[str, Any] = {}; total = 0
        for sh in job.shards:
            v = validate_tar_shard(sh.local_path, sh.kind)                                       # (1) سلامة كاملة
            errs = list(v["errors"])
            if v["ok"]: errs += random_sample_check(sh.local_path, v["offsets"], k=5)           # (2) قراءة عشوائية عبر الإزاحات
            if v["ok"] and v["n_samples"] != sh.n_samples: errs.append(f"عدد العينات {v['n_samples']} ≠ {sh.n_samples}")
            if errs:
                self.stats["validation_failures"] += 1
                dst = self._quarantine(sh)
                raise RuntimeError(f"فشل تحقق الشظية {sh.name} (لن تُرفع كمكتملة؛ حُفظت في {dst}): {errs[:4]}")
            idx_bytes, ext = index_bytes(sh.rows, v["offsets"])
            adds[sh.repo_path] = sh.local_path; adds[f"index/{sh.name}.{ext}"] = idx_bytes; total += sh.size
        adds.update(job.control)
        t0 = time.time()
        for attempt in range(1, 4):
            self._patient(lambda: self.store.commit(adds, job.message), f"commit {'+'.join(s.name for s in job.shards)}")
            bad = [s.name for s in job.shards if not self.store.verify(s.repo_path, s.size, s.sha256)]     # (3) وجود + حجم (+sha256 للـLFS)
            if not bad: break
            log.warning("التحقق بعد الرفع فشل لـ%s (محاولة %d/3) → إعادة", bad, attempt)
        else:
            raise RuntimeError(f"الشظايا {bad} لم تُتحقَّق في المستودع بعد 3 محاولات")
        dt = max(1e-6, time.time() - t0); mb = total / 1e6
        self.stats["shards_uploaded"] += len(job.shards); self.stats["bytes_uploaded"] += total; self.stats["upload_seconds"] += dt
        self.stats["last_speed_MBps"] = mb / dt; self.stats["avg_speed_MBps"] = self.stats["bytes_uploaded"] / 1e6 / self.stats["upload_seconds"]
        for sh in job.shards:
            self.state.mark_committed(sh.info)
            if not KEEP_LOCAL_SHARDS: sh.local_path.unlink(missing_ok=True)                    # (4) لا حذف قبل التأكد
            log.info("✔ %s مرفوعة ومتحقَّق منها: %.0f MB | %d عينة", sh.name, sh.size / 1e6, sh.n_samples)
        log.info("   commit واحد ذري: %.0f MB في %.1fs (%.1f MB/s)", mb, dt, mb / dt)

def make_uploader() -> UploadWorker:
    return UploadWorker(STORE, STATE)

print("uploader ready | store:", STORE.name, "| index format:", INDEX_EXT)

### CELL 23 — Checkpoint Manager
> ثلاث طبقات: (1) ملفات control المحلية الذرية كل CHECKPOINT_EVERY_SECONDS، (2) لقطة control داخل commit كل مجموعة شظايا (متسقة مع البيانات)،
> (3) commit للـ control وحده (محدود التكرار) للتقدّم الذي لا ينتج شظية. كلها تمر عبر نفس خيط الرفع كي لا تسبق لقطة قديمة لقطة أحدث.

In [ ]:
# CELL 23 - Checkpoint Manager
#: ثلاث طبقات: (1) ملفات control المحلية الذرية كل CHECKPOINT_EVERY_SECONDS، (2) لقطة control داخل commit كل مجموعة شظايا (متسقة مع البيانات)،
#: (3) commit للـ control وحده (محدود التكرار) للتقدّم الذي لا ينتج شظية. كلها تمر عبر نفس خيط الرفع كي لا تسبق لقطة قديمة لقطة أحدث.
SUBMIT_LOCK = threading.Lock()          # "أخذ اللقطة + وضعها في الطابور" عملية واحدة غير قابلة للتقطيع (يمنع تبديل ترتيب لقطتين)

def close_group(writers: Sequence["ShardWriter"], uploader: "UploadWorker", reason: str = "", on_commit: Optional[Callable[[], None]] = None,
                before_snapshot: Optional[Callable[[], None]] = None) -> List["ClosedShard"]:
    """يغلق كل الشظايا المفتوحة (train + review) معًا ويسجّلها في STATE ويأخذ لقطة control واحدة ثم يرسلها كمهمة commit ذرية واحدة.
    before_snapshot: يطبّق تقدّم الفيديوهات (cursor/عدّادات) على STATE داخل نفس القفل قبل اللقطة ⇒ لا لقطة تدّعي تقدّمًا لعينات خارج الـcommit.
    بلا شظايا (تقدّم بلا عينات) ⇒ commit للـ control وحده. يتجمّد المنتج إن كان الرفع السابق لم ينتهِ (backpressure)."""
    with SUBMIT_LOCK:
        closed: List[ClosedShard] = []
        for w in writers:
            cs = w.close()
            if cs is not None: closed.append(cs)
        with STATE.lock:
            if before_snapshot is not None: before_snapshot()
            for cs in closed:
                cs.info = dict(name=cs.name, kind=cs.kind, repo_path=cs.repo_path, index_path=f"index/{cs.name}.{INDEX_EXT}", size=cs.size, sha256=cs.sha256,
                               n_samples=cs.n_samples, videos=cs.videos, created_at=cs.created_at, upload_ok=False)
                STATE.register_shard(cs.info)                                           # قبل اللقطة ⇒ completed_shards.json يحتويها
            STATE.recompute()
            snap = {f"{control_prefix()}/{k}": v for k, v in STATE.control_bytes().items()}
            STATE._dirty_since_commit = False; STATE._last_remote_control = time.time()
        STATE.save_local(force=True)
        if closed:
            names = ", ".join(f"{c.name}({c.n_samples})" for c in closed)
            uploader.submit(UploadJob("shard", snap, f"add {names}{(' — ' + reason) if reason else ''}", closed, on_commit))
        else:
            uploader.submit(UploadJob("control", snap, f"control{(' — ' + reason) if reason else ''}", [], on_commit))
        return closed

def close_shard_and_enqueue(writer: "ShardWriter", uploader: "UploadWorker", reason: str = "") -> Optional["ClosedShard"]:
    """واجهة مبسّطة لشظية واحدة (تُستعمل في الاختبارات)."""
    cs = close_group([writer], uploader, reason)
    return cs[0] if cs else None

class CheckpointManager:
    def __init__(self, state: "StateManager", uploader: "UploadWorker"):
        self.state, self.uploader = state, uploader
        self._stop = threading.Event(); self.last_remote = time.time()
        self.thread = threading.Thread(target=self._loop, name="checkpoint", daemon=True); self.thread.start()
    def _loop(self) -> None:
        while not self._stop.wait(5.0):
            try:
                self.state.save_local()                                                 # محلي ذري (يحترم الفاصل الزمني داخليًا)
                if self.state._dirty_since_commit and time.time() - self.last_remote > REMOTE_CONTROL_MIN_INTERVAL and not self.uploader.fatal:
                    if not SUBMIT_LOCK.acquire(blocking=False): continue                # المنتج يغلق مجموعة الآن: لقطته أحدث وأشمل
                    try:
                        with self.state.lock: snap = {f"{control_prefix()}/{k}": v for k, v in self.state.control_bytes().items()}
                        if self.uploader.submit(UploadJob("control", snap, "control checkpoint"), block=False):
                            self.last_remote = time.time(); self.state._dirty_since_commit = False
                    finally: SUBMIT_LOCK.release()
            except Exception as e:
                log.warning("checkpoint loop: %r", e)
    def flush(self, remote: bool = True) -> None:
        self.state.save_local(force=True)
        if remote and not self.uploader.fatal:
            with SUBMIT_LOCK:
                with self.state.lock: snap = {f"{control_prefix()}/{k}": v for k, v in self.state.control_bytes().items()}
                self.uploader.submit(UploadJob("control", snap, "final control state"))
            self.uploader.drain()
    def stop(self) -> None:
        self._stop.set()

print("checkpoint manager ready | local every", CONFIG.CHECKPOINT_EVERY_SECONDS, "s | remote control-only min interval", REMOTE_CONTROL_MIN_INTERVAL, "s")

---
## الجزء 4 — التشغيل (24 → 24F)
لوحة الأداء، المنسّق، Benchmark، Smoke Test، قائمة الجاهزية (24E)، ثم التشغيل الكامل (24F) المحجوب ببوابة Smoke Test.

### CELL 24 — Performance Dashboard
> لوحة حية (تُحدَّث في مكانها) بكل أقسام المواصفات. التقدير الزمني يعتمد على السرعة المقاسة فعليًا لآخر مرحلة (الوصف) خلال آخر 5 دقائق.

In [ ]:
# CELL 24 - Performance Dashboard
#: لوحة حية (تُحدَّث في مكانها) بكل أقسام المواصفات. التقدير الزمني يعتمد على السرعة المقاسة فعليًا لآخر مرحلة (الوصف) خلال آخر 5 دقائق.
import uuid
try:
    import psutil
except ImportError:                                           # pragma: no cover
    psutil = None

class RateMeter:
    def __init__(self, horizon: float = 900.0):
        self.pts: "collections.deque" = collections.deque(); self.total = 0.0; self.horizon = horizon
    def add(self, n: float = 1.0) -> None:
        now = time.time(); self.total += n; self.pts.append((now, self.total))
        while self.pts and now - self.pts[0][0] > self.horizon: self.pts.popleft()
    def rate(self, window: float = 60.0) -> float:
        if len(self.pts) < 2: return 0.0
        now = time.time(); cut = now - window
        old = next((p for p in self.pts if p[0] >= cut), self.pts[0])
        dt = now - old[0]
        return (self.total - old[1]) / dt if dt > 1e-6 else 0.0

class LiveStats:
    METERS = ("src_frames", "selected_frames", "inspected", "accepted", "captioned")
    def __init__(self):
        self.lock = threading.Lock(); self.c: "collections.Counter" = collections.Counter(); self.g: Dict[str, Any] = {}
        self.t0 = time.time(); self.m = {k: RateMeter() for k in self.METERS}
    def inc(self, k: str, n: float = 1) -> None:
        with self.lock:
            self.c[k] += n
            if k in self.m: self.m[k].add(n)
    def set(self, k: str, v: Any) -> None:
        with self.lock: self.g[k] = v
    def get(self, k: str, default: Any = 0) -> Any:
        with self.lock: return self.c.get(k, self.g.get(k, default))
    def rate(self, k: str, window: float = 60.0) -> float:
        return self.m[k].rate(window) if k in self.m else 0.0

class HardwareMonitor:
    def __init__(self): self._t = 0.0; self._gpu: List[Dict[str, float]] = []
    def gpus(self) -> List[Dict[str, float]]:
        if time.time() - self._t < 4.0: return self._gpu
        self._t = time.time(); g: List[Dict[str, float]] = []
        rc, out, _ = run_cmd(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"], timeout=5)
        if rc == 0:
            for line in out.strip().splitlines():
                try: u, mu, mt = (float(x) for x in line.split(",")); g.append(dict(util=u, mem_used_gb=mu / 1024, mem_total_gb=mt / 1024))
                except ValueError: pass
        self._gpu = g; return g
    def host(self) -> Dict[str, float]:
        if psutil is None: return dict(cpu=0.0, ram_pct=0.0, ram_used_gb=0.0)
        vm = psutil.virtual_memory(); return dict(cpu=psutil.cpu_percent(interval=None), ram_pct=vm.percent, ram_used_gb=(vm.total - vm.available) / 1e9)

HW = HardwareMonitor()

def _bar(frac: float, width: int = 24) -> str:
    frac = max(0.0, min(1.0, frac)); n = int(round(frac * width)); return "█" * n + "░" * (width - n)

def estimate_eta(live: LiveStats, pending_videos: int, current_backlog: int) -> Tuple[Optional[float], str]:
    """ETA = (صور متبقية مقدّرة) / (سرعة الوصف المقاسة آخر 5 دقائق). الصور المتبقية = فيديوهات متبقية × متوسط المقبول/فيديو (من فيديوهات هذه الجلسة) + المتراكم."""
    key = "captioned" if CONFIG.ENABLE_CAPTIONING else "accepted"
    rate = live.rate(key, 300.0)
    done_vids = live.get("videos_finished", 0)
    if rate <= 0 or done_vids <= 0: return None, "لا بيانات سرعة كافية بعد"
    avg = live.get("accepted", 0) / done_vids
    remaining = pending_videos * avg + current_backlog
    return remaining / rate, f"{remaining:,.0f} صورة متبقية مقدّرة ÷ {rate:.2f} صورة/ث (آخر 5 دقائق، مرحلة {'الوصف' if CONFIG.ENABLE_CAPTIONING else 'القبول'})"

def render_dashboard(live: LiveStats, pool: Optional["CaptionPool"], uploader: Optional["UploadWorker"], writer_info: Dict[str, Any]) -> str:
    P = STATE.recompute(); el = time.time() - live.t0; pend = len(STATE.pending_videos())
    inspected, accepted = live.get("inspected"), live.get("accepted"); rejected = inspected - accepted
    rb = {k[len("rej_"):]: int(v) for k, v in live.c.items() if k.startswith("rej_")}
    dup = rb.get("exact_duplicate", 0) + rb.get("near_duplicate", 0)
    invalid = sum(rb.get(k, 0) for k in ("invalid_image", "low_resolution", "corrupt_frame", "uniform_frame", "encode_failed"))
    m = pool.metrics() if pool else {}
    sel, cand_total = P.get("selected_frames", 0), P.get("total_candidate_frames", 0)        # من STATE: مجموع الفيديوهات التي بدأت (خطط حتمية)
    eta, eta_note = estimate_eta(live, max(0, pend - (1 if P.get("current_video") else 0)), int(live.get("backlog", 0)))
    g = HW.gpus(); h = HW.host(); du = shutil.disk_usage(CONFIG.WORK_DIR)
    up = uploader.stats if uploader else {}
    L: List[str] = []
    A = L.append
    A("═" * 78); A(f" VIDEO → IMAGE DATASET  |  {time.strftime('%H:%M:%S')}  |  session {P.get('sessions', 0) + 1}"); A("═" * 78)
    tv = max(1, P.get("total_videos", 0)); dv = P.get("completed_videos", 0)
    A(f" VIDEO PROCESSING   [{_bar(dv / tv)}] {dv}/{tv}")
    wu = writer_info.get("done_uncommitted", 0)
    A(f"   total {tv} | completed {dv}" + (f" (+{wu} بانتظار الالتزام)" if wu else "") + f" | remaining {pend} | failed {P.get('failed_videos', 0)} | "
      f"skipped {sum(v['state']=='skipped' for v in STATE.videos.values())}")
    A(f"   current: {P.get('current_video')} | stage: {P.get('current_stage')} | decoder: {EXTRACT_STATS.get('backend')} (fallbacks {EXTRACT_STATS.get('fallbacks')})")
    A(" FRAME PROCESSING")
    A(f"   candidate frames {int(cand_total):,} | selected {int(sel):,} | sampling ratio {sel / cand_total if cand_total else 0:.4f} (target {CONFIG.SAMPLING_RATIO})")
    A(f"   decode progress {live.rate('src_frames', 30):.0f} source-frames/s | extraction {live.rate('selected_frames', 30):.1f} selected-frames/s | this session: selected {int(live.get('selected_frames')):,}")
    A(" IMAGE QUALITY")
    A(f"   inspected {int(inspected):,} | accepted {int(accepted):,} | rejected {int(rejected):,} | blur {rb.get('blur', 0)} | black {rb.get('black_frame', 0)} | "
      f"duplicates {dup} (exact {rb.get('exact_duplicate', 0)}, near {rb.get('near_duplicate', 0)}) | invalid {invalid}")
    A(" CAPTIONING")
    backlog = int(live.get("backlog", 0))
    A(f"   captioned {int(live.get('captioned')):,} | remaining (in pipeline) {backlog} | {live.rate('captioned', 60):.2f} captions/s | avg latency {m.get('avg_latency_s', 0):.2f}s/img")
    A(f"   batch size {m.get('batch_size')} | model {m.get('model', CONFIG.MODEL_ID if CONFIG.ENABLE_CAPTIONING else 'disabled')} ({m.get('engine', '-')}/{m.get('precision', '-')})")
    A(f"   failures {int(live.get('caption_failures'))} | retries {m.get('retries', 0)} | OOM recoveries {m.get('oom', 0)} | sent to review {int(live.get('review'))}")
    A(" HARDWARE")
    for i, gg in enumerate(g): A(f"   GPU{i}: util {gg['util']:.0f}% | mem {gg['mem_used_gb']:.1f}/{gg['mem_total_gb']:.1f} GB")
    if not g: A("   GPU: n/a")
    A(f"   CPU {h['cpu']:.0f}% | RAM {h['ram_pct']:.0f}% ({h['ram_used_gb']:.1f} GB) | disk used {(du.total - du.free) / 1e9:.1f} GB | free {du.free / 1e9:.1f} GB")
    A(" UPLOAD")
    A(f"   shard groups closed {writer_info.get('closed', 0)} | shards committed (total) {len(STATE.shards)} | uploaded this session {up.get('shards_uploaded', 0)} | open samples {writer_info.get('open_samples', 0)} | "
      f"{up.get('last_speed_MBps', 0):.1f} MB/s (avg {up.get('avg_speed_MBps', 0):.1f}) | "
      f"uploaded {up.get('bytes_uploaded', 0) / 1e9:.2f} GB | failures {up.get('failures', 0)} | store: {STORE.name}")
    A(" TIME")
    thr = (live.get("captioned") if CONFIG.ENABLE_CAPTIONING else accepted) / max(el, 1e-6)
    A(f"   elapsed {human_time(el)} | avg throughput {thr:.2f} img/s | ETA {human_time(eta) if eta else 'n/a'}")
    if eta is not None: A(f"   ({eta_note})")
    A("═" * 78)
    return "\n".join(L)

class DashboardRenderer:
    """معرّف عرض فريد لكل نسخة ⇒ كل تشغيل (خلية) يرسم لوحته في مخرجاته هو، لا في مخرجات خلية سابقة."""
    def __init__(self): self._shown = False; self._last = 0.0; self.display_id = "dashboard-" + uuid.uuid4().hex[:8]
    def update(self, text: str, force: bool = False) -> None:
        if not force and time.time() - self._last < CONFIG.DASHBOARD_REFRESH_SECONDS: return
        self._last = time.time()
        if HAVE_IPY:
            try:
                htm = HTML(f"<pre style='font-size:12px;line-height:1.25'>{_html.escape(text)}</pre>")
                if not self._shown: display(htm, display_id=self.display_id); self._shown = True
                else: update_display(htm, display_id=self.display_id)
                return
            except Exception: pass
        print(text, flush=True)

DASH = DashboardRenderer()
print("dashboard ready")

### CELL 24B — Pipeline Orchestrator
> خط أنابيب متدفّق بمراحل متوازية موصولة بطوابير محدودة (backpressure من الرفع/الوصف حتى الفك):
> [تنزيل مسبق] → [المنتج: ffprobe + خطة + فك] → analysis_q → [الحاسم: فحص/تكرار بالترتيب + ترميز] → chunk_q → [الكاتب: وصف + شظايا + رفع]
> ضمانات الاستئناف: الكاتب وحده يطبّق تقدّم الفيديوهات (cursor/عدّادات/done) على STATE لحظة إغلاق مجموعة الشظايا، وفي نفس الـcommit الذري،
> لذلك لا توجد لقطة control تدّعي تقدّمًا لعينات غير موجودة في المستودع. توقف في منتصف فيديو آمن: يُستأنف من الـcursor.

In [ ]:
# CELL 24B - Pipeline Orchestrator
#: خط أنابيب متدفّق بمراحل متوازية موصولة بطوابير محدودة (backpressure من الرفع/الوصف حتى الفك):
#:   [تنزيل مسبق] → [المنتج: ffprobe + خطة + فك] → analysis_q → [الحاسم: فحص/تكرار بالترتيب + ترميز] → chunk_q → [الكاتب: وصف + شظايا + رفع]
#: ضمانات الاستئناف: الكاتب وحده يطبّق تقدّم الفيديوهات (cursor/عدّادات/done) على STATE لحظة إغلاق مجموعة الشظايا، وفي نفس الـcommit الذري،
#: لذلك لا توجد لقطة control تدّعي تقدّمًا لعينات غير موجودة في المستودع. توقف في منتصف فيديو آمن: يُستأنف من الـcursor.
from concurrent.futures import TimeoutError as FutTimeout

PIPELINE_DIRTY = False            # True أثناء التشغيل وبعد أي فشل/إيقاف غير سليم: الاستدعاء التالي يعيد تحميل الحالة من المخزن البعيد
CAPTION_POOL: Optional["CaptionPool"] = None
LAST_PIPELINE: Optional["Pipeline"] = None
PREV_UPLOADER: Optional["UploadWorker"] = None

class PipelineError(RuntimeError):
    pass
class _Aborted(Exception):
    pass

def get_caption_pool(run_sanity: bool = True) -> Optional["CaptionPool"]:
    """يحمّل Qwen مرة واحدة ويعيد استعماله بين الاستدعاءات (تحميل 8B مكلف)."""
    global CAPTION_POOL
    if not CONFIG.ENABLE_CAPTIONING: return None
    if CAPTION_POOL is None: CAPTION_POOL = load_caption_pool(run_sanity=run_sanity)
    return CAPTION_POOL

def release_caption_pool() -> None:
    global CAPTION_POOL
    if CAPTION_POOL is not None:
        try: CAPTION_POOL.close()
        except Exception as e: log.warning("إغلاق مجمّع الوصف: %r", e)
        CAPTION_POOL = None
        try:
            import torch
            if torch.cuda.is_available(): gc.collect(); torch.cuda.empty_cache()
        except Exception: pass

def _attach_log_file() -> None:
    p = Path(CONFIG.WORK_DIR) / "pipeline.log"
    if any(isinstance(h, logging.FileHandler) and getattr(h, "baseFilename", "") == str(p.resolve()) for h in log.handlers): return
    fh = logging.FileHandler(p, encoding="utf-8"); fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(threadName)s | %(message)s"))
    fh.addFilter(_RedactFilter()); log.addHandler(fh)

class ByteGate:
    """ميزانية RAM لإطارات RGB الحيّة (من الفك حتى الترميز/الرفض). تمنع تضخم الذاكرة عندما يكون الوصف أبطأ من الفك."""
    def __init__(self, limit: int):
        self.limit, self.used = int(limit), 0; self.cv = threading.Condition()
    def acquire(self, n: int, abort: threading.Event) -> bool:
        with self.cv:
            while self.used > 0 and self.used + n > self.limit:           # إطار واحد دائمًا يمر ولو تجاوز الحد (تجنّب الجمود)
                if abort.is_set(): return False
                self.cv.wait(0.5)
            self.used += n; return True
    def release(self, n: int) -> None:
        with self.cv:
            self.used = max(0, self.used - n); self.cv.notify_all()

@dataclasses.dataclass
class VDelta:
    """تقدّم فيديو واحد منذ آخر إغلاق (لا يُكتب في STATE إلا لحظة الإغلاق)."""
    accepted: int = 0
    rejected: int = 0
    captioned: int = 0
    caption_failures: int = 0
    review: int = 0
    rej: "collections.Counter" = dataclasses.field(default_factory=collections.Counter)
    cursor: int = -1


class Pipeline:
    def __init__(self, pool: Optional["CaptionPool"], uploader: "UploadWorker", *, max_videos: int = 0, max_accepted: int = 0,
                 deadline: Optional[float] = None, only_videos: Optional[Sequence[str]] = None, show_ui: bool = True):
        self.pool, self.uploader = pool, uploader
        self.max_videos, self.max_accepted, self.deadline = int(max_videos or 0), int(max_accepted or 0), deadline
        self.only_videos = set(only_videos) if only_videos else None
        self.show_ui = show_ui
        self.live = LiveStats(); self.dash = DashboardRenderer(); PREVIEW.new_run()
        rp = RESOURCE_PLAN
        self.qc_pool = ThreadPoolExecutor(rp["qc_threads"], thread_name_prefix="qc")
        self.enc_pool = ThreadPoolExecutor(max(2, rp["qc_threads"] // 2), thread_name_prefix="enc")
        self.dl_pool = ThreadPoolExecutor(1, thread_name_prefix="dl")
        self.analysis_q: queue.Queue = queue.Queue(maxsize=max(8, rp["decode_queue_frames"] * 2))
        self.chunk_q: queue.Queue = queue.Queue(maxsize=max(1, rp["ready_queue_chunks"]) + (len(pool.engines) - 1 if pool is not None else 0))
        self.chunk_images = int(rp["chunk_images"])
        gb = CONFIG.DECODE_BYTE_BUDGET_GB or min(3.0, 0.25 * float(ENV.get("ram_available_gb", 8.0) or 8.0))
        self.mem = ByteGate(int(max(0.3, gb) * 1e9))
        self.stop_requested, self.abort, self.force_close = threading.Event(), threading.Event(), threading.Event()
        self.producer_done, self.decider_done, self.writer_done = threading.Event(), threading.Event(), threading.Event()
        self.stop_reason = ""; self.fatal: Optional[Tuple[str, BaseException]] = None
        self.abort_videos: set = set(); self.enc_fail: "collections.Counter" = collections.Counter(); self.unexpected = 0
        self.preview_q: "collections.deque" = collections.deque(maxlen=3)
        self.w_train, self.w_review = ShardWriter("train"), ShardWriter("review")      # الكاتب
        self.deltas: Dict[str, VDelta] = {}; self.vsess: Dict[str, VDelta] = {}; self.done_pending: List[str] = []; self.release_pending: List[Dict[str, Any]] = []
        self.group_t0: Optional[float] = None; self.closed_groups = 0; self.n_acc_decided = 0; self.n_written = 0; self.n_cap_skipped = 0
        self.session_videos: set = set(); self.threads: List[threading.Thread] = []; self.t0 = time.time()
        self.precision = (pool.plan.get("precision") if pool is not None else "") or ""

    # ------------------------------------------------------------------ أدوات مشتركة
    def request_stop(self, reason: str) -> None:
        if not self.stop_requested.is_set():
            self.stop_reason = reason; self.stop_requested.set(); log.warning("إيقاف آمن مطلوب (%s): إكمال ما في الطوابير ثم رفع آخر شظية", reason)

    def _stopping(self) -> bool:
        if self.stop_requested.is_set() or self.abort.is_set(): return True
        if self.deadline and time.time() > self.deadline: self.request_stop("MAX_RUNTIME_HOURS"); return True
        if (Path(CONFIG.WORK_DIR) / "STOP").exists(): self.request_stop("ملف STOP"); return True      # إيقاف يدوي من طرفية/Terminal
        return False

    def _fatal(self, stage: str, exc: BaseException) -> None:
        if self.fatal is None:
            self.fatal = (stage, exc)
            log.error("خطأ قاتل في مرحلة [%s]: %r\n%s", stage, exc, "".join(traceback.format_exception(type(exc), exc, exc.__traceback__))[-2500:])
        self.abort.set()

    def _put(self, q: "queue.Queue", item: Any) -> bool:
        while not self.abort.is_set():
            try: q.put(item, timeout=0.5); return True
            except queue.Full: continue
        return False

    # ------------------------------------------------------------------ المنتج: تنزيل ← تحليل ← فك
    def _select_videos(self) -> List[Dict[str, Any]]:
        todo = [v for v in STATE.pending_videos() if self.only_videos is None or v["video_id"] in self.only_videos]
        return todo[:self.max_videos] if self.max_videos else todo

    @staticmethod
    def _predict_skip(v: Dict[str, Any]) -> Optional[str]:
        exp = v.get("expected_duration")
        if CONFIG.ENFORCE_DURATION_LIMITS and exp and not (CONFIG.MIN_VIDEO_DURATION * 0.9 <= float(exp) <= CONFIG.MAX_VIDEO_DURATION * 1.1):
            return f"duration: expected_duration {float(exp):.0f}s خارج [{CONFIG.MIN_VIDEO_DURATION:.0f},{CONFIG.MAX_VIDEO_DURATION:.0f}] (تخطٍّ قبل التنزيل)"
        return None

    def _wait_for_disk(self) -> None:
        need, t0 = CONFIG.MIN_FREE_DISK_GB * 1.2, time.time()
        while free_disk_gb(VIDEO_DIR) < need:
            if self.abort.is_set(): raise DownloadError("أُلغي")
            self.force_close.set()                                          # اطلب من الكاتب إغلاق/رفع ما عنده كي تُحذف الفيديوهات المنتظرة
            if time.time() - t0 > CONFIG.DISK_WAIT_MINUTES * 60:
                raise DownloadError(f"مساحة القرص غير كافية بعد انتظار {CONFIG.DISK_WAIT_MINUTES:.0f} دقيقة ({free_disk_gb(VIDEO_DIR):.1f}GB متاح)")
            time.sleep(3)

    def _download_task(self, v: Dict[str, Any]) -> Path:
        vid, url = v["video_id"], v["url"]
        if str(url).startswith(("http://", "https://")): self._wait_for_disk()
        STATE.update_video(vid, state="downloading", download_status="running"); t0 = time.time()
        path = DOWNLOADER.ensure({"video_id": vid, "url": url}, v.get("sha256"), v.get("expected_size"))
        STATE.update_video(vid, state="downloaded", download_status="done"); self.live.inc("download_seconds", time.time() - t0)
        return path

    def _prefetch(self, todo: List[Dict[str, Any]], i: int, futs: Dict[str, Future]) -> None:
        for j in range(i, min(len(todo), i + 1 + max(0, CONFIG.PREFETCH_VIDEOS))):
            w = todo[j]
            if w["video_id"] in futs or self._predict_skip(w): continue
            futs[w["video_id"]] = self.dl_pool.submit(self._download_task, w)

    def _producer(self) -> None:
        futs: Dict[str, Future] = {}
        try:
            todo = self._select_videos(); self.live.set("todo", len(todo))
            log.info("المنتج: %d فيديو للمعالجة في هذه الجلسة", len(todo))
            for i, v in enumerate(todo):
                if self._stopping(): break
                vid = v["video_id"]; rec = {"video_id": vid, "url": v["url"]}
                self.session_videos.add(vid)
                try:
                    skip = self._predict_skip(v)
                    if skip: self._put(self.analysis_q, ("video_skipped", vid, skip, rec)); continue
                    self._prefetch(todo, i, futs)
                    self._process_video(v, rec, futs)
                except (KeyboardInterrupt, SystemExit): raise
                except Exception as e:                                     # خطأ غير متوقع في فيديو واحد لا يُسقط الباقي (حتى حدّ معقول)
                    self.unexpected += 1
                    log.error("خطأ غير متوقع في %s: %r\n%s", vid, e, traceback.format_exc()[-1500:])
                    self._put(self.analysis_q, ("video_failed", vid, f"unexpected: {type(e).__name__}: {str(e)[:200]}", False, rec))
                    if self.unexpected > CONFIG.MAX_UNEXPECTED_VIDEO_ERRORS: raise
        except BaseException as e:
            self._fatal("producer", e)
        finally:
            self.dl_pool.shutdown(wait=False, cancel_futures=True)
            self._put(self.analysis_q, ("producer_done",)); self.producer_done.set()
            STATE.set_stage("idle", None)

    def _process_video(self, v: Dict[str, Any], rec: Dict[str, Any], futs: Dict[str, Future]) -> None:
        vid = v["video_id"]; put = lambda it: self._put(self.analysis_q, it)
        STATE.set_stage("downloading", vid)
        try:
            path = futs.pop(vid).result()
        except PermanentDownloadError as e:
            put(("video_failed", vid, f"download: {e}", True, rec)); return
        except Exception as e:
            put(("video_failed", vid, f"download: {type(e).__name__}: {str(e)[:200]}", False, rec)); return
        STATE.set_stage("analyzing", vid)
        try:
            meta = analyze_video(str(path), vid)
        except VideoAnalysisError as e:
            put(("video_failed", vid, f"analysis: {e}", False, rec)); return
        skip = check_video_limits(meta)
        if skip: put(("video_skipped", vid, skip, rec)); return
        plan = build_plan(meta)
        cursor = int(v.get("cursor", -1))
        STATE.update_video(vid, state="processing", decode_status="running", duration=meta.duration, width=meta.width, height=meta.height, fps_avg=meta.avg_fps,
                           fps_mode="vfr" if meta.is_vfr else "cfr", codec=meta.codec, candidate_frames=int(len(plan.cand)), selected_frames=int(len(plan.sel)),
                           sampling_ratio_actual=plan.ratio_actual)
        for w in meta.warnings: log.warning("%s: %s", vid, w)
        if len(plan.sel) == 0: log.warning("%s: الخطة لا تحتوي إطارات مختارة (ratio=%s, مرشّحة=%d)", vid, CONFIG.SAMPLING_RATIO, len(plan.cand))
        if not put(("video_start", vid, rec, meta.summary())): return
        stopped = False
        if cursor >= 0 and not bool((plan.sel > cursor).any()):
            log.info("%s: لا إطارات بعد المؤشر %d (اكتمل سابقًا) → إنهاء مباشر بلا فك", vid, cursor)
        else:
            STATE.set_stage("decoding", vid); last = max(cursor, 0)
            gen = extract_frames(meta, plan, cursor=cursor, chain=decoder_chain(meta))
            try:
                for fr in gen:
                    if not fr.seed_only and (self._stopping() or vid in self.abort_videos): stopped = True; break
                    if not self.mem.acquire(int(fr.rgb.nbytes), self.abort): stopped = True; break
                    self.live.inc("src_frames", max(0, fr.frame_index - last)); last = max(last, fr.frame_index)
                    if not fr.seed_only: self.live.inc("selected_frames")
                    if not put(("frame", vid, fr, self.qc_pool.submit(analyze_frame, fr))): stopped = True; break
            except FrameExtractionError as e:
                put(("video_failed", vid, f"decode: {str(e)[:300]}", False, rec)); return
            finally:
                gen.close()
        if vid in self.abort_videos: put(("video_failed", vid, f"أكثر من {CONFIG.MAX_JPEG_FAILS_PER_VIDEO} فشل ترميز JPEG", False, rec)); return
        put(("video_paused" if stopped else "video_end", vid, rec))

    # ------------------------------------------------------------------ الحاسم: فحص/تكرار بالترتيب + ترميز + تجميع دفعات
    def _encode(self, fr: "RawFrame", fa: "FrameAnalysis") -> "Sample":
        try:
            jb, w, h = prepare_stored_image(fr.rgb)
            mi = prepare_model_input(fr.rgb, QWEN_FACTOR) if CONFIG.ENABLE_CAPTIONING else None
            return Sample(fr.video_id, fa.frame_index, fa.ts, jb, w, h, fa.width, fa.height, fa.pixel_sha256 or pixel_sha256(fr.rgb), fa.phash, fa.dhash,
                          fa.metrics, backend=fr.backend, model_input=mi)
        finally:
            self.mem.release(int(fr.rgb.nbytes))

    def _submit_chunk(self, batch: List[tuple]) -> None:
        resolved: List[tuple] = []; items: List[CapItem] = []
        for ev in batch:
            if ev[0] != "accept": resolved.append(ev); continue
            _, vid, idx, efut = ev
            try: s = efut.result()
            except Exception as e:
                self.enc_fail[vid] += 1; self.live.inc("encode_failures")
                if self.enc_fail[vid] == 1 or self.enc_fail[vid] % 10 == 0: log.warning("%s: فشل ترميز JPEG للإطار %s: %r", vid, idx, e)
                if self.enc_fail[vid] > CONFIG.MAX_JPEG_FAILS_PER_VIDEO: self.abort_videos.add(vid)
                self.n_acc_decided -= 1
                resolved.append(("reject", vid, idx, REJECT_ENCODE, True)); continue                  # True: كانت محسوبة "مقبولة" في العدّاد الحي
            resolved.append(("accept", vid, idx, s))
            if s.model_input is not None: items.append(CapItem(s.sample_id, s.model_input))
        fut = self.pool.submit(items) if (items and self.pool is not None) else None
        self._put(self.chunk_q, (resolved, fut))

    def _decider(self) -> None:
        decider: Optional[VideoDecider] = None; events: List[tuple] = []; n_acc = 0; t_first = 0.0
        def flush() -> None:
            nonlocal events, n_acc
            if events: batch, events, n_acc = events, [], 0; self._submit_chunk(batch)
        try:
            while not self.abort.is_set():
                try: item = self.analysis_q.get(timeout=0.5)
                except queue.Empty:
                    if events and time.time() - t_first > CONFIG.CHUNK_LINGER_SECONDS: flush()
                    continue
                kind = item[0]
                if kind == "producer_done": flush(); break
                if not events: t_first = time.time()
                if kind == "frame":
                    _, vid, fr, fut = item
                    try: fa = fut.result()
                    except Exception as e:
                        self.live.inc("analysis_errors")
                        if self.live.get("analysis_errors") <= 3: log.warning("%s: فشل تحليل الإطار %s: %r", vid, fr.frame_index, e)
                        fa = FrameAnalysis(fr.frame_index, fr.ts, int(fr.rgb.shape[1]), int(fr.rgb.shape[0]), REJECT_INVALID, {}, None, 0, 0, None, fr.seed_only)
                    reason = decider.decide(fa) if decider is not None else REJECT_INVALID
                    if reason == "seed": self.mem.release(int(fr.rgb.nbytes)); continue
                    self.live.inc("inspected")
                    if reason is None:
                        self.live.inc("accepted"); self.n_acc_decided += 1; n_acc += 1
                        events.append(("accept", vid, fa.frame_index, self.enc_pool.submit(self._encode, fr, fa)))
                        if self.max_accepted and self.n_acc_decided >= self.max_accepted: self.request_stop("max_accepted")
                    else:
                        self.mem.release(int(fr.rgb.nbytes)); events.append(("reject", vid, fa.frame_index, reason, False))
                else:
                    if kind == "video_start": decider = VideoDecider(item[1], CROSS_MIH)
                    events.append(item)
                if n_acc >= self.chunk_images or len(events) >= 4 * self.chunk_images or (events and time.time() - t_first > CONFIG.CHUNK_LINGER_SECONDS): flush()
        except BaseException as e:
            self._fatal("decider", e)
        finally:
            self._put(self.chunk_q, None); self.decider_done.set()

    # ------------------------------------------------------------------ الكاتب: وصف ← شظايا ← رفع
    def _note_progress(self) -> None:
        if self.group_t0 is None: self.group_t0 = time.time()

    def _apply_deltas(self) -> None:
        """يُستدعى داخل STATE.lock من close_group قبل اللقطة."""
        for vid, d in self.deltas.items():
            v = STATE.videos.get(vid)
            if v is None: continue
            v["accepted"] += d.accepted; v["rejected"] += d.rejected; v["captioned"] += d.captioned
            v["caption_failures"] += d.caption_failures; v["review_flagged"] += d.review
            for k, n in d.rej.items(): v["rejected_by_reason"][k] = v["rejected_by_reason"].get(k, 0) + n
            if d.cursor > v["cursor"]: v["cursor"] = d.cursor
            v["updated_at"] = time.time()
        for vid in self.done_pending:
            v = STATE.videos.get(vid)
            if v is not None and v["state"] not in ("failed", "failed_permanent", "skipped"):
                v.update(state="done", decode_status="done", upload_status="committed", error=None, updated_at=time.time())
        self.deltas = {}; self.done_pending = []; STATE._dirty_since_commit = True

    def _close_group(self, reason: str) -> None:
        if self.group_t0 is None and self.w_train.n == 0 and self.w_review.n == 0 and not self.release_pending and not self.done_pending and not self.deltas: return
        recs, self.release_pending = self.release_pending, []
        def on_commit() -> None:
            for r in recs: DOWNLOADER.release(r)                            # "نزّل ← عالج ← ارفع ← احذف": لا حذف قبل التزام النتائج
        n_tr, n_rv = self.w_train.n, self.w_review.n
        close_group([self.w_train, self.w_review], self.uploader, reason, on_commit=on_commit, before_snapshot=self._apply_deltas)
        self.group_t0 = None; self.closed_groups += 1
        log.info("مجموعة شظايا أُغلقت (%s): train=%d review=%d عينة | في الطابور للرفع", reason, n_tr, n_rv)

    def _ensure_shard_space(self, w: "ShardWriter") -> None:
        need = CONFIG.SHARD_TARGET_SIZE_GB * 1.15 + 0.5
        if w.n > 0 or free_disk_gb(SHARD_DIR) >= need: return
        log.warning("مساحة غير كافية لشظية جديدة (%.1fGB < %.1fGB) → إغلاق/رفع ما هو مفتوح وانتظار تفريغ القرص", free_disk_gb(SHARD_DIR), need)
        self._close_group("disk space"); self.uploader.drain(timeout=CONFIG.DISK_WAIT_MINUTES * 60 + 600)
        if free_disk_gb(SHARD_DIR) < need: raise DownloadError(f"مساحة القرص غير كافية لشظية جديدة: {free_disk_gb(SHARD_DIR):.1f}GB متاح، مطلوب {need:.1f}GB")

    def _route(self, s: "Sample", res: Optional["CapResult"]) -> Tuple["ShardWriter", str]:
        if not CONFIG.ENABLE_CAPTIONING: return self.w_train, ""
        s.caption_precision = self.precision
        if res is None: s.caption_error = "no caption result"; return self.w_review, "caption_missing"
        s.caption_model = res.model
        if res.error: s.caption_error = res.error; return self.w_review, "caption_failed"
        p = res.parsed or {}
        s.caption = p
        if not p.get("training_prompt"): s.caption_error = "empty caption"; return self.w_review, "caption_empty"
        if p.get("review"): return self.w_review, str(p.get("review_reason") or "low_score")
        return self.w_train, ""

    def _bump(self, vid: str, idx: Optional[int] = None, reason: Optional[str] = None, **kw: int) -> None:
        """يزيد عدّادات الفيديو في نسختين: الدلتا (تُصفَّر عند كل إغلاق وتُكتب في STATE) والمجموع التراكمي لهذه الجلسة (للتقارير)."""
        for d in (self.deltas.setdefault(vid, VDelta()), self.vsess.setdefault(vid, VDelta())):
            for k, n in kw.items(): setattr(d, k, getattr(d, k) + n)
            if reason: d.rej[reason] += 1
            if idx is not None and idx > d.cursor: d.cursor = idx

    def _handle_chunk(self, events: List[tuple], results: Dict[str, "CapResult"]) -> None:
        good: List[Dict[str, Any]] = []
        for ev in events:
            k = ev[0]
            if k == "video_start":
                self.deltas.setdefault(ev[1], VDelta()); self.live.set("current_video", ev[1])
            elif k == "reject":
                _, vid, idx, reason, was_counted = ev
                self._bump(vid, idx, reason, rejected=1)
                if was_counted: self.live.inc("accepted", -1)                  # رُفضت عند الترميز بعد أن عُدّت مقبولة في الحاسم
                self.live.inc("rej_" + reason); self._note_progress()
            elif k == "accept":
                _, vid, idx, s = ev
                res = results.get(s.sample_id)
                if CONFIG.ENABLE_CAPTIONING and res is not None and res.parsed and res.parsed.get("reject_reason") == REJECT_CAPTION_BANNED:
                    # وصف يحوي عبارة محظورة بعد إعادة المحاولة ⇒ يُتخطّى الإطار كله (لا صورة ولا وصف ولا سجل مراجعة) ويُحسب كرفض بسبب caption_banned_phrase
                    s.model_input = None; self.n_cap_skipped += 1; self.live.inc("accepted", -1)
                    self._bump(vid, idx, REJECT_CAPTION_BANNED, rejected=1); self.live.inc("rej_" + REJECT_CAPTION_BANNED); self._note_progress()
                    continue
                w, why = self._route(s, res); s.model_input = None
                rec = build_record(s, STATE.videos[vid])
                if why: rec["review_reason"] = why
                self._ensure_shard_space(w)
                full = w.add(s, rec, why)
                self._bump(vid, idx, accepted=1); self.n_written += 1; self._note_progress()
                has_prompt = bool(rec.get("training_prompt"))
                if CONFIG.ENABLE_CAPTIONING:
                    if has_prompt: self._bump(vid, captioned=1); self.live.inc("captioned")
                    else: self._bump(vid, caption_failures=1); self.live.inc("caption_failures")
                if why: self._bump(vid, review=1); self.live.inc("review")
                elif has_prompt: good.append(dict(jpeg=s.jpeg, rec=rec))
                if full: self._close_group("size")
            elif k == "video_end":
                _, vid, rec = ev; self.done_pending.append(vid); self.release_pending.append(rec); self._note_progress()
                d = self.vsess.setdefault(vid, VDelta()); self.live.inc("videos_finished"); self.live.inc("accepted_finished", d.accepted)
                v = STATE.videos[vid]
                log.info("✔ %s: مرشّحة=%s مختارة=%s | هذه الجلسة: مقبولة=%d مرفوضة=%d (%s) موصوفة=%d مراجعة=%d", vid, v["candidate_frames"], v["selected_frames"],
                         d.accepted, d.rejected, dict(d.rej.most_common(3)), d.captioned, d.review)
            elif k == "video_paused":
                self.release_pending.append(ev[2]); self._note_progress()
            elif k == "video_skipped":
                _, vid, reason, rec = ev; STATE.update_video(vid, state="skipped", error=reason, decode_status="skipped")
                self.release_pending.append(rec); self.live.inc("videos_skipped"); self._note_progress(); log.warning("تخطّي %s: %s", vid, reason)
            elif k == "video_failed":
                _, vid, reason, permanent, rec = ev; STATE.mark_failed(vid, reason, permanent)
                self.release_pending.append(rec); self.live.inc("videos_failed"); self._note_progress()
        if CONFIG.ENABLE_RANDOM_SAMPLE_PREVIEW and good:
            self.preview_q.append(random.sample(good, min(max(1, CONFIG.PREVIEW_SAMPLES_PER_BATCH), len(good))))

    def _await(self, fut: Optional[Future]) -> Dict[str, "CapResult"]:
        if fut is None: return {}
        while True:
            try: return {r.sample_id: r for r in fut.result(timeout=1.0)}
            except FutTimeout:
                if self.abort.is_set(): raise _Aborted()

    def _tick(self) -> None:
        if self.group_t0 is None: return
        if CONFIG.SHARD_MAX_AGE_MINUTES and time.time() - self.group_t0 > CONFIG.SHARD_MAX_AGE_MINUTES * 60: self._close_group("age")
        elif self.force_close.is_set(): self.force_close.clear(); self._close_group("disk pressure")

    def _writer(self) -> None:
        try:
            while True:
                try: ch = self.chunk_q.get(timeout=0.5)
                except queue.Empty:
                    if self.abort.is_set(): return
                    self._tick(); continue
                if ch is None: break
                events, fut = ch
                self._handle_chunk(events, self._await(fut)); self._tick()
            self._close_group("final")                                       # آخر شظية (حتى لو لم تبلغ الحجم المستهدف) + تقدّم الفيديوهات
        except _Aborted: pass
        except BaseException as e: self._fatal("writer", e)
        finally: self.writer_done.set()

    # ------------------------------------------------------------------ الواجهة (الخيط الرئيسي فقط: كل العرض هنا كي يصل لمخرجات الخلية)
    def writer_info(self) -> Dict[str, Any]:
        return dict(closed=self.closed_groups, open_samples=self.w_train.n + self.w_review.n, review_open=self.w_review.n,
                    done_uncommitted=len(self.done_pending), backlog=max(0, self.n_acc_decided - self.n_written - self.n_cap_skipped))

    def _ui_tick(self, force: bool = False) -> None:
        if not self.show_ui: return
        while self.preview_q:
            try: picks = self.preview_q.popleft()
            except IndexError: break
            try: PREVIEW.show_from_batch(picks)
            except Exception as e: log.warning("المعاينة: %r", e)
        self.live.set("backlog", self.writer_info()["backlog"])
        try: self.dash.update(render_dashboard(self.live, self.pool, self.uploader, self.writer_info()), force=force)
        except Exception as e: log.warning("اللوحة: %r", e)

    def summary(self) -> Dict[str, Any]:
        c = self.live.c
        return dict(elapsed_s=time.time() - self.t0, videos_started=len(self.session_videos), videos_finished=int(c["videos_finished"]),
                    videos_failed=int(c["videos_failed"]), videos_skipped=int(c["videos_skipped"]), inspected=int(c["inspected"]), accepted=int(c["accepted"]),
                    written=self.n_written, captioned=int(c["captioned"]), review=int(c["review"]), caption_failures=int(c["caption_failures"]),
                    shard_groups=self.closed_groups, stop_reason=self.stop_reason or ("fatal" if self.fatal else "completed"),
                    caption_pool=(self.pool.metrics() if self.pool is not None else None), caption_skipped_banned=self.n_cap_skipped,
                    fatal=(f"{self.fatal[0]}: {self.fatal[1]!r}" if self.fatal else None))

    def run(self) -> Dict[str, Any]:
        for name, fn in (("producer", self._producer), ("decider", self._decider), ("writer", self._writer)):
            t = threading.Thread(target=fn, name=name, daemon=True); t.start(); self.threads.append(t)
        try:
            while not self.writer_done.is_set():
                time.sleep(0.4); self._ui_tick()
                if self.uploader.fatal and self.fatal is None: self._fatal("uploader", self.uploader.fatal)
                if self.abort.is_set(): break
        except KeyboardInterrupt:
            log.warning("⏹ Interrupt → إيقاف آمن (تفريغ الطوابير ثم الرفع). اضغط مرة ثانية للإلغاء الفوري (يُهمل ما لم يُرفع).")
            self.request_stop("KeyboardInterrupt")
            try:
                while not self.writer_done.is_set() and not self.abort.is_set(): time.sleep(0.4); self._ui_tick()
            except KeyboardInterrupt: self._fatal("user", RuntimeError("إلغاء فوري من المستخدم"))
        for t in self.threads: t.join(timeout=60)
        self._ui_tick(force=True)
        for ex in (self.qc_pool, self.enc_pool): ex.shutdown(wait=False, cancel_futures=True)
        return self.summary()


def print_run_plan() -> None:
    P = STATE.progress; pend = STATE.pending_videos()
    done = sum(v["state"] == "done" for v in STATE.videos.values()); sk = sum(v["state"] == "skipped" for v in STATE.videos.values())
    print("=" * 78); print(" RUN PLAN"); print("=" * 78)
    print(f"  videos: total {len(STATE.videos)} | done {done} | skipped {sk} | pending/failed-retry {len(pend)} | resumed: {STATE.resumed}")
    print(f"  shards committed: {len(STATE.shards)} | next shard: {shard_basename(STATE.next_shard_index('train'))} | store: {STORE.name}"
          + (f" ({CONFIG.HF_DATASET_REPO})" if STORE.name == "huggingface" else " (LOCAL — لن يُرفع شيء إلى Hugging Face)"))
    print(f"  sampling: {CONFIG.sampling_method()} ratio={CONFIG.SAMPLING_RATIO} seed={CONFIG.SAMPLING_SEED} target_fps={CONFIG.TARGET_FPS}"
          + (f" | MAX_IMAGES_PER_VIDEO={CONFIG.MAX_IMAGES_PER_VIDEO}" if CONFIG.MAX_IMAGES_PER_VIDEO else ""))
    print(f"  decoder preference: {CONFIG.DECODER_PREFERENCE} | chain(h264): {decoder_chain()} | qc threads: {RESOURCE_PLAN['qc_threads']}")
    print(f"  captioning: {'ON ' + CONFIG.MODEL_ID if CONFIG.ENABLE_CAPTIONING else 'OFF'} | plans: {[(p['precision'], p['layout']) for p in (CAPTION_PLANS or [])] or 'n/a'}")
    est = (STATE.benchmark or {}).get("estimate")
    if est: print(f"  benchmark estimate: {est.get('summary', '')}")
    print("  إيقاف آمن في أي لحظة: زر Interrupt في Kaggle أو إنشاء ملف  " + str(Path(CONFIG.WORK_DIR) / "STOP"))
    print("=" * 78)


SMOKE_RUNNING = False             # يضبطه اختبار الدخان (24D) أثناء تشغيله كي لا تحجبه البوابة نفسها

def _check_gate() -> None:
    """بوابة الأمان: لا تشغيل كامل قبل نجاح اختبار الدخان بنفس العتاد والإعدادات (REQUIRE_SMOKE_TEST). تُطبَّق داخل run_pipeline نفسه كي لا تُتجاوز باستدعاء مباشر."""
    if not CONFIG.REQUIRE_SMOKE_TEST or SMOKE_RUNNING: return
    fn = globals().get("smoke_test_passed")
    if fn is None: return                                       # الخلية 24D غير محمَّلة (اختبارات الوحدة للمنسّق)
    if not STATE.pending_videos(): return                       # لا شيء ليُعالَج: لا معنى لحجب التشغيل (إعادة تشغيل خلية بعد اكتمال كل شيء)
    ok, why = fn()
    if not ok: raise PipelineError(f"بوابة الأمان: {why}. شغّل run_smoke_test() (الخلية 24D/24E) — أو اضبط REQUIRE_SMOKE_TEST=False إن كنت تتحمّل المسؤولية.")

CONTENT_CFG_KEYS = ("SAMPLING_RATIO", "SAMPLING_SEED", "SAMPLING_METHOD", "EXACT_PER_VIDEO_SAMPLING", "TARGET_FPS", "MAX_IMAGES_PER_VIDEO", "MIN_IMAGE_WIDTH", "MIN_IMAGE_HEIGHT",
                    "IMAGE_FORMAT", "IMAGE_QUALITY", "CROSS_VIDEO_DEDUP", "DEDUP_HAMMING_THRESHOLD", "ENABLE_CAPTIONING", "MODEL_ID", "CAPTION_LANGUAGE")

def content_fingerprint() -> Dict[str, Any]:
    """المعلمات التي تغيّر محتوى Dataset (لا سرعة التنفيذ): أخذ العيّنات، الحد الأدنى للدقة، الترميز، إزالة التكرار، الوصف + عتبات QC."""
    d: Dict[str, Any] = {k: getattr(CONFIG, k, None) for k in CONTENT_CFG_KEYS}
    try: d.update({f"qc.{k}": v for k, v in dataclasses.asdict(QC).items()})
    except Exception: pass
    return json.loads(json.dumps(d, default=str))

def note_config_changes(verbose: bool = True) -> List[Dict[str, Any]]:
    """عند كل تشغيل: يقارن معلمات المحتوى الحالية بما سُجِّل مع الحالة (progress['content_cfg']). إن تغيّرت بعد وجود صور مقبولة يُسجَّل الفرق في progress['config_changes']
    (سجلّ مصدر/provenance يُلتزم مع control) ويُحذَّر منه. لا يُمنع التغيير: قد يكون مقصودًا (مثل خفض MAX_IMAGES_PER_VIDEO لما تبقى من الفيديوهات)، لكن Dataset سيحمل خليط معلمات."""
    cur = content_fingerprint()
    with STATE.lock:
        STATE.recompute(); P = STATE.progress; prev = P.get("content_cfg"); hist = P.setdefault("config_changes", [])
        diff = {k: [prev.get(k), cur.get(k)] for k in sorted(set(prev) | set(cur)) if prev.get(k) != cur.get(k)} if isinstance(prev, dict) else {}
        if diff and int(P.get("accepted_images", 0)) > 0:
            hist.append(dict(at=time.time(), session=int(P.get("sessions", 0)), changes=diff, accepted_before=int(P["accepted_images"]), videos_done_before=int(P["completed_videos"])))
            if verbose:
                log.warning("⚠ تغيّرت معلمات المحتوى منذ آخر تشغيل (بعد %d صورة مقبولة في %d فيديو مكتمل): %s — الفيديوهات المكتملة لا تُعاد؛ الباقية تُعالَج بالمعلمات الجديدة فيحمل Dataset خليطًا (السجل في control/progress.json ← config_changes).",
                            P["accepted_images"], P["completed_videos"], " | ".join(f"{k}: {a!r} → {b!r}" for k, (a, b) in diff.items())[:600])
        P["content_cfg"] = cur; STATE._dirty_since_commit = True
        return list(hist)

def run_pipeline(max_videos: int = 0, max_accepted: int = 0, only_videos: Optional[Sequence[str]] = None, max_runtime_hours: Optional[float] = None,
                 show_ui: bool = True) -> Dict[str, Any]:
    """يشغّل المعالجة حتى اكتمال كل الفيديوهات المعلّقة (أو حتى حدّ الوقت/العدد/الإيقاف اليدوي) ثم يفرّغ ويرفع كل شيء. يمكن استدعاؤه مجددًا بأمان."""
    global PIPELINE_DIRTY, LAST_PIPELINE, PREV_UPLOADER
    _check_gate()
    _attach_log_file(); shard_stale_gc()
    if PREV_UPLOADER is not None and PREV_UPLOADER.thread.is_alive():          # رفع الجلسة السابقة ما زال جاريًا (مثلًا بعد Interrupt): لا نبدأ قبل أن ينتهي كي لا تتداخل الالتزامات
        log.warning("انتظار انتهاء رفع الجلسة السابقة قبل البدء...")
        PREV_UPLOADER.thread.join(timeout=CONFIG.UPLOAD_PATIENCE_MINUTES * 60 + 300)
    if PIPELINE_DIRTY:
        log.warning("التشغيل السابق انتهى بشكل غير سليم → إعادة تحميل الحالة من المخزن البعيد (يُهمل أي تقدّم لم يُرفع ويُعاد حسابه بالنتيجة نفسها)")
        STATE.reload_from_remote(MANIFEST_ROWS)
    note_config_changes()
    stop_file = Path(CONFIG.WORK_DIR) / "STOP"
    if stop_file.exists(): stop_file.unlink()
    mrt = CONFIG.MAX_RUNTIME_HOURS if max_runtime_hours is None else max_runtime_hours
    deadline = (time.time() + mrt * 3600 - CONFIG.STOP_MARGIN_MINUTES * 60) if mrt and mrt > 0 else None
    keep = {DOWNLOADER.local_path(dict(video_id=v["video_id"], url=v["url"])).name for v in STATE.pending_videos() if str(v["url"]).startswith("http")}
    DOWNLOADER.sweep_orphans(keep=keep | {k + ".part" for k in keep})
    pool = get_caption_pool()
    uploader = make_uploader(); PREV_UPLOADER = uploader; ckpt = CheckpointManager(STATE, uploader)
    pl = Pipeline(pool, uploader, max_videos=max_videos or CONFIG.MAX_VIDEOS_THIS_RUN, max_accepted=max_accepted, deadline=deadline, only_videos=only_videos, show_ui=show_ui)
    LAST_PIPELINE = pl; PIPELINE_DIRTY = True; STATE.set_stage("running")
    try:
        res = pl.run()
        try:
            uploader.drain()                                                   # كل ما أُغلق يجب أن يُرفع ويُتحقَّق منه
            if pl.fatal is None: ckpt.flush(remote=True)
        except Exception as e:
            if pl.fatal is None: pl._fatal("uploader", e)
            res = pl.summary()
        if pl.fatal is None: PIPELINE_DIRTY = False
    finally:
        ckpt.stop(); uploader.close(); STATE.set_stage("idle", None)
        try: STATE.save_local(force=True)
        except Exception: pass
    if pl.fatal is not None:
        raise PipelineError(f"توقف خط الأنابيب في مرحلة [{pl.fatal[0]}]: {pl.fatal[1]!r} — ما أُلتزم قبل الخطأ آمن في المخزن؛ أعد تشغيل الخلية للاستئناف.") from pl.fatal[1]
    res["uploader"] = dict(uploader.stats)
    log.info("اكتملت الجلسة: %s", {k: v for k, v in res.items() if k != "uploader"})
    return res

print("orchestrator ready | qc threads:", RESOURCE_PLAN["qc_threads"], "| chunk images:", RESOURCE_PLAN["chunk_images"], "| ready chunks:", RESOURCE_PLAN["ready_queue_chunks"])

### CELL 24C — Benchmark Mode
> قياس فعلي على عتادك وعلى فيديوهاتك الحقيقية، ثم ضبط تلقائي آمن وتقدير واقعي للزمن والحجم. كل رقم هنا إما مقيس أو موسوم "assumption".
> يُضبط تلقائيًا (ويُحفظ في control/benchmark_results.json ويُعاد تطبيقه في الجلسات التالية): المفكّك الأسرع الصالح لونيًا، المشفّر، عتبات الضبابية (بحدود آمنة)،
> حجم دفعة Qwen وميزانية الرموز، SDPA مقابل FlashAttention-2، Transformers مقابل vLLM. لا يغيّر أبدًا SAMPLING_RATIO ولا أي إعداد حدّدتَه أنت يدويًا.
> صدق: NVDEC/PyNvVideoCodec/FlashAttention/vLLM/Qwen الحقيقي لا يمكن اختبارها في بيئة التطوير (بلا GPU)؛ منطق الاختيار والتقدير مُختبر بأرقام اصطناعية وبمحرك Mock.

In [ ]:
# CELL 24C - Benchmark Mode
#: قياس فعلي على عتادك وعلى فيديوهاتك الحقيقية، ثم ضبط تلقائي آمن وتقدير واقعي للزمن والحجم. كل رقم هنا إما مقيس أو موسوم "assumption".
#: يُضبط تلقائيًا (ويُحفظ في control/benchmark_results.json ويُعاد تطبيقه في الجلسات التالية): المفكّك الأسرع الصالح لونيًا، المشفّر، عتبات الضبابية (بحدود آمنة)،
#: حجم دفعة Qwen وميزانية الرموز، SDPA مقابل FlashAttention-2، Transformers مقابل vLLM. لا يغيّر أبدًا SAMPLING_RATIO ولا أي إعداد حدّدتَه أنت يدويًا.
#: صدق: NVDEC/PyNvVideoCodec/FlashAttention/vLLM/Qwen الحقيقي لا يمكن اختبارها في بيئة التطوير (بلا GPU)؛ منطق الاختيار والتقدير مُختبر بأرقام اصطناعية وبمحرك Mock.
import contextlib

BENCH_VERSION = 2                                    # 2: بداية الدفعة 16 + سقف VRAM قابل للضبط + 32B bf16 (نتائج الإصدار 1 أو عتاد آخر لا تُعاد استعمالها)
BENCH_COLOR_MAD_MAX = PYNV_MAX_COLOR_MAD            # نفس عتبة الخلية 10 (فرق لوني متوسط بين المفكّك وFFmpeg، مقياس 0-255)
BENCH_VRAM_CAP = 0.85                                # القيمة الافتراضية للدوال النقية؛ التشغيل الفعلي يستعمل CONFIG.BENCH_VRAM_TARGET (0.80: هامش KV-cache/NVDEC وتقلّب أطوال الأوصاف)
_BENCH_LADDER_UP = (16, 20, 24, 28, 32, 40, 48, 56, 64, 80, 96, 128, 160, 192, 256)
_BENCH_CAPTION = ("A wide shot of an outdoor scene during daytime. In the foreground, a figure stands near the left edge facing the camera, while behind them "
                  "a row of buildings with large windows recedes toward the horizon. Soft natural light falls from the upper right, casting gentle shadows "
                  "on the pavement. The color palette is dominated by warm beige and muted blue, with sharp focus across the frame and a slightly elevated "
                  "camera angle. Photographic style, natural color grading, no visible text.")

# ---------------------------------------------------------------- أدوات عامة
def _scrub(x: Any, limit: int = 300) -> str:
    """نص آمن للتخزين: يحجب أي سرّ مسجَّل (HF_TOKEN) ويقصّ الطول."""
    s = str(x)
    for sec in list(_SECRETS):
        if sec: s = s.replace(sec, "***")
    return s[:limit]

def _jsonable(o: Any) -> Any:
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return _jsonable(o.tolist())
    if isinstance(o, np.generic): return _jsonable(o.item())
    if isinstance(o, float) and not math.isfinite(o): return None
    if isinstance(o, Path): return str(o)
    return o

def env_fingerprint() -> str:
    """بصمة العتاد والحزم: نتائج القياس تُعدّ صالحة فقط لنفس البصمة (GPU، إصدارات torch/transformers/vllm/…، ffmpeg، النوى)."""
    parts = dict(gpus=[(g.get("name"), g.get("vram_total_mb"), g.get("compute_cap")) for g in ENV.get("gpus", [])],
                 pk={k: pkg_version(k) for k in ("torch", "transformers", "accelerate", "bitsandbytes", "flash-attn", "vllm", "PyNvVideoCodec", "pyarrow", "Pillow")},
                 ffmpeg=str(ENV.get("ffmpeg")), py=list(sys.version_info[:2]), cores=ENV.get("cpu_cores"))
    return hashlib.sha1(json.dumps(parts, sort_keys=True, default=str).encode()).hexdigest()[:12]

_BENCH_CFG_KEYS = ("MODEL_ID", "CAPTION_PRECISION", "CAPTION_ALLOW_FALLBACK_PLANS", "BENCH_START_BATCH", "BENCH_VRAM_TARGET", "BENCH_STABLE_FRACTION", "CAPTION_MAX_NEW_TOKENS", "MODEL_MIN_PIXELS", "MODEL_MAX_PIXELS", "CAPTION_REPETITION_PENALTY", "IMAGE_FORMAT",
                   "IMAGE_QUALITY", "MAX_STORED_IMAGE_DIMENSION", "JPEG_SUBSAMPLING", "TARGET_FPS", "ENABLE_GPU_DECODING", "ENABLE_CAPTIONING")

def bench_fingerprint() -> str:
    d = {k: getattr(CONFIG, k) for k in _BENCH_CFG_KEYS}
    return hashlib.sha1((env_fingerprint() + json.dumps(d, sort_keys=True, default=str)).encode()).hexdigest()[:12]

@contextlib.contextmanager
def _isolated_dedup():
    """القياس لا يجوز أن يلوّث حالة إزالة التكرار العالمية (CROSS_VIDEO_DEDUP) وإلا رُفضت إطارات الفيديو الأول فعليًا كمكرّرة."""
    ed = EXACT_DEDUPER; saved = (ed._video, ed._global, ed.hits)
    ed._video, ed._global, ed.hits = set(), set(), 0
    try: yield
    finally: ed._video, ed._global, ed.hits = saved

# ---------------------------------------------------------------- الفك
def _bench_decode_one(meta: "VideoMeta", sub: "FramePlan", backend: str, keep: int) -> Tuple[Dict[str, Any], List["RawFrame"]]:
    """يفك الخطة الفرعية بمفكّك واحد بلا أي بديل. يعيد (نتيجة، أول keep إطارًا). startup_s = زمن أول إطار (تشغيل العملية/NVDEC)، والسرعات مقيسة بعده."""
    t0 = time.perf_counter(); t_first: Optional[float] = None; n = 0; first_idx = last_idx = -1; kept: List["RawFrame"] = []
    try:
        for fr in extract_frames(meta, sub, cursor=-1, chain=[backend]):
            if t_first is None: t_first = time.perf_counter() - t0; first_idx = fr.frame_index
            n += 1; last_idx = fr.frame_index
            if len(kept) < keep: kept.append(fr)
        dt = time.perf_counter() - t0
    except Exception as e:
        return dict(ok=False, frames=n, error=_scrub(f"{type(e).__name__}: {e}", 240)), kept
    steady = max(1e-6, dt - (t_first or 0.0))
    ok = n >= max(2, int(0.98 * len(sub.sel)))
    r = dict(ok=ok, frames=n, seconds=dt, startup_s=t_first, selected_fps=((n - 1) / steady if n > 1 else 0.0),
             source_fps=(max(0, last_idx - first_idx) / steady if n > 1 else 0.0))
    if not ok: r["error"] = f"أُنتج {n} من {len(sub.sel)} إطارًا"
    return r, kept

def _color_mad(ref: Dict[int, np.ndarray], frames: Sequence["RawFrame"]) -> Optional[float]:
    ds: List[float] = []
    for fr in frames:
        r = ref.get(fr.frame_index)
        if r is None: continue
        if r.shape != fr.rgb.shape: return float("inf")                       # دوران/أبعاد مختلفة ⇒ غير صالح
        ds.append(float(np.abs(r.astype(np.int16) - fr.rgb.astype(np.int16)).mean()))
    return float(np.mean(ds)) if ds else None

def choose_decoder(agg: Dict[str, Dict[str, Any]], default_order: Sequence[str] = tuple(DECODER_ORDER_DEFAULT)) -> Dict[str, Any]:
    """دالة نقية. agg[backend] = {videos, videos_ok, selected_fps, color_mad, valid}. تختار الأسرع بين الصالحين (نجح على كل الفيديوهات ولونه ≤ العتبة).
    التفضيل يُضبط فقط إن كان الأسرع يتفوق ≥10% على أول صالح في الترتيب الافتراضي (لتجنّب التذبذب بسبب الضجيج). مفكّك بلون مختلف يُعطَّل عالميًا."""
    out: Dict[str, Any] = dict(best=None, preference=None, disable=[], failing=[], gain_vs_default=None, note="")
    out["disable"] = [b for b, a in agg.items() if a.get("color_mad") is not None and a["color_mad"] > BENCH_COLOR_MAD_MAX and b != "ffmpeg_cpu"]
    out["failing"] = [b for b, a in agg.items() if a.get("videos_ok", 0) == 0]
    valid = {b: a for b, a in agg.items() if a.get("valid")}
    if not valid: out["note"] = "لا مفكّك صالح على فيديوهات القياس"; return out
    best = max(valid, key=lambda b: valid[b]["selected_fps"]); out["best"] = best
    first = next((b for b in default_order if b in valid), best)
    gain = valid[best]["selected_fps"] / max(1e-9, valid[first]["selected_fps"]); out["gain_vs_default"] = gain
    if best != first and gain >= 1.10: out["preference"] = best
    out["note"] = (f"{best} أسرع {gain:.2f}× من {first} (الافتراضي)" if best != first else f"الافتراضي ({first}) هو الأسرع الصالح")
    return out

# ---------------------------------------------------------------- Qwen: قياس الدفعات
def _cuda_devices(eng: "CaptionEngine") -> List[int]:
    try: import torch
    except Exception: return []
    if not torch.cuda.is_available(): return []
    if (getattr(eng, "plan", None) or {}).get("layout") == "shard": return list(range(torch.cuda.device_count()))
    return [int(getattr(eng, "gpu_index", 0))]

def _reset_peak(eng: "CaptionEngine") -> None:
    try:
        import torch
        for d in _cuda_devices(eng): torch.cuda.reset_peak_memory_stats(d)
    except Exception: pass

def _vram_peak(eng: "CaptionEngine") -> Tuple[Optional[float], Optional[float]]:
    """(أقصى ذاكرة محجوزة GB، أعلى نسبة من سعة GPU) — بعد آخر reset."""
    try:
        import torch
        devs = _cuda_devices(eng)
        if not devs: return None, None
        pk = [torch.cuda.max_memory_reserved(d) for d in devs]; tot = [torch.cuda.get_device_properties(d).total_memory for d in devs]
        return max(pk) / 1e9, max(p / t for p, t in zip(pk, tot))
    except Exception: return None, None

def _count_tokens(eng: "CaptionEngine", text: str) -> int:
    tok = getattr(getattr(eng, "processor", None), "tokenizer", None)
    if tok is not None:
        try: return len(tok(text, add_special_tokens=False)["input_ids"])
        except Exception: pass
    return int(len(text.split()) * 1.35) + 1

def _time_batch(eng: "CaptionEngine", imgs: List[np.ndarray], bs: int, iters: int = 2) -> Dict[str, Any]:
    """يقيس bs صورة × iters مرة عبر engine.generate مباشرة (بلا تحليل/تحقق)، مع قياس الذروة. يرمي CaptionOOM كما هو."""
    batch = [imgs[i % len(imgs)] for i in range(bs)]
    _reset_peak(eng); t0 = time.perf_counter(); outs: List[str] = []
    for _ in range(iters): outs += eng.generate(batch, CONFIG.CAPTION_MAX_NEW_TOKENS)
    dt = max(1e-6, time.perf_counter() - t0); toks = sum(_count_tokens(eng, o) for o in outs); peak, frac = _vram_peak(eng)
    return dict(batch=bs, seconds=dt, img_per_s=bs * iters / dt, tokens=int(toks), tokens_per_s=toks / dt, peak_gb=peak, vram_frac=frac, oom=False)

def sweep_batches(eng: "CaptionEngine", imgs: List[np.ndarray], max_batch: int = 32, iters: int = 2, start: Optional[int] = None,
                  vram_target: Optional[float] = None) -> List[Dict[str, Any]]:
    """start=None (السلوك القديم): يرفع الدفعة (1,2,4,6,8,12,16,24,32,…) حتى OOM أو ذاكرة > 90% أو لا تحسّن ≥5% مرتين متتاليتين أو بلوغ max_batch.
    start=N (الافتراضي في BenchmarkRun = CONFIG.BENCH_START_BATCH=16): يبدأ من N؛ إن فشلت (OOM أو ذاكرة > vram_target) ينزل N/2…1 حتى تنجح واحدة؛ وإن نجحت يصعد
    (16→20→24→28→32→40→…) حتى: OOM، أو الذاكرة تتجاوز vram_target (يُسجَّل التشغيل ويُستثنى من الاختيار)، أو التوقّع الخطي للذاكرة يتجاوز السقف بهامش (لا نخاطر بـOOM)،
    أو لا تحسّن ≥5% مرتين متتاليتين، أو بلوغ max_batch."""
    runs: List[Dict[str, Any]] = []; best = 0.0; stale = 0
    def attempt(bs: int) -> Optional[Dict[str, Any]]:
        try:
            if not runs: eng.generate([imgs[i % len(imgs)] for i in range(min(2, bs))], 24)         # إحماء: نوى CUDA والمخصِّص
            r = _time_batch(eng, imgs, bs, iters)
        except CaptionOOM as e:
            runs.append(dict(batch=bs, oom=True, error=_scrub(e, 100)))
            try:
                import torch; gc.collect(); torch.cuda.empty_cache()
            except Exception: pass
            return None
        except Exception as e:
            runs.append(dict(batch=bs, error=_scrub(f"{type(e).__name__}: {e}", 160))); return None
        runs.append(r); log.info("   bench batch=%-3d %.2f img/s | %.0f tok/s | VRAM %s", bs, r["img_per_s"], r["tokens_per_s"],
                                 f"{r['peak_gb']:.1f}GB ({r['vram_frac']:.0%})" if r.get("peak_gb") else "n/a")
        return r
    if start is None:
        for bs in [b for b in (1, 2, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256) if b <= max_batch]:
            r = attempt(bs)
            if r is None: break
            stale = stale + 1 if r["img_per_s"] < 1.05 * best else 0
            best = max(best, r["img_per_s"])
            if stale >= 2 or (r.get("vram_frac") or 0) > 0.90: break
        return runs
    cap = float(vram_target if vram_target is not None else BENCH_VRAM_CAP); start = max(1, min(int(start), int(max_batch)))
    r = attempt(start)
    if r is None or (r.get("vram_frac") or 0) > cap:                                  # البداية فشلت أو تجاوزت السقف ⇒ نزول
        b = start // 2
        while b >= 1:
            r2 = attempt(b)
            if r2 is not None and (r2.get("vram_frac") or 0) <= cap: break
            if r2 is None and "oom" not in runs[-1]: break                              # خطأ غير OOM: لا فائدة من الاستمرار
            b = b // 2 if b > 1 else 0
        return runs
    best = r["img_per_s"]
    for bs in [b for b in _BENCH_LADDER_UP if start < b <= max_batch]:
        ok = [x for x in runs if not x.get("oom") and "error" not in x and x.get("vram_frac") is not None]
        if len(ok) >= 2:                                                                 # توقّع خطي لذاكرة الدفعة التالية؛ إن تجاوزت السقف بهامش لا نجرّبها
            a1, a2 = ok[-2], ok[-1]
            if a2["batch"] > a1["batch"]:
                slope = (a2["vram_frac"] - a1["vram_frac"]) / (a2["batch"] - a1["batch"])
                if a2["vram_frac"] + max(0.0, slope) * (bs - a2["batch"]) > min(0.97, cap + 0.10): break
        r = attempt(bs)
        if r is None: break
        stale = stale + 1 if r["img_per_s"] < 1.05 * best else 0
        best = max(best, r["img_per_s"])
        if stale >= 2 or (r.get("vram_frac") or 0) > cap: break
    return runs

def choose_batch(runs: Sequence[Dict[str, Any]], avg_item_tokens: float, vram_cap: Optional[float] = None, stable_fraction: Optional[float] = None) -> Dict[str, Any]:
    """دالة نقية. best = أعلى img/s ضمن الدفعات الآمنة (بلا OOM وذاكرة ≤ vram_cap)؛ knee = أصغر دفعة بـ≥95% من أفضل أداء؛
    stable_batch (نقطة بدء AdaptiveBatcher) = min(knee, stable_fraction × best) كي يبقى هامش لتقلّب أطوال الأوصاف؛ max_tested_batch = best (لا يتجاوز ما اختُبر).
    الافتراضيات (None): vram_cap=BENCH_VRAM_CAP (0.85)، stable_fraction=0.75؛ BenchmarkRun يمرّر CONFIG.BENCH_VRAM_TARGET وCONFIG.BENCH_STABLE_FRACTION."""
    vram_cap = BENCH_VRAM_CAP if vram_cap is None else vram_cap; stable_fraction = 0.75 if stable_fraction is None else stable_fraction
    ok = [r for r in runs if not r.get("oom") and "error" not in r and r.get("img_per_s")]
    if not ok:
        return dict(ok=False, note="لم تنجح أي دفعة (OOM حتى عند الحجم 1؟). خفّض MODEL_MAX_PIXELS، أو (بجودة أقل) CAPTION_ALLOW_FALLBACK_PLANS=True مع CAPTION_PRECISION=int4")
    safe = [r for r in ok if r.get("vram_frac") is None or r["vram_frac"] <= vram_cap] or ok[:1]
    best = max(safe, key=lambda r: (round(r["img_per_s"], 6), -r["batch"]))
    knee = min((r for r in safe if r["img_per_s"] >= 0.95 * best["img_per_s"]), key=lambda r: r["batch"])
    stable = max(1, min(knee["batch"], int(round(stable_fraction * best["batch"]))))
    return dict(ok=True, best_batch=best["batch"], best_img_per_s=best["img_per_s"], knee_batch=knee["batch"], stable_batch=stable,
                max_tested_batch=best["batch"], token_budget=int(max(1024, best["batch"] * avg_item_tokens)),
                peak_vram_gb=best.get("peak_gb"), tokens_per_s=best.get("tokens_per_s"), vram_cap=vram_cap)

def retune_pool(pool: "CaptionPool") -> None:
    """يطبّق قيم القياس على AdaptiveBatcher الحالية (المجمّع قد حُمِّل قبل القياس بقيم افتراضية)."""
    nb = initial_batcher()
    for c in pool.captioners:
        b = c.batcher; b.bs, b.max_bs, b.ceiling, b.token_budget = nb.bs, nb.max_bs, nb.max_bs, nb.token_budget; b.streak = 0

# vLLM: يُقاس في عملية منفصلة (لا نلوّث عملية الـNotebook ولا ذاكرة GPU). مُختبَر منطقيًا فقط محليًا؛ التشغيل الفعلي يحتاج GPU + vllm.
_VLLM_BENCH_SCRIPT = r'''
import json, sys, time
from PIL import Image
a = json.load(open(sys.argv[1]))
from vllm import LLM, SamplingParams
imgs = [Image.open(p).convert("RGB") for p in a["images"]]
dt = "bfloat16" if a["dtype"] == "bf16" else "float16"
llm = LLM(model=a["model_id"], trust_remote_code=True, dtype=dt, gpu_memory_utilization=0.88, enforce_eager=False, tensor_parallel_size=int(a.get("tp", 1)),
          max_model_len=4096, limit_mm_per_prompt={"image": 1}, seed=0)
sp = SamplingParams(temperature=0.0, max_tokens=int(a["max_tokens"]), top_k=-1, repetition_penalty=float(a["rep"]))
mk = lambda ims: [{"prompt": a["prompt"], "multi_modal_data": {"image": im}} for im in ims]
llm.generate(mk(imgs[:2]), sampling_params=SamplingParams(temperature=0.0, max_tokens=16), use_tqdm=False)      # إحماء
t = time.perf_counter(); outs = llm.generate(mk(imgs), sampling_params=sp, use_tqdm=False); dt_s = time.perf_counter() - t
print("BENCH_JSON:" + json.dumps(dict(ok=True, n=len(imgs), seconds=dt_s, img_per_s=len(imgs) / dt_s, texts=[o.outputs[0].text for o in outs],
                                      tokens=sum(len(o.outputs[0].token_ids) for o in outs))))
'''

def bench_vllm_subprocess(imgs: List[np.ndarray], prompt_text: str, dtype: str, tp: int = 1, timeout: float = 1800.0) -> Dict[str, Any]:
    d = Path(CONFIG.TEMP_DIR) / "bench_vllm"; shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True, exist_ok=True)
    try:
        from PIL import Image as _Im
        paths = []
        for i, im in enumerate(imgs): p = d / f"img_{i:03d}.png"; _Im.fromarray(im).save(p); paths.append(str(p))
        (d / "bench.py").write_text(_VLLM_BENCH_SCRIPT, encoding="utf-8")
        (d / "args.json").write_text(json.dumps(dict(model_id=CONFIG.MODEL_ID, images=paths, prompt=prompt_text, dtype=dtype, tp=tp,
                                                     max_tokens=CONFIG.CAPTION_MAX_NEW_TOKENS, rep=CONFIG.CAPTION_REPETITION_PENALTY)), encoding="utf-8")
        p = subprocess.run([sys.executable, str(d / "bench.py"), str(d / "args.json")], capture_output=True, text=True, timeout=timeout)
        for line in reversed((p.stdout or "").splitlines()):
            if line.startswith("BENCH_JSON:"): return json.loads(line[len("BENCH_JSON:"):])
        return dict(ok=False, error=_scrub(f"rc={p.returncode} {(p.stderr or p.stdout or '')[-300:]}", 340))
    except subprocess.TimeoutExpired: return dict(ok=False, error=f"انتهت المهلة ({timeout:.0f}s)")
    except Exception as e: return dict(ok=False, error=_scrub(f"{type(e).__name__}: {e}", 240))
    finally: shutil.rmtree(d, ignore_errors=True)

# ---------------------------------------------------------------- التقدير الواقعي (دالة نقية قابلة للاختبار بأرقام اصطناعية)
def compute_estimate(bench: Dict[str, Any], pending: Sequence[Dict[str, Any]], *, session_hours: float = 12.0, weekly_gpu_hours: float = 30.0) -> Dict[str, Any]:
    """يحوّل السرعات المقيسة إلى زمن/حجم متوقعين لكل الفيديوهات المعلّقة. المراحل تعمل بالتوازي في الأنبوب ⇒ الزمن = أبطأ مرحلة (+ زمن تحليل كل فيديو التسلسلي).
    ما لم يُقَس يُوسَم في assumptions. weekly_gpu_hours افتراض (حصة GPU الأسبوعية المعتادة على Kaggle؛ عدّله إن اختلفت حصتك)."""
    est: Dict[str, Any] = dict(assumptions=[], warnings=[], stages={})
    note = est["assumptions"].append; warn = est["warnings"].append
    n_pend = len(pending)
    if n_pend == 0:
        est["summary"] = "لا فيديوهات معلّقة"; return est
    items = (bench.get("videos") or {}).get("items") or []
    known = [float(p["expected_duration"]) for p in pending if p.get("expected_duration")]
    bdur = [float(v["duration"]) for v in items if v.get("duration")]
    if known and len(known) == n_pend: mean_d = float(np.mean(known))
    elif bdur:
        mean_d = float(np.mean(known + bdur)) if known else float(np.mean(bdur))
        note(f"مدة {n_pend - len(known)} فيديو غير معروفة: افتُرض متوسط {mean_d / 60:.0f} دقيقة (من فيديوهات القياس). أضف expected_duration في المانيفست لتقدير أدق")
    else:
        mean_d = (CONFIG.MIN_VIDEO_DURATION + CONFIG.MAX_VIDEO_DURATION) / 2.0
        note(f"لا مدة معروفة ولا فيديوهات قياس: افتُرض {mean_d / 60:.0f} دقيقة (منتصف [MIN,MAX])")
    d = np.array([float(p["expected_duration"]) if p.get("expected_duration") else mean_d for p in pending], dtype=np.float64)
    if items:
        cand_rate = float(np.mean([v["n_candidates"] / v["duration"] for v in items if v.get("duration")]))
        src_rate = float(np.mean([v["n_frames"] / v["duration"] for v in items if v.get("duration")]))
        if len(items) < 5: note(f"معدل الإطارات المرشّحة ({cand_rate:.1f}/ث) من {len(items)} فيديو قياس فقط؛ إن اختلفت FPS فيديوهاتك الأخرى (24/30/60) تتغير الأرقام")
    else:
        cand_rate, src_rate = min(CONFIG.TARGET_FPS, 30.0), 30.0; note("لا فيديوهات قياس: افتُرض 30 إطارًا/ث")
    sel_i = d * cand_rate * CONFIG.SAMPLING_RATIO
    if CONFIG.MAX_IMAGES_PER_VIDEO: sel_i = np.minimum(sel_i, CONFIG.MAX_IMAGES_PER_VIDEO)
    n_cand, n_sel, n_src = float((d * cand_rate).sum()), float(sel_i.sum()), float((d * src_rate).sum())
    qc = bench.get("qc") or {}; enc = bench.get("encode") or {}; shd = bench.get("shard") or {}; up = bench.get("upload") or {}; cap = bench.get("caption") or {}
    accept = qc.get("accept_fraction")
    if not accept:
        accept = 0.9; note("نسبة القبول 0.9 (افتراض: لم تُقَس)")
    accept = float(min(1.0, max(0.02, accept)))
    note(f"نسبة القبول {accept:.0%} مقيسة على أول ~{qc.get('n_frames', '?')} إطارًا من فيديوهات القياس فقط (تختلف مع المحتوى: مقدّمات سوداء/لقطات ثابتة)") if qc.get("accept_fraction") else None
    n_acc = n_sel * accept
    avg_jpg = float(enc.get("avg_jpeg_bytes") or 350_000); avg_json = float(shd.get("avg_json_bytes") or 2_500)
    if not enc.get("avg_jpeg_bytes"): note("حجم JPEG 350KB (افتراض: لم يُقَس)")
    total_bytes = n_acc * (avg_jpg + avg_json)
    # ---- المراحل
    st: Dict[str, Dict[str, Any]] = est["stages"]
    def stage(name: str, units: float, rate: Optional[float], unit: str) -> None:
        if rate and rate > 0: st[name] = dict(units=float(units), rate=float(rate), unit=unit, hours=float(units / rate / 3600.0))
        else: warn(f"مرحلة {name}: لم تُقَس (تُستبعد من التقدير)")
    dec = bench.get("decode") or {}; best_b = (dec.get("choice") or {}).get("best"); a = (dec.get("agg") or {}).get(best_b or "", {})
    if a.get("source_fps"): stage("decode", n_src, a["source_fps"], "source-frames")
    elif a.get("selected_fps"): stage("decode", n_sel, a["selected_fps"], "selected-frames")
    else: warn("مرحلة decode: لم تُقَس")
    if "decode" in st:
        ov = float(np.mean([v.get("analysis_s", 0.0) for v in items]) if items else 0.0) + float(a.get("startup_s") or 0.0)
        st["decode"]["hours"] += n_pend * ov / 3600.0; st["decode"]["per_video_overhead_s"] = ov
    stage("qc", n_sel, qc.get("analyze_fps"), "selected-frames")
    if qc.get("decide_ms"): stage("dedup/decide", n_sel, 1000.0 / qc["decide_ms"], "selected-frames")
    stage("encode", n_acc, enc.get("enc_fps_parallel"), "accepted-images")
    if CONFIG.ENABLE_CAPTIONING:
        ips = cap.get("img_per_s_total")
        if ips and cap.get("mock"): warn("الوصف قيس بمحرك MOCK (اختبار فقط): تقدير الزمن غير معتمد")
        if ips: ips = ips / 1.10; note("سرعة الوصف مقسومة على 1.10 (هامش للتقلّب: أوصاف أطول، إعادة محاولات، حشو الدفعات)")
        stage("caption", n_acc, ips, "accepted-images")
    if shd.get("validate_MBps"): stage("validate", total_bytes / 1e6, shd["validate_MBps"], "MB")
    if up.get("measured") and up.get("MBps"):
        n_sh = max(1.0, total_bytes / (CONFIG.SHARD_TARGET_SIZE_GB * 1e9))
        stage("upload", total_bytes / 1e6, up["MBps"], "MB"); st["upload"]["hours"] += n_sh * float(up.get("commit_latency_s") or 0.0) / 3600.0
    elif up: warn("سرعة الرفع إلى HF لم تُقَس (مخزن محلي): تُستبعد من التقدير")
    dl = [v.get("download_MBps") for v in items if v.get("download_MBps")]
    if dl:
        mb = float(np.mean([v.get("size_mb", 0) for v in items])) * n_pend; stage("download", mb, float(np.mean(dl)), "MB")
    if not st: est["summary"] = "لا قياسات كافية للتقدير"; return est
    bott = max(st, key=lambda k: st[k]["hours"]); total_h = st[bott]["hours"]
    usable = max(1.0, (session_hours or 12.0) - CONFIG.STOP_MARGIN_MINUTES / 60.0 - 0.25)
    eff_rate = n_acc / max(1e-9, total_h * 3600.0)                               # صور مقبولة/ثانية فعلية للأنبوب
    est.update(videos=n_pend, candidate_frames=n_cand, selected_frames=n_sel, accept_fraction=accept, accepted_images=n_acc, dataset_gb=total_bytes / 1e9,
               bottleneck=bott, total_hours=total_h, session_hours_usable=usable, sessions_needed=int(math.ceil(total_h / usable)),
               weekly_gpu_hours_assumed=weekly_gpu_hours, weeks_of_gpu_quota=total_h / weekly_gpu_hours, effective_images_per_s=eff_rate,
               shard_count=int(math.ceil(total_bytes / (CONFIG.SHARD_TARGET_SIZE_GB * 1e9))))
    est["budgets"] = []
    for bh in (10.0, 30.0, 100.0):
        imgs = min(n_acc, eff_rate * 3600.0 * bh)
        est["budgets"].append(dict(gpu_hours=bh, images=imgs, images_per_video=imgs / n_pend, max_images_per_video=int(imgs // n_pend),
                                   equivalent_sampling_ratio=float(imgs / max(1.0, n_cand * accept)), dataset_gb=imgs * (avg_jpg + avg_json) / 1e9))
    if total_h > usable * 3: warn(f"الزمن المقدّر {total_h:,.0f} ساعة ≈ {est['sessions_needed']:,} جلسة بطول {usable:.1f}h فعّالة ≈ {est['weeks_of_gpu_quota']:.1f} أسبوعًا من حصة {weekly_gpu_hours:.0f}h/أسبوع (افتراض). "
                                      f"العنق: {bott}. لخفض الحجم: MAX_IMAGES_PER_VIDEO أو SAMPLING_RATIO (انظر الجدول).")
    if total_bytes / 1e9 > 100: warn(f"حجم Dataset المتوقع {total_bytes / 1e12:.2f} TB؛ تحقق من حدود تخزين مستودع Hugging Face لحسابك قبل الرفع الكامل")
    if est["shard_count"] > 8000:
        warn(f"≈ {est['shard_count']:,} شظية في مجلد data/ (وفهرس لكل شظية في index/): توصي Hugging Face بألّا يتجاوز المجلد الواحد 10,000 ملف؛ "
             f"ارفع SHARD_TARGET_SIZE_GB (مثلًا إلى {max(2.0, math.ceil(CONFIG.SHARD_TARGET_SIZE_GB * est['shard_count'] / 5000.0)):.0f}) قبل البدء — لا يمكن تغييره لاحقًا دون خلط أحجام الشظايا")
    est["summary"] = (f"{n_pend} فيديو → ≈{n_acc:,.0f} صورة مقبولة (~{total_bytes / 1e9:,.1f} GB) | زمن مقدّر ≈ {total_h:,.1f}h (عنق الزجاجة: {bott}) "
                      f"≈ {est['sessions_needed']:,} جلسة | {eff_rate:.2f} صورة/ث")
    return est

def print_estimate(est: Dict[str, Any]) -> None:
    print("=" * 78); print(" REALISTIC ESTIMATE  (من القياسات الفعلية أعلاه؛ التقدير ≠ ضمان)"); print("=" * 78)
    if "total_hours" not in est:
        print("  ", est.get("summary", "لا تقدير"))
    else:
        print(f"  الفيديوهات المعلّقة       : {est['videos']:,}")
        print(f"  إطارات مرشّحة (≤{CONFIG.TARGET_FPS:.0f}fps)  : {est['candidate_frames']:,.0f}   → مختارة ({CONFIG.SAMPLING_RATIO:.0%}): {est['selected_frames']:,.0f}   → مقبولة (~{est['accept_fraction']:.0%}): {est['accepted_images']:,.0f}")
        print(f"  حجم Dataset المتوقع       : {est['dataset_gb']:,.1f} GB في ≈ {est['shard_count']:,} شظية")
        print("  زمن كل مرحلة (تعمل بالتوازي؛ الأبطأ يحدّد الإجمالي):")
        for k, s in sorted(est["stages"].items(), key=lambda kv: -kv[1]["hours"]):
            print(f"     {'▶' if k == est['bottleneck'] else ' '} {k:<14}{s['hours']:>12,.1f} h   ({s['rate']:,.2f} {s['unit']}/s)")
        print(f"  الإجمالي المقدّر          : {est['total_hours']:,.1f} h  ≈ {est['sessions_needed']:,} جلسة × {est['session_hours_usable']:.1f}h فعّالة  |  {est['effective_images_per_s']:.2f} صورة/ث")
        print("  ماذا يتّسع في ميزانية GPU محددة (المعدّل الفعلي المقيس):")
        for b in est["budgets"]:
            print(f"     {b['gpu_hours']:>5.0f} h → {b['images']:>12,.0f} صورة ({b['dataset_gb']:,.1f} GB) | MAX_IMAGES_PER_VIDEO ≈ {b['max_images_per_video']:,} | أو SAMPLING_RATIO ≈ {b['equivalent_sampling_ratio']:.4f}")
    for w in est.get("warnings", []): print("  ⚠", w)
    for s in est.get("assumptions", []): print("  • افتراض:", s)
    print("=" * 78)

# ---------------------------------------------------------------- تقرير القياس
def print_benchmark_report(b: Dict[str, Any]) -> None:
    def sec(name: str) -> Dict[str, Any]: return b.get(name) or {}
    def bad(name: str) -> bool:
        s = sec(name)
        if s and not s.get("ok", True): print(f"  ✖ {name}: فشل — {s.get('error')}"); return True
        return not s
    print("=" * 78); print(" BENCHMARK REPORT  (قياس فعلي على هذا العتاد)"); print("=" * 78)
    e = b.get("env", {})
    print(f"  GPU: {e.get('gpus') or 'لا يوجد'} | CPU cores {e.get('cpu_cores')} | RAM {e.get('ram_total_gb', 0):.0f} GB | fingerprint {b.get('bench_fp')}")
    v = sec("videos")
    if not bad("videos"):
        for it in v.get("items", []):
            print(f"  video {it['video_id']}: {it['duration'] / 60:.1f} min | {it['width']}x{it['height']} {it['codec']} {it['avg_fps']:.2f}fps{' VFR' if it['vfr'] else ''} | {it['size_mb']:.0f} MB | "
                  f"ffprobe {it['analysis_s']:.1f}s | خطة العينة {it['plan_s'] * 1000:.0f}ms ({it['n_candidates']:,} مرشّح → {it['n_selected']:,} مختار)"
                  + (f" | تنزيل {it['download_MBps']:.1f} MB/s" if it.get("download_MBps") else ""))
        for n in v.get("notes", []): print("   -", n)
    d = sec("decode")
    if not bad("decode"):
        print("  DECODING (أول إطارات مختارة بنسبة العينة الحقيقية؛ السرعة بعد أول إطار)")
        print(f"     {'backend':<16}{'sel fps':>9}{'src fps':>9}{'startup':>9}{'colour MAD':>12}  verdict")
        for k, a in (d.get("agg") or {}).items():
            mad = a.get("color_mad"); verdict = "صالح" if a.get("valid") else ("لون مختلف → مُعطَّل" if (mad or 0) > BENCH_COLOR_MAD_MAX else f"فشل ({'; '.join(a.get('errors', []))[:70]})")
            print(f"     {k:<16}{a['selected_fps']:>9.1f}{a['source_fps']:>9.1f}{(a.get('startup_s') or 0):>8.2f}s{('—' if mad is None else f'{mad:.2f}'):>12}  {verdict}")
        print("     →", (d.get("choice") or {}).get("note"))
    q = sec("qc")
    if not bad("qc"):
        print(f"  QUALITY/DEDUP: تحليل {q['analyze_fps']:.1f} إطار/ث بـ{q['threads']} خيوط (×{q['parallel_speedup']:.1f} على خيط واحد) | قرار+تكرار {q['decide_ms']:.2f} ms/إطار | قبول {q['accept_fraction']:.0%} من {q['n_frames']} | رفض: {q['reasons']}")
        if q.get("calibration"): print(f"     معايرة الضبابية: lap_floor={q['calibration']['lap_floor']:.2f} ten_floor={q['calibration']['ten_floor']:.1f} (n={q['calibration']['n']}) → {'طُبّقت' if q.get('calibration_applied') else 'لم تُطبَّق: ' + str(q.get('calibration_note'))}")
    en = sec("encode")
    if not bad("encode"):
        print(f"  IMAGE ENCODING: مشفّر {en['chosen']} | {en['enc_ms_single']:.1f} ms/صورة (خيط) → {en['enc_fps_parallel']:.1f} صورة/ث بالتوازي | متوسط JPEG {en['avg_jpeg_bytes'] / 1e3:.0f} KB | تجهيز مدخل النموذج {en['model_input_ms']:.1f} ms")
    s = sec("shard")
    if not bad("shard"): print(f"  SHARDS: كتابة {s['write_MBps']:.0f} MB/s | تحقق كامل {s['validate_MBps']:.0f} MB/s | فهرس {s['index_ms']:.0f} ms | JSON/عينة {s['avg_json_bytes']:.0f} B")
    u = sec("upload")
    if not bad("upload"):
        print(f"  UPLOAD: {u.get('store')} | " + (f"{u['MBps']:.1f} MB/s (حمولة {u['payload_MB']}MB) | زمن commit صغير {u['commit_latency_s']:.1f}s" if u.get("measured") else u.get("note", "")))
    c = sec("caption")
    if c and c.get("skipped"): print("  CAPTION: تخطّي —", c["skipped"])
    elif not bad("caption"):
        print(f"  CAPTION: {c['model']} | {c['engine']}/{c['precision']}/{c['layout']} attn={c.get('attn')} | نسخ المحرك {c['replicas']}{' | MOCK (ليس Qwen!)' if c.get('mock') else ''}")
        print(f"     {'batch':>6}{'img/s':>8}{'tok/s':>9}{'peak GB':>9}{'VRAM%':>7}")
        for r in c["runs"]:
            print(f"     {r['batch']:>6}" + ("   OOM" if r.get("oom") else (f"   خطأ: {r['error']}" if r.get("error") else f"{r['img_per_s']:>8.2f}{r['tokens_per_s']:>9.1f}{(r.get('peak_gb') or 0):>9.1f}{((r.get('vram_frac') or 0) * 100):>6.0f}%")))
        print(f"     ⇒ stable_batch={c.get('stable_batch')} max_tested_batch={c.get('max_tested_batch')} token_budget={c.get('token_budget')} | أفضل {c.get('best_img_per_s', 0):.2f} img/s × {c['replicas']} = {c.get('img_per_s_total', 0):.2f} img/s")
        if c.get("pipeline_img_per_s") is not None: print(f"     مسار الأنبوب الفعلي (تحليل+تحقق): {c['pipeline_img_per_s']:.2f} img/s | وصف صالح {c['caption_ok_rate']:.0%} | متوسط {c['mean_words']:.0f} كلمة | أخطاء {c['errors']} | OOM/إعادات {c.get('oom_events', 0)}/{c.get('retries', 0)}")
        fa, vl = c.get("flash_attn") or {}, c.get("vllm") or {}
        print("     FlashAttention-2:", (f"{fa['fa2_img_per_s']:.2f} vs SDPA {fa['sdpa_img_per_s']:.2f} img/s → " + ("اعتُمد" if fa.get("kept") else "لم يُعتمد") if fa.get("tested") and fa.get("fa2_img_per_s") else fa.get("reason") or fa.get("error", "لم يُختبر")))
        print("     vLLM            :", (f"{vl['img_per_s']:.2f} img/s (الأفضل Transformers {vl['transformers_img_per_s']:.2f}) → " + ("اعتُمد" if vl.get("selected") else "لم يُعتمد") if vl.get("tested") and vl.get("img_per_s") else vl.get("reason") or vl.get("error", "لم يُختبر")))
    ap = b.get("applied") or {}
    if ap: print("  APPLIED:", {k: v for k, v in ap.items() if v not in (None, [], {}, "")})
    print(f"  المدة الكلية للقياس: {human_time(b.get('duration_s', 0))}")
    print("=" * 78)

# ---------------------------------------------------------------- مشغّل القياس
class BenchmarkRun:
    def __init__(self, n_videos: int = 2, decode_frames: int = 480, qc_frames: int = 240, caption_images: Optional[int] = None, max_batch: Optional[int] = None, caption: bool = True,
                 try_flash_attn: bool = True, try_vllm: bool = True, upload_test: bool = True, only_videos: Optional[Sequence[str]] = None, apply: bool = True):
        self.n, self.decode_frames, self.qc_frames = max(1, int(n_videos)), max(30, int(decode_frames)), max(16, int(qc_frames))
        big_gpu = max([float(g.get("vram_total_mb", 0)) for g in (ENV.get("gpus") or [])] or [0.0]) / 1024 >= 40      # RTX Pro 6000 (96GB) وأمثالها: دفعات أكبر بكثير
        max_batch = int(max_batch) if max_batch else (192 if big_gpu else 32)
        caption_images = int(caption_images) if caption_images else (192 if big_gpu else 32)      # GPU كبيرة: صور متنوعة بقدر أكبر دفعة مُجرَّبة (بلا تكرار نفس الصورة داخل الدفعة)
        self.caption_images, self.max_batch = max(4, int(caption_images)), max(1, int(max_batch))
        self.do_caption = bool(caption) and CONFIG.ENABLE_CAPTIONING
        self.try_flash, self.try_vllm = bool(try_flash_attn) and bool(apply), bool(try_vllm) and bool(apply)     # يعيدان تحميل النموذج ⇒ فقط عند السماح بالتطبيق
        self.do_upload, self.only, self.apply = bool(upload_test), (set(only_videos) if only_videos else None), bool(apply)
        self.res: Dict[str, Any] = {}; self.vids: List[Dict[str, Any]] = []
        self.frames: List["RawFrame"] = []; self.accepted: List["RawFrame"] = []; self.laps: List[float] = []; self.tens: List[float] = []
        self.failing_pairs: List[Tuple[str, str]] = []; self.applied: Dict[str, Any] = {}; self.sample_jpegs: List[bytes] = []; self.t0 = time.time()

    def step(self, name: str, fn: Callable[[], Optional[Dict[str, Any]]]) -> Dict[str, Any]:
        t0 = time.time(); print(f"  ▸ {name} ...", flush=True)
        try:
            out = fn() or {}; out = dict(out, ok=out.get("ok", True), seconds=round(time.time() - t0, 2))
        except Exception as e:
            log.error("benchmark/%s فشل: %s", name, _scrub(repr(e)))
            out = dict(ok=False, error=_scrub(f"{type(e).__name__}: {e}", 300), seconds=round(time.time() - t0, 2), trace=_scrub(traceback.format_exc()[-700:], 700))
        self.res[name] = out
        return out

    # ---- 1) فيديوهات القياس: تنزيل + ffprobe + خطة العينة
    def _videos(self) -> Dict[str, Any]:
        todo = [v for v in STATE.pending_videos() if self.only is None or v["video_id"] in self.only]
        if not todo: raise RuntimeError("لا توجد فيديوهات معلّقة للقياس (الحالة فارغة أو كلها منتهية)")
        notes: List[str] = []
        for v in todo[: self.n * 4 + 4]:
            if len(self.vids) >= self.n: break
            vid = v["video_id"]; rec = {"video_id": vid, "url": v["url"]}; is_http = str(v["url"]).startswith(("http://", "https://"))
            try:
                b0 = DOWNLOADER.stats["bytes"]; t0 = time.perf_counter()
                path = DOWNLOADER.ensure(rec, v.get("sha256"), v.get("expected_size")); dl_s = time.perf_counter() - t0; dl_b = DOWNLOADER.stats["bytes"] - b0
                t1 = time.perf_counter(); meta = analyze_video(str(path), vid); an_s = time.perf_counter() - t1
            except Exception as e:
                notes.append(f"{vid}: {_scrub(e, 120)}"); continue
            skip = check_video_limits(meta)
            if skip: notes.append(f"{vid}: سيُتخطّى في التشغيل الفعلي ({skip[:90]})"); continue
            t2 = time.perf_counter(); plan = build_plan(meta); plan_s = time.perf_counter() - t2
            info = dict(video_id=vid, duration=meta.duration, width=meta.width, height=meta.height, codec=meta.codec, avg_fps=meta.avg_fps, vfr=meta.is_vfr, n_frames=meta.n_frames,
                        n_candidates=int(len(plan.cand)), n_selected=int(len(plan.sel)), ratio_actual=plan.ratio_actual, size_mb=meta.file_size / 1e6, analysis_s=an_s, plan_s=plan_s,
                        hashes_per_s=(len(plan.cand) / plan_s if plan_s > 0 else None), local=not is_http,
                        download_MBps=((dl_b / 1e6 / dl_s) if (is_http and dl_b > 0 and dl_s > 0.2) else None))
            self.vids.append(dict(info=info, meta=meta, plan=plan))
        if not self.vids: raise RuntimeError("تعذّر تحضير أي فيديو للقياس: " + " | ".join(notes[:4]))
        return dict(items=[x["info"] for x in self.vids], notes=notes)

    # ---- 2) الفك: كل مفكّك متاح على كل فيديو، بنفس الخطة، مع مطابقة اللون مع FFmpeg CPU
    def _decode(self) -> Dict[str, Any]:
        per: Dict[str, Dict[str, Dict[str, Any]]] = collections.defaultdict(dict)
        qc_per_video = int(math.ceil(self.qc_frames / max(1, len(self.vids))))
        for vi in self.vids:
            meta, plan = vi["meta"], vi["plan"]; k = min(self.decode_frames, len(plan.sel))
            if k < 2: continue
            sub = dataclasses.replace(plan, sel=plan.sel[:k]); chain = decoder_chain(meta)
            order = (["ffmpeg_cpu"] if DECODER_STATUS.get("ffmpeg_cpu", {}).get("available") else []) + [b for b in chain if b != "ffmpeg_cpu"]
            if "ffmpeg_cpu" in order: order = [b for b in order if b != "opencv"]           # OpenCV مسار طوارئ فقط (ألوانه تختلف)
            fb = meta.width * meta.height * 3; keep_qc = max(8, min(qc_per_video, int(0.8e9 // fb))); ref: Dict[int, np.ndarray] = {}
            for bi, backend in enumerate(order):
                r, frames = _bench_decode_one(meta, sub, backend, keep_qc if bi == 0 else 6)
                if r.get("ok"):
                    if bi == 0:
                        ref = {f.frame_index: f.rgb for f in frames[:6]}; r["color_mad"] = 0.0; r["reference"] = True
                        self.frames += frames[:keep_qc]
                    else:
                        r["color_mad"] = _color_mad(ref, frames)
                else: self.failing_pairs.append((backend, meta.codec))
                per[backend][meta.video_id] = r
                log.info("   bench decode %-15s %s: %s", backend, meta.video_id, f"{r['selected_fps']:.1f} sel-fps | {r['source_fps']:.0f} src-fps" if r.get("ok") else r.get("error"))
        if not per: raise RuntimeError("لا إطارات مختارة كافية لقياس الفك")
        agg: Dict[str, Dict[str, Any]] = {}
        for b, dv in per.items():
            oks = [r for r in dv.values() if r.get("ok")]; mads = [r["color_mad"] for r in oks if r.get("color_mad") is not None]
            a = dict(videos=len(dv), videos_ok=len(oks), selected_fps=float(np.mean([r["selected_fps"] for r in oks])) if oks else 0.0,
                     source_fps=float(np.mean([r["source_fps"] for r in oks])) if oks else 0.0, startup_s=float(np.mean([r["startup_s"] or 0 for r in oks])) if oks else None,
                     color_mad=(max(mads) if mads else None), errors=[r["error"] for r in dv.values() if not r.get("ok")][:2])
            a["valid"] = bool(oks) and len(oks) == len(dv) and (a["color_mad"] is None or a["color_mad"] <= BENCH_COLOR_MAD_MAX)
            agg[b] = a
        return dict(agg=agg, per_video={b: dv for b, dv in per.items()}, choice=choose_decoder(agg), frames_per_video=min(self.decode_frames, min(len(x["plan"].sel) for x in self.vids)))

    # ---- 3) الجودة وإزالة التكرار: سرعة التحليل المتوازي + سرعة القرار + نسب القبول/الرفض + معايرة الضبابية
    def _qc(self) -> Dict[str, Any]:
        if len(self.frames) < 8: raise RuntimeError("إطارات غير كافية لقياس الجودة")
        nthr = RESOURCE_PLAN["qc_threads"]; by_vid: Dict[str, List["RawFrame"]] = collections.defaultdict(list)
        for f in self.frames: by_vid[f.video_id].append(f)
        n_all, t_an, t_dec = 0, 0.0, 0.0; reasons: "collections.Counter" = collections.Counter()
        with _isolated_dedup(), ThreadPoolExecutor(nthr) as ex:
            for vid, fs in by_vid.items():
                t = time.perf_counter(); fas = list(ex.map(analyze_frame, fs)); t_an += time.perf_counter() - t
                dec = VideoDecider(vid, None); t = time.perf_counter(); rs = [dec.decide(fa) for fa in fas]; t_dec += time.perf_counter() - t
                for fr, fa, r in zip(fs, fas, rs):
                    reasons[r or "accepted"] += 1
                    if fa.reject is None and fa.metrics: self.laps.append(fa.metrics["lap_var"]); self.tens.append(fa.metrics["tenengrad"])
                    if r is None: self.accepted.append(fr)
                n_all += len(fs)
        sample = self.frames[: min(12, len(self.frames))]; t = time.perf_counter()
        for f in sample: analyze_frame(f)
        single_ms = (time.perf_counter() - t) / len(sample) * 1000.0
        out = dict(n_frames=n_all, threads=nthr, analyze_fps=n_all / max(1e-9, t_an), single_thread_ms=single_ms, parallel_speedup=(single_ms * n_all / 1000.0) / max(1e-9, t_an),
                   decide_ms=t_dec / n_all * 1000.0, accept_fraction=reasons["accepted"] / n_all, reasons={k: v for k, v in reasons.items() if k != "accepted"})
        cal = calibrate_blur_thresholds(self.laps, self.tens)
        if cal: out["calibration"] = cal
        return out

    # ---- 4) المشفّرات: اختيار الأسرع المكافئ جودةً + حجم JPEG الحقيقي + سرعة التجهيز
    def _encode(self) -> Dict[str, Any]:
        src = [f.rgb for f in (self.accepted or self.frames)[:24]]
        if not src: raise RuntimeError("لا إطارات")
        be = benchmark_encoders(src[:8]); t = time.perf_counter(); packed = [prepare_stored_image(a) for a in src]; single = (time.perf_counter() - t) / len(src)
        nthr = max(2, RESOURCE_PLAN["qc_threads"] // 2); t = time.perf_counter()
        with ThreadPoolExecutor(nthr) as ex: list(ex.map(prepare_stored_image, src * 2))
        par_fps = 2 * len(src) / max(1e-9, time.perf_counter() - t)
        t = time.perf_counter(); [prepare_model_input(a, QWEN_FACTOR) for a in src]; mi = (time.perf_counter() - t) / len(src) * 1000.0
        self.sample_jpegs = [p[0] for p in packed[: self.caption_images]]
        return dict(encoders=be, chosen=ENCODER_CHOICE, enc_ms_single=single * 1000.0, enc_fps_parallel=par_fps, enc_threads=nthr, model_input_ms=mi,
                    avg_jpeg_bytes=float(np.mean([len(p[0]) for p in packed])), n=len(src), format=CONFIG.IMAGE_FORMAT, quality=CONFIG.IMAGE_QUALITY)

    # ---- 5) الشظايا: كتابة + تحقق كامل + فهرس (شظية مؤقتة تُحذف)
    def _shard(self) -> Dict[str, Any]:
        src = (self.accepted or self.frames)[:12]
        if not src: raise RuntimeError("لا إطارات")
        packed = [(prepare_stored_image(f.rgb), f) for f in src]; w = ShardWriter("bench", target_bytes=int(100e6)); cs = None; n, total, jlen = 0, 0, 0
        try:
            t0 = time.perf_counter()
            while total < 48e6 and n < 600:
                (jb, sw, sh), f = packed[n % len(packed)]
                s = Sample("bench", n, n / 30.0, jb, sw, sh, f.rgb.shape[1], f.rgb.shape[0], hashlib.sha256(jb).hexdigest(), 0, 0, dict(quality_score=0.5), backend="bench")
                s.caption = dict(training_prompt=_BENCH_CAPTION, caption_words=len(_BENCH_CAPTION.split())); s.caption_model = "benchmark"
                rec = build_record(s, {}); jlen += len(json.dumps(rec, ensure_ascii=False)); w.add(s, rec); total += len(jb); n += 1
            cs = w.close(); t_write = time.perf_counter() - t0
            t = time.perf_counter(); v = validate_tar_shard(cs.local_path, "bench"); t_val = time.perf_counter() - t
            t = time.perf_counter(); index_bytes(cs.rows, v["offsets"]); t_idx = time.perf_counter() - t
            if not v["ok"]: raise RuntimeError(f"شظية القياس فشلت التحقق: {v['errors'][:2]}")
            return dict(samples=n, bytes=cs.size, write_MBps=cs.size / 1e6 / max(1e-9, t_write), validate_MBps=cs.size / 1e6 / max(1e-9, t_val), index_ms=t_idx * 1000.0, avg_json_bytes=jlen / n)
        finally:
            try: w.close()
            except Exception: pass
            if cs is not None: cs.local_path.unlink(missing_ok=True)
            for p in (SHARD_DIR / "data").glob("bench-*"): p.unlink(missing_ok=True)

    # ---- 6) الرفع: حمولة 20MB عشوائية (غير قابلة للضغط) تُحذف بعد القياس
    def _upload(self) -> Dict[str, Any]:
        if STORE.name == "local":
            return dict(store="local", measured=False, MBps=None, note="مخزن محلي (DRY_RUN أو لا HF_TOKEN/اتصال): سرعة الرفع إلى Hugging Face غير مقيسة")
        if not self.do_upload: return dict(store=STORE.name, measured=False, MBps=None, note="upload_test=False")
        tag = f"_benchmark/probe-{int(time.time())}"; small, big = f"{tag}-small.bin", f"{tag}-big.bin"; paths: List[str] = []; payload = os.urandom(20_000_000)
        try:
            t = time.perf_counter(); STORE.commit({small: os.urandom(1024)}, "benchmark: small commit"); lat = time.perf_counter() - t; paths.append(small)
            t = time.perf_counter(); STORE.commit({big: payload}, "benchmark: 20MB upload"); dt = time.perf_counter() - t; paths.append(big)
            return dict(store=STORE.name, measured=True, MBps=len(payload) / 1e6 / dt, commit_latency_s=lat, payload_MB=20, verified=bool(STORE.verify(big, len(payload))),
                        note="الشظايا الأكبر عادةً أسرع من حمولة 20MB (تكلفة commit ثابتة)")
        finally:
            if paths:
                try: STORE.commit({}, "benchmark cleanup", deletes=paths)
                except Exception as e: log.warning("تعذّر حذف ملفات قياس الرفع %s من المستودع (احذفها يدويًا): %s", paths, _scrub(e, 120))

    # ---- 7) Qwen: دفعات + فحص جودة المسار الفعلي + FlashAttention + vLLM
    def _show_samples(self, imgs_src: List[bytes], results: List["CapResult"]) -> None:
        good = [(i, r) for i, r in enumerate(results) if r.parsed and r.parsed.get("training_prompt")]
        for i, r in random.sample(good, min(2, len(good))):
            cap = r.parsed["training_prompt"]
            if HAVE_IPY and i < len(imgs_src):
                try: display(HTML(render_sample_html(imgs_src[i], {"Sample": r.sample_id, "Model": r.model, "Latency": f"{r.latency:.2f}s/img", "Words": len(cap.split())}, cap,
                                                      title="BENCHMARK SAMPLE (real model output)", flags=r.parsed.get("caption_flags", [])))); continue
                except Exception: pass
            print(f"\n---- BENCHMARK SAMPLE [{r.sample_id}] model={r.model} ----\n{cap}\n----")

    def _flash(self, pool: "CaptionPool", imgs: List[np.ndarray], bs: int, base_ips: float) -> Tuple["CaptionPool", Dict[str, Any]]:
        usable = VLM_COMPAT.get("flash_attn_usable") and pool.plan.get("precision") in ("bf16", "fp16") and pool.engine_name == "transformers"
        if not usable: return pool, dict(tested=False, reason="غير قابل: flash-attn غير مثبّت أو GPU < Ampere أو دقة مكمّمة أو المحرك ليس Transformers")
        if CONFIG.ATTN_IMPLEMENTATION != "auto": return pool, dict(tested=False, reason=f"ATTN_IMPLEMENTATION مضبوط يدويًا = {CONFIG.ATTN_IMPLEMENTATION}")
        if getattr(pool.engines[0], "attn", None) == "flash_attention_2":
            return pool, dict(tested=False, kept=True, reason="flash_attention_2 مفعّلة تلقائيًا (متاحة وتعمل) — لا مقارنة ولا إعادة تحميل (النموذج كبير)")
        out: Dict[str, Any] = dict(tested=True, sdpa_img_per_s=base_ips, kept=False)
        release_caption_pool(); CONFIG.ATTN_IMPLEMENTATION = "flash_attention_2"
        try:
            p2 = get_caption_pool(run_sanity=True)                                     # فحص السلامة يرفض نتائج FA2 التالفة
            r = _time_batch(p2.engines[0], imgs, bs); out["fa2_img_per_s"] = r["img_per_s"]
            if r["img_per_s"] >= 1.10 * base_ips: out["kept"] = True; self.applied["attn"] = "flash_attention_2"; return p2, out
            release_caption_pool()
        except Exception as e:
            out["error"] = _scrub(f"{type(e).__name__}: {e}", 200); release_caption_pool()
        CONFIG.ATTN_IMPLEMENTATION = "auto"                                            # رجوع إلى SDPA (إعادة تحميل)
        return get_caption_pool(run_sanity=True), out

    def _vllm(self, pool: "CaptionPool", imgs: List[np.ndarray], tf_ips: float) -> Tuple["CaptionPool", Dict[str, Any]]:
        if not VLM_COMPAT.get("vllm_usable"): return pool, dict(tested=False, reason="vllm غير مثبّت أو لا GPU")
        if CONFIG.CAPTION_ENGINE not in ("auto", "vllm"): return pool, dict(tested=False, reason=f"CAPTION_ENGINE مضبوط يدويًا = {CONFIG.CAPTION_ENGINE}")
        prompt = getattr(pool.engines[0], "prompt_text", None)
        if not prompt: return pool, dict(tested=False, reason="لا نص تلقين متاح")
        out: Dict[str, Any] = dict(tested=True, transformers_img_per_s=tf_ips, selected=False); dtype = pool.plan.get("dtype") or "bf16"
        tp = int(pool.plan.get("n_gpu", 1)) if pool.plan.get("layout") == "shard" else 1
        release_caption_pool()                                                         # نحرّر ذاكرة GPU قبل أن تحجزها عملية vLLM
        r = bench_vllm_subprocess(imgs, prompt, dtype, tp)
        if not r.get("ok"): out["error"] = r.get("error")
        else:
            parsed = [process_raw_caption(PREFILL + t) for t in r["texts"]]; ok = sum(bool(p["training_prompt"]) and not p["review"] for p in parsed) / max(1, len(parsed))
            out.update(img_per_s=r["img_per_s"], caption_ok_rate=ok, tokens=r.get("tokens"))
            if r["img_per_s"] >= 1.30 * tf_ips and ok >= 0.8 and CONFIG.CAPTION_ENGINE == "auto":
                out["selected"] = True; CONFIG.CAPTION_ENGINE = "vllm"; self.applied["caption_engine"] = "vllm"
        try: return get_caption_pool(run_sanity=True), out                           # يحمّل vLLM إن اختير (مع رجوع تلقائي إلى Transformers إن فشل) وإلا Transformers
        except Exception as e:
            out["error_reload"] = _scrub(e, 200)
            if CONFIG.CAPTION_ENGINE == "vllm": CONFIG.CAPTION_ENGINE = "transformers"; out["selected"] = False; self.applied.pop("caption_engine", None)
            return get_caption_pool(run_sanity=True), out

    def _caption(self) -> Dict[str, Any]:
        if not self.do_caption: return dict(skipped="الوصف معطّل أو غير مطلوب في هذا القياس")
        src = self.accepted or self.frames
        if len(src) < 4: raise RuntimeError("لا صور مقبولة كافية لقياس الوصف")
        imgs = [prepare_model_input(f.rgb, QWEN_FACTOR) for f in src[: self.caption_images]]
        pool = get_caption_pool(run_sanity=True)
        if pool is None: return dict(skipped="لا مجمّع وصف")
        eng = pool.engines[0]; replicas = len(pool.engines) if pool.plan.get("layout") == "replicate" else 1; mock = isinstance(eng, MockEngine)
        avg_tok = float(np.mean([eng.visual_tokens(i.shape[1], i.shape[0]) + 400 + CONFIG.CAPTION_MAX_NEW_TOKENS for i in imgs]))
        runs = sweep_batches(eng, imgs, self.max_batch, start=CONFIG.BENCH_START_BATCH, vram_target=CONFIG.BENCH_VRAM_TARGET)
        pick = choose_batch(runs, avg_tok, CONFIG.BENCH_VRAM_TARGET, CONFIG.BENCH_STABLE_FRACTION)
        out: Dict[str, Any] = dict(engine=eng.name, precision=pool.plan.get("precision"), layout=pool.plan.get("layout"), replicas=replicas, mock=mock, attn=getattr(eng, "attn", None),
                                   model=eng.model_id, runs=runs, n_images=len(imgs), avg_item_tokens=avg_tok, **pick)
        if not pick["ok"]: out["error"] = pick.get("note"); return out
        STATE.benchmark["caption"] = dict(stable_batch=pick["stable_batch"], max_tested_batch=pick["max_tested_batch"], token_budget=pick["token_budget"]); retune_pool(pool)
        items = [CapItem(f"bench_{i:03d}", im) for i, im in enumerate(imgs)]
        t = time.perf_counter(); results = pool.submit(items).result(); dt = time.perf_counter() - t; met = pool.metrics()
        good = [r for r in results if r.parsed and r.parsed.get("training_prompt") and not r.parsed.get("review")]; flags: "collections.Counter" = collections.Counter()
        for r in results:
            if r.parsed: flags.update(r.parsed.get("caption_flags", []))
        out.update(pipeline_img_per_s=len(items) / dt, caption_ok_rate=len(good) / len(items), errors=sum(1 for r in results if r.error),
                   mean_words=float(np.mean([len(r.parsed["training_prompt"].split()) for r in good])) if good else 0.0, flags=dict(flags.most_common(6)),
                   oom_events=met.get("oom", 0), retries=met.get("retries", 0))
        self._show_samples(self.sample_jpegs, results)
        base = pick["best_img_per_s"]
        if self.try_flash and not mock: pool, out["flash_attn"] = self._flash(pool, imgs, pick["best_batch"], base)
        if out.get("flash_attn", {}).get("kept"): base = max(base, out["flash_attn"].get("fa2_img_per_s", base))
        if self.try_vllm and not mock: pool, out["vllm"] = self._vllm(pool, imgs, base)
        if out.get("vllm", {}).get("selected"):                                       # vLLM يدمج الطلبات داخليًا: دفعات إدخال أكبر تُبقي المجدول مشغولًا
            base = out["vllm"]["img_per_s"]; v_stable = max(pick["stable_batch"], 16); v_max = max(pick["max_tested_batch"], 32)
            STATE.benchmark["caption"] = dict(stable_batch=v_stable, max_tested_batch=v_max, token_budget=int(v_max * avg_tok)); out.update(stable_batch=v_stable, max_tested_batch=v_max, token_budget=int(v_max * avg_tok))
        else: STATE.benchmark["caption"] = dict(stable_batch=pick["stable_batch"], max_tested_batch=pick["max_tested_batch"], token_budget=pick["token_budget"])
        retune_pool(pool); eng2 = pool.engines[0]
        out.update(engine_final=eng2.name, attn_final=getattr(eng2, "attn", None), best_img_per_s=base, img_per_s_total=base * replicas)
        return out

    # ---- التطبيق + التقدير + الحفظ
    def _apply(self) -> None:
        a = self.applied; r = self.res
        ch = (r.get("decode") or {}).get("choice") or {}
        for b in ch.get("disable", []):
            DECODER_STATUS[b]["available"] = False; DECODER_STATUS[b]["reason"] = "benchmark: اختلاف لون عن FFmpeg أكبر من العتبة"
        a["disabled_decoders"] = list(ch.get("disable", []))
        a["decoder_health"] = {f"{b}|{c}": 2 for b, c in self.failing_pairs if DECODER_STATUS.get(b, {}).get("available") and b not in ch.get("disable", [])}
        for k in a["decoder_health"]: b, c = k.split("|"); DECODER_HEALTH[(b, c)] = 2             # لا نجرّب مفكّكًا فشل في القياس لهذا الترميز
        if ch.get("preference") and CONFIG.DECODER_PREFERENCE == "auto": CONFIG.DECODER_PREFERENCE = ch["preference"]
        a["decoder_preference"] = ch.get("preference") if CONFIG.DECODER_PREFERENCE == ch.get("preference") else None
        en = r.get("encode") or {}
        if en.get("ok") and en.get("chosen"): a["encoder_choice"] = en["chosen"]
        q = r.get("qc") or {}; cal = q.get("calibration")
        if cal:
            if CONFIG.BLUR_THRESHOLD is not None: q["calibration_applied"] = False; q["calibration_note"] = "BLUR_THRESHOLD مضبوط يدويًا"
            elif cal["n"] < 100: q["calibration_applied"] = False; q["calibration_note"] = f"عينات قليلة ({cal['n']}<100): تبقى العتبات الافتراضية المحافظة"
            else:
                QC.lap_floor, QC.ten_floor = float(cal["lap_floor"]), float(cal["ten_floor"]); q["calibration_applied"] = True
                STATE.benchmark["qc_calibration"] = dict(lap_floor=QC.lap_floor, ten_floor=QC.ten_floor, n=cal["n"], source="benchmark")
        c = r.get("caption") or {}
        if c.get("ok") and c.get("stable_batch"):
            a["caption_batch"] = dict(stable=c["stable_batch"], max=c["max_tested_batch"], token_budget=c["token_budget"])

    def finish(self) -> Dict[str, Any]:
        r = self.res
        r.update(version=BENCH_VERSION, created_at=time.time(), duration_s=time.time() - self.t0, env_fp=env_fingerprint(), bench_fp=bench_fingerprint(),
                 env=dict(gpus=[f"{g['name']} {g['vram_total_mb'] / 1024:.0f}GB cc{g.get('compute_cap')}" for g in ENV.get("gpus", [])], cpu_cores=ENV.get("cpu_cores"),
                          ram_total_gb=ENV.get("ram_total_gb"), python=ENV.get("python"), torch=ENV.get("torch")),
                 params=dict(n_videos=self.n, decode_frames=self.decode_frames, qc_frames=self.qc_frames, caption_images=self.caption_images, max_batch=self.max_batch, apply=self.apply),
                 ok_core=bool((r.get("decode") or {}).get("ok") and (r.get("qc") or {}).get("ok") and (not self.do_caption or (r.get("caption") or {}).get("ok", False) or (r.get("caption") or {}).get("skipped"))))
        if self.apply: self._apply()
        r["applied"] = self.applied
        est = compute_estimate(r, STATE.pending_videos(), session_hours=(CONFIG.MAX_RUNTIME_HOURS or 12.0)); r["estimate"] = est
        return r

def persist_benchmark(commit: bool = True) -> None:
    """يحفظ STATE.benchmark محليًا وعن بُعد. بعد تشغيل نظيف: لقطة control كاملة (متسقة لأن STATE لم يتغيّر بعد آخر commit) كي تحمّلها الجلسة التالية (load() يشترط progress+manifest).
    إن كان PIPELINE_DIRTY (تشغيل سابق فاشل: STATE قد يسبق المستودع) نلتزم benchmark_results.json وحده فلا نُعلن تقدّمًا غير مرفوع."""
    STATE.benchmark["updated_at"] = time.time(); STATE.benchmark = _jsonable(STATE.benchmark)
    STATE.save_local(force=True); atomic_write_json(Path(CONFIG.WORK_DIR) / "benchmark_results.json", STATE.benchmark)
    if not commit: return
    with STATE.lock:
        snap = ({f"{control_prefix()}/benchmark_results.json": json.dumps(STATE.benchmark, ensure_ascii=False, indent=2, default=str).encode()} if PIPELINE_DIRTY
                else {f"{control_prefix()}/{k}": v for k, v in STATE.control_bytes().items()})
    with_retries(lambda: STORE.commit(snap, "benchmark results"), "benchmark control commit", retries=3, backoff=CONFIG.DOWNLOAD_BACKOFF)

_DECODER_STATUS_ORIG = {k: dict(v) for k, v in DECODER_STATUS.items()}          # لقطة قبل أي تطبيق لقرارات القياس (لإعادة القياس في الجلسة نفسها)

def _reset_benchmark_effects() -> None:
    """إعادة القياس: نُلغي أثر القياس السابق (مفكّكات مُعطَّلة، صحة المفكّكات، تفضيلات FA2/vLLM/المفكّك التي ضبطها هو لا أنت) كي يُقيَّم كل شيء من جديد."""
    ap = (STATE.benchmark or {}).get("applied") or {}
    for k in ap.get("disabled_decoders", []):                                    # فقط ما عطّله القياس نفسه (لا نلمس أي تعديل آخر)
        if k in _DECODER_STATUS_ORIG: DECODER_STATUS[k] = dict(_DECODER_STATUS_ORIG[k])
    for k in ap.get("decoder_health", {}): DECODER_HEALTH.pop(tuple(k.split("|")), None)
    if ap.get("decoder_preference") and CONFIG.DECODER_PREFERENCE == ap["decoder_preference"]: CONFIG.DECODER_PREFERENCE = "auto"
    reload_model = False
    if ap.get("attn") and CONFIG.ATTN_IMPLEMENTATION == ap["attn"]: CONFIG.ATTN_IMPLEMENTATION = "auto"; reload_model = True
    if ap.get("caption_engine") and CONFIG.CAPTION_ENGINE == ap["caption_engine"]: CONFIG.CAPTION_ENGINE = "auto"; reload_model = True
    if reload_model: release_caption_pool()                                      # النموذج المحمَّل بإعدادات القياس السابق لا يصلح خط أساس

def run_benchmark(n_videos: int = 2, decode_frames: int = 480, qc_frames: int = 240, caption_images: Optional[int] = None, max_batch: Optional[int] = None, caption: bool = True,
                  try_flash_attn: bool = True, try_vllm: bool = True, upload_test: bool = True, only_videos: Optional[Sequence[str]] = None,
                  apply: bool = True, persist: bool = True) -> Dict[str, Any]:
    """يقيس كل شيء على أول n_videos فيديو معلّق. الفيديوهات المنزَّلة تبقى (ستُعالَج أولًا في التشغيل الفعلي فلا تُنزَّل مرتين). المدة: دقائق إلى ~30 دقيقة (تحميل Qwen + FlashAttention/vLLM الاختياريان)."""
    print("=" * 78); print(" BENCHMARK MODE — قياس فعلي (لا يكتب أي شيء من Dataset)"); print("=" * 78)
    _reset_benchmark_effects()
    b = BenchmarkRun(n_videos, decode_frames, qc_frames, caption_images, max_batch, caption, try_flash_attn, try_vllm, upload_test, only_videos, apply)
    saved_health, saved_stats = dict(DECODER_HEALTH), dict(EXTRACT_STATS)
    try:
        b.step("videos", b._videos); b.step("decode", b._decode)
    finally:
        DECODER_HEALTH.clear(); DECODER_HEALTH.update(saved_health); EXTRACT_STATS.clear(); EXTRACT_STATS.update(saved_stats)      # محاولات القياس لا تحتسب إخفاقات إنتاجية
    try:
        b.step("qc", b._qc); b.step("encode", b._encode); b.step("shard", b._shard); b.step("upload", b._upload)
        if b.do_caption:
            b.frames = []                                                              # حرّر ذاكرة الإطارات الكاملة قبل تحميل النموذج
            gc.collect(); b.step("caption", b._caption)
        else: b.res["caption"] = dict(ok=True, skipped="الوصف معطّل")
        res = b.finish()
    finally:
        b.frames, b.accepted = [], []; gc.collect()
    prev = {k: v for k, v in STATE.benchmark.items() if k in ("smoke_test",)}
    STATE.benchmark.update(_jsonable(res)); STATE.benchmark.update(prev)
    if persist:
        try: persist_benchmark(commit=True)
        except Exception as e: log.warning("تعذّر حفظ نتائج القياس عن بُعد (محفوظة محليًا): %s", _scrub(e, 160))
    print_benchmark_report(STATE.benchmark); print_estimate(STATE.benchmark["estimate"])
    return STATE.benchmark

def refresh_estimate(show: bool = False) -> Optional[Dict[str, Any]]:
    """يعيد حساب التقدير من القياسات المخزنة وإعداداتك الحالية (SAMPLING_RATIO / MAX_IMAGES_PER_VIDEO ...) دون إعادة القياس."""
    b = STATE.benchmark
    if not b or not (b.get("videos") or {}).get("items"): return None
    est = compute_estimate(b, STATE.pending_videos(), session_hours=(CONFIG.MAX_RUNTIME_HOURS or 12.0)); b["estimate"] = _jsonable(est)
    if show: print_estimate(est)
    return est

def benchmark_is_current() -> Tuple[bool, str]:
    b = STATE.benchmark or {}
    if not b.get("bench_fp"): return False, "لا يوجد قياس"
    if b.get("version") != BENCH_VERSION: return False, "قياس من إصدار أقدم (قديم: عتاد/نموذج مختلف) — يلزم قياس جديد"
    if b["bench_fp"] != bench_fingerprint(): return False, "العتاد/الحزم/إعدادات الصور أو النموذج تغيّرت منذ آخر قياس"
    return True, "ok"

def apply_stored_benchmark(verbose: bool = True) -> List[str]:
    """جلسة جديدة (kernel جديد): يعيد تطبيق قرارات القياس المخزنة (المفكّك، المشفّر، FA2/vLLM) إن كانت البصمة مطابقة. الإعدادات اليدوية لا تُلمس."""
    b = STATE.benchmark or {}; ap = b.get("applied") or {}; changed: List[str] = []
    if not ap: return changed
    ok, why = benchmark_is_current()
    if not ok:
        if verbose: print(f"benchmark محفوظ لكنه غير مطبَّق: {why} → أعد run_benchmark()")
        return changed
    for name in ap.get("disabled_decoders", []):
        if DECODER_STATUS.get(name, {}).get("available"): DECODER_STATUS[name]["available"] = False; DECODER_STATUS[name]["reason"] = "benchmark"; changed.append(f"disable:{name}")
    for k in ap.get("decoder_health", {}):
        bk, cd = k.split("|"); DECODER_HEALTH[(bk, cd)] = 2
    if ap.get("decoder_preference") and CONFIG.DECODER_PREFERENCE == "auto": CONFIG.DECODER_PREFERENCE = ap["decoder_preference"]; changed.append(f"decoder={ap['decoder_preference']}")
    if ap.get("encoder_choice") in ENCODERS and ENCODER_CHOICE != ap["encoder_choice"]:
        globals()["ENCODER_CHOICE"] = ap["encoder_choice"]; changed.append(f"encoder={ap['encoder_choice']}")
    if ap.get("attn") and CONFIG.ATTN_IMPLEMENTATION == "auto": CONFIG.ATTN_IMPLEMENTATION = ap["attn"]; changed.append(f"attn={ap['attn']}")
    if ap.get("caption_engine") and CONFIG.CAPTION_ENGINE == "auto": CONFIG.CAPTION_ENGINE = ap["caption_engine"]; changed.append(f"engine={ap['caption_engine']}")
    if verbose and changed: print("benchmark: أُعيد تطبيق قرارات القياس السابقة:", changed)
    return changed

APPLIED_FROM_BENCH = apply_stored_benchmark(verbose=False)
_cur, _why = benchmark_is_current()
print("benchmark mode ready | stored:", bool(STATE.benchmark.get("bench_fp")), "| current:", _cur, f"({_why})", "| re-applied:", APPLIED_FROM_BENCH or "—",
      "| run_benchmark(n_videos=2) لقياس فعلي")

### CELL 24D — Smoke Test (اختبار تكاملي صغير) + بوابة التشغيل الكامل
> يشغّل المسار الحقيقي كاملًا على أول فيديو صالح بعدد صغير من الصور (SMOKE_TEST_IMAGES): تنزيل ← ffprobe ← فك ← جودة/تكرار ← وصف Qwen ← شظايا ← تحقق ← رفع ذري ← control،
> ثم يحاكي جلسة جديدة (حالة طازجة من المخزن + مجلدات محلية مفرغة) ويستأنف ويتحقق من عدم التكرار واستمرار الترقيم.
> كل ذلك داخل مساحة معزولة `_smoke_test/` في المخزن نفسه (Hugging Face أو محلي) وحالة معزولة في الذاكرة، تُحذف بعد الانتهاء؛ حالة التشغيل الحقيقية لا تُمسّ.
> النجاح يُسجَّل في benchmark_results.json مع بصمة العتاد/الإعدادات؛ run_pipeline() يرفض البدء بدونه (CONFIG.REQUIRE_SMOKE_TEST) ويُعاد الاختبار تلقائيًا إن تغيّر العتاد أو إعدادات مؤثرة.

In [ ]:
# CELL 24D - Smoke Test (اختبار تكاملي صغير) + بوابة التشغيل الكامل
#: يشغّل المسار الحقيقي كاملًا على أول فيديو صالح بعدد صغير من الصور (SMOKE_TEST_IMAGES): تنزيل ← ffprobe ← فك ← جودة/تكرار ← وصف Qwen ← شظايا ← تحقق ← رفع ذري ← control،
#: ثم يحاكي جلسة جديدة (حالة طازجة من المخزن + مجلدات محلية مفرغة) ويستأنف ويتحقق من عدم التكرار واستمرار الترقيم.
#: كل ذلك داخل مساحة معزولة `_smoke_test/` في المخزن نفسه (Hugging Face أو محلي) وحالة معزولة في الذاكرة، تُحذف بعد الانتهاء؛ حالة التشغيل الحقيقية لا تُمسّ.
#: النجاح يُسجَّل في benchmark_results.json مع بصمة العتاد/الإعدادات؛ run_pipeline() يرفض البدء بدونه (CONFIG.REQUIRE_SMOKE_TEST) ويُعاد الاختبار تلقائيًا إن تغيّر العتاد أو إعدادات مؤثرة.
import copy

SMOKE_PREFIX = "_smoke_test/"
# إعدادات لا تؤثر على صحة النظام (حدود تشغيل، مسارات، عدد الفيديوهات، نسبة العيّنة...): تغييرها لا يُبطل نتيجة الاختبار.
# (نسبة العيّنة وسقف الصور لكل فيديو تحديدًا: مسار العمل بعد Benchmark هو ضبطهما دون إعادة الاختبار.)
SMOKE_IGNORED_CFG = {"WORK_DIR", "TEMP_DIR", "HF_TOKEN", "VIDEO_MANIFEST", "TOTAL_VIDEOS", "MAX_VIDEOS_THIS_RUN", "MAX_RUNTIME_HOURS", "STOP_MARGIN_MINUTES", "WORKER_ID", "NUM_WORKERS",
                     "DASHBOARD_REFRESH_SECONDS", "CHECKPOINT_EVERY_SECONDS", "REQUIRE_SMOKE_TEST", "SMOKE_TEST_IMAGES", "AUTO_RESUME", "KEEP_VIDEOS", "ENFORCE_DURATION_LIMITS",
                     "MIN_VIDEO_DURATION", "MAX_VIDEO_DURATION", "PREVIEW_SAMPLES_PER_BATCH", "ENABLE_RANDOM_SAMPLE_PREVIEW", "MAX_UNEXPECTED_VIDEO_ERRORS", "DISK_WAIT_MINUTES",
                     "PREFETCH_VIDEOS", "UPLOAD_PATIENCE_MINUTES", "DOWNLOAD_RETRIES", "UPLOAD_RETRIES", "DOWNLOAD_BACKOFF", "DOWNLOAD_TIMEOUT",
                     "SAMPLING_RATIO", "SAMPLING_SEED", "MAX_IMAGES_PER_VIDEO"}

def smoke_fingerprint() -> str:
    d = {k: v for k, v in dataclasses.asdict(CONFIG).items() if k not in SMOKE_IGNORED_CFG}
    return hashlib.sha1((env_fingerprint() + json.dumps(d, sort_keys=True, default=str)).encode()).hexdigest()[:12]

def smoke_test_passed() -> Tuple[bool, str]:
    st = (STATE.benchmark or {}).get("smoke_test")
    if not st: return False, "لم يُشغَّل اختبار الدخان بعد"
    if not st.get("passed"): return False, "فشل آخر اختبار دخان: " + str(st.get("failure") or "راجع checks في benchmark_results.json")[:200]
    if st.get("fingerprint") != smoke_fingerprint(): return False, "العتاد/الحزم/إعدادات مؤثرة تغيّرت منذ آخر اختبار ناجح (يُعاد الاختبار تلقائيًا)"
    return True, "ok"

def require_smoke_test() -> None:
    if not CONFIG.REQUIRE_SMOKE_TEST: return
    ok, why = smoke_test_passed()
    if not ok:
        root = " السبب الجذري: لا توجد قائمة فيديوهات (راجع رسالة الخلية 07 واضبط USER_SETTINGS في الخلية 03)؛ دونها لا يعمل Benchmark ولا اختبار الدخان." if not globals().get("MANIFEST_ROWS") else ""
        raise PipelineError(f"بوابة الأمان: {why}.{root} شغّل الخلية 24E (Preflight: تقيس وتختبر تلقائيًا) أولًا — أو run_smoke_test() — أو REQUIRE_SMOKE_TEST=False إن كنت تتحمّل المسؤولية.")

class PrefixStore(RemoteStore):
    """يغلّف مخزنًا حقيقيًا (HFStore أو LocalStore) فتُكتب وتُقرأ كل المسارات تحت بادئة معزولة. نفس مسار الرفع الحقيقي بلا أي تماس مع data/ وcontrol/ الحقيقيين."""
    def __init__(self, inner: RemoteStore, prefix: str):
        self.inner, self.prefix = inner, prefix; self.name = f"{inner.name}[{prefix}]"
    def _p(self, path: str) -> str: return self.prefix + path
    def exists(self, path): return self.inner.exists(self._p(path))
    def size(self, path): return self.inner.size(self._p(path))
    def sizes(self, paths):
        r = self.inner.sizes([self._p(p) for p in paths]); return {p: r.get(self._p(p)) for p in paths}
    def read_bytes(self, path): return self.inner.read_bytes(self._p(path))
    def read_range(self, path, start, length): return self.inner.read_range(self._p(path), start, length)
    def list_files(self, prefix=""): return sorted(f[len(self.prefix):] for f in self.inner.list_files(self.prefix + prefix))
    def commit(self, adds, message, deletes=()):
        self.inner.commit({self._p(k): v for k, v in adds.items()}, "[smoke-test] " + message, [self._p(d) for d in deletes])
    def verify(self, path, size, sha256=None): return self.inner.verify(self._p(path), size, sha256)

def _scan_secrets(blobs: Iterable[bytes]) -> List[str]:
    """أي سرّ مسجَّل (HF_TOKEN) ظهر في أي من البيانات؟ يعيد قائمة بمواضع الظهور (بلا طباعة السرّ نفسه)."""
    secrets = [s.encode("utf-8") for s in list(_SECRETS) if s]; hits: List[str] = []
    for i, b in enumerate(blobs):
        if not b: continue
        for sec in secrets:
            if sec in b: hits.append(f"blob#{i}")
    return hits

def _tar_records(path: os.PathLike) -> Dict[str, Dict[str, Any]]:
    out: Dict[str, Dict[str, Any]] = {}
    with tarfile.open(path, "r:") as tf:
        for m in tf:
            if m.isfile() and m.name.endswith(".json"): out[m.name[:-5]] = json.loads(tf.extractfile(m).read().decode("utf-8"))
    return out

class _Sandbox:
    """يعزل كل ما قد يلمسه تشغيل المسار الحقيقي: الحالة، المخزن، مجلدات الشظايا/control، حالة إزالة التكرار العالمية، مؤشرات التشغيل. يُعيد كل شيء في الخروج حتى عند الأخطاء."""
    NAMES = ("STATE", "STORE", "SHARD_DIR", "CONTROL_DIR", "MANIFEST_ROWS", "PIPELINE_DIRTY", "LAST_PIPELINE", "PREV_UPLOADER", "CROSS_MIH", "SMOKE_RUNNING")
    def __enter__(self) -> "_Sandbox":
        g = globals(); self.saved = {n: g.get(n) for n in self.NAMES}
        ed = EXACT_DEDUPER; self.dedup = (ed._video, ed._global, ed.hits); self.keep = CONFIG.KEEP_VIDEOS; self.auto = CONFIG.AUTO_RESUME; self.extract = dict(EXTRACT_STATS)
        ed._video, ed._global, ed.hits = set(), set(), 0
        g["CROSS_MIH"] = MIHIndex(d=CONFIG.DEDUP_HAMMING_THRESHOLD) if CONFIG.CROSS_VIDEO_DEDUP else None
        g["SMOKE_RUNNING"] = True
        CONFIG.KEEP_VIDEOS = True          # الفيديو المنزَّل يبقى كي يعيد التشغيل الحقيقي استعماله بدل تنزيله ثانيةً
        CONFIG.AUTO_RESUME = True          # اختبار الاستئناف يتطلبه حتى لو اخترتَ AUTO_RESUME=False للتشغيل الحقيقي (الحالة هنا معزولة)
        return self
    def __exit__(self, *exc) -> bool:
        g = globals()
        for n, v in self.saved.items(): g[n] = v
        ed = EXACT_DEDUPER; ed._video, ed._global, ed.hits = self.dedup
        CONFIG.KEEP_VIDEOS, CONFIG.AUTO_RESUME = self.keep, self.auto
        EXTRACT_STATS.clear(); EXTRACT_STATS.update(self.extract)
        return False

class SmokeTest:
    def __init__(self, images: int, max_attempt_videos: int = 3):
        self.N = max(8, int(images)); self.max_attempts = max(1, int(max_attempt_videos)); self.t0 = time.time()
        self.real_state, self.real_store, self.rows = STATE, STORE, list(MANIFEST_ROWS)
        self.root = Path(CONFIG.WORK_DIR) / "_smoke"; self.sb_store = PrefixStore(self.real_store, SMOKE_PREFIX)
        self.checks: List[Dict[str, Any]] = []; self.vid: Optional[str] = None; self.info: Dict[str, Any] = {}; self.skipped: List[str] = []

    # ---- أدوات
    def check(self, name: str, ok: bool, detail: Any = "") -> bool:
        self.checks.append(dict(name=name, ok=bool(ok), detail=_scrub(detail, 300)))
        print(("  ✔ " if ok else "  ✖ ") + name + (f"  [{_scrub(detail, 160)}]" if detail != "" else ""), flush=True)
        return bool(ok)

    def cleanup_remote(self) -> bool:
        try:
            files = self.real_store.list_files(SMOKE_PREFIX)
            if files: self.real_store.commit({}, "smoke test cleanup", deletes=files)
            return not self.real_store.list_files(SMOKE_PREFIX)
        except Exception as e:
            log.warning("تعذّر حذف مساحة الاختبار %s من المستودع (احذفها يدويًا): %s", SMOKE_PREFIX, _scrub(e, 160)); return False

    def new_state(self, load: bool) -> "StateManager":
        shutil.rmtree(self.root / "control", ignore_errors=True); (self.root / "control").mkdir(parents=True)
        st = StateManager(self.sb_store, self.root / "control")
        st.benchmark = {k: copy.deepcopy(v) for k, v in (self.real_state.benchmark or {}).items() if k != "smoke_test"}
        resumed = st.load() if load else False
        if resumed: st.fail_unfinished_pending_for_retry()
        st.init_videos(self.rows); globals()["STATE"] = st; globals()["MANIFEST_ROWS"] = self.rows
        return st

    def run_phase(self, vid: str, images: int, label: str) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
        print(f"\n— {label}: حتى {images} صورة من الفيديو '{vid}' ...", flush=True)
        try:
            res = run_pipeline(max_videos=1, max_accepted=images, only_videos=[vid], max_runtime_hours=0.6, show_ui=True)
        except KeyboardInterrupt: raise
        except Exception as e:
            return None, _scrub(f"{type(e).__name__}: {e}", 400)
        if res.get("stop_reason") == "KeyboardInterrupt": raise KeyboardInterrupt("أُوقف اختبار الدخان يدويًا")
        return res, None

    def read_store(self, tag: str) -> Dict[str, Any]:
        """يعيد تنزيل كل شظية من المخزن (كما سيفعل أي مستهلك لاحقًا) ويتحقق منها ومن الفهارس وملفات control."""
        S = self.sb_store; files = S.list_files(""); vdir = self.root / f"verify_{tag}"; shutil.rmtree(vdir, ignore_errors=True); vdir.mkdir(parents=True)
        shard_files = sorted(f for f in files if f.startswith("data/") and f.endswith(".tar")); out: Dict[str, Any] = dict(shards=[], records={}, errors=[], index_ok=True, files=files)
        for sp in shard_files:
            local = vdir / Path(sp).name
            if not S.download(sp, local): out["errors"].append(f"تعذّر تنزيل {sp}"); continue
            kind = "review" if Path(sp).name.startswith("review") else "train"; v = validate_tar_shard(local, kind)
            out["errors"] += [f"{sp}: {e}" for e in v["errors"][:3]]
            recs = _tar_records(local) if v["ok"] else {}
            out["shards"].append(dict(path=sp, kind=kind, n=v["n_samples"], size=local.stat().st_size, ok=v["ok"])); out["records"].update({k: dict(r, _shard=sp, _kind=kind) for k, r in recs.items()})
            name = Path(sp).stem; ip = next((f for f in (f"index/{name}.parquet", f"index/{name}.jsonl") if f in files), None)
            if ip is None: out["index_ok"] = False; out["errors"].append(f"لا فهرس للشظية {name}")
            else:
                b = S.read_bytes(ip) or b""
                try:
                    if ip.endswith(".jsonl"): rows = sum(1 for l in b.decode("utf-8").splitlines() if l.strip())
                    else:
                        import pyarrow.parquet as pq
                        rows = pq.read_table(io.BytesIO(b)).num_rows
                    if rows != v["n_samples"]: out["index_ok"] = False; out["errors"].append(f"فهرس {name}: {rows} صفًا ≠ {v['n_samples']} عينة")
                except Exception as e: out["index_ok"] = False; out["errors"].append(f"فهرس {name} غير قابل للقراءة: {type(e).__name__}")
        shutil.rmtree(vdir, ignore_errors=True)
        return out

    # ---- المحاولة على فيديو واحد: True نجاح كل الفحوص، False فشل فحص، None الفيديو نفسه غير صالح (جرّب التالي)
    def attempt(self, vid: str) -> Optional[bool]:
        self.checks = []; self.vid = vid; N = self.N
        self.cleanup_remote(); shutil.rmtree(self.root, ignore_errors=True); (self.root / "shards").mkdir(parents=True)
        globals()["STORE"] = self.sb_store; globals()["SHARD_DIR"] = self.root / "shards"; globals()["CONTROL_DIR"] = self.root / "control"; globals()["PIPELINE_DIRTY"] = False
        st = self.new_state(load=False); shown0 = PREVIEW.shown
        res, err = self.run_phase(vid, N, "المرحلة 1 (جلسة جديدة)")
        if res is None:
            self.check("المرحلة 1: المسار الحقيقي اكتمل بلا خطأ قاتل", False, err); return False
        v = st.videos[vid]; done1 = v["state"] == "done"
        if res["videos_failed"] or res["videos_skipped"] or v["state"] in ("failed", "failed_permanent", "skipped") or (done1 and res["written"] == 0):
            why = v.get("error") or ("كل إطارات الفيديو رُفضت في الفحص" if done1 else "غير معروف")
            self.skipped.append(f"{vid}: {_scrub(why, 160)}"); print(f"  ⚠ الفيديو '{vid}' غير صالح للاختبار ({_scrub(why, 140)}) → أجرّب الفيديو التالي"); return None
        self.check("المرحلة 1: المسار الحقيقي اكتمل بلا خطأ قاتل", True, f"stop={res['stop_reason']} | {res['elapsed_s']:.0f}s")
        self.check("لا أخطاء رفع/تحقق ولا مشاكل تشغيل", res["uploader"]["failures"] == 0 and PIPELINE_DIRTY is False and not res.get("fatal"), f"uploader={res['uploader']['failures']} dirty={PIPELINE_DIRTY}")
        need = 1 if done1 else max(1, int(0.9 * N))                    # فيديو انتهى كله قبل بلوغ العدد المطلوب: يكفي أنه أنتج صورًا (يُسجَّل أن الاستئناف لم يُختبر)
        self.check(f"عدد الصور كافٍ (≥ {need} من {N})", res["written"] >= need, f"written={res['written']} inspected={res['inspected']} accepted={res['accepted']} stop={res['stop_reason']}")
        rs = self.read_store("p1"); recs = rs["records"]; n_tr = sum(1 for r in recs.values() if r["_kind"] == "train")
        self.check("الشظايا أُعيد تنزيلها من المخزن ونجح التحقق الكامل (TAR/صور/JSON/checksum/أبعاد)", bool(rs["shards"]) and all(s["ok"] for s in rs["shards"]) and not rs["errors"], rs["errors"][:2] or f"{len(rs['shards'])} شظية، {len(recs)} عينة")
        self.check("الفهارس موجودة بعدد الصفوف الصحيح", rs["index_ok"], f"format={INDEX_EXT}")
        self.check("عدد العينات في المخزن = المكتوب = مجموع المقبول في الحالة", len(recs) == res["written"] == int(st.recompute()["accepted_images"]), f"store={len(recs)} written={res['written']} state={st.progress['accepted_images']}")
        self.check("لا sample_id مكرر", len(recs) == len({r["sample_id"] for r in recs.values()}))
        geo = [k for k, r in recs.items() if not (r["width"] <= r["original_width"] and r["height"] <= r["original_height"]
                                                 and abs(r["width"] / r["height"] - r["original_width"] / r["original_height"]) <= 0.012 * (r["original_width"] / r["original_height"]))]
        small = [k for k, r in recs.items() if r["width"] < CONFIG.MIN_IMAGE_WIDTH or r["height"] < CONFIG.MIN_IMAGE_HEIGHT]
        self.check("الصور المخزّنة: لا قصّ ولا upscale ولا تغيير في نسبة الأبعاد (≤ الأصل، النسبة ثابتة) ولا أقل من الحد الأدنى", not geo and not small, (geo + small)[:3])
        sizes = S_sizes = self.sb_store.sizes([s["repo_path"] for s in st.shards]); self.check("كل شظية مسجّلة في الحالة موجودة في المخزن بالحجم نفسه وبعلامة upload_ok", all(S_sizes[s["repo_path"]] == s["size"] and s.get("upload_ok") for s in st.shards), f"{len(st.shards)} شظية")
        cfiles = {n: self.sb_store.read_bytes(f"{control_prefix()}/{n}") for n in CONTROL_FILES}
        try: prog_ok = bool(json.loads(cfiles["progress.json"] or b"{}")) and all((json.loads(l) if l.strip() else True) for l in (cfiles["videos_manifest.jsonl"] or b"").decode("utf-8").splitlines())
        except Exception: prog_ok = False
        self.check("ملفات control الستة موجودة وقابلة للقراءة", all(b is not None for b in cfiles.values()) and prog_ok, [k for k, b in cfiles.items() if b is None] or ("JSON غير صالح" if not prog_ok else ""))    # failed_videos.jsonl فارغ بلا إخفاقات: طبيعي
        try: cs = [c["name"] for c in json.loads(cfiles["completed_shards.json"])]
        except Exception: cs = None
        self.check("completed_shards.json البعيد = شظايا الحالة", cs == [s["name"] for s in st.shards])
        self.check("لا ملفات شظايا محلية متبقية بعد الرفع", not list((self.root / "shards").rglob("*.tar")) and not list((self.root / "shards").rglob("*.partial")))
        # الإطارات من خطة العيّنة الحتمية (يتحقق أن المفكّك الحقيقي يطابق ffprobe/الخطة)
        try:
            vr = st.videos[vid]
            meta = analyze_video(str(DOWNLOADER.ensure({"video_id": vid, "url": vr["url"]}, vr.get("sha256"), vr.get("expected_size"))), vid); sel = set(int(i) for i in build_plan(meta).sel)
            bad = [r["frame_index"] for r in recs.values() if r["frame_index"] not in sel]
            self.check("كل إطار محفوظ ينتمي إلى خطة العيّنة الحتمية للفيديو (المفكّك يطابق الخطة)", not bad, f"decoder={sorted({r.get('decoder') for r in recs.values()})} خارج الخطة: {bad[:3]}")
            self.info["decoder"] = sorted({r.get("decoder") for r in recs.values()})
        except Exception as e: self.check("تحليل الفيديو للتحقق من الخطة", False, f"{type(e).__name__}: {e}")
        if CONFIG.ENABLE_CAPTIONING:
            mock = any(r.get("caption_model") == "mock-test-only" for r in recs.values()); ok_rate = n_tr / max(1, len(recs))
            words = [r.get("caption_words", 0) for r in recs.values() if r["_kind"] == "train"]
            self.info["captions"] = dict(ok_rate=ok_rate, mean_words=(float(np.mean(words)) if words else 0.0), mock=mock, model=sorted({r.get("caption_model") for r in recs.values()}))
            self.check(f"جودة الوصف: ≥ 50% من الصور دخلت شظايا التدريب (لا للمراجعة)", ok_rate >= 0.5, f"{n_tr}/{len(recs)} | متوسط {self.info['captions']['mean_words']:.0f} كلمة")
            self.check("كل سجل تدريب يحمل training_prompt غير فارغ ونموذج الوصف الصحيح", all(r.get("training_prompt") and (r.get("caption_model") == (CONFIG.MODEL_ID if not mock else "mock-test-only")) for r in recs.values() if r["_kind"] == "train"),
                       "MOCK (ليس Qwen)" if mock else CONFIG.MODEL_ID)
            if CONFIG.ENABLE_RANDOM_SAMPLE_PREVIEW and n_tr: self.check("عُرضت معاينة عشوائية حقيقية (صورة + وصف + بيانات)", PREVIEW.shown > shown0, f"+{PREVIEW.shown - shown0}")
        cur1 = int(v["cursor"])
        logp = Path(CONFIG.WORK_DIR) / "pipeline.log"; tail = logp.read_bytes()[-3_000_000:] if logp.exists() else b""
        self.check("لا Token في ملفات control ولا في سجل التشغيل", not _scan_secrets(list(cfiles.values()) + [tail]), f"{len(_SECRETS)} سرّ مُسجَّل" if _SECRETS else "لا Token مستعمل (DRY_RUN/عام)")
        n1, shard_names1, accepted1 = len(recs), [s["name"] for s in st.shards], int(v["accepted"])
        # ---- المرحلة 2: جلسة جديدة تمامًا (حالة من المخزن، مجلدات محلية فارغة) ⇒ استئناف
        if done1:
            self.info["resume_exercised"] = False
            self.check("الاستئناف", True, "اكتمل الفيديو داخل المرحلة 1 (مسار الاستئناف لم يُختبر: فيديو قصير؟)"); return all(c["ok"] for c in self.checks)
        self.info["resume_exercised"] = True; shutil.rmtree(self.root / "shards", ignore_errors=True); shutil.rmtree(self.root / "control", ignore_errors=True)
        (self.root / "shards").mkdir(parents=True); st2 = self.new_state(load=True)
        self.check("المرحلة 2: الجلسة الجديدة استعادت الحالة من المخزن (cursor والشظايا والعدّادات)", st2.resumed and st2.videos[vid]["cursor"] == cur1 and [s["name"] for s in st2.shards] == shard_names1 and st2.videos[vid]["accepted"] == accepted1,
                   f"resumed={st2.resumed} cursor={st2.videos[vid]['cursor']}/{cur1}")
        N2 = max(8, N // 2); res2, err2 = self.run_phase(vid, N2, "المرحلة 2 (استئناف في جلسة جديدة)")
        if res2 is None: self.check("المرحلة 2: الاستئناف اكتمل بلا خطأ قاتل", False, err2); return False
        self.check("المرحلة 2: الاستئناف اكتمل بلا خطأ قاتل", res2["uploader"]["failures"] == 0 and PIPELINE_DIRTY is False, f"written={res2['written']}")
        rs2 = self.read_store("p2"); recs2 = rs2["records"]; new = {k: r for k, r in recs2.items() if k not in recs}
        self.check("شظايا المرحلتين سليمة بعد إعادة التنزيل والتحقق", bool(rs2["shards"]) and all(s["ok"] for s in rs2["shards"]) and not rs2["errors"], rs2["errors"][:2])
        self.check("لا sample_id مكرر بين المرحلتين ولا تكرار لأي إطار قديم", len(recs2) == n1 + res2["written"] and len(new) == res2["written"] > 0 and set(recs) <= set(recs2), f"{n1} + {res2['written']} = {len(recs2)}")
        self.check("كل إطار جديد بعد مؤشر الاستئناف (إطارات التهيئة seed لا تُخرَج)", all(r["frame_index"] > cur1 for r in new.values()), f"cursor={cur1}")
        self.check("ترقيم الشظايا استمر ولم يُعاد استعمال اسم", len({s['name'] for s in st2.shards}) == len(st2.shards) > len(shard_names1) and not (set(shard_names1) - {s["name"] for s in st2.shards}), str([s["name"] for s in st2.shards]))
        self.check("عدّاد الفيديو في الحالة = مجموع المخزن", st2.videos[vid]["accepted"] == len(recs2), f"{st2.videos[vid]['accepted']} vs {len(recs2)}")
        return all(c["ok"] for c in self.checks)

    def run(self) -> Dict[str, Any]:
        pend = [v["video_id"] for v in self.real_state.pending_videos()]
        print("=" * 78); print(f" SMOKE TEST — اختبار تكاملي حقيقي مصغّر ({self.N} صورة) داخل {SMOKE_PREFIX} في المخزن: {self.real_store.name}"); print("=" * 78)
        if not pend:
            print("  لا فيديوهات معلّقة: لا شيء لاختباره (لا يُسجَّل شيء ولا تتأثر البوابة)."); return dict(passed=None, note="no pending videos")
        if CONFIG.ENABLE_CAPTIONING and not (CONFIG.ALLOW_MOCK_CAPTIONER and not ENV.get("gpus")) and not RESOURCE_PLAN.get("has_gpu"):
            print("  ⚠ لا GPU في هذه الجلسة: تحميل Qwen سيفشل (انظر الرسالة). شغّل الجلسة بمسرّع GPU أو عطّل ENABLE_CAPTIONING عن قصد.")
        passed: Optional[bool] = False; failure = ""
        try:
            with _Sandbox():
                for vid in pend[: self.max_attempts]:
                    passed = self.attempt(vid)
                    if passed is not None: break
                if passed is None: failure = "كل الفيديوهات المرشّحة للاختبار فشلت/تُخطّيت: " + " | ".join(self.skipped)[:300]; self.check("وجود فيديو صالح للاختبار", False, failure); passed = False
        except KeyboardInterrupt:
            self.cleanup_remote(); shutil.rmtree(self.root, ignore_errors=True); raise
        except Exception as e:
            failure = _scrub(f"{type(e).__name__}: {e}", 300); self.check("خطأ غير متوقع في اختبار الدخان", False, failure); passed = False
        clean = self.cleanup_remote()
        if passed: shutil.rmtree(self.root, ignore_errors=True)
        if not failure and not passed: failure = next((f"{c['name']} — {c['detail']}" for c in self.checks if not c["ok"]), "فشل غير محدد")
        self.info["cleanup_remote_ok"] = clean
        return self.finish(bool(passed), failure=failure)

    def finish(self, passed: bool, note: str = "", failure: str = "") -> Dict[str, Any]:
        r = dict(passed=bool(passed), finished_at=time.time(), duration_s=time.time() - self.t0, fingerprint=smoke_fingerprint(), env_fp=env_fingerprint(), store=self.real_store.name,
                 images=self.N, video=self.vid, checks=self.checks, note=note, failure=failure or None, skipped_videos=self.skipped, **self.info)
        print("=" * 78)
        print((" ✔ SMOKE TEST PASSED" if passed else " ✖ SMOKE TEST FAILED") + f"  | {sum(c['ok'] for c in self.checks)}/{len(self.checks)} فحصًا | {human_time(r['duration_s'])} | المخزن: {r['store']}")
        if passed: print("   النظام يعمل بنهايته (حقيقيًا) على عتادك وإعداداتك. يمكن الآن تشغيل التشغيل الكامل." + ("  ⚠ المخزن محلي: رفع HF الحقيقي لم يُختبر." if self.real_store.name == "local" else ""))
        else: print(f"   السبب: {failure}\n   لن يبدأ التشغيل الكامل (REQUIRE_SMOKE_TEST) قبل إصلاحه وإعادة run_smoke_test().")
        if not self.info.get("cleanup_remote_ok", True): print(f"   ⚠ تعذّر حذف {SMOKE_PREFIX} من المستودع: احذفه يدويًا (ملفات صغيرة).")
        print("=" * 78)
        return r

def run_smoke_test(images: Optional[int] = None, persist: bool = True, max_attempt_videos: int = 3) -> Dict[str, Any]:
    """اختبار تكاملي حقيقي مصغّر + محاكاة استئناف. يُسجَّل في STATE.benchmark['smoke_test'] ويُحفظ؛ يفتح البوابة فقط عند النجاح بنفس العتاد/الإعدادات."""
    t = SmokeTest(images or CONFIG.SMOKE_TEST_IMAGES, max_attempt_videos)
    res = t.run()
    if res.get("passed") is None: return res
    STATE.benchmark["smoke_test"] = _jsonable(res)
    if persist:
        try: persist_benchmark(commit=True)
        except Exception as e: log.warning("تعذّر حفظ نتيجة اختبار الدخان عن بُعد (محفوظة محليًا): %s", _scrub(e, 160))
    return res

_ok, _why = smoke_test_passed()
print("smoke test ready | gate:", "REQUIRED" if CONFIG.REQUIRE_SMOKE_TEST else "off", "| passed for current setup:", _ok, f"({_why})", "| run_smoke_test() لتشغيله")

### CELL 24E — Preflight (قائمة الجاهزية) = فحوص + Benchmark (إن لزم) + Smoke Test (إن لزم)
> تنفيذ هذه الخلية وحدها يجيب: هل النظام جاهز للتشغيل الكامل؟ وإن لم يكن فلماذا. لا تبدأ أي معالجة جماعية؛ الخلية 24F ترفض البدء قبل نجاح اختبار الدخان.
> الترتيب مقصود: القياس أولًا (يضبط المفكّك/الدفعة/المحرّك على عتادك) ثم اختبار الدخان يجرّب الإعدادات المضبوطة فعلًا من الطرف إلى الطرف.

▶ **شغّل هذه الخلية قبل التشغيل الكامل**: تجيب هل النظام جاهز؟ (فحوص + Benchmark + Smoke Test). لا تبدأ أي معالجة جماعية.

In [ ]:
# CELL 24E - Preflight (قائمة الجاهزية) = فحوص + Benchmark (إن لزم) + Smoke Test (إن لزم)
#: تنفيذ هذه الخلية وحدها يجيب: هل النظام جاهز للتشغيل الكامل؟ وإن لم يكن فلماذا. لا تبدأ أي معالجة جماعية؛ الخلية 24F ترفض البدء قبل نجاح اختبار الدخان.
#: الترتيب مقصود: القياس أولًا (يضبط المفكّك/الدفعة/المحرّك على عتادك) ثم اختبار الدخان يجرّب الإعدادات المضبوطة فعلًا من الطرف إلى الطرف.

def _pf(items: List[Dict[str, Any]], ok: Optional[bool], name: str, detail: str = "") -> Optional[bool]:
    items.append(dict(name=name, ok=ok, detail=detail))
    print(f"  {({True: '✔', False: '✖', None: '⚠'})[ok]} {name}" + (f"  — {detail}" if detail else ""), flush=True)
    return ok

def _hardware_items(items: List[Dict[str, Any]]) -> None:
    gpus = ENV.get("gpus") or []
    mock_ok = bool(CONFIG.ALLOW_MOCK_CAPTIONER and not gpus)
    if gpus:
        _pf(items, True, "GPU", ", ".join(f"{g['name']} {g['vram_total_mb'] / 1024:.0f}GB" for g in gpus))
        _cc = ENV.get("cuda_check") or {}
        if _cc.get("ok") is False and not (mock_ok and not CONFIG.ENABLE_CAPTIONING): _pf(items, False, "نواة CUDA", _cc.get("note") or "فشل تشغيل نواة CUDA بسيطة")
        elif _cc.get("note"): _pf(items, None, "نواة CUDA", _cc["note"])
    elif not CONFIG.ENABLE_CAPTIONING: _pf(items, None, "GPU", "لا GPU (مقبول: الوصف معطّل عن قصد؛ الفك على CPU)")
    elif mock_ok: _pf(items, None, "GPU", "لا GPU — وضع MOCK للاختبار فقط: الأوصاف ليست من Qwen ولا تصلح كبيانات تدريب")
    else: _pf(items, False, "GPU", "لا GPU: لا يمكن تشغيل Qwen3-VL-8B عمليًا. فعّل مسرّع GPU في إعدادات الجلسة (أو ENABLE_CAPTIONING=False عن قصد)")
    plans = CAPTION_PLANS or []
    if CONFIG.ENABLE_CAPTIONING and gpus:
        if plans: _pf(items, True, "خطط تحميل Qwen (بالترتيب)", " → ".join(f"{p['precision']}/{p['layout']}" for p in plans[:3]))
        else: _pf(items, False, "خطط تحميل Qwen", "لا خطة تلائم ذاكرة GPU المتاحة")
    ff = ENV.get("ffmpeg")
    _pf(items, bool(ff), "FFmpeg/ffprobe", str(ff) if ff else "غير موجود: لا يمكن فك الفيديو (على Kaggle متوفر افتراضيًا)")
    avail = [k for k, v in DECODER_STATUS.items() if v.get("available")]
    _pf(items, bool(avail), "المفكّكات المتاحة", ", ".join(f"{k}" for k in avail) or "لا يوجد")
    need = CONFIG.MIN_FREE_DISK_GB
    wd, vd = free_disk_gb(CONFIG.WORK_DIR), free_disk_gb(VIDEO_DIR)
    _pf(items, wd >= need and vd >= need, "مساحة القرص", f"WORK {wd:.1f}GB | TEMP {vd:.1f}GB (الحد الأدنى {need:.0f}GB)")
    miss = [k for k, v in DEP_STATUS.items() if str(v).startswith(("FAILED", "missing"))]
    if miss: _pf(items, None, "حزم غير متوفرة", ", ".join(miss) + " (تُستعمل بدائل موثّقة إن وُجدت؛ انظر README)")

def preflight(run_benchmark_if_needed: bool = True, run_smoke_if_needed: bool = True, force_benchmark: bool = False, force_smoke: bool = False,
              benchmark_kwargs: Optional[Dict[str, Any]] = None, smoke_kwargs: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    items: List[Dict[str, Any]] = []; t0 = time.time(); bench_ran = smoke_ran = False
    print("=" * 78); print(" PREFLIGHT — هل النظام جاهز للتشغيل الكامل؟"); print("=" * 78)
    n_rows, pend = len(MANIFEST_ROWS), STATE.pending_videos()
    _pf(items, n_rows > 0, "المانيفست", f"{n_rows} فيديو صالح | {len(STATE.videos)} في الحالة | {len(pend)} معلّق" if n_rows else "لا فيديوهات: اضبط USER_SETTINGS في أعلى الخلية 03 (VIDEO_MANIFEST مجلد/ملف، أو VIDEO_URLS) ثم شغّل 03 و07 — انظر رسالة الخلية 07")
    if n_rows:
        nl = sum(not r.get("license") for r in MANIFEST_ROWS)
        if nl: _pf(items, None, "الرخصة/المصدر", f"{nl} فيديو بلا حقل license (يُسجَّل فارغًا في كل JSON — الأفضل إكماله)")
    if STORE.name == "huggingface":
        vis = getattr(STORE, "private", None); word = {True: "خاص", False: "عام", None: "الخصوصية غير معروفة"}[vis]
        warn_pub = vis is False and CONFIG.HF_PRIVATE_REPO          # طلبتَ خاصًا لكن المستودع موجود مسبقًا وعام: ما يُرفع سيكون مرئيًا للجميع
        _pf(items, None if warn_pub else True, "المخزن البعيد", f"Hugging Face dataset: {CONFIG.HF_DATASET_REPO} ({word})"
            + ("  ⚠ المستودع عام رغم HF_PRIVATE_REPO=True (موجود مسبقًا): كل ما يُرفع سيكون مرئيًا للجميع — غيّر خصوصيته من Settings في Hugging Face إن لم يكن مقصودًا" if warn_pub else ""))
    else: _pf(items, None, "المخزن البعيد", f"محلي ({STORE.name}) — لن يُرفع شيء إلى Hugging Face. أضِف HF_TOKEN إلى Kaggle Secrets واضبط DRY_RUN=False")
    _hardware_items(items)
    hard_fail = [i["name"] for i in items if i["ok"] is False]
    if hard_fail:
        print("-" * 78); print(f" ✖ NOT READY — أصلح أولًا: {', '.join(hard_fail)}  (لم يُشغَّل Benchmark ولا Smoke Test قبل ذلك)"); print("=" * 78)
        return dict(ready=False, items=items, reasons=hard_fail, benchmark_ran=False, smoke_ran=False, duration_s=time.time() - t0)
    if not pend:
        _pf(items, None, "Benchmark / Smoke Test", "لا فيديوهات معلّقة: لا شيء لقياسه أو اختباره")
        print("-" * 78); print(" ✔ لا عمل متبقٍّ: كل الفيديوهات معالجة أو مُتخطّاة. انتقل إلى الخلايا 25–27 (فحص وإحصاءات)."); print("=" * 78)
        return dict(ready=True, items=items, reasons=[], benchmark_ran=False, smoke_ran=False, nothing_pending=True, duration_s=time.time() - t0)
    # ---- Benchmark
    cur, why = benchmark_is_current()
    if force_benchmark or (not cur and run_benchmark_if_needed):
        print(f"\n— Benchmark: {'مُجبَر' if force_benchmark else why} → قياس فعلي (دقائق؛ لا يكتب أي شيء من Dataset) ...", flush=True)
        try: run_benchmark(**(benchmark_kwargs or {})); bench_ran = True
        except KeyboardInterrupt: raise
        except Exception as e:
            log.error("Benchmark فشل: %r", e); _pf(items, None, "Benchmark", f"فشل ({_scrub(f'{type(e).__name__}: {e}', 200)}) — يستمر Smoke Test بالإعدادات الافتراضية؛ راجع السجل")
        else: _pf(items, True, "Benchmark", "قيس وطُبِّق وحُفظ في control/benchmark_results.json")
    elif cur: _pf(items, True, "Benchmark", "قياس محفوظ ومطابق للعتاد/الإعدادات الحالية (يُعاد تطبيقه تلقائيًا)")
    else: _pf(items, None, "Benchmark", f"غير مُشغَّل ({why}) — موصى به: preflight(force_benchmark=True)")
    # ---- Smoke test
    ok, why = smoke_test_passed()
    if force_smoke or (not ok and run_smoke_if_needed):
        if not force_smoke: print(f"\n— Smoke Test: {why}", flush=True)
        r = run_smoke_test(**(smoke_kwargs or {})); smoke_ran = True; ok, why = smoke_test_passed()
        if r.get("passed") is None: _pf(items, None, "Smoke Test", "لا فيديوهات معلّقة")
        else: _pf(items, bool(ok), "Smoke Test", (f"{sum(c['ok'] for c in r['checks'])}/{len(r['checks'])} فحصًا | {human_time(r['duration_s'])}" + (" | ⚠ المخزن محلي: رفع HF لم يُختبر" if STORE.name == "local" else "")) if ok else _scrub(why, 240))
    else: _pf(items, bool(ok), "Smoke Test", "نجح سابقًا بنفس العتاد/الإعدادات" if ok else why)
    reasons = [i["name"] for i in items if i["ok"] is False]
    if not reasons and not bench_ran: refresh_estimate(show=True)
    print("-" * 78)
    ran = [n for n, f in (("Benchmark", bench_ran), ("Smoke Test", smoke_ran)) if f]
    if reasons: print(f" ✖ NOT READY — {', '.join(reasons)}.  التشغيل الكامل (الخلية 24F) محجوب حتى يُصلَح.")
    else: print(f" ✔ READY — يمكن تشغيل الخلية 24F الآن  ({human_time(time.time() - t0)} | " + ("نُفِّذ: " + " + ".join(ran) if ran else "لا إعادة قياس/اختبار: سبق نجاحهما بنفس العتاد/الإعدادات") + ")")
    print("=" * 78)
    return dict(ready=not reasons, items=items, reasons=reasons, benchmark_ran=bench_ran, smoke_ran=smoke_ran, duration_s=time.time() - t0)

PREFLIGHT = preflight()

### CELL 24F — FULL RUN (التشغيل الكامل) — محجوب ببوابة Smoke Test
> يعالج كل الفيديوهات المعلّقة حتى النهاية أو حتى حدّ الوقت/الإيقاف اليدوي، ثم يفرّغ ويرفع كل شيء. آمن للتشغيل مجددًا (يستأنف من control/ على Hugging Face).
> إيقاف آمن: زر Interrupt (مرة واحدة) أو إنشاء الملف WORK_DIR/STOP. على Kaggle يُضبط MAX_RUNTIME_HOURS=11 تلقائيًا إن لم تحدده (حدّ الجلسة 12 ساعة).

▶ **التشغيل الكامل** — لا يبدأ إلا بعد نجاح Smoke Test (بوابة أمان). آمن للتشغيل مجددًا: يكمل من آخر نقطة.

In [ ]:
# CELL 24F - FULL RUN (التشغيل الكامل) — محجوب ببوابة Smoke Test
#: يعالج كل الفيديوهات المعلّقة حتى النهاية أو حتى حدّ الوقت/الإيقاف اليدوي، ثم يفرّغ ويرفع كل شيء. آمن للتشغيل مجددًا (يستأنف من control/ على Hugging Face).
#: إيقاف آمن: زر Interrupt (مرة واحدة) أو إنشاء الملف WORK_DIR/STOP. على Kaggle يُضبط MAX_RUNTIME_HOURS=11 تلقائيًا إن لم تحدده (حدّ الجلسة 12 ساعة).
try:
    require_smoke_test()                                          # يرفع PipelineError إن لم ينجح الاختبار التكاملي بنفس العتاد والإعدادات؛ run_pipeline نفسه يتحقق مرة ثانية
except PipelineError as _gate_err:
    if not (CONFIG.AUTO_PREFLIGHT and MANIFEST_ROWS and "preflight" in globals()): raise      # بلا فيديوهات لا فائدة من أي اختبار (الرسالة تشرح السبب الجذري)؛ وبلا تشغيل 24E لم تُعرَّف preflight
    print("ℹ لم يُشغَّل الاختبار التكاملي بعد على هذا العتاد/الإعدادات — أشغّله الآن تلقائيًا (Preflight: قياس + Smoke Test) ثم أكمل إن نجح (AUTO_PREFLIGHT=True).")
    _pf_res = preflight()
    if not _pf_res.get("ready"):
        raise PipelineError("Preflight لم ينجح فلن يبدأ التشغيل الكامل: " + ", ".join(_pf_res.get("reasons") or ["انظر الجدول أعلاه"])) from None
    require_smoke_test()

if (ENV.get("on_kaggle") or ENV.get("on_marimo")) and not os.environ.get("PIPE_MAX_RUNTIME_HOURS") and not CONFIG.MAX_RUNTIME_HOURS:
    CONFIG.MAX_RUNTIME_HOURS = 11.0
    print("ℹ Kaggle: MAX_RUNTIME_HOURS=11 (إيقاف آمن قبل حدّ الجلسة 12 ساعة؛ أعد تشغيل الخلية في جلسة جديدة للاستئناف). عدّلها عبر CONFIG أو PIPE_MAX_RUNTIME_HOURS.")

refresh_estimate(show=False)
print_run_plan()

def print_run_summary(res: Dict[str, Any]) -> None:
    P = STATE.recompute(); up = res.get("uploader", {})
    print("=" * 78); print(" RUN SUMMARY"); print("=" * 78)
    print(f"  سبب التوقف: {res.get('stop_reason')} | المدة: {human_time(res.get('elapsed_s', 0))}")
    print(f"  هذه الجلسة: فيديو بدأ {res.get('videos_started')} | اكتمل {res.get('videos_finished')} | فشل {res.get('videos_failed')} | تُخطّي {res.get('videos_skipped')}")
    print(f"              فُحص {res.get('inspected'):,} | قُبل {res.get('accepted'):,} | كُتب {res.get('written'):,} | موصوف {res.get('captioned'):,} | للمراجعة {res.get('review'):,} | فشل وصف {res.get('caption_failures'):,}")
    print(f"              شظايا مرفوعة {up.get('shards_uploaded', 0)} ({up.get('bytes_uploaded', 0) / 1e9:.2f} GB) | فشل رفع {up.get('failures', 0)}")
    print(f"  الإجمالي (كل الجلسات): فيديو {P['completed_videos']}/{P['total_videos']} | صور مقبولة {P['accepted_images']:,} | موصوفة {P['captioned_images']:,} | شظايا {P['uploaded_shards']}")
    cp = res.get("caption_pool") or {}
    if cp.get("banned_first"): print(f"  فلتر العبارات المحظورة: وُسم {cp['banned_first']:,} | صُحّح بإعادة المحاولة {cp.get('banned_fixed', 0):,} | تُخطّي {cp.get('banned_skipped', 0):,} إطارًا")
    left = len(STATE.pending_videos())
    print("  ✔ اكتملت كل الفيديوهات." if not left else f"  ↻ متبقٍّ {left} فيديو: أعد تشغيل هذه الخلية (في هذه الجلسة أو جلسة جديدة) للمتابعة من control/ بلا تكرار.")
    print("=" * 78)

RUN_SUMMARY = run_pipeline()
print_run_summary(RUN_SUMMARY)

---
## الجزء 5 — التحقق والإحصاءات والاستئناف (25–28)
فاحص عشوائي للعيّنات، مدقّق تكامل Dataset، الإحصاءات النهائية، حالة الاستئناف والتعليمات.

### CELL 25 — RANDOM DATASET INSPECTOR  (+ قارئ Dataset المشترك للخليتين 26 و27)
> يعرض عيّنات عشوائية حقيقية من Dataset المرفوع: الصورة + وصف Qwen الفعلي + كل البيانات، ويتحقق من سلامة كل عيّنة (checksum/أبعاد) أثناء ذلك.
> لا يحمّل أي شظية كاملة: يقرأ فهرس الشظية الصغير ثم بايتات الصورة والـJSON فقط عبر إزاحات الفهرس (HTTP Range على Hugging Face عبر HfFileSystem، seek محليًا).
> يعمل في أي جلسة (حتى بعد اكتمال المشروع): يكفي تشغيل الخلايا 01–06 ثم هذه الخلية. عدّل المتغيرات أدناه ثم أعد تشغيل الخلية.

يُشغَّل أثناء العمل أو بعده (أو في جلسة جديدة للفحص فقط).

In [ ]:
# CELL 25 - RANDOM DATASET INSPECTOR  (+ قارئ Dataset المشترك للخليتين 26 و27)
#: يعرض عيّنات عشوائية حقيقية من Dataset المرفوع: الصورة + وصف Qwen الفعلي + كل البيانات، ويتحقق من سلامة كل عيّنة (checksum/أبعاد) أثناء ذلك.
#: لا يحمّل أي شظية كاملة: يقرأ فهرس الشظية الصغير ثم بايتات الصورة والـJSON فقط عبر إزاحات الفهرس (HTTP Range على Hugging Face عبر HfFileSystem، seek محليًا).
#: يعمل في أي جلسة (حتى بعد اكتمال المشروع): يكفي تشغيل الخلايا 01–06 ثم هذه الخلية. عدّل المتغيرات أدناه ثم أعد تشغيل الخلية.
import itertools

INSPECT_COUNT = 5              # كم عيّنة عشوائية
INSPECT_SAMPLE_ID = None       # مثال: "myvideo_00012345" لعرض عيّنة محددة (يتجاهل العدد)
INSPECT_SOURCE = "auto"        # auto = المخزن الحالي | local = مجلد local_repo (DRY_RUN) | hf = مستودع Hugging Face (يتطلب HF_TOKEN)
INSPECT_KIND = "train"         # train | review | any
INSPECT_VIDEO_ID = None        # تقييد العيّنة بفيديو معيّن
INSPECT_SEED = None            # رقم لتكرار نفس الاختيار

# ---------------------------------------------------------------- قارئ Dataset (يُستعمل أيضًا في 26 و27)
def dataset_store(source: str = "auto") -> "RemoteStore":
    if source == "auto": return STORE
    if source == "local": return STORE if STORE.name == "local" else LocalStore(Path(CONFIG.WORK_DIR) / "local_repo")
    if source == "hf":
        if STORE.name == "huggingface": return STORE
        tok = get_hf_token()
        if not tok: raise RuntimeError("لا HF_TOKEN: لا يمكن القراءة من Hugging Face (أضِفه إلى Kaggle Secrets باسم HF_TOKEN)")
        return HFStore(CONFIG.HF_DATASET_REPO, tok, private=CONFIG.HF_PRIVATE_REPO)
    raise ValueError("source: auto | local | hf")

def shard_catalog(store: "RemoteStore", kind: str = "any") -> List[Dict[str, Any]]:
    """قائمة الشظايا المكتملة: حالة هذه الجلسة + completed_shards.json في المخزن (كل العاملين عند NUM_WORKERS>1) + عند الغياب فهارس index/ مباشرة."""
    by_name: Dict[str, Dict[str, Any]] = {}
    if store is STORE:
        for s in STATE.shards: by_name[s["name"]] = dict(s)
    try:
        for f in store.list_files("control/"):
            if f.endswith("completed_shards.json"):
                for s in json.loads(store.read_bytes(f) or b"[]"): by_name.setdefault(s["name"], dict(s))
    except Exception as e: log.warning("قراءة completed_shards.json فشلت: %s", type(e).__name__)
    if not by_name:
        for f in store.list_files("index/"):
            nm = Path(f).stem; by_name[nm] = dict(name=nm, kind="review" if nm.startswith("review") else "train", repo_path=f"data/{nm}.tar", index_path=f, n_samples=None)
    cat = [s for _, s in sorted(by_name.items())]
    return [s for s in cat if kind == "any" or s.get("kind") == kind]

def parse_index(blob: bytes, ext: str, columns: Optional[Sequence[str]] = None) -> List[Dict[str, Any]]:
    if ext == "parquet":
        import pyarrow.parquet as pq
        return pq.read_table(io.BytesIO(blob), columns=list(columns) if columns else None).to_pylist()
    rows = [json.loads(l) for l in blob.decode("utf-8").splitlines() if l.strip()]
    return [{c: r.get(c) for c in columns} for r in rows] if columns else rows

def read_shard_index(store: "RemoteStore", name: str, columns: Optional[Sequence[str]] = None, index_path: Optional[str] = None) -> Optional[List[Dict[str, Any]]]:
    """صفوف فهرس شظية (Parquet أو JSONL حسب ما كُتب). None إن لم يوجد."""
    cands = ([index_path] if index_path else []) + [f"index/{name}.{e}" for e in dict.fromkeys((INDEX_EXT, "parquet", "jsonl"))]
    for p in dict.fromkeys(cands):
        b = store.read_bytes(p)
        if b is not None: return parse_index(b, p.rsplit(".", 1)[-1], columns)
    return None

def iter_shard_indexes(store: "RemoteStore", catalog: Sequence[Dict[str, Any]], columns: Optional[Sequence[str]] = None, threads: int = 8, window: int = 16) -> Iterator[Tuple[Dict[str, Any], Optional[List[Dict[str, Any]]]]]:
    """يمرّ على فهارس الشظايا بالترتيب مع قراءة متوازية محدودة (ذاكرة محدودة: لا تُحمَّل كل الفهارس معًا)."""
    def load(sh: Dict[str, Any]) -> Tuple[Dict[str, Any], Optional[List[Dict[str, Any]]]]:
        try: return sh, read_shard_index(store, sh["name"], columns, sh.get("index_path"))
        except Exception as e:
            log.warning("فهرس %s غير قابل للقراءة: %s: %s", sh["name"], type(e).__name__, str(e)[:100]); return sh, None
    with ThreadPoolExecutor(threads, thread_name_prefix="idx") as ex:
        it = iter(catalog); futs: "collections.deque" = collections.deque(ex.submit(load, s) for s in itertools.islice(it, window))
        while futs:
            res = futs.popleft().result(); nxt = next(it, None)
            if nxt is not None: futs.append(ex.submit(load, nxt))
            yield res

def fetch_sample(store: "RemoteStore", row: Dict[str, Any]) -> Dict[str, Any]:
    """يقرأ صورة + JSON عيّنة واحدة عبر إزاحات الفهرس فقط ويتحقق منها (checksum، أبعاد، تطابق sample_id، وصف غير فارغ في شظايا التدريب)."""
    path, probs = f"data/{row['shard']}.tar", []
    img = store.read_range(path, int(row["img_offset"]), int(row["img_size"])); js = store.read_range(path, int(row["json_offset"]), int(row["json_size"]))
    if img is None or js is None or len(img) != int(row["img_size"]) or len(js) != int(row["json_size"]):
        return dict(ok=False, problems=["قراءة الإزاحات فشلت أو أعادت طولًا خاطئًا (الشظية مبتورة/الفهرس لا يطابقها)"], rec=None, jpeg=None)
    try: rec = json.loads(js.decode("utf-8"))
    except Exception as e: return dict(ok=False, problems=[f"JSON غير صالح: {type(e).__name__}"], rec=None, jpeg=img)
    if hashlib.sha256(img).hexdigest() != rec.get("image_sha256"): probs.append("checksum الصورة ≠ image_sha256 في JSON")
    if row.get("image_sha256") and row["image_sha256"] != rec.get("image_sha256"): probs.append("image_sha256 في الفهرس ≠ JSON")
    try:
        im = Image.open(io.BytesIO(img)); im.load()
        if im.size != (rec.get("width"), rec.get("height")): probs.append(f"الأبعاد {im.size} ≠ JSON {(rec.get('width'), rec.get('height'))}")
    except Exception as e: probs.append(f"الصورة غير قابلة للفتح: {type(e).__name__}")
    if rec.get("sample_id") != row.get("sample_id"): probs.append("sample_id في JSON ≠ الفهرس")
    if row.get("kind") == "train" and rec.get("caption_status") != "disabled" and not str(rec.get("training_prompt", "")).strip(): probs.append("training_prompt فارغ في شظية تدريب")
    return dict(ok=not probs, problems=probs, rec=rec, jpeg=img)

# ---------------------------------------------------------------- المفتش
def _inspector_meta(rec: Dict[str, Any], row: Dict[str, Any], verdict: str) -> Dict[str, Any]:
    m = {"Sample ID": rec["sample_id"], "Video ID": rec["video_id"], "Frame": rec["frame_index"], "Timestamp": f'{rec["timestamp_seconds"]:.3f}s',
         "Stored": f'{rec["width"]}x{rec["height"]} {rec.get("image_format", "")} ({rec.get("file_size", 0) / 1024:.0f} KB)', "Original": f'{rec.get("original_width")}x{rec.get("original_height")}',
         "Shard": f'{row["shard"]} ({row["kind"]}) @ img {row["img_offset"]:,} + {row["img_size"]:,} B', "Caption model": rec.get("caption_model") or "?",
         "Caption words / score": f'{rec.get("caption_words", "")} / {rec.get("caption_score", "")}', "Quality score": rec.get("quality_score", ""),
         "Visual style": rec.get("visual_style", ""), "Lighting": rec.get("lighting", ""), "Camera": rec.get("camera_view", ""),
         "Palette": ", ".join(map(str, rec.get("color_palette", []))), "Objects": ", ".join([str(o) for o in rec["objects"]][:12]) if isinstance(rec.get("objects"), list) else rec.get("objects", ""),
         "Visible text": ", ".join(map(str, rec.get("visible_text", []))) or "—", "Source / license": f'{rec.get("source_name")} / {rec.get("license")}',
         "Decoder": rec.get("decoder", ""), "Sampling": f'{(rec.get("sampling") or {}).get("method")} u={(rec.get("sampling") or {}).get("hash_u")}', "Integrity": verdict}
    if row.get("review_reason") or rec.get("review_reason"): m["Review reason"] = row.get("review_reason") or rec.get("review_reason")
    return m

def inspect_random(n: int = 5, sample_id: Optional[str] = None, source: str = "auto", kind: str = "train", video_id: Optional[str] = None, seed: Optional[int] = None,
                   show: bool = True) -> List[Dict[str, Any]]:
    store = dataset_store(source); rng = random.Random(seed); cat = shard_catalog(store, kind)
    print("=" * 78); print(f" RANDOM DATASET INSPECTOR | المصدر: {store.name}" + (f" ({CONFIG.HF_DATASET_REPO})" if store.name == "huggingface" else "") + f" | النوع: {kind} | شظايا: {len(cat)}"); print("=" * 78)
    if not cat:
        print("  لا توجد شظايا في المخزن بعد (لم يكتمل أي commit). شغّل 24F أولًا، أو جرّب INSPECT_KIND='any' / INSPECT_SOURCE='hf'."); return []
    cache: Dict[str, List[Dict[str, Any]]] = {}
    def rows_of(sh: Dict[str, Any]) -> List[Dict[str, Any]]:
        if sh["name"] not in cache: cache[sh["name"]] = read_shard_index(store, sh["name"], index_path=sh.get("index_path")) or []
        return cache[sh["name"]]
    picks: List[Tuple[Dict[str, Any], Dict[str, Any]]] = []
    if sample_id:
        vid = sample_id.rsplit("_", 1)[0].replace("~", ".")
        cand = [s for s in cat if vid in (s.get("videos") or {}) or s["name"] in (STATE.videos.get(vid, {}).get("shards") or [])] or cat
        for sh in cand:
            hit = next((r for r in rows_of(sh) if r["sample_id"] == sample_id), None)
            if hit: picks.append((sh, hit)); break
        if not picks: print(f"  العيّنة '{sample_id}' غير موجودة في شظايا النوع '{kind}' (جرّب kind='any')."); return []
    else:
        pool = [s for s in cat if video_id is None or video_id in (s.get("videos") or {}) or s.get("videos") is None]
        if not pool: print(f"  لا شظايا تحتوي الفيديو '{video_id}'."); return []
        for _ in range(30 * max(1, n)):
            if len(picks) >= n: break
            sh = rng.choices(pool, weights=[max(1, s.get("n_samples") or 1) for s in pool])[0]
            rows = [r for r in rows_of(sh) if video_id is None or r["video_id"] == video_id]
            if not rows: continue
            r = rng.choice(rows)
            if all(r["sample_id"] != p[1]["sample_id"] for p in picks): picks.append((sh, r))
    out: List[Dict[str, Any]] = []
    for i, (sh, row) in enumerate(picks, 1):
        f = fetch_sample(store, row); rec = f["rec"]
        verdict = "✔ checksum + أبعاد + sample_id + وصف: مُتحقَّق منها بقراءة الإزاحات" if f["ok"] else "✖ " + " | ".join(f["problems"])
        out.append(dict(sample_id=row["sample_id"], shard=row["shard"], ok=f["ok"], problems=f["problems"], rec=rec))
        if not show or rec is None or f["jpeg"] is None:
            if show: print(f"  [{i}/{len(picks)}] {row['sample_id']}: {verdict}")
            continue
        htm = render_sample_html(f["jpeg"], _inspector_meta(rec, row, verdict), rec.get("training_prompt") or "(لا وصف: شظية مراجعة أو الوصف معطّل)", title=f"RANDOM DATASET SAMPLE {i}/{len(picks)}", flags=rec.get("caption_flags", []))
        (PREVIEW_DIR / f"inspector_{i}.html").write_text(htm, encoding="utf-8")
        if HAVE_IPY:
            try: display(HTML(htm))
            except Exception: pass
        else:
            print(f"\n---- [{i}/{len(picks)}] {row['sample_id']} | {rec['width']}x{rec['height']} @ {rec['timestamp_seconds']:.2f}s | shard {row['shard']} ----\n{rec.get('training_prompt') or '(لا وصف)'}\n"
                  f"style: {rec.get('visual_style')} | lighting: {rec.get('lighting')} | camera: {rec.get('camera_view')} | palette: {rec.get('color_palette')} | model: {rec.get('caption_model')}\n{verdict}")
    bad = [o for o in out if not o["ok"]]
    print(f"\n  فُحصت {len(out)} عيّنة: {len(out) - len(bad)} سليمة" + (f" | ✖ {len(bad)} فيها مشاكل: {[(o['sample_id'], o['problems'][:1]) for o in bad[:3]]}" if bad else " ✔") + "  (لم تُحمَّل أي شظية كاملة)")
    return out

def inspect_sample(sample_id: str, source: str = "auto") -> List[Dict[str, Any]]:
    return inspect_random(sample_id=sample_id, source=source, kind="any")

INSPECTION = inspect_random(INSPECT_COUNT, INSPECT_SAMPLE_ID, INSPECT_SOURCE, INSPECT_KIND, INSPECT_VIDEO_ID, INSPECT_SEED)

### CELL 26 — DATASET INTEGRITY VALIDATOR  (فحص تكامل Dataset كاملًا في المخزن)
> quick (الافتراضي): أحجام كل الشظايا + قراءة كل الفهارس + التكرار + تطابق الحالة مع المخزن + عيّنات عشوائية عبر Range + بضع شظايا كاملة (تنزيل + تحقق + sha256 + مطابقة الإزاحات).
> full: كل الشظايا تُنزَّل وتُفحص كاملة واحدة بعد أخرى (تُحذف بعد الفحص؛ قابل للاستئناف عبر integrity_progress.json). لا يعدّل شيئًا في Dataset أبدًا (قراءة فقط).
> الفحوص: ✖ = خلل حقيقي في Dataset، ⚠ = تحذير/معلومة. التقرير يُحفظ في WORK_DIR/integrity_report.json (ويمكن commit له عبر commit_report=True).

يُشغَّل بعد اكتمال العمل (أو دوريًا): quick افتراضيًا، و`INTEGRITY_MODE = "full"` للفحص الكامل القابل للاستئناف.

In [ ]:
# CELL 26 - DATASET INTEGRITY VALIDATOR  (فحص تكامل Dataset كاملًا في المخزن)
#: quick (الافتراضي): أحجام كل الشظايا + قراءة كل الفهارس + التكرار + تطابق الحالة مع المخزن + عيّنات عشوائية عبر Range + بضع شظايا كاملة (تنزيل + تحقق + sha256 + مطابقة الإزاحات).
#: full: كل الشظايا تُنزَّل وتُفحص كاملة واحدة بعد أخرى (تُحذف بعد الفحص؛ قابل للاستئناف عبر integrity_progress.json). لا يعدّل شيئًا في Dataset أبدًا (قراءة فقط).
#: الفحوص: ✖ = خلل حقيقي في Dataset، ⚠ = تحذير/معلومة. التقرير يُحفظ في WORK_DIR/integrity_report.json (ويمكن commit له عبر commit_report=True).
INTEGRITY_MODE = "quick"        # quick | full
INTEGRITY_SOURCE = "auto"       # auto | local | hf
INTEGRITY_SHARD_SAMPLE = 3      # quick: عدد الشظايا الكاملة المنزَّلة (تشمل دائمًا أحدث شظية)
INTEGRITY_SAMPLE_READS = 60     # quick: عدد العيّنات العشوائية المقروءة عبر Range من كل الشظايا
INTEGRITY_SEED = None

_TOKEN_RE = re.compile(rb"hf_[A-Za-z0-9]{20,}")
_M64 = (1 << 64) - 1

def _h64(s: Any) -> int:
    return hash(str(s)) & _M64          # تجزئة عملية-محلية (تتغير بين العمليات؛ تكفي لكشف التكرار داخل هذا الفحص: تصادم ≈ n²/2⁶⁵)

def _dup_count(a: "np.ndarray") -> Tuple[int, "np.ndarray"]:
    if len(a) < 2: return 0, a[:0]
    s = np.sort(a); same = s[1:] == s[:-1]
    return int(same.sum()), np.unique(s[1:][same])

def _dup_examples(store: "RemoteStore", cat: Sequence[Dict[str, Any]], dups: "np.ndarray", key_fn: Callable[[Dict[str, Any]], str], cols: Sequence[str], limit: int = 4) -> List[str]:
    """تمريرة ثانية (تُنفَّذ فقط عند وجود تكرار): أمثلة فعلية للمفاتيح المكرّرة لتسهيل التتبّع."""
    want, seen, out = set(int(x) for x in dups[:5000]), {}, []
    for sh, rows in iter_shard_indexes(store, cat, cols):
        for r in rows or []:
            k = key_fn(r); h = _h64(k)
            if h in want:
                if h in seen: out.append(f"{seen[h]} ~ {r['sample_id']}")
                else: seen[h] = r["sample_id"]
                if len(out) >= limit: return out
    return out

def validate_dataset(mode: str = "quick", source: str = "auto", shard_sample: int = 3, sample_reads: int = 60, seed: Optional[int] = None,
                     commit_report: bool = False, resume: bool = True) -> Dict[str, Any]:
    t0 = time.time(); store = dataset_store(source); rng = random.Random(seed); full = (mode == "full")
    items: List[Dict[str, Any]] = []; E: List[str] = []; W: List[str] = []; stats: Dict[str, Any] = {}
    def item(ok: Optional[bool], name: str, detail: Any = "") -> None:
        items.append(dict(name=name, ok=ok, detail=str(detail)[:500]))
        if ok is False: E.append(f"{name}: {str(detail)[:200]}")
        elif ok is None: W.append(f"{name}: {str(detail)[:200]}")
        print(f"  {({True: '✔', False: '✖', None: '⚠'})[ok]} {name}" + (f"  — {str(detail)[:260]}" if detail != "" else ""), flush=True)
    print("=" * 78); print(f" DATASET INTEGRITY VALIDATOR | {mode.upper()} | المصدر: {store.name}" + (f" ({CONFIG.HF_DATASET_REPO})" if store.name == "huggingface" else "")); print("=" * 78)
    cat = shard_catalog(store, "any"); stats["shards"] = len(cat)
    if not cat:
        item(None, "الشظايا", "لا شظايا في المخزن بعد: لا شيء للفحص")
        return _finish_report(store, mode, items, E, W, stats, t0, commit_report)
    # ---- 1) الشظايا موجودة بالحجم المسجَّل
    sizes = store.sizes([s["repo_path"] for s in cat]); known = {s["repo_path"]: sizes.get(s["repo_path"]) for s in cat}
    missing = [s["name"] for s in cat if known[s["repo_path"]] is None]; wrong = [(s["name"], known[s["repo_path"]], s["size"]) for s in cat if s.get("size") and known[s["repo_path"]] not in (None, s["size"])]
    stats["shards_bytes"] = int(sum(v or 0 for v in known.values()))
    item(not missing and not wrong, "كل الشظايا موجودة بالحجم المسجَّل", f"{len(cat)} شظية | {human_bytes(stats['shards_bytes'])}" + (f" | مفقودة: {missing[:4]}" if missing else "") + (f" | حجم مختلف: {wrong[:3]}" if wrong else ""))
    # ---- 2) تمريرة الفهارس
    cols = [c for c in INDEX_COLUMNS if c != "training_prompt"]
    a = dict(rows=0, img_bytes=0, below_min=0, bad_offsets=0, bad_link=0, empty_train=0, short_train=0, kinds=collections.Counter(), reasons=collections.Counter(), per_video=collections.Counter())
    miss_idx: List[str] = []; cnt_mismatch: List[Any] = []; ids_l: List["np.ndarray"] = []; vph_l: List["np.ndarray"] = []; pix_l: List["np.ndarray"] = []
    K = max(0, int(sample_reads)); reservoir: List[Dict[str, Any]] = []; seen_n = 0
    for sh, rows in iter_shard_indexes(store, cat, cols):
        if rows is None: miss_idx.append(sh["name"]); continue
        n = len(rows); size = known.get(sh["repo_path"])
        if sh.get("n_samples") is not None and sh["n_samples"] != n: cnt_mismatch.append((sh["name"], n, sh["n_samples"]))
        ids = np.empty(n, dtype=np.uint64); vph = np.empty(n, dtype=np.uint64); pix = np.empty(n, dtype=np.uint64)
        for i, r in enumerate(rows):
            ids[i] = _h64(r["sample_id"]); vph[i] = _h64(f'{r["video_id"]}|{r["pixel_sha256"]}'); pix[i] = _h64(r["pixel_sha256"])
            if r["shard"] != sh["name"] or r["kind"] != sh["kind"]: a["bad_link"] += 1
            if (r["width"] or 0) < CONFIG.MIN_IMAGE_WIDTH or (r["height"] or 0) < CONFIG.MIN_IMAGE_HEIGHT: a["below_min"] += 1
            io_, is_, jo, js = r["img_offset"], r["img_size"], r["json_offset"], r["json_size"]
            if None in (io_, is_, jo, js) or io_ % 512 or jo % 512 or is_ <= 0 or js <= 0 or (size and (io_ + is_ > size or jo + js > size)): a["bad_offsets"] += 1
            a["per_video"][r["video_id"]] += 1; a["img_bytes"] += r["file_size"] or 0; a["kinds"][sh["kind"]] += 1
            if sh["kind"] == "train":
                if CONFIG.ENABLE_CAPTIONING and r["caption_status"] != "ok": a["empty_train"] += 1
                elif CONFIG.ENABLE_CAPTIONING and (r["caption_words"] or 0) < CONFIG.CAPTION_MIN_WORDS: a["short_train"] += 1
            else: a["reasons"][r.get("review_reason") or "غير محدد"] += 1
            seen_n += 1
            if K:
                if len(reservoir) < K: reservoir.append(r)
                else:
                    j = rng.randrange(seen_n)
                    if j < K: reservoir[j] = r
        a["rows"] += n; ids_l.append(ids); vph_l.append(vph); pix_l.append(pix)
    stats.update(rows=a["rows"], train_rows=a["kinds"]["train"], review_rows=a["kinds"]["review"], image_bytes=a["img_bytes"], videos_with_samples=len(a["per_video"]))
    item(not miss_idx, "كل شظية لها فهرس قابل للقراءة", f"مفقود/تالف: {miss_idx[:5]}" if miss_idx else f"{len(cat) - len(miss_idx)}/{len(cat)} فهرس")
    item(not cnt_mismatch, "عدد صفوف كل فهرس = عدد عينات الشظية المسجَّل", cnt_mismatch[:3] if cnt_mismatch else f"Σ = {a['rows']:,} صورة ({a['kinds']['train']:,} تدريب + {a['kinds']['review']:,} مراجعة)")
    item(a["bad_link"] == 0, "كل صف يشير إلى شظيته ونوعها الصحيح", f"{a['bad_link']} صفًا مخالفًا" if a["bad_link"] else "")
    item(a["bad_offsets"] == 0, "إزاحات الفهرس صالحة (من مضاعفات 512، داخل حدود الشظية، أطوال موجبة)", f"{a['bad_offsets']} صفًا مخالفًا" if a["bad_offsets"] else "")
    ids_all = np.concatenate(ids_l) if ids_l else np.empty(0, np.uint64); n_dup, dh = _dup_count(ids_all)
    ex_id = _dup_examples(store, cat, dh, lambda r: str(r["sample_id"]), ["sample_id"]) if n_dup else []
    item(n_dup == 0, "لا sample_id مكرر في Dataset كله", f"{n_dup} تكرارًا، مثال: {ex_id}" if n_dup else f"{len(ids_all):,} معرّفًا فريدًا")
    v_dup, vdh = _dup_count(np.concatenate(vph_l) if vph_l else np.empty(0, np.uint64))
    ex_px = _dup_examples(store, cat, vdh, lambda r: str(r["video_id"]) + "|" + str(r["pixel_sha256"]), ["sample_id", "video_id", "pixel_sha256"]) if v_dup else []
    item(v_dup == 0, "لا إطار مكرر حرفيًا (pixel_sha256) داخل الفيديو الواحد", f"{v_dup} تكرارًا، أمثلة: {ex_px}" if v_dup else "")
    p_dup, _ = _dup_count(np.concatenate(pix_l) if pix_l else np.empty(0, np.uint64)); stats.update(duplicate_ids=n_dup, duplicate_pixels_within_video=v_dup, duplicate_pixels_global=p_dup)
    item(True if p_dup == v_dup else None, "تكرار حرفي بين فيديوهات مختلفة", "لا شيء" if p_dup == v_dup else f"{p_dup - v_dup} إطارًا متطابق البكسلات في أكثر من فيديو (طبيعي إن تشاركت الفيديوهات مشاهد؛ فلترة المصدر تتحكم بها CROSS_VIDEO_DEDUP)")
    item(a["below_min"] == 0, f"كل الصور ≥ {CONFIG.MIN_IMAGE_WIDTH}x{CONFIG.MIN_IMAGE_HEIGHT}", f"{a['below_min']} صورة أصغر" if a["below_min"] else "")
    if CONFIG.ENABLE_CAPTIONING:
        item(a["empty_train"] == 0, "كل صور شظايا التدريب لها وصف صالح (caption_status=ok)", f"{a['empty_train']} صورة بلا وصف" if a["empty_train"] else "")
        if a["short_train"]: item(None, "أوصاف تدريب أقصر من CAPTION_MIN_WORDS", f"{a['short_train']} (ربما رُفعت بحد أدنى أقل في جلسة سابقة)")
    # ---- 3) تطابق الحالة مع المخزن
    if store is STORE and not PIPELINE_DIRTY:
        P = STATE.recompute()
        if CONFIG.NUM_WORKERS == 1:
            item(P["accepted_images"] == a["rows"], "مجموع الصور في الفهارس = الصور المقبولة في الحالة (progress.json)", f"الفهارس {a['rows']:,} | الحالة {P['accepted_images']:,}")
        mism = [(v_, STATE.videos[v_]["accepted"], a["per_video"].get(v_, 0)) for v_ in STATE.videos if STATE.videos[v_]["accepted"] != a["per_video"].get(v_, 0)]
        item(not mism, "عدّاد كل فيديو في الحالة = عدد صوره في الفهارس", f"{len(mism)} فيديو مخالف، أمثلة (id, الحالة, الفهرس): {mism[:3]}" if mism else f"{len(STATE.videos)} فيديو")
        try: rn = [s["name"] for s in json.loads(store.read_bytes(f"{control_prefix()}/completed_shards.json") or b"[]")]
        except Exception: rn = None
        item(rn == [s["name"] for s in STATE.shards], "completed_shards.json في المخزن = شظايا الحالة", "" if rn == [s["name"] for s in STATE.shards] else f"المخزن {len(rn or [])} | الحالة {len(STATE.shards)}")
    elif store is STORE: item(None, "مطابقة الحالة", "PIPELINE_DIRTY: حالة الذاكرة غير موثوقة حتى إعادة تحميلها من المخزن (شغّل 24F مجددًا)")
    # ---- 4) عيّنات عشوائية عبر Range (كل الشظايا)
    if K and reservoir:
        def one(r: Dict[str, Any]) -> List[str]:
            f = fetch_sample(store, r); pr = list(f["problems"]); rec = f["rec"]
            if rec:
                pr += [f"{k}: الفهرس {r[k]!r} ≠ JSON {rec.get(k)!r}" for k in ("video_id", "frame_index", "width", "height", "image_sha256", "pixel_sha256", "caption_words") if r.get(k) != rec.get(k)]
            return [f"{r['sample_id']}: {p}" for p in pr]
        with ThreadPoolExecutor(4) as ex: bad = [p for ps in ex.map(one, reservoir) for p in ps]
        stats["sample_reads"] = len(reservoir)
        item(not bad, f"{len(reservoir)} عيّنة عشوائية من كل الشظايا مقروءة عبر Range وسليمة (checksum/أبعاد/JSON/تطابق الفهرس)", bad[:3] if bad else "")
    # ---- 5) شظايا كاملة: تنزيل + تحقق + sha256 + مطابقة الإزاحات
    if full: chosen = list(cat)
    else:
        chosen = {cat[-1]["name"]: cat[-1]}
        for s_ in rng.sample(list(cat), min(len(cat), max(0, int(shard_sample)))): chosen.setdefault(s_["name"], s_)
        chosen = list(chosen.values())[: max(1, int(shard_sample))]
    prog_p = Path(CONFIG.WORK_DIR) / "integrity_progress.json"
    try: done_prev = json.loads(prog_p.read_text(encoding="utf-8")).get("validated", {}) if (full and resume and prog_p.exists()) else {}
    except Exception: done_prev = {}
    tmp = Path(CONFIG.TEMP_DIR) / "integrity"; tmp.mkdir(parents=True, exist_ok=True); shard_errs: List[str] = []; n_ok = n_skip = 0
    for i, sh in enumerate(chosen, 1):
        if full and done_prev.get(sh["name"]) == sh.get("sha256") and sh.get("sha256"): n_skip += 1; continue
        need = (known.get(sh["repo_path"]) or sh.get("size") or 0) / 1e9 * 1.2 + 0.2
        if shutil.disk_usage(tmp).free / 1e9 < need: shard_errs.append(f"{sh['name']}: مساحة القرص لا تكفي للتنزيل ({need:.1f}GB) — تم الإيقاف"); break
        local = tmp / f"{sh['name']}.tar"
        try:
            if not store.download(sh["repo_path"], local): shard_errs.append(f"{sh['name']}: تعذّر تنزيل الشظية"); continue
            v = validate_tar_shard(local, sh["kind"]); errs = list(v["errors"])
            if sh.get("sha256") and sha256_file(local) != sh["sha256"]: errs.append("sha256 الملف المنزَّل ≠ sha256 المحسوب قبل الرفع (الرفع غير مطابق للأصل)")
            if sh.get("n_samples") is not None and v["n_samples"] != sh["n_samples"]: errs.append(f"عدد العينات {v['n_samples']} ≠ {sh['n_samples']}")
            irows = read_shard_index(store, sh["name"], index_path=sh.get("index_path")) or []
            bad_off = [r["sample_id"] for r in irows if {k: r.get(k) for k in ("img_offset", "img_size", "json_offset", "json_size")} != v["offsets"].get(r["sample_id"])]
            if bad_off: errs.append(f"{len(bad_off)} إزاحة في الفهرس لا تطابق موضع العنصر الفعلي في الشظية (مثال {bad_off[0]})")
            if {r["sample_id"] for r in irows} != set(v["offsets"]): errs.append("معرّفات الفهرس ≠ معرّفات عناصر الشظية")
            if errs: shard_errs += [f"{sh['name']}: {e}" for e in errs[:3]]
            else:
                n_ok += 1
                if full:
                    done_prev[sh["name"]] = sh.get("sha256"); atomic_write_bytes(prog_p, json.dumps(dict(validated=done_prev)).encode())
        except Exception as e: shard_errs.append(f"{sh['name']}: {type(e).__name__}: {str(e)[:120]}")
        finally: local.unlink(missing_ok=True)
        if full and i % 10 == 0: print(f"    … {i}/{len(chosen)} شظية فُحصت ({n_ok} سليمة)", flush=True)
    shutil.rmtree(tmp, ignore_errors=True); stats["shards_fully_validated"] = n_ok + n_skip
    item(not shard_errs, f"{n_ok + n_skip}/{len(chosen)} شظية نُزّلت وفُحصت كاملة (TAR، صور، JSON، checksum، sha256، إزاحات الفهرس)" + (f" ({n_skip} سبق فحصها)" if n_skip else ""), shard_errs[:3] if shard_errs else "")
    if not full and len(chosen) < len(cat): item(None, "الفحص الكامل لبقية الشظايا", f"لم يُفحص {len(cat) - len(chosen)} شظية بالتنزيل الكامل في وضع quick: شغّل INTEGRITY_MODE='full' (يمكن إيقافه واستئنافه)")
    # ---- 6) ملخص المراجعة
    if a["kinds"]["review"]:
        frac = a["kinds"]["review"] / max(1, a["rows"]); top = ", ".join(f"{k}: {v:,}" for k, v in a["reasons"].most_common(5))
        item(None if frac > 0.3 else True, f"شظايا المراجعة: {a['kinds']['review']:,} صورة ({frac:.1%} من الكل) — محفوظة لا محذوفة", f"الأسباب: {top}" + (" | النسبة مرتفعة: راجع جودة الأوصاف/الموجّه" if frac > 0.3 else ""))
    # ---- 7) الأسرار + 8) ملفات شاردة
    allf = store.list_files(""); leaks = []
    for f in [x for x in allf if x.startswith("control/") or x == "README.md"]:
        b = store.read_bytes(f) or b""
        if _TOKEN_RE.search(b) or any(s.encode() in b for s in _SECRETS if s): leaks.append(f)
    item(not leaks, "لا Token ولا سرّ في ملفات control وREADME", f"موجود في: {leaks[:3]}" if leaks else f"{len([x for x in allf if x.startswith('control/')])} ملف control فُحصت")
    cat_names = {s["name"] for s in cat}
    orphan = [f for f in allf if f.startswith("data/") and f.endswith(".tar") and Path(f).stem not in cat_names]
    junk = [f for f in allf if f.startswith("_smoke_test/") or f.endswith((".tmp", ".partial", ".bak"))]
    odd = [f for f in allf if not (f in ("README.md", ".gitattributes") or f.startswith(("data/", "index/", "control/")) or f.endswith("integrity_report.json"))] if allf else []
    item(None if (orphan or junk) else True, "لا شظايا يتيمة ولا ملفات مؤقتة/اختبار في المستودع", f"يتيمة (ليست في completed_shards): {orphan[:3]} | مؤقتة: {junk[:3]}" if (orphan or junk) else "")
    if odd and not junk: item(None, "ملفات غير متوقعة في جذر المستودع", odd[:4])
    return _finish_report(store, mode, items, E, W, stats, t0, commit_report)

def _finish_report(store: "RemoteStore", mode: str, items: List[Dict[str, Any]], E: List[str], W: List[str], stats: Dict[str, Any], t0: float, commit_report: bool) -> Dict[str, Any]:
    rep = dict(ok=not E, mode=mode, store=store.name, errors=E, warnings=W, items=items, stats=stats, duration_s=round(time.time() - t0, 1), finished_at=time.time())
    blob = json.dumps(rep, ensure_ascii=False, indent=1, default=str).encode("utf-8")
    try: atomic_write_bytes(Path(CONFIG.WORK_DIR) / "integrity_report.json", blob)
    except Exception as e: log.warning("تعذّر حفظ integrity_report.json: %s", e)
    if commit_report:
        try: store.commit({"control/integrity_report.json": blob}, f"integrity report ({mode}): {'OK' if not E else str(len(E)) + ' errors'}")
        except Exception as e: log.warning("تعذّر رفع integrity_report.json: %s", str(e)[:150])
    print("-" * 78)
    print((f" ✔ DATASET INTEGRITY OK ({mode})" if not E else f" ✖ DATASET INTEGRITY FAILED ({mode}) — {len(E)} خلل") + f" | تحذيرات: {len(W)} | {human_time(rep['duration_s'])}")
    for e_ in E[:8]: print("   ✖", e_)
    print("=" * 78)
    return rep

INTEGRITY = validate_dataset(INTEGRITY_MODE, INTEGRITY_SOURCE, INTEGRITY_SHARD_SAMPLE, INTEGRITY_SAMPLE_READS, INTEGRITY_SEED)

### CELL 27 — FINAL STATISTICS  (إحصاءات Dataset النهائية)
> تُحسب من فهارس Dataset الفعلية في المخزن (مصدر الحقيقة؛ تشمل كل العاملين) + حالة الجلسة (STATE) عندما يكون المخزن هو الحالي.
> ذاكرة محدودة: الفهارس تُقرأ شظية بعد شظية؛ النسب المئوية من عيّنة عشوائية منتظمة (حتى STATS_SAMPLE قيمة لكل عمود، ودقيقة تمامًا إن قلّ العدد عنها)،
> والفئات (الأنماط/الدقات/الأعلام...) بعدّادات محدودة. للقراءة فقط: لا تعدّل شيئًا في Dataset (إلا commit=True فتضيف control/final_statistics.json).

In [ ]:
# CELL 27 - FINAL STATISTICS  (إحصاءات Dataset النهائية)
#: تُحسب من فهارس Dataset الفعلية في المخزن (مصدر الحقيقة؛ تشمل كل العاملين) + حالة الجلسة (STATE) عندما يكون المخزن هو الحالي.
#: ذاكرة محدودة: الفهارس تُقرأ شظية بعد شظية؛ النسب المئوية من عيّنة عشوائية منتظمة (حتى STATS_SAMPLE قيمة لكل عمود، ودقيقة تمامًا إن قلّ العدد عنها)،
#: والفئات (الأنماط/الدقات/الأعلام...) بعدّادات محدودة. للقراءة فقط: لا تعدّل شيئًا في Dataset (إلا commit=True فتضيف control/final_statistics.json).
STATS_SOURCE = "auto"          # auto | local | hf
STATS_SAVE = True              # يحفظ WORK_DIR/final_statistics.json
STATS_TOP = 8                  # حجم قوائم «الأكثر شيوعًا»
STATS_SAMPLE = 100_000         # حجم عيّنة النسب المئوية لكل عمود رقمي

_STAT_COLS = ["video_id", "kind", "width", "height", "original_width", "original_height", "file_size", "quality_score", "caption_score", "caption_words",
              "caption_flags", "visual_style", "review_reason", "source_name", "license"]


class _Sample:
    """مُراكِم رقمي بذاكرة محدودة: العدد/المجموع/الأدنى/الأقصى دقيقة دائمًا؛ النسب المئوية دقيقة إن n ≤ cap وإلا من عيّنة منتظمة (أصغر cap مفاتيح عشوائية)."""
    def __init__(self, cap: int, seed: int):
        self.cap, self.rng = max(1000, int(cap)), np.random.default_rng(seed)
        self.keys = np.empty(0, np.float64); self.vals = np.empty(0, np.float64); self.thr = 1.0
        self.n = 0; self.s = 0.0; self.mn = math.inf; self.mx = -math.inf

    def add(self, arr: Any) -> None:
        a = np.asarray(arr, dtype=np.float64); a = a[~np.isnan(a)]
        if not a.size: return
        self.n += int(a.size); self.s += float(a.sum()); self.mn = min(self.mn, float(a.min())); self.mx = max(self.mx, float(a.max()))
        k = self.rng.random(a.size)
        if self.thr < 1.0:
            m = k < self.thr; a, k = a[m], k[m]
        self.vals = np.concatenate([self.vals, a]); self.keys = np.concatenate([self.keys, k])
        if self.vals.size > self.cap:
            idx = np.argpartition(self.keys, self.cap - 1)[: self.cap]; self.vals, self.keys = self.vals[idx], self.keys[idx]; self.thr = float(self.keys.max())

    def summary(self, qs: Sequence[int] = (5, 25, 50, 75, 95)) -> Dict[str, Any]:
        if not self.n: return dict(count=0)
        p = np.percentile(self.vals, list(qs))
        return dict(count=self.n, mean=self.s / self.n, min=self.mn, max=self.mx, exact=self.n <= self.cap, **{f"p{q}": float(v) for q, v in zip(qs, p)})

    def histogram(self, bins: int = 10, lo: Optional[float] = None, hi: Optional[float] = None) -> List[int]:
        if not self.vals.size: return []
        return [int(x) for x in np.histogram(self.vals, bins=bins, range=(self.mn if lo is None else lo, self.mx if hi is None else hi))[0]]


class _Cat:
    """عدّاد فئات بذاكرة محدودة: عند تجاوز cap مفتاحًا يُبقى الأكثر شيوعًا ويُجمَّع الباقي في «أخرى» (approx=True)."""
    def __init__(self, cap: int = 5000):
        self.c: "collections.Counter" = collections.Counter(); self.cap = cap; self.other = 0; self.approx = False

    def add(self, items: Iterable[Any]) -> None:
        self.c.update(items)
        if len(self.c) > self.cap:
            keep = dict(self.c.most_common(self.cap // 2)); self.other += sum(self.c.values()) - sum(keep.values()); self.c = collections.Counter(keep); self.approx = True

    def top(self, n: int) -> List[List[Any]]: return [[k, v] for k, v in self.c.most_common(n)]
    def total(self) -> int: return sum(self.c.values()) + self.other
    def distinct(self) -> int: return len(self.c) + (1 if self.other else 0)


def _col(rows: Sequence[Dict[str, Any]], key: str) -> "np.ndarray":
    return np.array([(float(r[key]) if r.get(key) is not None else math.nan) for r in rows], dtype=np.float64)


def _pcts(s: Dict[str, Any], fmt: str = "{:.2f}") -> str:
    if not s.get("count"): return "—"
    f = fmt.format
    return f"متوسط {f(s['mean'])} | أدنى {f(s['min'])} | p5 {f(s['p5'])} | p25 {f(s['p25'])} | وسيط {f(s['p50'])} | p75 {f(s['p75'])} | p95 {f(s['p95'])} | أعلى {f(s['max'])}" + ("" if s.get("exact") else "  (≈ عيّنة)")


def _bars(counts: Sequence[int], lo: float, hi: float, width: int = 28) -> List[str]:
    if not counts: return []
    top = max(counts) or 1; step = (hi - lo) / len(counts)
    return [f"{lo + i * step:6.2f}–{lo + (i + 1) * step:5.2f} │{'█' * int(round(width * c / top)):<{width}}│ {c:,}" for i, c in enumerate(counts)]


def _pct(a: float, b: float) -> str:
    return f"{100.0 * a / b:.1f}%" if b else "—"


def build_filter_report(out: Dict[str, Any]) -> Dict[str, Any]:
    """التقرير النهائي للفلاتر: (1) الفيديوهات المرفوضة وأسبابها (بعد التنزيل من حالة الجلسة + أثناء البحث قبل التنزيل)، (2) نسبة الإطارات المرفوضة لكل فلتر من الإطارات المفحوصة
    (مقبول + مرفوض)، (3) متوسط عدد كلمات الوصف في صور التدريب (من فهارس Dataset). أجزاء (1) و(2) تحتاج حالة الجلسة؛ الجزء (3) دائمًا."""
    cw = out["dataset"]["caption_words"]; rep_: Dict[str, Any] = dict(state_available="state" in out, mean_caption_words=(cw.get("mean") if cw.get("count") else None),
                                                                      caption_words=cw if cw.get("count") else {}, caption_words_images=int(cw.get("count") or 0))
    if "state" not in out: return rep_
    sd = out["state"]; examined = int(sd["accepted"] + sd["rejected"]); n_skip = int(sd["skipped"])
    rep_.update(videos_total=int(sd["videos_total"]), videos_rejected=n_skip, videos_rejected_by_reason=sd.get("skipped_by_reason", []), videos_rejected_examples=sd.get("skipped_examples", {}),
                videos_rejected_at_search=int(sum((sd.get("search_rejections") or {}).values())), search_rejected_by_reason=sorted((sd.get("search_rejections") or {}).items(), key=lambda kv: -kv[1]),
                frames_examined=examined, frames_accepted=int(sd["accepted"]), frames_rejected=int(sd["rejected"]), frames_rejected_pct=(100.0 * sd["rejected"] / examined if examined else None),
                frames_rejected_by_filter=[dict(filter=k, count=int(n), pct=(100.0 * n / examined if examined else None)) for k, n in sd["rejected_by_reason"]])
    RS = globals().get("RUN_SUMMARY") if isinstance(globals().get("RUN_SUMMARY"), dict) else None
    if RS and RS.get("caption_pool"):
        cp = RS["caption_pool"]; rep_["caption_retry_this_session"] = dict(flagged_first_try=int(cp.get("banned_first", 0)), retried=int(cp.get("banned_retried", 0)),
                                                                           fixed_by_retry=int(cp.get("banned_fixed", 0)), skipped_after_retry=int(cp.get("banned_skipped", 0)))
    return rep_


def _print_filter_report(fr: Dict[str, Any]) -> None:
    print("-" * 78); print(" FILTER REPORT | تقرير الفلاتر")
    if fr.get("state_available"):
        v, tot = fr["videos_rejected"], fr["videos_total"]
        print(f"  فيديوهات مرفوضة: {v:,} من {tot:,} ({_pct(v, tot)})" + (" — الأسباب: " + " | ".join(f"{k}: {n:,}" for k, n in fr["videos_rejected_by_reason"]) if fr["videos_rejected_by_reason"] else ""))
        for k, ex in (fr.get("videos_rejected_examples") or {}).items(): print(f"      مثال {k}: {ex}")
        if fr["videos_rejected_at_search"]: print(f"  رُفضت أثناء البحث (قبل التنزيل، لا تدخل القائمة): {fr['videos_rejected_at_search']:,} — " + " | ".join(f"{k}: {n:,}" for k, n in fr["search_rejected_by_reason"]))
        e = fr["frames_examined"]
        print(f"  إطارات مفحوصة: {e:,} → مقبولة {fr['frames_accepted']:,} | مرفوضة {fr['frames_rejected']:,} ({_pct(fr['frames_rejected'], e)})")
        for f in fr["frames_rejected_by_filter"]: print(f"      {f['filter']:<24} {f['count']:>9,}  ({_pct(f['count'], e)} من المفحوصة)")
        cr = fr.get("caption_retry_this_session")
        if cr: print(f"  إعادة محاولة الوصف (هذه الجلسة): وُسم {cr['flagged_first_try']:,} | أُعيد {cr['retried']:,} | صُحّح بالإعادة {cr['fixed_by_retry']:,} | تُخطّي بعدها {cr['skipped_after_retry']:,}")
    else: print("  (تقرير الرفض يحتاج حالة الجلسة الحالية وهذا المصدر مختلف عنها)")
    mw = fr.get("mean_caption_words")
    print("  متوسط عدد كلمات الوصف: " + (f"{mw:.1f} كلمة (صور التدريب: {fr['caption_words_images']:,})" if mw is not None else "—"))


def final_statistics(source: str = "auto", save: bool = True, show: bool = True, top: int = 8, sample: int = 100_000, seed: int = 0, commit: bool = False) -> Dict[str, Any]:
    t0 = time.time(); store = dataset_store(source); cat = shard_catalog(store, "any")
    out: Dict[str, Any] = dict(generated_at=time.time(), store=store.name, repo=(CONFIG.HF_DATASET_REPO if store.name == "huggingface" else None), shards=len(cat))
    # ------------------------------------------------------------------ 1) من فهارس Dataset (كل العاملين)
    names = ("quality", "cap_score", "cap_words", "width", "height", "mpix", "file_kb")
    S = {k: _Sample(sample, seed + i) for i, k in enumerate(names)}
    C = {k: _Cat() for k in ("style", "res", "orient", "flags", "review_reason", "source", "license")}
    per_video: "collections.Counter" = collections.Counter(); kinds: "collections.Counter" = collections.Counter()
    upscaled = downscaled = unreadable = img_bytes = 0
    for sh, rows in iter_shard_indexes(store, cat, _STAT_COLS):
        if rows is None: unreadable += 1; continue
        if not rows: continue
        n = len(rows); kind = sh.get("kind") or rows[0].get("kind") or "train"; kinds[kind] += n
        w, h, ow, oh, fs = (_col(rows, k) for k in ("width", "height", "original_width", "original_height", "file_size"))
        S["width"].add(w); S["height"].add(h); S["mpix"].add(w * h / 1e6); S["file_kb"].add(fs / 1024.0); S["quality"].add(_col(rows, "quality_score"))
        ok_o = ~(np.isnan(ow) | np.isnan(oh)); upscaled += int(((w > ow) | (h > oh))[ok_o].sum()); downscaled += int(((w < ow) | (h < oh))[ok_o].sum()); img_bytes += int(np.nansum(fs))
        per_video.update(r["video_id"] for r in rows)
        dims = [(int(a), int(b)) for a, b in zip(w, h) if not (math.isnan(a) or math.isnan(b))]
        C["res"].add(f"{a}x{b}" for a, b in dims); C["orient"].add(("landscape" if a > b else "portrait" if a < b else "square") for a, b in dims)
        C["source"].add((r.get("source_name") or "—") for r in rows); C["license"].add((r.get("license") or "—") for r in rows)
        C["flags"].add(f for r in rows for f in (r.get("caption_flags") or ()))
        if kind == "train":
            S["cap_score"].add(_col(rows, "caption_score")); S["cap_words"].add(_col(rows, "caption_words"))
            C["style"].add(((r.get("visual_style") or "unspecified").strip().lower()[:40] or "unspecified") for r in rows)
        else:
            C["review_reason"].add((r.get("review_reason") or "غير محدد") for r in rows)
    sizes = store.sizes([s["repo_path"] for s in cat]) if cat else {}
    total_b = int(sum(v or 0 for v in sizes.values())); kind_b = collections.Counter()
    for s in cat: kind_b[s.get("kind") or "train"] += int(sizes.get(s["repo_path"]) or 0)
    n_rows = int(sum(kinds.values())); pv = np.array(sorted(per_video.values()), dtype=np.float64)
    out["dataset"] = dict(
        images=n_rows, train_images=int(kinds["train"]), review_images=int(kinds["review"]), review_fraction=(kinds["review"] / n_rows if n_rows else 0.0), unreadable_indexes=unreadable,
        videos_with_images=len(per_video), images_per_video=(dict(min=int(pv.min()), median=float(np.median(pv)), mean=float(pv.mean()), max=int(pv.max())) if pv.size else {}),
        most_images=[list(x) for x in per_video.most_common(top)], fewest_images=[list(x) for x in sorted(per_video.items(), key=lambda kv: kv[1])[:top]],
        quality_score=S["quality"].summary(), quality_hist=S["quality"].histogram(10, 0.0, 1.0), caption_score=S["cap_score"].summary(), caption_words=S["cap_words"].summary(),
        width=S["width"].summary(), height=S["height"].summary(), megapixels=S["mpix"].summary(), file_kb=S["file_kb"].summary(),
        resolutions=C["res"].top(top), distinct_resolutions=C["res"].distinct(), orientation=C["orient"].top(5), upscaled_images=upscaled, downscaled_images=downscaled,
        visual_styles=C["style"].top(top), distinct_styles=C["style"].distinct(), caption_flags=C["flags"].top(top), review_reasons=C["review_reason"].top(top),
        sources=C["source"].top(top), licenses=C["license"].top(top), approx_categories=any(c.approx for c in C.values()))
    out["storage"] = dict(shards=len(cat), train_shards=sum(1 for s in cat if s.get("kind") != "review"), review_shards=sum(1 for s in cat if s.get("kind") == "review"), total_bytes=total_b,
                          train_bytes=int(kind_b["train"]), review_bytes=int(kind_b["review"]), image_bytes=img_bytes, bytes_per_image=(total_b / n_rows if n_rows else None),
                          avg_shard_bytes=(total_b / len(cat) if cat else None), tar_overhead_fraction=((total_b - img_bytes) / total_b if total_b and total_b >= img_bytes else None))
    # ------------------------------------------------------------------ 2) من حالة الجلسة (فقط إن كان المخزن هو الحالي)
    if store is STORE:
        with STATE.lock: vs = [dict(v) for v in STATE.videos.values()]; sessions = int(STATE.progress.get("sessions", 0))
        stc = collections.Counter(v["state"] for v in vs); rb: "collections.Counter" = collections.Counter()
        for v in vs: rb.update(v.get("rejected_by_reason") or {})
        sm = lambda k: int(sum(v[k] or 0 for v in vs))
        done = [v for v in vs if v["state"] == "done"]; dur_h = sum((v["duration"] or 0) for v in done) / 3600.0
        rates = sorted(((v["accepted"] / v["selected_frames"], v["video_id"], v["selected_frames"]) for v in done if (v["selected_frames"] or 0) >= 20))
        RS = globals().get("RUN_SUMMARY") if isinstance(globals().get("RUN_SUMMARY"), dict) else None
        sd = dict(videos_total=len(vs), done=int(stc["done"]), failed=int(stc["failed"]), failed_permanent=int(stc["failed_permanent"]), skipped=int(stc["skipped"]),
                  pending=int(len(vs) - stc["done"] - stc["failed_permanent"] - stc["skipped"]), candidate_frames=sm("candidate_frames"), selected_frames=sm("selected_frames"),
                  accepted=sm("accepted"), rejected=sm("rejected"), captioned=sm("captioned"), caption_failures=sm("caption_failures"), review_flagged=sm("review_flagged"),
                  rejected_by_reason=[list(x) for x in rb.most_common()], source_video_hours=dur_h, sessions=sessions,
                  failed_videos=[dict(video_id=v["video_id"], state=v["state"], attempts=v["attempts"], error=redact_text(v["error"], 200)) for v in vs if v["state"] in ("failed", "failed_permanent")][:top],
                  done_without_images=[v["video_id"] for v in done if not v["accepted"]][:top],
                  lowest_acceptance=[dict(video_id=i, rate=r, selected=s) for r, i, s in rates[:5]], highest_acceptance=[dict(video_id=i, rate=r, selected=s) for r, i, s in rates[::-1][:5]])
        skipped = [v for v in vs if v["state"] == "skipped"]; by_code: "collections.Counter" = collections.Counter(skip_reason_code(v.get("error")) for v in skipped)
        sd["skipped_by_reason"] = [list(x) for x in by_code.most_common()]
        sd["skipped_examples"] = {c: redact_text(next((v.get("error") for v in skipped if skip_reason_code(v.get("error")) == c), ""), 160) for c in by_code}
        sd["search_rejections"] = dict(STATE.progress.get("search_rejections") or {})
        sd["selected_over_candidates"] = sd["selected_frames"] / sd["candidate_frames"] if sd["candidate_frames"] else None
        sd["accepted_over_selected"] = sd["accepted"] / sd["selected_frames"] if sd["selected_frames"] else None
        if RS and RS.get("elapsed_s"): sd["last_run"] = dict(elapsed_s=float(RS["elapsed_s"]), accepted=int(RS.get("accepted", 0)), images_per_hour=float(RS.get("accepted", 0)) / float(RS["elapsed_s"]) * 3600.0, stop_reason=RS.get("stop_reason"))
        out["state"] = sd
    out["sampling"] = dict(method=CONFIG.sampling_method(), ratio=CONFIG.SAMPLING_RATIO, seed=CONFIG.SAMPLING_SEED, target_fps=CONFIG.TARGET_FPS, max_images_per_video=CONFIG.MAX_IMAGES_PER_VIDEO,
                           trim_start=CONFIG.TRIM_START_RATIO, trim_end=CONFIG.TRIM_END_RATIO,
                           captioning=bool(CONFIG.ENABLE_CAPTIONING), caption_model=(CONFIG.MODEL_ID if CONFIG.ENABLE_CAPTIONING else None), min_image=[CONFIG.MIN_IMAGE_WIDTH, CONFIG.MIN_IMAGE_HEIGHT], jpeg_quality=CONFIG.IMAGE_QUALITY)
    out["filter_report"] = build_filter_report(out)
    out["duration_s"] = round(time.time() - t0, 1)
    blob = json.dumps(out, ensure_ascii=False, indent=1, default=str).encode("utf-8")
    if save:
        try: atomic_write_bytes(Path(CONFIG.WORK_DIR) / "final_statistics.json", blob)
        except Exception as e: log.warning("تعذّر حفظ final_statistics.json: %s", e)
    if commit:
        try: store.commit({"control/final_statistics.json": blob}, "final statistics")
        except Exception as e: log.warning("تعذّر رفع final_statistics.json: %s", str(e)[:150])
    if show: _print_stats(out, top)
    return out


def _print_stats(o: Dict[str, Any], top: int) -> None:
    D, St = o["dataset"], o["storage"]; P = lambda k, v: print(f"  {k:<14}: {v}")
    print("=" * 78); print(f" FINAL DATASET STATISTICS | المصدر: {o['store']}" + (f" ({o['repo']})" if o.get("repo") else "")); print("=" * 78)
    if "state" in o:
        s = o["state"]
        P("الفيديوهات", f"{s['videos_total']:,} إجمالًا | ✔ {s['done']:,} مكتمل | ✖ {s['failed_permanent']:,} فشل نهائي | ↻ {s['failed']:,} فشل (سيُعاد) | ⏭ {s['skipped']:,} متخطّى | … {s['pending']:,} معلّق")
        if s["candidate_frames"]: P("الإطارات", f"{s['candidate_frames']:,} مرشّح → {s['selected_frames']:,} مختار ({_pct(s['selected_frames'], s['candidate_frames'])}) → {s['accepted']:,} مقبول ({_pct(s['accepted'], s['selected_frames'])}) | {s['rejected']:,} مرفوض")
        if s["rejected_by_reason"]: P("أسباب الرفض", " | ".join(f"{k}: {v:,}" for k, v in s["rejected_by_reason"][:top]))
        P("الأوصاف", f"{s['captioned']:,} وُصفت | {s['caption_failures']:,} فشل وصف | {s['review_flagged']:,} إلى المراجعة")
        if s["source_video_hours"]: P("زمن المصدر", f"{s['source_video_hours']:,.1f} ساعة فيديو مكتملة → {D['images'] / max(s['source_video_hours'], 1e-9):,.0f} صورة/ساعة فيديو")
        P("الجلسات", f"{s['sessions']:,} استئناف" + (f" | آخر تشغيل: {human_time(s['last_run']['elapsed_s'])} → {s['last_run']['images_per_hour']:,.0f} صورة/ساعة ({s['last_run']['stop_reason']})" if s.get("last_run") else ""))
    print("-" * 78)
    P("الصور", f"{D['images']:,} = {D['train_images']:,} تدريب + {D['review_images']:,} مراجعة ({D['review_fraction']:.1%})" + (f" | ⚠ {D['unreadable_indexes']} فهرس غير قابل للقراءة" if D["unreadable_indexes"] else ""))
    ipv = D["images_per_video"]
    if ipv: P("لكل فيديو", f"{D['videos_with_images']:,} فيديو له صور | أدنى {ipv['min']:,} | وسيط {ipv['median']:,.0f} | متوسط {ipv['mean']:,.1f} | أعلى {ipv['max']:,}")
    P("الدقة", f"{D['distinct_resolutions']:,} دقة مختلفة | الأكثر: " + ", ".join(f"{k} ({v:,})" for k, v in D["resolutions"][:5]) + (" | " + ", ".join(f"{k} {v:,}" for k, v in D["orientation"]) if D["orientation"] else ""))
    P("الحجم الأصلي", f"مصغَّرة {D['downscaled_images']:,} | مكبَّرة {D['upscaled_images']:,}" + ("  ✖ يجب أن تكون 0 (لا تكبير)" if D["upscaled_images"] else "  (لا تكبير ✔)"))
    P("MPix", _pcts(D["megapixels"], "{:.2f}")); P("جودة الصورة", _pcts(D["quality_score"], "{:.3f}"))
    for line in _bars(D["quality_hist"], 0.0, 1.0): print("      " + line)
    P("درجة الوصف", _pcts(D["caption_score"], "{:.3f}")); P("كلمات الوصف", _pcts(D["caption_words"], "{:.0f}"))
    if D["visual_styles"]: P("الأنماط", f"{D['distinct_styles']:,} نمطًا | " + " | ".join(f"{k}: {v:,}" for k, v in D["visual_styles"]))
    if D["caption_flags"]: P("أعلام الوصف", " | ".join(f"{k}: {v:,}" for k, v in D["caption_flags"]))
    if D["review_reasons"]: P("أسباب المراجعة", " | ".join(f"{k}: {v:,}" for k, v in D["review_reasons"]))
    if len(D["sources"]) > 1 or (D["sources"] and D["sources"][0][0] != "—"): P("المصادر", " | ".join(f"{k}: {v:,}" for k, v in D["sources"][:5]) + " || الرخص: " + " | ".join(f"{k}: {v:,}" for k, v in D["licenses"][:5]))
    print("-" * 78)
    P("التخزين", f"{St['shards']:,} شظية ({St['train_shards']:,} تدريب + {St['review_shards']:,} مراجعة) | {human_bytes(St['total_bytes'])}" + (f" | {human_bytes(St['bytes_per_image'])}/صورة" if St["bytes_per_image"] else "") + (f" | متوسط الشظية {human_bytes(St['avg_shard_bytes'])}" if St["avg_shard_bytes"] else ""))
    if St["tar_overhead_fraction"] is not None: P("", f"الصور {human_bytes(St['image_bytes'])} + JSON/TAR {St['tar_overhead_fraction']:.1%} من الحجم")
    sp = o["sampling"]
    P("أخذ العيّنات", f"{sp['method']} | ratio={sp['ratio']} قص {sp.get('trim_start', 0):.0%}/{sp.get('trim_end', 0):.0%} seed={sp['seed']} target_fps={sp['target_fps']}" + (f" | MAX_IMAGES_PER_VIDEO={sp['max_images_per_video']}" if sp["max_images_per_video"] else "") + f" | الوصف: {sp['caption_model'] if sp['captioning'] else 'معطّل'}")
    if "state" in o:
        s = o["state"]
        if s["failed_videos"]:
            print("  فيديوهات فاشلة:")
            for f in s["failed_videos"]: print(f"    ✖ {f['video_id']} [{f['state']}, محاولات {f['attempts']}] {f['error']}")
        if s["done_without_images"]: print(f"  ⚠ مكتملة بلا أي صورة مقبولة: {s['done_without_images']}")
        if s["lowest_acceptance"]: print("  أدنى نسب القبول: " + " | ".join(f"{x['video_id']} {x['rate']:.0%} ({x['selected']:,})" for x in s["lowest_acceptance"]))
    if o.get("filter_report"): _print_filter_report(o["filter_report"])
    if D["approx_categories"]: print("  ملاحظة: كثرة الفئات المتميزة ⇒ أعداد الذيل (الفئات النادرة) تقريبية.")
    print(f"  ({human_time(o['duration_s'])} | final_statistics.json)"); print("=" * 78)


FINAL_STATS = final_statistics(STATS_SOURCE, STATS_SAVE, True, STATS_TOP, STATS_SAMPLE)

### CELL 28 — RESUME STATUS & INSTRUCTIONS  (حالة الاستئناف + التعليمات)
> resume_status() لا يعدّل شيئًا: يعرض ما سيحدث لو شغّلت الخلية 24F الآن (فيديوهات متبقية/متوقفة في منتصفها/فاشلة، شظايا، بوابة الأمان، تغيّر المعلمات).
> retry_failed_videos() يعيد الفيديوهات الفاشلة نهائيًا إلى pending ليُعاد تجريبها في التشغيل التالي. print_resume_instructions() = دليل الإيقاف والاستئناف.

In [ ]:
# CELL 28 - RESUME STATUS & INSTRUCTIONS  (حالة الاستئناف + التعليمات)
#: resume_status() لا يعدّل شيئًا: يعرض ما سيحدث لو شغّلت الخلية 24F الآن (فيديوهات متبقية/متوقفة في منتصفها/فاشلة، شظايا، بوابة الأمان، تغيّر المعلمات).
#: retry_failed_videos() يعيد الفيديوهات الفاشلة نهائيًا إلى pending ليُعاد تجريبها في التشغيل التالي. print_resume_instructions() = دليل الإيقاف والاستئناف.


def _local_leftovers() -> Dict[str, Any]:
    out: Dict[str, Any] = dict(local_shards=0, local_shard_bytes=0, kept_videos=0, kept_video_bytes=0)
    for d, k in ((SHARD_DIR, "local_shard"), (VIDEO_DIR, "kept_video")):
        try:
            fs = [p for p in Path(d).iterdir() if p.is_file()]
            out[k + "s"] = len(fs); out[k + "_bytes"] = int(sum(p.stat().st_size for p in fs))
        except Exception: pass
    return out


def resume_status(show: bool = True) -> Dict[str, Any]:
    with STATE.lock:
        vs = [dict(v) for v in STATE.videos.values()]; P = dict(STATE.progress); shards = list(STATE.shards)
    stc = collections.Counter(v["state"] for v in vs)
    todo = [v for v in vs if v["state"] not in ("done", "failed_permanent", "skipped")]
    partial = [dict(video_id=v["video_id"], accepted=int(v["accepted"]), cursor=int(v["cursor"])) for v in todo if v["accepted"] > 0 or v["cursor"] >= 0]
    retry = [v["video_id"] for v in todo if v["state"] == "failed"]
    dead = [dict(video_id=v["video_id"], attempts=int(v["attempts"]), error=redact_text(v["error"], 160)) for v in vs if v["state"] == "failed_permanent"]
    def gate(fn: str, why: str) -> Tuple[bool, str]:          # الدالة تعيش في 24C/24D: لا ننهار إن لم تُشغَّل (فحص Dataset دون المعالجة)
        f = globals().get(fn)
        return f() if f is not None else (False, why)
    smoke_ok, smoke_why = gate("smoke_test_passed", "الخلية 24D لم تُشغَّل في هذه الجلسة"); bench_ok, bench_why = gate("benchmark_is_current", "الخلية 24C لم تُشغَّل في هذه الجلسة")
    gate_ok = (not CONFIG.REQUIRE_SMOKE_TEST) or smoke_ok
    damaged = list(P.get("damaged_shards") or []); stop_file = Path(CONFIG.WORK_DIR) / "STOP"
    nxt = dict(train=shard_basename(STATE.next_shard_index("train")), review=shard_basename(STATE.next_shard_index("review"), "review"))
    S: Dict[str, Any] = dict(
        store=STORE.name, repo=(CONFIG.HF_DATASET_REPO if STORE.name == "huggingface" else None), resumed=bool(STATE.resumed), sessions=int(P.get("sessions", 0)),
        videos_total=len(vs), done=int(stc["done"]), skipped=int(stc["skipped"]), failed_permanent=len(dead), to_process=len(todo), in_progress=partial, will_retry=retry, failed_permanent_videos=dead[:20],
        shards_committed=len(shards), next_shards=nxt, damaged_shards=damaged, accepted_images=int(P.get("accepted_images", 0)),
        smoke_test_ok=smoke_ok, smoke_test_reason=smoke_why, benchmark_current=bench_ok, benchmark_reason=bench_why, gate_ok=gate_ok,
        pipeline_dirty=bool(PIPELINE_DIRTY), stop_file_present=stop_file.exists(), config_changes=list(P.get("config_changes") or []), **_local_leftovers())
    if not vs: key, msg = "no_videos", "لا فيديوهات في الحالة: اضبط VIDEO_MANIFEST أو VIDEO_URLS في USER_SETTINGS (أعلى الخلية 03) ثم شغّل 03 و07"
    elif damaged: key, msg = "damaged", f"⚠ {len(damaged)} شظية مسجَّلة لكنها مفقودة/مختلفة الحجم في المخزن ({', '.join(damaged[:3])}{' …' if len(damaged) > 3 else ''}): راجع المستودع (أو شغّل الخلية 26) قبل المتابعة"
    elif not todo: key, msg = "finished", "كل الفيديوهات منتهية (مكتملة/متخطّاة/فاشلة نهائيًا) ⇒ شغّل الخلايا 25 → 27 للتحقق والإحصاءات" + (" — وللفاشلة نهائيًا: retry_failed_videos()" if dead else "")
    elif not gate_ok: key, msg = "need_preflight", f"شغّل الخلية 24E (Benchmark + Smoke Test) أولًا — {smoke_why}"
    else: key, msg = "run", f"شغّل الخلية 24F: سيعالج {len(todo):,} فيديو" + (f" (منها {len(partial)} متوقفة في منتصفها وتكمل من آخر مؤشر مُلتزَم)" if partial else "") + (" — الحالة في الذاكرة غير موثوقة وستُعاد من المخزن تلقائيًا" if PIPELINE_DIRTY else "")
    S.update(next_action=key, next_action_text=msg, can_run=(key == "run"))
    if show:
        print("=" * 78); print(" RESUME STATUS | حالة الاستئناف"); print("=" * 78)
        print(f"  المخزن     : {S['store']}" + (f" ({S['repo']})" if S["repo"] else " (محلي — لن يُرفع شيء إلى Hugging Face)") + f" | استُؤنف من حالة سابقة: {'نعم' if S['resumed'] else 'لا (بداية جديدة)'} | جلسات سابقة: {S['sessions']}")
        print(f"  الفيديوهات : {S['videos_total']:,} | ✔ {S['done']:,} مكتمل | … {S['to_process']:,} متبقٍ | ✖ {S['failed_permanent']:,} فشل نهائي | ⏭ {S['skipped']:,} متخطّى")
        print(f"  الشظايا    : {S['shards_committed']:,} مُلتزَمة ({S['accepted_images']:,} صورة) | التالية: {nxt['train']} / {nxt['review']}")
        if partial: print("  متوقفة في منتصفها: " + ", ".join(f"{p['video_id']} ({p['accepted']:,} صورة، مؤشر {p['cursor']})" for p in partial[:5]) + (" …" if len(partial) > 5 else ""))
        if retry: print(f"  ↻ ستُعاد تلقائيًا (فشلت أقل من {MAX_VIDEO_ATTEMPTS} مرات): {retry[:5]}" + (" …" if len(retry) > 5 else ""))
        for d in dead[:5]: print(f"  ✖ فشل نهائي: {d['video_id']} (محاولات {d['attempts']}): {d['error']}")
        if S["local_shards"] or S["kept_videos"]: print(f"  محلي      : {S['local_shards']} ملف شظية ({human_bytes(S['local_shard_bytes'])}) | {S['kept_videos']} فيديو محفوظ ({human_bytes(S['kept_video_bytes'])})")
        print(f"  البوابات   : Smoke Test {'✔' if smoke_ok else '✖ ' + smoke_why} | Benchmark {'✔ حالي' if bench_ok else '… ' + bench_why}" + ("" if CONFIG.REQUIRE_SMOKE_TEST else " | REQUIRE_SMOKE_TEST=False"))
        if S["stop_file_present"]: print(f"  ⚠ ملف الإيقاف موجود ({stop_file}) — يُحذف تلقائيًا عند بدء 24F")
        for ch in S["config_changes"][-3:]:
            print(f"  ⚠ تغيّرت معلمات المحتوى (جلسة {ch['session']}، بعد {ch['accepted_before']:,} صورة): " + " | ".join(f"{k}: {a!r} → {b!r}" for k, (a, b) in ch["changes"].items())[:300])
        print("-" * 78); print("  ▶ الخطوة التالية: " + msg); print("=" * 78)
    return S


def retry_failed_videos(video_ids: Optional[Sequence[str]] = None, include_skipped: bool = False) -> List[str]:
    """يعيد الفيديوهات الفاشلة نهائيًا (failed_permanent؛ ويمكن المتخطّاة skipped) إلى pending مع تصفير المحاولات، ليُعاد تجريبها عند تشغيل 24F التالي (بعد إصلاح الرابط/الشبكة/الإعداد).
    يحافظ على المؤشر والصور المُلتزَمة لكل فيديو، ولا يمسّ المكتمل أبدًا. سجلّ الإخفاقات (failed_videos.jsonl) يبقى للتاريخ."""
    want = None if video_ids is None else set(video_ids); changed: List[str] = []
    with STATE.lock:
        for v in STATE.videos.values():
            if v["state"] == "failed_permanent" or (include_skipped and v["state"] == "skipped"):
                if want is None or v["video_id"] in want:
                    v.update(state="pending", attempts=0, error=None, download_status="pending" if v.get("download_status") != "done" else "done", updated_at=time.time()); changed.append(v["video_id"])
        if changed: STATE.recompute(); STATE._dirty_since_commit = True
    if changed:
        try: STATE.save_local(force=True)
        except Exception as e: log.warning("تعذّر حفظ الحالة المحلية: %s", e)
        print(f"أُعيد {len(changed)} فيديو إلى pending: {changed[:8]}" + (" …" if len(changed) > 8 else "") + " — شغّل الخلية 24F (أو run_pipeline()) لإعادة المحاولة.")
    else: print("لا فيديوهات مطابقة (فاشلة نهائيًا" + (" أو متخطّاة" if include_skipped else "") + ").")
    if want is not None and (missing := sorted(want - {v for v in STATE.videos})): print("معرّفات غير موجودة في الحالة:", missing[:5])
    return changed


def print_resume_instructions() -> None:
    w = Path(CONFIG.WORK_DIR)
    print("""
══════════════════════════ كيف توقف وتستأنف (الحالة كلها في مستودع Hugging Face: control/) ══════════════════════════
▸ إيقاف آمن (لا يضيع شيء ولا يتلف شيء): زر Interrupt في Kaggle مرة واحدة، أو أنشئ الملف  %s/STOP ، أو دع MAX_RUNTIME_HOURS (على Kaggle = 11 ساعة تلقائيًا).
    النظام يُنهي الدفعات الجارية ويرفع آخر شظية ويلتزم الحالة. ضغطة Interrupt ثانية = إلغاء فوري: يُهمل ما لم يُرفع ويُعاد عمله لاحقًا (لا تلف).
▸ استئناف بعد انقطاع/انتهاء جلسة (أو على جهاز آخر): افتح نفس الـ Notebook بنفس HF_DATASET_REPO و HF_TOKEN في Kaggle Secrets و VIDEO_MANIFEST، ثم Run All (الخلايا 01 → 24F).
    • الخلية 06 تقرأ control/ وتتخطّى كل فيديو مكتمل، وتكمل المتوقف في منتصفه من آخر مؤشر مُلتزَم؛ ما عُولج بعد آخر التزام يُعاد حسابه بالنتيجة نفسها (قابلية إعادة الإنتاج).
    • الخلية 24E لا تعيد Benchmark/Smoke Test إلا إن تغيّر العتاد أو الحزم أو إعدادات المحتوى (البصمة محفوظة في control/benchmark_results.json).
    • لا تحذف شيئًا يدويًا: الفيديوهات لا تُحذف إلا بعد التزام شظاياها، والشظايا المحلية غير المرفوعة تُنظَّف وتُعاد.
▸ للتحقق من الحالة قبل التشغيل: resume_status() — وهي تقول لك أي خلية تشغّل الآن.
▸ فيديو فشل نهائيًا (3 محاولات: رابط منتهٍ، انقطاع شبكة طويل…): أصلح السبب ثم  retry_failed_videos()  (أو retry_failed_videos(["id1", "id2"]) ؛ include_skipped=True للمتخطّاة)  ثم 24F.
▸ تغيير الإعدادات بين الجلسات: التشغيل لا يُمنع، لكن أي تغيير في معلمات المحتوى (SAMPLING_RATIO/SEED/TARGET_FPS/MAX_IMAGES_PER_VIDEO/الدقة الدنيا/QC/نموذج الوصف…)
    يُسجَّل في control/progress.json ← config_changes ويُحذَّر منه: الفيديوهات المكتملة لا تُعاد، والباقية تأخذ المعلمات الجديدة (Dataset خليط). للتجانس الكامل ابدأ مستودعًا جديدًا.
▸ عدة Notebooks بالتوازي: في كل Notebook اضبط NUM_WORKERS=N و WORKER_ID=0..N-1 (config.json أو PIPE_NUM_WORKERS / PIPE_WORKER_ID) وأعطِ كلًّا منها GPU/قرصًا خاصًا؛
    كل عامل يأخذ شريحة ثابتة من الفيديوهات (تجزئة video_id) ويكتب control/wNN/ وشظايا train-wNN-*؛ الخلايا 25 → 27 تجمع كل العاملين من المستودع.
▸ فحص Dataset بلا أي معالجة (جلسة جديدة بعد اكتمال العمل): شغّل 01 → 24D (تعريفات فقط) ثم 25 (عيّنات عشوائية) و26 (مدقق التكامل: INTEGRITY_MODE="full" للتنزيل الكامل القابل للاستئناف) و27 (الإحصاءات) — وتجاوز 24E و24F.
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════""" % w)


RESUME = resume_status()
print_resume_instructions()